# Recent developments in geometric machine learning

In [ ]:
# 1. Cài đặt các thư viện hệ thống cần thiết (Đây là lý do gây lỗi manimpango)
!sudo apt update
!sudo apt install libcairo2-dev libpango1.0-dev ffmpeg

# 2. Cài đặt LaTeX (để vẽ công thức toán học đẹp như 3Blue1Brown)
!sudo apt install texlive texlive-latex-extra texlive-fonts-extra texlive-latex-recommended dvipng dvisvgm

# 3. Bây giờ mới cài đặt Manim
!pip install manim
# from manim import *
!pip install pydub
!pip install gTTS
!pip install manimpango
!pip install "manim-voiceover[gtts,azure]" --force-reinstall
!pip install edge-tts


In [ ]:
from manim import *

config.video_dir = "/content/drive/MyDrive/lab01"


In [ ]:
import numpy as np
from manim_voiceover import VoiceoverScene
from manim_voiceover.services.base import SpeechService
import os
import asyncio
import threading
import hashlib
import shutil
from pathlib import Path


## slide 30-33
Generic construction + Linear equivariant layers

## VoiceOver

In [ ]:
from manim import *
import numpy as np
from manim_voiceover import VoiceoverScene
from manim_voiceover.services.base import SpeechService
import os
import asyncio
import threading
import hashlib
from pathlib import Path

# ===================================================================
# ─── CUSTOM EDGETTS SERVICE (BẺ KHÓA HOÀN TOÀN LỖI CACHE KEYERROR) ───
# ===================================================================
class EdgeTTSService(SpeechService):
    def __init__(self, voice="en-US-GuyNeural", cache_dir="clean_edge_cache", **kwargs):
        self.voice = voice
        super().__init__(cache_dir=cache_dir, **kwargs)

    def get_cached_result(self, input_data, cache_dir):
        # GIẢI PHÁP: Ghi đè để vô hiệu hóa hoàn toàn hàm check cache gốc bị lỗi của thư viện
        return None

    def generate_from_text(self, text: str, cache_dir: str = None, path: str = None) -> dict:
        import edge_tts

        if cache_dir is None:
            cache_dir = "clean_edge_cache"

        cache_dir_path = Path(cache_dir)
        os.makedirs(cache_dir_path, exist_ok=True)

        if path is None:
            hash_name = hashlib.md5(text.encode()).hexdigest()
            path = f"{hash_name}.mp3"

        final_path_str = str(cache_dir_path / path)

        async def _download():
            communicate = edge_tts.Communicate(text, self.voice)
            await communicate.save(final_path_str)

        def _worker():
            loop = asyncio.new_event_loop()
            asyncio.set_event_loop(loop)
            try:
                loop.run_until_complete(_download())
            finally:
                loop.close()

        # Cô lập Event Loop trong một Thread riêng để tránh lỗi đóng băng hệ thống của Colab
        thread = threading.Thread(target=_worker)
        thread.start()
        thread.join()

        # Trả về đúng tên file tương đối để khớp với trình quản lý VoiceoverTracker
        return {"original_audio": path}


In [ ]:
%%manim -qh -v WARNING EquivariantVsInvariant29

from manim import *
from manim_voiceover import VoiceoverScene
from manim_voiceover.services.base import SpeechService
import asyncio
import threading
import hashlib
import os
from pathlib import Path

# ─── MAIN SCENE GRAPH ───────────────────────────────────────────
class EquivariantVsInvariant29(VoiceoverScene):
    def construct(self):
        self.camera.background_color = BLACK

        # Cấu hình giọng đọc en-US-GuyNeural chuyên nghiệp
        self.set_speech_service(EdgeTTSService(voice="en-US-GuyNeural"))

        # ==========================================
        # PART 1: TITLE & BASIC DEFINITIONS
        # ==========================================
        title = Text("Core Mechanisms: Equivariance & Invariance", font_size=36, color=WHITE, weight=BOLD)

        with self.voiceover(text="As a reminder, two core mechanisms that power all Geometric Neural Networks: Equivariance and Invariance.") as tracker:
            self.play(Write(title))
            self.play(title.animate.to_edge(UP, buff=0.4))
            intro = Text(
                "Every Geometric Neural Network (CNN, GNN, DeepSets) is built from these 2 blocks.",
                font_size=22, color=LIGHT_GREY
            ).next_to(title, DOWN, buff=0.3)
            self.play(FadeIn(intro))

        # ==========================================
        # PART 2: EQUIVARIANT LAYER
        # ==========================================
        eq_title = Text("1. Equivariant Layer", font_size=28, color=BLUE_C).to_edge(LEFT, buff=1.0).shift(UP*1.0)

        in_shape_eq = RegularPolygon(n=3, color=WHITE, fill_opacity=0.5).scale(0.6).move_to(LEFT * 4 + DOWN * 0.5)
        in_label_eq = MathTex("x", font_size=24).next_to(in_shape_eq, DOWN)
        in_group_eq = VGroup(in_shape_eq, in_label_eq)

        box_eq = Rectangle(height=1.5, width=2.5, color=BLUE_E, fill_opacity=0.3).move_to(LEFT * 0 + DOWN * 0.5)
        text_eq = Text("Feature\nExtraction (f)", font_size=20, color=BLUE_C).move_to(box_eq)
        layer_eq = VGroup(box_eq, text_eq)

        out_shape_eq = RegularPolygon(n=3, color=BLUE_C, fill_opacity=0.8).scale(0.8).move_to(RIGHT * 4 + DOWN * 0.5)
        out_label_eq = MathTex("f(x)", font_size=24, color=BLUE_C).next_to(out_shape_eq, DOWN)
        out_group_eq = VGroup(out_shape_eq, out_label_eq)

        arr1 = Arrow(in_shape_eq.get_right(), box_eq.get_left(), buff=0.2)
        arr2 = Arrow(box_eq.get_right(), out_shape_eq.get_left(), buff=0.2)

        with self.voiceover(text="First is the Equivariant Layer. Imagine passing a geometric structure, like a triangle, through a neural network to extract features."):
            self.play(Write(eq_title))
            self.play(FadeIn(in_group_eq), FadeIn(layer_eq), GrowArrow(arr1))
            self.play(GrowArrow(arr2), FadeIn(out_group_eq))

        action_eq = Text("Input transforms → Output transforms (Structure preserved)", font_size=20, color=YELLOW).next_to(box_eq, DOWN, buff=1.0)
        math_eq = MathTex(r"f(g \cdot x) = g \cdot f(x)", font_size=32, color=BLUE_C).next_to(action_eq, DOWN, buff=0.3)

        with self.voiceover(text="Equivariance means: if the input is transformed, for example, rotated by an angle, the output will rotate in the exact same way. The geometric structure is perfectly preserved."):
            self.play(Write(action_eq))
            self.play(
                Rotate(in_shape_eq, angle=PI/2),
                Rotate(out_shape_eq, angle=PI/2),
                run_time=2
            )
            self.play(Write(math_eq))

        self.play(FadeOut(VGroup(eq_title, in_group_eq, layer_eq, out_group_eq, arr1, arr2, action_eq, math_eq)))

        # ==========================================
        # PART 3: INVARIANT POOLING
        # ==========================================
        inv_title = Text("2. Invariant / Pooling Layer", font_size=28, color=GREEN_C).to_edge(LEFT, buff=1.0).shift(UP*1.0)

        in_shape_inv = RegularPolygon(n=3, color=WHITE, fill_opacity=0.5).scale(0.6).move_to(LEFT * 4 + DOWN * 0.5)
        in_label_inv = MathTex("x", font_size=24).next_to(in_shape_inv, DOWN)
        in_group_inv = VGroup(in_shape_inv, in_label_inv)

        box_inv = Rectangle(height=1.5, width=2.5, color=GREEN_E, fill_opacity=0.3).move_to(LEFT * 0 + DOWN * 0.5)
        text_inv = Text("Global Pool\n(Sum/Max) (g)", font_size=20, color=GREEN_C).move_to(box_inv)
        layer_inv = VGroup(box_inv, text_inv)

        gauge = VGroup(
            Circle(radius=0.5, color=WHITE, stroke_width=4),
            Text("0.95", font_size=24, color=GREEN_C)
        ).move_to(RIGHT * 4 + DOWN * 0.5)
        out_label_inv = MathTex("g(x)", font_size=24, color=GREEN_C).next_to(gauge, DOWN)
        out_group_inv = VGroup(gauge, out_label_inv)

        arr3 = Arrow(in_shape_inv.get_right(), box_inv.get_left(), buff=0.2)
        arr4 = Arrow(box_inv.get_right(), gauge.get_left(), buff=0.2)

        with self.voiceover(text="The second concept is the Invariant Layer, often used in pooling. Here, features are aggregated into a single global value."):
            self.play(Write(inv_title))
            self.play(FadeIn(in_group_inv), FadeIn(layer_inv), GrowArrow(arr3))
            self.play(GrowArrow(arr4), FadeIn(out_group_inv))

        action_inv = Text("Input transforms → Output STILL (Information distilled)", font_size=20, color=YELLOW).next_to(box_inv, DOWN, buff=1.0)
        math_inv = MathTex(r"g(h \cdot x) = g(x)", font_size=32, color=GREEN_C).next_to(action_inv, DOWN, buff=0.3)

        with self.voiceover(text="Invariance means: no matter how you rotate, flip, or transform the input, the output remains exactly the same. The network has distilled the core information."):
            self.play(Write(action_inv))
            self.play(Rotate(in_shape_inv, angle=PI), run_time=2)
            self.play(gauge[1].animate.set_color(WHITE).scale(1.2), run_time=0.3)
            self.play(gauge[1].animate.set_color(GREEN_C).scale(1/1.2), run_time=0.3)
            self.play(Write(math_inv))

        self.play(FadeOut(VGroup(inv_title, in_group_inv, layer_inv, out_group_inv, arr3, arr4, action_inv, math_inv)))

        # ==========================================
        # PART 4: THE GENERIC PIPELINE
        # ==========================================
        pipe_title = Text("3. The Generic Construction", font_size=28, color=YELLOW).to_edge(LEFT, buff=1.0).shift(UP*1.0)
        desc_pipe = Text(
            "Use multiple Equivariant layers for deep structural analysis,\n"
            "then use an Invariant layer at the end for final prediction.",
            font_size=22, color=LIGHT_GREY, line_spacing=1.3
        ).next_to(pipe_title, DOWN, buff=0.4, aligned_edge=LEFT)

        nodes = VGroup()
        labels = VGroup()
        nodes.add(RegularPolygon(n=4, color=WHITE).scale(0.4)); labels.add(Text("Input", font_size=16))
        nodes.add(Rectangle(height=0.8, width=1.5, color=BLUE_C, fill_opacity=0.5)); labels.add(Text("Equivariant 1", font_size=16))
        nodes.add(Rectangle(height=0.8, width=1.5, color=BLUE_C, fill_opacity=0.5)); labels.add(Text("Equivariant 2", font_size=16))
        nodes.add(Rectangle(height=0.8, width=1.5, color=GREEN_C, fill_opacity=0.5)); labels.add(Text("Invariant Pool", font_size=16))
        nodes.add(Circle(radius=0.3, color=YELLOW, fill_opacity=0.8)); labels.add(Text("Prediction", font_size=16))

        nodes.arrange(RIGHT, buff=0.8).next_to(desc_pipe, DOWN, buff=1.0).shift(RIGHT*0.5)
        for i in range(5):
            labels[i].next_to(nodes[i], DOWN, buff=0.2)
        arrows = VGroup(*[Arrow(nodes[i].get_right(), nodes[i+1].get_left(), buff=0.1) for i in range(4)])

        with self.voiceover(text="So what is the big picture? In practice, architectures like CNNs or GNNs chain multiple Equivariant layers to process deep spatial structures."):
            self.play(Write(pipe_title), FadeIn(desc_pipe))
            self.play(FadeIn(nodes[0]), Write(labels[0]))
            for i in range(3):
                self.play(GrowArrow(arrows[i]), FadeIn(nodes[i+1]), Write(labels[i+1]), run_time=0.4)

        box_feature = SurroundingRectangle(VGroup(nodes[1], nodes[2]), color=BLUE_C, buff=0.12)
        text_feature = Text("Geometric Feature Extraction", font_size=16, color=BLUE_C).next_to(box_feature, UP)
        box_pool = SurroundingRectangle(nodes[3], color=GREEN_C, buff=0.12)
        text_pool = Text("Aggregation", font_size=16, color=GREEN_C).next_to(box_pool, UP)

        with self.voiceover(text="Finally, they use an Invariant layer to make a prediction that is independent of the viewpoint. Models are usually divided into two clear parts: geometric feature extraction and aggregation."):
            self.play(GrowArrow(arrows[3]), FadeIn(nodes[4]), Write(labels[4]))
            self.play(Create(box_feature), Write(text_feature))
            self.play(Create(box_pool), Write(text_pool))

        self.play(FadeOut(Group(*self.mobjects)))

        # ==========================================
        # PART 5: CONCRETE NN EXAMPLE
        # ==========================================
        title_ex = Text("Example: DeepSets Computation", font_size=32, color=YELLOW, weight=BOLD).to_edge(UP, buff=0.3)
        eq_title_ex = Text("1. Equivariant Layer (Shared MLP)", font_size=24, color=BLUE_B).next_to(title_ex, DOWN, buff=0.5).to_edge(LEFT, buff=1.0)

        x_text = MathTex(r"X =", font_size=28)
        nodes_x = VGroup(*[VGroup(Circle(radius=0.35, color=col, fill_opacity=0.5), Text(str(val), font_size=28)) for val, col in zip([1, 2, 3], [RED_C, BLUE_C, GREEN_C])]).arrange(RIGHT, buff=1.2)
        x_group = VGroup(x_text, nodes_x).arrange(RIGHT, buff=0.5)

        mlps = VGroup(*[VGroup(Rectangle(width=0.8, height=0.5, color=BLUE_E, fill_opacity=0.5), Text("x 2", font_size=20, color=WHITE)) for _ in range(3)])
        arrows_in = VGroup()
        for i in range(3):
            mlps[i].next_to(nodes_x[i], DOWN, buff=0.6)
            arrows_in.add(Arrow(nodes_x[i].get_bottom(), mlps[i].get_top(), buff=0.1, color=LIGHT_GREY))

        h_text = MathTex(r"H =", font_size=28)
        nodes_h = VGroup()
        arrows_out = VGroup()
        for i, (val, col) in enumerate(zip([2, 4, 6], [RED_C, BLUE_C, GREEN_C])):
            circle = Circle(radius=0.35, color=col, fill_opacity=0.8)
            label = Text(str(val), font_size=28).move_to(circle)
            nodes_h.add(VGroup(circle, label).next_to(mlps[i], DOWN, buff=0.6))
            arrows_out.add(Arrow(mlps[i].get_bottom(), nodes_h[-1].get_top(), buff=0.1, color=BLUE_C))
        h_text.next_to(nodes_h, LEFT, buff=0.5).align_to(x_text, LEFT)

        network_group = VGroup(x_text, nodes_x, arrows_in, mlps, h_text, nodes_h, arrows_out).next_to(eq_title_ex, DOWN, buff=0.5).shift(RIGHT * 1.5)

        with self.voiceover(text="Let's look at a concrete example using DeepSets. We have an Equivariant layer applying a multiply-by-two function to each input element: 1, 2, and 3 become 2, 4, and 6."):
            self.play(Write(title_ex), Write(eq_title_ex))
            self.play(FadeIn(x_text), LaggedStart(*[FadeIn(node) for node in nodes_x], lag_ratio=0.2))
            self.play(LaggedStart(*[FadeIn(mlp) for mlp in mlps], lag_ratio=0.2), LaggedStart(*[GrowArrow(arr) for arr in arrows_in], lag_ratio=0.2))
            self.play(FadeIn(h_text), LaggedStart(*[FadeIn(node) for node in nodes_h], lag_ratio=0.2), LaggedStart(*[GrowArrow(arr) for arr in arrows_out], lag_ratio=0.2))

        test1_desc = Text("Equivariance Test: Swap Input → Output swaps", font_size=20, color=YELLOW).next_to(network_group, DOWN, buff=0.6)

        with self.voiceover(text="If we swap the positions of the input elements, their positions in the output automatically swap as well. This is permutation equivariance!"):
            self.play(Write(test1_desc))
            self.play(Swap(nodes_x[0], nodes_x[2], path_arc=PI/2), Swap(nodes_h[0], nodes_h[2], path_arc=-PI/2), run_time=1.5)
            self.play(Swap(nodes_x[0], nodes_x[2], path_arc=-PI/2), Swap(nodes_h[0], nodes_h[2], path_arc=PI/2), run_time=1.0)

        self.play(FadeOut(test1_desc), FadeOut(eq_title_ex))

        # Invariant Block
        inv_title_ex = Text("2. Invariant Layer (Sum Pooling)", font_size=24, color=GREEN_C).next_to(network_group, DOWN, buff=0.4).to_edge(LEFT, buff=1.0)

        sum_node = Polygon(ORIGIN, RIGHT*1.2, RIGHT*0.9+DOWN*0.6, RIGHT*0.3+DOWN*0.6, color=GREEN_E, fill_opacity=0.5)
        sum_text = MathTex(r"\sum", font_size=32).move_to(sum_node)
        sum_group = VGroup(sum_node, sum_text)

        out_circle = Circle(radius=0.4, color=YELLOW, fill_opacity=1.0)
        out_val = Text("12", font_size=28, color=BLACK, weight=BOLD).move_to(out_circle)
        final_out = VGroup(out_circle, out_val)

        with self.voiceover(text="Next, we pass these values through an Invariant layer using Sum Pooling, getting a total of 12."):
            self.play(network_group.animate.scale(0.7).next_to(title_ex, DOWN, buff=0.2))

            sum_group.next_to(nodes_h, DOWN, buff=0.8)
            sum_arrows = VGroup(*[Arrow(nodes_h[i].get_bottom(), sum_node.get_top(), buff=0.1, color=GREEN_C) for i in range(3)])

            final_out.next_to(sum_group, DOWN, buff=0.5)
            out_arr = Arrow(sum_group.get_bottom(), out_circle.get_top(), buff=0.1, color=YELLOW)
            out_text = MathTex(r"Output =", font_size=28).next_to(final_out, LEFT, buff=0.4)

            self.play(Write(inv_title_ex))
            self.play(FadeIn(sum_group), LaggedStart(*[GrowArrow(arr) for arr in sum_arrows], lag_ratio=0.2))
            self.play(GrowArrow(out_arr), Write(out_text), FadeIn(final_out))

        test2_desc = Text("Invariance Test:\nSwap Input → Output STILL", font_size=20, color=YELLOW).next_to(final_out, RIGHT, buff=0.5)

        with self.voiceover(text="Now, even if you shuffle the input elements a thousand more times, the final sum will always be 12. Our prediction is completely invariant to the order!"):
            self.play(Write(test2_desc))
            self.play(
                Swap(nodes_x[0], nodes_x[1], path_arc=PI/2),
                Swap(nodes_h[0], nodes_h[1], path_arc=-PI/2),
                run_time=1.5
            )
            self.play(out_circle.animate.set_stroke(WHITE, width=6).scale(1.2), run_time=0.3)
            self.play(out_circle.animate.set_stroke(YELLOW, width=2).scale(1/1.2), run_time=0.3)

        self.play(FadeOut(Group(*self.mobjects)))


In [ ]:
%%manim -qh -v WARNING LinearEquivariantTheory30
class LinearEquivariantTheory30(VoiceoverScene):
    def construct(self):
        self.camera.background_color = BLACK

        # Initialize voiceover service (Using EdgeTTS English)
        self.set_speech_service(EdgeTTSService(voice="en-US-GuyNeural"))

        # ==========================================
        # PART 0: GENERIC CONSTRUCTION PIPELINE
        # ==========================================
        title_generic = Text("Generic Construction", font_size=40, color=YELLOW, weight=BOLD)

        with self.voiceover(text="Now, let's look at the generic construction of a geometric neural network.") as tracker:
            self.play(Write(title_generic))
            self.play(title_generic.animate.to_edge(UP, buff=0.4))

        eq_box1 = Rectangle(width=2.5, height=1.2, color=BLUE_C, fill_opacity=0.2)
        eq_text1 = Text("Equivariant\nLayer", font_size=22, color=BLUE_C).move_to(eq_box1)
        layer1 = VGroup(eq_box1, eq_text1)

        eq_box2 = Rectangle(width=2.5, height=1.2, color=BLUE_C, fill_opacity=0.2)
        eq_text2 = Text("Equivariant\nLayer", font_size=22, color=BLUE_C).move_to(eq_box2)
        layer2 = VGroup(eq_box2, eq_text2)

        inv_box = Rectangle(width=2.5, height=1.2, color=GREEN_C, fill_opacity=0.2)
        inv_text = Text("Invariant\n(Pooling)", font_size=22, color=GREEN_C).move_to(inv_box)
        layer3 = VGroup(inv_box, inv_text)

        pipeline = VGroup(layer1, layer2, layer3).arrange(RIGHT, buff=0.8).next_to(title_generic, DOWN, buff=0.6)
        arrow1 = Arrow(layer1.get_right(), layer2.get_left(), buff=0.1, color=WHITE)
        arrow2 = Arrow(layer2.get_right(), layer3.get_left(), buff=0.1, color=WHITE)

        with self.voiceover(text="As we discussed, these models stack multiple equivariant layers to process structural information, followed by an invariant pooling layer at the end."):
            self.play(FadeIn(layer1))
            self.play(GrowArrow(arrow1), FadeIn(layer2))
            self.play(GrowArrow(arrow2), FadeIn(layer3))

        types_title = Text("Equivariant layers (Example):", font_size=26, color=WHITE).next_to(pipeline, DOWN, buff=0.8).to_edge(LEFT, buff=1.5)

        with self.voiceover(text="There are many types of equivariant layers. Some common examples include..."):
            self.play(Write(types_title))

        b1 = Text("- Linear equivariant + nonlinearity", font_size=24, color=LIGHT_GREY).next_to(types_title, DOWN, buff=0.3, aligned_edge=LEFT).shift(RIGHT * 0.5)
        b2 = Text("- Group convolutions", font_size=24, color=LIGHT_GREY).next_to(b1, DOWN, buff=0.3, aligned_edge=LEFT)
        b3 = Text("- Equivariant polynomials", font_size=24, color=LIGHT_GREY).next_to(b2, DOWN, buff=0.3, aligned_edge=LEFT)

        with self.voiceover(text="Linear equivariant layers with non-linearities, Group convolutions, or Equivariant polynomials."):
            self.play(FadeIn(b1), FadeIn(b2), FadeIn(b3), run_time=tracker.duration if tracker.duration > 1.5 else 1.5)

        arch_text = Text("e.g., CNNs, GNNs, DeepSets, ...", font_size=26, color=GREEN_C, slant=ITALIC).next_to(b3, DOWN, buff=0.6).align_to(types_title, LEFT)

        with self.voiceover(text="These are the building blocks of famous architectures like CNNs, Graph Neural Networks, and DeepSets."):
            self.play(Write(arch_text))

        self.play(FadeOut(Group(*self.mobjects)))


        # ==========================================
        # PART 1: LINEAR EQUIVARIANT LAYERS
        # ==========================================
        title = Text("Linear Equivariant Layers", font_size=36, color=WHITE, weight=BOLD)
        subtitle = Text("Parameter Sharing", font_size=28, color=LIGHT_GREY)
        header = VGroup(title, subtitle).arrange(DOWN, buff=0.2)

        with self.voiceover(text="Let's focus on the most fundamental one: the Linear Equivariant Layer, and its core principle, Parameter Sharing."):
            self.play(Write(header))
            self.play(header.animate.to_edge(UP, buff=0.4))

        layer_eq = MathTex(r"f(x) = \sigma(W x + b)", font_size=42)
        layer_eq.next_to(header, DOWN, buff=0.8)

        with self.voiceover(text="A standard linear layer applies a weight matrix W to the input x, followed by an activation function."):
            self.play(Write(layer_eq))

        w_box = SurroundingRectangle(layer_eq[0][5], color=YELLOW, buff=0.1)
        w_text = Text("Symmetry creates structure in W (Parameter Sharing)", font_size=24, color=YELLOW).next_to(w_box, DOWN, buff=0.5)

        with self.voiceover(text="To make this layer equivariant to a specific symmetry, we must force a specific structure onto the weight matrix W. This mathematically results in parameter sharing."):
            self.play(Create(w_box))
            self.play(FadeIn(w_text))

        self.play(FadeOut(VGroup(header, layer_eq, w_box, w_text)))


        # ==========================================
        # PART 2: EQUIVARIANCE CONSTRAINT
        # ==========================================
        constraint_title = Text("1. Equivariance Constraint", font_size=32, color=YELLOW).to_edge(UP, buff=0.5)

        with self.voiceover(text="How do we define this mathematical constraint? It can be written as an elegant equation."):
            self.play(Write(constraint_title))

        main_eq = MathTex(r"W P_g x = P_g W x", font_size=48)

        with self.voiceover(text="W times P_g times x, equals P_g times W times x."):
            self.play(Write(main_eq))

        desc_group = VGroup()
        x_lbl = MathTex("x", font_size=28, color=WHITE)
        x_txt = Text(": Input data", font_size=24, color=LIGHT_GREY)
        desc_group.add(VGroup(x_lbl, x_txt).arrange(RIGHT, buff=0.2))

        w_lbl = MathTex("W", font_size=28, color=GREEN_C)
        w_txt = Text(": Computational Neural Network layer", font_size=24, color=LIGHT_GREY)
        desc_group.add(VGroup(w_lbl, w_txt).arrange(RIGHT, buff=0.2))

        p_lbl = MathTex("P_g", font_size=28, color=BLUE_C)
        p_txt = Text(": Symmetry transformation (Rotate, Flip, Permute...)", font_size=24, color=LIGHT_GREY)
        desc_group.add(VGroup(p_lbl, p_txt).arrange(RIGHT, buff=0.2))

        desc_group.arrange(DOWN, aligned_edge=LEFT, buff=0.4).next_to(main_eq, DOWN, buff=1.0)

        with self.voiceover(text="Here, x is the input data. W is the learnable weight matrix."):
            self.play(
                main_eq[0][0].animate.set_color(GREEN_C),
                main_eq[0][7].animate.set_color(GREEN_C)
            )
            self.play(FadeIn(desc_group[0], shift=RIGHT*0.5))
            self.play(FadeIn(desc_group[1], shift=RIGHT*0.5))

        with self.voiceover(text="And P_g is the symmetry transformation matrix, representing actions like rotating, flipping, or permuting."):
            self.play(
                main_eq[0][1:3].animate.set_color(BLUE_C),
                main_eq[0][5:7].animate.set_color(BLUE_C)
            )
            self.play(FadeIn(desc_group[2], shift=RIGHT*0.5))

        self.play(FadeOut(desc_group))
        self.play(main_eq.animate.to_edge(UP, buff=1.5))

        ex_title = Text("Visual example:", font_size=28, color=YELLOW).next_to(main_eq, DOWN, buff=0.5)

        with self.voiceover(text="Let's visualize this with an example."):
            self.play(Write(ex_title))

        grid_x = VGroup(*[Square(side_length=0.6, fill_opacity=0.8, fill_color=color) for color in [RED, BLUE, GREEN, ORANGE]])
        grid_x.arrange_in_grid(rows=2, cols=2, buff=0.05).move_to(LEFT * 4 + DOWN * 1.5)
        lbl_x = MathTex("x", font_size=28, color=WHITE).next_to(grid_x, DOWN, buff=0.3)

        with self.voiceover(text="We start with an input grid x."):
            self.play(FadeIn(grid_x), Write(lbl_x))

        grid_pgx = grid_x.copy().move_to(ORIGIN + DOWN * 1.5)
        lbl_pgx = MathTex("P_g x", font_size=28, color=BLUE_C).next_to(grid_pgx, DOWN, buff=0.3)
        arrow_pg = Arrow(grid_x.get_right(), grid_pgx.get_left(), buff=0.5, color=BLUE_C)
        txt_pg = Text("Rotate 90°", font_size=20, color=BLUE_C).next_to(arrow_pg, UP, buff=0.1)

        with self.voiceover(text="First, we apply a geometric transformation, such as a 90-degree rotation. This gives us P_g times x."):
            self.play(GrowArrow(arrow_pg), Write(txt_pg))
            self.play(TransformFromCopy(grid_x, grid_pgx))
            self.play(Rotate(grid_pgx, angle=-PI/2), run_time=1.5)
            self.play(Write(lbl_pgx))

        feat_w = Circle(radius=0.5, fill_opacity=0.8, fill_color=RED).move_to(RIGHT * 4 + DOWN * 1.5)
        lbl_wx = MathTex("W(P_g x)", font_size=28, color=GREEN_C).next_to(feat_w, DOWN, buff=0.3)
        arrow_w = Arrow(grid_pgx.get_right(), feat_w.get_left(), buff=0.5, color=GREEN_C)
        txt_w = Text("Extract", font_size=20, color=GREEN_C).next_to(arrow_w, UP, buff=0.1)

        with self.voiceover(text="Then, we apply the neural network layer W to extract features. The result is W applied to P_g of x."):
            self.play(GrowArrow(arrow_w), Write(txt_w))
            self.play(TransformFromCopy(grid_pgx, feat_w), Write(lbl_wx))

        self.play(FadeOut(Group(*self.mobjects)))

        # --- SCENE 2: COMMUTATIVE DIAGRAM ---
        title = Text("2. Commutative Diagram", font_size=32, color=YELLOW).to_edge(UP, buff=0.5)
        info_text = Text("Path 1: Transform first, Compute later", font_size=24, color=BLUE_A).next_to(title, DOWN, buff=0.5)

        with self.voiceover(text="This brings us to a fundamental concept: The Commutative Diagram."):
            self.play(Write(title))
            self.play(Write(info_text))

        node_x = MathTex("x", font_size=36).move_to(LEFT * 3.0 + UP * 1.0)
        node_Px = MathTex("P_g x", font_size=36, color=BLUE_C).move_to(RIGHT * 3.0 + UP * 1.0)
        node_Wx = MathTex("W x", font_size=36, color=GREEN_C).move_to(LEFT * 3.0 + DOWN * 1.5)

        final_left = MathTex("W(P_g x)", font_size=36, color=YELLOW)
        final_eq = MathTex("=", font_size=36, color=WHITE)
        final_right = MathTex("P_g(W x)", font_size=36, color=YELLOW)
        node_final = VGroup(final_left, final_eq, final_right).arrange(RIGHT, buff=0.2).move_to(RIGHT * 3.0 + DOWN * 1.5)

        self.play(FadeIn(node_x))

        arrow_P_top = Arrow(node_x.get_right(), node_Px.get_left(), color=BLUE_C, buff=0.3)
        lbl_P_top = MathTex("P_g", font_size=32, color=BLUE_C).next_to(arrow_P_top, UP, buff=0.1)
        arrow_W_right = Arrow(node_Px.get_bottom(), node_final.get_top(), color=GREEN_C, buff=0.3)
        lbl_W_right = MathTex("W", font_size=32, color=GREEN_C).next_to(arrow_W_right, RIGHT, buff=0.1)

        with self.voiceover(text="Path number one: We first transform the input with P_g, and then apply the neural network W. This gives us the left side of our equation."):
            self.play(GrowArrow(arrow_P_top), Write(lbl_P_top))
            self.play(FadeIn(node_Px))
            self.play(GrowArrow(arrow_W_right), Write(lbl_W_right))
            self.play(FadeIn(final_left))

        info_text_2 = Text("Path 2: Compute first, Transform later", font_size=24, color=GREEN_A).move_to(info_text)

        with self.voiceover(text="Path number two: We apply the neural network W first, and then apply the geometric transformation P_g to the feature maps."):
            self.play(Transform(info_text, info_text_2))
            arrow_W_left = Arrow(node_x.get_bottom(), node_Wx.get_top(), color=GREEN_C, buff=0.3)
            lbl_W_left = MathTex("W", font_size=32, color=GREEN_C).next_to(arrow_W_left, LEFT, buff=0.1)
            arrow_P_bot = Arrow(node_Wx.get_right(), node_final.get_left(), color=BLUE_C, buff=0.3)
            lbl_P_bot = MathTex("P_g", font_size=32, color=BLUE_C).next_to(arrow_P_bot, DOWN, buff=0.1)

            self.play(GrowArrow(arrow_W_left), Write(lbl_W_left))
            self.play(FadeIn(node_Wx))
            self.play(GrowArrow(arrow_P_bot), Write(lbl_P_bot))
            self.play(FadeIn(final_eq), FadeIn(final_right))

        conclusion = Text("EQUIVARIANCE:\nBoth paths yield the exact same result!", font_size=20, color=YELLOW, weight=BOLD).to_edge(DOWN, buff=0.5)

        with self.voiceover(text="For a layer to be equivariant, both paths must commute. Meaning, it doesn't matter which path you take, you must end up with the exact same result."):
            self.play(Transform(info_text, conclusion))
            final_box = SurroundingRectangle(node_final, color=YELLOW, buff=0.2)
            self.play(Create(final_box))

        self.play(FadeOut(Group(*self.mobjects)))


        # ==========================================
        # PART 3: DERIVING THE CONSTRAINT
        # ==========================================
        title = Text("3. Deriving the Constraint for Matrix W", font_size=32, color=YELLOW).to_edge(UP, buff=0.5)
        intro_text = Text("Explaining the Permutation Constraint formula:", font_size=28, color=BLUE_A).next_to(title, DOWN, buff=0.5)
        eq_explain = MathTex(r"P_\tau W = W P_\tau", font_size=48).next_to(intro_text, DOWN, buff=0.5)

        with self.voiceover(text="Now, let's derive the constraint for the weight matrix, specifically for the permutation group."):
            self.play(Write(title))
            self.play(Write(intro_text))
            self.play(Write(eq_explain))

        desc_group = VGroup()
        p_lbl = MathTex(r"P_\tau", font_size=28, color=BLUE_C)
        p_txt = Text(": Permutation Matrix (swaps rows/columns)", font_size=24, color=LIGHT_GREY)
        desc_group.add(VGroup(p_lbl, p_txt).arrange(RIGHT, buff=0.2))

        w_lbl = MathTex("W", font_size=28, color=GREEN_C)
        w_txt = Text(": Learnable Weights Matrix", font_size=24, color=LIGHT_GREY)
        desc_group.add(VGroup(w_lbl, w_txt).arrange(RIGHT, buff=0.2))

        desc_group.arrange(DOWN, aligned_edge=LEFT, buff=0.4).next_to(eq_explain, DOWN, buff=0.8)

        with self.voiceover(text="Here, P_tau is the permutation matrix that shuffles rows and columns, and W is our parameter matrix."):
            self.play(
                eq_explain[0][0:2].animate.set_color(BLUE_C),
                eq_explain[0][2].animate.set_color(GREEN_C),
                eq_explain[0][4].animate.set_color(GREEN_C),
                eq_explain[0][5:7].animate.set_color(BLUE_C)
            )
            for desc in desc_group:
                self.play(FadeIn(desc, shift=RIGHT*0.5))

        self.play(FadeOut(VGroup(intro_text, eq_explain, desc_group)))

        step1_txt = Text("From the original equation (valid for all x):", font_size=24, color=LIGHT_GREY)
        step1_math = MathTex(r"W P_g = P_g W", font_size=36)
        step1 = VGroup(step1_txt, step1_math).arrange(DOWN, buff=0.3)

        step2_txt = Text("Assuming G is the Permutation Group, we use the orthogonal matrix property:", font_size=24, color=LIGHT_GREY)
        step2_math = MathTex(r"P_\tau^{-1} = P_\tau^\top", font_size=36, color=BLUE_C)
        step2 = VGroup(step2_txt, step2_math).arrange(DOWN, buff=0.3)

        step3_txt = Text("Multiply both sides by the transpose from the right:", font_size=24, color=LIGHT_GREY)
        step3_math = MathTex(r"W P_\tau P_\tau^\top = P_\tau W P_\tau^\top", font_size=36)
        step3_final = MathTex(r"W = P_\tau W P_\tau^\top", font_size=42, color=YELLOW)
        step3 = VGroup(step3_txt, step3_math, step3_final).arrange(DOWN, buff=0.3)

        derivation_group = VGroup(step1, step2, step3).arrange(DOWN, buff=0.8).next_to(title, DOWN, buff=0.5)

        with self.voiceover(text="Starting from the original commutative equation, we drop the x."):
            self.play(FadeIn(step1_txt), Write(step1_math))

        with self.voiceover(text="Since permutation matrices are orthogonal, their inverse is simply their transpose."):
            self.play(FadeIn(step2_txt), Write(step2_math))

        with self.voiceover(text="By multiplying both sides by the transpose from the right, we arrive at the final condition. The matrix W must be invariant to conjugation by the permutation matrix."):
            self.play(FadeIn(step3_txt), Write(step3_math))
            self.play(Transform(step3_math, step3_final))
            box = SurroundingRectangle(step3_math, color=RED_C, buff=0.2)
            self.play(Create(box))

        self.play(FadeOut(Group(*self.mobjects)))


        # ==========================================
        # PART 4: VISUALIZING PARAMETER SHARING
        # ==========================================
        header = VGroup(Text("Linear Equivariant Layers", font_size=36, color=WHITE, weight=BOLD), Text("Parameter Sharing", font_size=28, color=LIGHT_GREY)).arrange(DOWN, buff=0.2).to_edge(UP, buff=0.4)
        grid_title = Text("The Essence of Parameter Sharing in W", font_size=30, color=BLUE_B).next_to(header, DOWN, buff=0.4)

        with self.voiceover(text="What does this mathematical constraint actually mean in practice? It leads directly to Parameter Sharing."):
            self.play(Write(header))
            self.play(Write(grid_title))

        cond_text1_part1 = Text("If ", font_size=28)
        cond_text1_math1 = MathTex(r"(\tau(i), \tau(j)) = (k, l)", font_size=28)
        cond_text1_part2 = Text(" with transformation ", font_size=28)
        cond_text1_math2 = MathTex(r"\tau \in G", font_size=28)
        cond_text1 = VGroup(cond_text1_part1, cond_text1_math1, cond_text1_part2, cond_text1_math2).arrange(RIGHT, buff=0.1)
        cond_text2 = MathTex(r"\text{then } W_{ij} = W_{kl}", font_size=36, color=RED_C)
        cond_group = VGroup(cond_text1, cond_text2).arrange(DOWN, buff=0.3).next_to(grid_title, DOWN, buff=0.4)

        with self.voiceover(text="It says: If a symmetry transformation tau maps the index pair i, j to a new pair k, l ..."):
            self.play(Write(cond_text1))

        grid_size = 5
        square_size = 0.6
        matrix_w = VGroup()
        squares = {}
        for row in range(grid_size):
            for col in range(grid_size):
                sq = Square(side_length=square_size, stroke_color=WHITE, stroke_width=1, fill_color=BLACK, fill_opacity=1)
                sq.move_to(RIGHT * (col * square_size) + DOWN * (row * square_size))
                matrix_w.add(sq)
                squares[(row, col)] = sq
        matrix_w.move_to(DOWN * 1.5)
        w_label = MathTex(r"W \in \mathbb{R}^{n \times n}", font_size=32).next_to(matrix_w, LEFT, buff=0.8)

        with self.voiceover(text="Consider a dense weight matrix W."):
            self.play(FadeIn(matrix_w), Write(w_label))

        i, j = 1, 1
        k, l = 3, 4
        cell_ij = squares[(i, j)]
        cell_kl = squares[(k, l)]
        label_ij = MathTex("(i, j)", font_size=20).next_to(cell_ij, UP, buff=0.1)

        with self.voiceover(text="We select an element at position i, j."):
            self.play(cell_ij.animate.set_fill(RED_C, opacity=0.8))
            self.play(FadeIn(label_ij))

        arrow_tau = CurvedArrow(cell_ij.get_center(), cell_kl.get_center(), angle=-PI/2, color=YELLOW)
        label_tau = MathTex(r"\tau", font_size=24, color=YELLOW).next_to(arrow_tau, DOWN, buff=0.1)
        label_kl = MathTex("(k, l)", font_size=20).next_to(cell_kl, RIGHT, buff=0.1)

        with self.voiceover(text="If the geometric transformation maps this position to k, l..."):
            self.play(Create(arrow_tau), Write(label_tau))

        with self.voiceover(text="Then the element at k, l MUST share the exact same weight value."):
            self.play(cell_kl.animate.set_fill(RED_C, opacity=0.8))
            self.play(FadeIn(label_kl))
            self.play(Write(cond_text2))

        extra_pairs = [((0, 2), (2, 0), BLUE_C), ((4, 1), (0, 4), GREEN_C)]
        with self.voiceover(text="This rule applies across the entire matrix. Symmetric positions are forced to share parameters, greatly reducing the model's complexity."):
            for (i1, j1), (k1, l1), col in extra_pairs:
                sq1 = squares[(i1, j1)]
                sq2 = squares[(k1, l1)]
                arr = CurvedArrow(sq1.get_center(), sq2.get_center(), angle=-PI/3, color=col)
                self.play(sq1.animate.set_fill(col, opacity=0.8), run_time=0.4)
                self.play(Create(arr), run_time=0.4)
                self.play(sq2.animate.set_fill(col, opacity=0.8), run_time=0.4)
                self.play(FadeOut(arr), run_time=0.4)

        self.play(FadeOut(Group(*self.mobjects)))


        # ==========================================
        # PART 5: 1D TRANSLATION EXAMPLE (CIRCULANT MATRIX)
        # ==========================================
        title_ex = Text("Example: The Essence of 1D Translation (Circulant Matrix)", font_size=32, color=YELLOW).to_edge(UP, buff=0.4)

        with self.voiceover(text="Let's view a classic example: 1D Cyclic Translation, which is the foundation of Convolutions."):
            self.play(Write(title_ex))

        sq_size = 0.8
        colors_list = [RED_C, BLUE_C, GREEN_C, ORANGE]
        values_list = ["1", "2", "3", "4"]

        def create_box(val_str, color):
            box = Square(side_length=sq_size, color=WHITE, stroke_width=1)
            box.set_fill(color, opacity=0.3)
            txt = Text(val_str, font_size=32, color=color).move_to(box.get_center())
            return VGroup(box, txt)

        row0 = VGroup(*[create_box(values_list[idx], colors_list[idx]) for idx in range(4)]).arrange(RIGHT, buff=0).move_to(UP * 1.5)
        stacked_matrix = VGroup(row0)

        with self.voiceover(text="We begin with a base weight vector containing four unique parameters."):
            self.play(FadeIn(row0, shift=DOWN*0.5))

        with self.voiceover(text="If the data shifts, the neural network weights must shift cyclically as well to maintain equivariance. We duplicate and shift the vector step by step."):
            for step in range(1, 4):
                prev_row = stacked_matrix[step-1]
                new_row = prev_row.copy()
                self.play(new_row.animate.shift(DOWN * sq_size), run_time=0.4)

                last_element = new_row[-1]
                other_elements = VGroup(*new_row[:-1])
                target_pos = new_row[0].get_center()
                arc_path = ArcBetweenPoints(last_element.get_center(), target_pos, angle=PI/1.5)

                self.play(
                    MoveAlongPath(last_element, arc_path),
                    other_elements.animate.shift(RIGHT * sq_size),
                    run_time=0.8
                )
                new_row_reordered = VGroup(last_element, *other_elements)
                stacked_matrix.add(new_row_reordered)

        self.play(stacked_matrix.animate.move_to(ORIGIN), run_time=1)
        mat_h = stacked_matrix.height + 0.2
        mat_w = 0.2
        left_bracket = VGroup(Line(UP*mat_h/2, DOWN*mat_h/2), Line(UP*mat_h/2, UP*mat_h/2 + RIGHT*mat_w), Line(DOWN*mat_h/2, DOWN*mat_h/2 + RIGHT*mat_w)).next_to(stacked_matrix, LEFT, buff=0.1)
        right_bracket = VGroup(Line(UP*mat_h/2, DOWN*mat_h/2), Line(UP*mat_h/2, UP*mat_h/2 + LEFT*mat_w), Line(DOWN*mat_h/2, DOWN*mat_h/2 + LEFT*mat_w)).next_to(stacked_matrix, RIGHT, buff=0.1)

        desc_ex = Text(
            "1D cyclic translation creates a Circulant Matrix structure.\n"
            "Identical parameters always lie on the same diagonal (Parameter Sharing).",
            font_size=24, color=LIGHT_GREY, line_spacing=1.3
        ).next_to(stacked_matrix, DOWN, buff=0.8)

        with self.voiceover(text="This stacked operation forms a Circulant Matrix. Notice how the same parameters align perfectly along the diagonals. This is the visual essence of parameter sharing in CNNs."):
            self.play(Create(left_bracket), Create(right_bracket))
            self.play(Write(desc_ex))

        self.play(FadeOut(Group(*self.mobjects)))


## slide 35-36
Example: Shift in 1D

In [ ]:
%%manim -qh -v WARNING Shiftin1D35

class Shiftin1D35(VoiceoverScene):
    def construct(self):
        self.camera.background_color = BLACK

        # Khởi tạo dịch vụ tạo giọng đọc tiếng Anh
        # self.set_speech_service(GTTSService(lang="en", tld="com"))
        self.set_speech_service(EdgeTTSService(voice="en-US-GuyNeural"))

        # ==========================================
        # PART 1: INTUITION ON CYCLIC SHIFT
        # ==========================================
        title = Text("Example 1: 1D Shift", font_size=36, color=BLUE, weight=BOLD)
        step1_title = Text("Understanding Cyclic Shift", font_size=28, color=YELLOW)

        with self.voiceover(text="Let's build our intuition on the one dimensional cyclic shift."):
            self.play(Write(title))
            self.play(title.animate.to_edge(UP, buff=0.4))
            step1_title.next_to(title, DOWN, buff=0.5)
            self.play(Write(step1_title))

        # Create 4 squares representing 1D vector
        boxes = VGroup(*[Square(side_length=0.8, color=WHITE) for _ in range(4)])
        boxes.arrange(RIGHT, buff=0.2)
        boxes.next_to(step1_title, DOWN, buff=1.0)
        labels = VGroup(*[MathTex(f"x_{i+1}", font_size=32).move_to(boxes[i]) for i in range(4)])
        array_group = VGroup(boxes, labels)

        with self.voiceover(text="Imagine a simple one-dimensional vector with four elements, x 1 through x 4."):
            self.play(FadeIn(array_group))

        desc1 = VGroup(
            Text("Group", font_size=24),
            MathTex("G", font_size=28),
            Text(": Shift operation with wrap-around.", font_size=24)
        ).arrange(RIGHT, buff=0.1).next_to(array_group, DOWN, buff=1.0)

        targets = [boxes[(i+1)%4].get_center() for i in range(4)]
        shift_anims = [labels[i].animate.move_to(targets[i]) for i in range(4)]

        with self.voiceover(text="A cyclic shift operation moves each element to the right, where the last element wraps around back to the beginning."):
            self.play(Write(desc1))
            self.play(*shift_anims, run_time=1.5)

        desc2 = MathTex(r"\text{Operation: } t_1(j) = j + 1 \pmod{n}", font_size=32, color=GREEN_C)
        desc2.move_to(desc1)

        with self.voiceover(text="Mathematically, the new index is calculated using addition modulo n."):
            self.play(Transform(desc1, desc2))

        self.play(FadeOut(Group(*self.mobjects)))


        # ==========================================
        # PART 2: BUILDING WEIGHT MATRIX W
        # ==========================================
        step2_title = VGroup(
            Text("Structure of Circular Matrix", font_size=28, color=YELLOW),
            MathTex("W", font_size=32, color=YELLOW)
        ).arrange(RIGHT, buff=0.15).to_edge(UP, buff=1.0)

        p_elements = [
            ["a", "b", "c", "d"],
            ["d", "a", "b", "c"],
            ["c", "d", "a", "b"],
            ["b", "c", "d", "a"]
        ]
        p_matrix = Matrix(p_elements)
        p_label = MathTex("W = ").next_to(p_matrix, LEFT)
        p_group = VGroup(p_label, p_matrix).next_to(step2_title, DOWN, buff=0.8)

        with self.voiceover(text="How does this dictate the structure of our weight matrix W?"):
            self.play(Write(step2_title))
            self.play(FadeIn(p_group))

        matrix_desc1 = VGroup(
            Text("To be compatible with the shift, the rows of", font_size=22),
            MathTex("W", font_size=26),
            Text("must also shift.", font_size=22)
        ).arrange(RIGHT, buff=0.1).next_to(p_group, DOWN, buff=0.8)

        with self.voiceover(text="To be compatible with the input shift, the rows of the matrix must also shift cyclically."):
            self.play(Write(matrix_desc1))

        matrix_desc2 = VGroup(
            Text("Result: We only need exactly", font_size=24, color=GREEN_C),
            MathTex("n", font_size=28, color=GREEN_C),
            Text("parameters (", font_size=24, color=GREEN_C),
            MathTex("n = 4", font_size=28, color=GREEN_C),
            Text(").", font_size=24, color=GREEN_C)
        ).arrange(RIGHT, buff=0.1).next_to(matrix_desc1, DOWN, buff=0.5)

        with self.voiceover(text="This results in a Circulant Matrix. We now only need exactly n parameters instead of n squared."):
            self.play(Write(matrix_desc2))

        colors = [RED_C, BLUE_C, GREEN_C, ORANGE]
        entries = p_matrix.get_entries()

        with self.voiceover(text="Notice how the identical parameters share the exact same diagonal stripes."):
            for offset in range(4):
                diag_group = VGroup(*[entries[i * 4 + ((i + offset) % 4)] for i in range(4)])
                self.play(diag_group.animate.set_color(colors[offset]), run_time=0.6)

        self.play(FadeOut(Group(*self.mobjects)))


        # ==========================================
        # PART 3: VISUALIZING 1D CONVOLUTION (CNN MECHANISM)
        # ==========================================
        step3_title = Text("How 1D CNN Operates", font_size=28, color=YELLOW)
        step3_title.to_edge(UP, buff=1.0)

        with self.voiceover(text="Step 3. Let's see how this translates to the mechanism of a 1D Convolutional Neural Network."):
            self.play(Write(step3_title))

        in_labels = ["x_1", "x_2", "x_3", "x_4", "x_1", "x_2"]
        in_boxes = VGroup(*[Square(side_length=0.8, color=WHITE) for _ in range(6)])
        in_boxes.arrange(RIGHT, buff=0)
        in_texts = VGroup(*[MathTex(lbl, font_size=32).move_to(in_boxes[i]) for i, lbl in enumerate(in_labels)])

        for i in [4, 5]:
            in_boxes[i].set_stroke(opacity=0.3)
            in_texts[i].set_opacity(0.3)

        input_group = VGroup(in_boxes, in_texts).next_to(step3_title, DOWN, buff=1.0)
        input_title = Text("Input X (with Circular Padding)", font_size=20, color=LIGHT_GREY).next_to(input_group, UP, buff=0.3)

        with self.voiceover(text="We start with our input sequence and apply circular padding by copying the initial elements to the end."):
            self.play(FadeIn(input_group), Write(input_title))

        f_labels = ["w_1", "w_2", "w_3"]
        f_boxes = VGroup(*[Square(side_length=0.8, color=BLUE_C, fill_color=BLUE_E, fill_opacity=0.5) for _ in range(3)])
        f_boxes.arrange(RIGHT, buff=0)
        f_texts = VGroup(*[MathTex(lbl, font_size=32).move_to(f_boxes[i]) for i, lbl in enumerate(f_labels)])

        filter_group = VGroup(f_boxes, f_texts)
        filter_group.next_to(in_boxes[0:3], DOWN, buff=0.1)

        f_title = Text("Filter W", font_size=20, color=BLUE_C).next_to(filter_group, DOWN, buff=0.2)
        filter_unit = VGroup(filter_group, f_title)

        y_labels = ["y_1", "y_2", "y_3", "y_4"]
        y_boxes = VGroup(*[Square(side_length=0.8, color=GREEN_C) for _ in range(4)])
        y_boxes.arrange(RIGHT, buff=0)
        y_texts = VGroup(*[MathTex(lbl, font_size=32, color=GREEN_C).move_to(y_boxes[i]) for i, lbl in enumerate(y_labels)])

        out_boxes_group = VGroup(y_boxes, y_texts)
        out_boxes_group.next_to(input_group, DOWN, buff=3.0).align_to(in_boxes[0], LEFT)
        out_title = Text("Output Y", font_size=20, color=GREEN_C).next_to(out_boxes_group, LEFT, buff=0.5)

        with self.voiceover(text="Then, we introduce a learnable filter, or kernel, that will slide across the input."):
            self.play(FadeIn(filter_unit))
            self.play(FadeIn(y_boxes), Write(out_title))

        with self.voiceover(text="As the filter slides step by step, it computes the output. When it reaches the end, the circular padding allows it to seamlessly wrap around, just like our cyclic shift."):
            for i in range(4):
                if i > 0:
                    self.play(filter_unit.animate.shift(RIGHT * 0.8), run_time=0.8)

                if i == 2:
                    self.play(
                        in_boxes[4].animate.set_stroke(YELLOW, opacity=1),
                        in_texts[4].animate.set_opacity(1).set_color(YELLOW),
                        run_time=0.4
                    )
                if i == 3:
                    self.play(
                        in_boxes[5].animate.set_stroke(YELLOW, opacity=1),
                        in_texts[5].animate.set_opacity(1).set_color(YELLOW),
                        run_time=0.4
                    )

                self.play(Write(y_texts[i]), run_time=0.5)

        math_eq = MathTex(
            r"y_i = \sum_{j} w_j \cdot x_{(i+j) \pmod n}",
            font_size=40
        ).next_to(step3_title, DOWN, buff=0.8)

        final_conclusion = Text(
            "This sliding window with wrap-around\n"
            "is EXACTLY what the Circulant Matrix does!\n"
            "Therefore, 1D CNNs with Circular Padding\n"
            "are naturally Translation Equivariant.",
            font_size=20, color=GREEN_C, line_spacing=1.2
        ).next_to(math_eq, DOWN, buff=0.6)

        if final_conclusion.width > config.frame_width - 1:
            final_conclusion.scale_to_fit_width(config.frame_width - 1)

        with self.voiceover(text="This sliding window with wrap-around is mathematically identical to our Circulant Matrix. Therefore, 1D CNNs with Circular Padding are naturally translation equivariant!"):
            self.play(FadeOut(VGroup(input_group, input_title, filter_unit, out_boxes_group, out_title)))
            self.play(Write(math_eq))
            self.play(Write(final_conclusion))

        self.play(FadeOut(Group(*self.mobjects)))


## slide 37-39
Example: all permutations of a set

In [ ]:
%%manim -qh -v WARNING AllpermutationsOfAset37

from manim import *
from manim_voiceover import VoiceoverScene
from manim_voiceover.services.gtts import GTTSService

class AllpermutationsOfAset37(VoiceoverScene):
    def construct(self):
        self.camera.background_color = BLACK

        # Khởi tạo dịch vụ tạo giọng đọc tiếng Anh
        self.set_speech_service(EdgeTTSService(voice="en-US-GuyNeural"))

        # ==========================================
        # PHẦN 1: TIÊU ĐỀ
        # ==========================================
        title = Text("Example 2: All permutations of a set", font_size=36, color=BLUE_C, weight=BOLD)
        step1_title = Text("Understanding Permutation Symmetry", font_size=28, color=YELLOW)

        with self.voiceover(text="Next, let's explore example number two: all permutations of a set, and understand permutation symmetry."):
            self.play(Write(title))
            self.play(title.animate.to_edge(UP, buff=0.4))
            step1_title.next_to(title, DOWN, buff=0.5)
            self.play(Write(step1_title))

        # ==========================================
        # PHẦN 1.1: VÍ DỤ THỰC TẾ - 3D POINT CLOUD
        # ==========================================
        # Lưu ý: Bạn có thể đổi to_edge(UP) thành next_to(step1_title, DOWN, buff=0.5) nếu nối tiếp code cũ
        pose_title = Text("Input: Set of Vectors (e.g., 3D Point Cloud)", font_size=24, color=GREEN_C).to_edge(UP, buff=1.0)

        with self.voiceover(text="Consider an input consisting of a set of vectors, such as a 3D point cloud representing an object."):
            self.play(FadeIn(pose_title))

        # 1. TẠO POINT CLOUD HÌNH TORUS (BÁNH DONUT) TRONG KHÔNG GIAN 3D
        point_cloud = VGroup()
        for u in np.linspace(0, TAU, 16, endpoint=False):
            for v in np.linspace(0, TAU, 10, endpoint=False):
                # Phương trình tham số của hình Torus
                x = (1.5 + 0.5 * np.cos(v)) * np.cos(u)
                y = (1.5 + 0.5 * np.cos(v)) * np.sin(u)
                z = 0.5 * np.sin(v)
                point_cloud.add(Dot(np.array([x, y, z]), radius=0.03, color=GRAY_D))

        # Xoay nghiêng khối Point Cloud để lộ ra cấu trúc 3D
        point_cloud.rotate(PI/3, axis=RIGHT).rotate(PI/4, axis=UP)
        point_cloud.move_to(LEFT * 3.5 + DOWN * 0.2)

        # Trích xuất 6 điểm đặc trưng phân tán để theo dõi (Tracked points)
        highlight_indices = [15, 42, 68, 95, 122, 149]
        colors = [RED, GREEN_C, BLUE, YELLOW, PURPLE, ORANGE]
        highlight_dots = VGroup()

        for idx, color in zip(highlight_indices, colors):
            dot = point_cloud[idx]
            dot.set_color(color).set_radius(0.09) # Làm to và tô màu các điểm theo dõi
            highlight_dots.add(dot)

        self.play(FadeIn(point_cloud, lag_ratio=0.01), run_time=2)

        # 2. KHỞI TẠO MA TRẬN ĐẠI DIỆN
        matrix_elements = [
            [r"x_1", r"y_1", r"z_1"],
            [r"x_2", r"y_2", r"z_2"],
            [r"x_3", r"y_3", r"z_3"],
            [r"x_4", r"y_4", r"z_4"],
            [r"x_5", r"y_5", r"z_5"],
            [r"x_6", r"y_6", r"z_6"]
        ]
        pose_matrix = Matrix(matrix_elements, left_bracket="[", right_bracket="]").scale(0.8)

        # Nhấn mạnh đây là ma trận Nx3
        matrix_group = VGroup(MathTex(r"X \in \mathbb{R}^{N \times 3} = ", font_size=32), pose_matrix).arrange(RIGHT).move_to(RIGHT*2.5 + DOWN*0.2)

        for i, color in enumerate(colors):
            pose_matrix.get_rows()[i].set_color(color)

        with self.voiceover(text="We can represent this point cloud as an N by 3 coordinate matrix. Let's track a few specific points."):
            self.play(FadeIn(matrix_group))
            # Chớp sáng các điểm trên mây để kết nối với màu trong ma trận
            self.play(LaggedStart(*[Flash(dot, color=dot.get_color(), line_length=0.15) for dot in highlight_dots], lag_ratio=0.15))

        pose_desc = Text(
            # "Đám mây điểm (Point Cloud) được lưu dưới dạng Ma trận N x 3.\n"
            # "Hoán vị các hàng (đổi thứ tự lưu trữ) KHÔNG làm thay đổi hình dáng vật thể!",
            "Point clouds are stored as N x 3 matrices.\n"
            "Permuting rows (changing the storage order) does NOT change the shape of the object!",
            font_size=20, color=LIGHT_GREY
        ).to_edge(DOWN, buff=0.5)

        # 3. ANIMATION HOÁN VỊ (PERMUTATION INVARIANCE)
        with self.voiceover(text="Notice that if we permute the rows, meaning we change the order in which the points are stored in memory, the physical shape of the object remains completely unchanged."):
            self.play(Write(pose_desc))

            # Đổi chỗ Hàng 1 và Hàng 3
            self.play(
                Swap(pose_matrix.get_rows()[0], pose_matrix.get_rows()[2]),
                # Phóng to nhẹ các điểm tương ứng trên Point Cloud để thấy chúng ĐỨNG YÊN
                Indicate(highlight_dots[0], scale_factor=1.5),
                Indicate(highlight_dots[2], scale_factor=1.5),
                run_time=1.5
            )

            # Đổi chỗ Hàng 2 và Hàng 5
            self.play(
                Swap(pose_matrix.get_rows()[1], pose_matrix.get_rows()[4]),
                Indicate(highlight_dots[1], scale_factor=1.5),
                Indicate(highlight_dots[4], scale_factor=1.5),
                run_time=1.5
            )

        self.play(FadeOut(VGroup(pose_title, point_cloud, matrix_group, pose_desc)))

        # ==========================================
        # PHẦN 2: TẠO TẬP HỢP CHUYÊN NGHIỆP VỚI NGOẶC NHỌN
        # ==========================================
        elements = VGroup()
        for i in range(4):
            circle = Circle(radius=0.4, fill_opacity=0.8, fill_color=DARK_GREY, stroke_color=WHITE)
            label = MathTex(f"x_{i+1}", font_size=32)
            group = VGroup(circle, label)
            elements.add(group)

        elements.arrange(RIGHT, buff=1.0).next_to(step1_title, DOWN, buff=1.5)
        left_brace = MathTex(r"\{", font_size=80).next_to(elements, LEFT, buff=0.4)
        right_brace = MathTex(r"\}", font_size=80).next_to(elements, RIGHT, buff=0.4)
        set_group = VGroup(left_brace, elements, right_brace)

        with self.voiceover(text="In mathematics, a collection like this is called a set."):
            self.play(FadeIn(left_brace), FadeIn(right_brace), LaggedStart(*[FadeIn(e) for e in elements], lag_ratio=0.2))

        desc1 = Text("In a mathematical set, the order of elements does not matter.", font_size=24)
        desc1.next_to(set_group, DOWN, buff=1.0)

        with self.voiceover(text="And a fundamental property of a set is that the order of its elements does not matter."):
            self.play(Write(desc1))

        # ==========================================
        # PHẦN 3: ANIMATION HOÁN VỊ (SWAP)
        # ==========================================
        action_txt = Text("Action: Swapping elements (Permutation)", font_size=20, color=YELLOW).next_to(desc1, DOWN, buff=0.5)

        with self.voiceover(text="If we perform a permutation, for example, swapping the first and third elements..."):
            self.play(Write(action_txt))
            self.play(
                elements[0][0].animate.set_stroke(YELLOW, width=5),
                elements[2][0].animate.set_stroke(YELLOW, width=5)
            )
            self.play(Swap(elements[0], elements[2], path_arc=PI/2), run_time=1.5)

        with self.voiceover(text="Or swapping the second and fourth elements..."):
            self.play(
                elements[1][0].animate.set_stroke(ORANGE, width=5),
                elements[3][0].animate.set_stroke(ORANGE, width=5)
            )
            self.play(Swap(elements[1], elements[3], path_arc=-PI/2), run_time=1.5)

        self.play(*[e[0].animate.set_stroke(WHITE, width=2) for e in elements])

        desc2 = Text("Swapping any elements leaves the structural meaning unchanged.", font_size=24, color=GREEN_C).move_to(desc1)

        with self.voiceover(text="The structural meaning of the set is perfectly preserved."):
            self.play(Transform(desc1, desc2), FadeOut(action_txt))

        # ==========================================
        # PHẦN 4: KẾT LUẬN VỀ CHIA SẺ THAM SỐ
        # ==========================================
        legend = VGroup(
            Circle(radius=0.15, fill_color=RED_E, fill_opacity=1, stroke_width=0),
            Text("Itself (Self)", font_size=22),
            Circle(radius=0.15, fill_color=BLUE_E, fill_opacity=1, stroke_width=0),
            Text("Others (Interaction)", font_size=22)
        ).arrange(RIGHT, buff=0.3)

        desc3 = VGroup(
            Text("Because all elements are equal, an element only distinguishes between:", font_size=22),
            legend
        ).arrange(DOWN, buff=0.2).move_to(desc1)

        with self.voiceover(text="Because all elements in a set are structurally equal, an element can only distinguish between two things: itself, and all the other elements interacting with it."):
            self.play(Transform(desc1, desc3))
            self.play(elements[0][0].animate.set_fill(RED_E))
            self.play(
                elements[1][0].animate.set_fill(BLUE_E),
                elements[2][0].animate.set_fill(BLUE_E),
                elements[3][0].animate.set_fill(BLUE_E),
                run_time=1.0
            )

        self.play(FadeOut(Group(*[m for m in self.mobjects if m != title])))

        # ==========================================
        # PART 1.5: MATHEMATICAL NOTATION FOR PERMUTATION
        # ==========================================
        notation_title = Text("Mathematical Notation & Visual Matrix", font_size=28, color=YELLOW).next_to(title, DOWN, buff=0.5)

        with self.voiceover(text="Let's formalize this with mathematical notation. A permutation can be represented by a Permutation Matrix, denoted as P tau."):
            self.play(Write(notation_title))
            tau_desc = Text("Definition of permutation:", font_size=24, color=LIGHT_GREY)
            tau_map = MathTex(r"\tau = \begin{pmatrix} 1 & 2 & 3 & 4 \\ 3 & 1 & 4 & 2 \end{pmatrix}", font_size=32, color=WHITE)
            tau_group = VGroup(tau_desc, tau_map).arrange(RIGHT, buff=0.5).next_to(notation_title, DOWN, buff=0.5)
            self.play(FadeIn(tau_group))

        proof_title = Text("Matrix Multiplication", font_size=24, color=GREEN_C).next_to(tau_group, DOWN, buff=0.5)
        self.play(Write(proof_title))

        box_colors = [RED_C, BLUE_C, GREEN_C, ORANGE]
        x_col = Matrix([[r"x_1"], [r"x_2"], [r"x_3"], [r"x_4"]]).scale(0.8)
        for i in range(4):
            x_col.get_entries()[i].set_color(box_colors[i])

        p_matrix = Matrix([
            [0, 0, 1, 0],
            [1, 0, 0, 0],
            [0, 0, 0, 1],
            [0, 1, 0, 0]
        ]).scale(0.8)

        res_col = Matrix([[r"x_3"], [r"x_1"], [r"x_4"], [r"x_2"]]).scale(0.8)
        res_indices = [2, 0, 3, 1]
        for i, idx in enumerate(res_indices):
            res_col.get_entries()[i].set_color(box_colors[idx])

        eq_group = VGroup(
            MathTex("P_\\tau", font_size=36, color=BLUE_C),
            p_matrix,
            MathTex(r"\times", font_size=36),
            MathTex("X", font_size=36, color=WHITE),
            x_col,
            MathTex("=", font_size=36),
            res_col
        ).arrange(RIGHT, buff=0.2).next_to(proof_title, DOWN, buff=0.5)

        with self.voiceover(text="When we multiply this matrix by our data vector X, let's see what happens step by step."):
            self.play(FadeIn(eq_group[:6]))
            self.play(FadeIn(res_col.get_brackets()))

        desc_mul = Text("Multiply the row of the matrix by the column of data.", font_size=20, color=LIGHT_GREY).next_to(eq_group, DOWN, buff=0.5)
        self.play(Write(desc_mul))

        with self.voiceover(text="The matrix multiplication takes a row of P tau and multiplies it by the column of X."):
            for i in range(4):
                row = p_matrix.get_rows()[i]
                col = x_col.get_columns()[0]
                self.play(row.animate.set_color(YELLOW), col.animate.set_color(YELLOW), run_time=0.4)

                one_idx = res_indices[i]
                math_str = " + ".join([f"0 \\times x_{j+1}" if j != one_idx else f"1 \\times x_{j+1}" for j in range(4)])
                math_str += f" = x_{one_idx+1}"
                detail_mul = MathTex(math_str, font_size=24, color=YELLOW).next_to(desc_mul, DOWN, buff=0.3)

                self.play(FadeIn(detail_mul, shift=UP*0.2), run_time=0.5)
                res_entry = res_col.get_entries()[i]
                self.play(FadeIn(res_entry, scale=1.5), run_time=0.5)
                self.play(
                    row.animate.set_color(WHITE),
                    FadeOut(detail_mul),
                    run_time=0.4
                )
                for j in range(4):
                    x_col.get_entries()[j].set_color(box_colors[j])

        concl = Text("A '1' at column j simply 'picks' x_j out!", font_size=24, color=GREEN_C, weight=BOLD).next_to(desc_mul, DOWN, buff=0.5)

        with self.voiceover(text="A number '1' at column J simply picks out the element x J and places it in the output."):
            self.play(Write(concl))

        # =======================================================
        # FIX LAYOUT: DỌN DẸP PHẦN TRÊN VÀ ĐẨY PHƯƠNG TRÌNH LÊN GIỮA
        # =======================================================
        self.play(
            FadeOut(desc_mul),
            FadeOut(concl),
            FadeOut(notation_title),
            FadeOut(tau_group),
            FadeOut(proof_title),
            eq_group.animate.move_to(ORIGIN) # Đẩy nguyên cụm phương trình lên chính giữa màn hình (hoặc có thể dùng UP*0.5)
        )

        dyn_title = Text("Experiment: Swapping rows in Matrix P", font_size=24, color=YELLOW).next_to(eq_group, DOWN, buff=0.8)
        swap_desc = Text("Swap Row 1 and Row 2... The output is swapped accordingly", font_size=20, color=LIGHT_GREY).next_to(dyn_title, DOWN, buff=0.3)

        with self.voiceover(text="If we swap the rows of the permutation matrix, the corresponding output elements swap perfectly along with it."):
            self.play(Write(dyn_title), Write(swap_desc))
            self.play(
                p_matrix.get_rows()[0].animate.set_color(RED_C),
                p_matrix.get_rows()[1].animate.set_color(BLUE_C),
                run_time=0.5
            )
            self.play(
                Swap(p_matrix.get_rows()[0], p_matrix.get_rows()[1]),
                Swap(res_col.get_entries()[0], res_col.get_entries()[1]),
                run_time=1.5
            )
            self.play(
                p_matrix.get_rows()[0].animate.set_color(WHITE),
                p_matrix.get_rows()[1].animate.set_color(WHITE),
                run_time=0.5
            )

        self.play(FadeOut(Group(*[m for m in self.mobjects if m != title])))

        # ==========================================
        # PART 2: BUILDING MATRIX W
        # ==========================================
        step2_title = VGroup(
            Text("Convert to Matrix", font_size=28, color=YELLOW),
            MathTex("W", font_size=32, color=YELLOW)
        ).arrange(RIGHT, buff=0.15).to_edge(UP, buff=1.0)

        p_elements = [
            [r"\alpha", r"\beta", r"\beta", r"\beta"],
            [r"\beta", r"\alpha", r"\beta", r"\beta"],
            [r"\beta", r"\beta", r"\alpha", r"\beta"],
            [r"\beta", r"\beta", r"\beta", r"\alpha"]
        ]
        p_matrix_w = Matrix(p_elements)
        p_label = MathTex("W = ").next_to(p_matrix_w, LEFT)
        p_group = VGroup(p_label, p_matrix_w).next_to(step2_title, DOWN, buff=1.0)

        with self.voiceover(text="Now, how does this restrict our learnable weight matrix W?"):
            self.play(Write(step2_title))
            self.play(FadeIn(p_group))

        matrix_desc1 = VGroup(
            Text("Diagonal (", font_size=24, color=RED_C),
            MathTex(r"\alpha", font_size=28, color=RED_C),
            Text(") represents 'Itself' (Red)", font_size=24, color=RED_C)
        ).arrange(RIGHT, buff=0.1).next_to(p_group, DOWN, buff=1.0)
        diagonals = VGroup(*[p_matrix_w.get_entries()[i * 4 + i] for i in range(4)])

        with self.voiceover(text="To remain equivariant to permutations, W can only have two unique parameters. The diagonal parameter, alpha, represents an element's interaction with itself."):
            self.play(Write(matrix_desc1))
            self.play(diagonals.animate.set_color(RED_C))

        matrix_desc2 = VGroup(
            Text("Off-diagonal (", font_size=24, color=BLUE_C),
            MathTex(r"\beta", font_size=28, color=BLUE_C),
            Text(") represents 'Others' (Blue)", font_size=24, color=BLUE_C)
        ).arrange(RIGHT, buff=0.1).next_to(matrix_desc1, DOWN, buff=0.5)
        off_diagonals = VGroup(*[p_matrix_w.get_entries()[i * 4 + j] for i in range(4) for j in range(4) if i != j])

        with self.voiceover(text="The off-diagonal parameter, beta, represents its identical interaction with all other elements in the set."):
            self.play(Write(matrix_desc2))
            self.play(off_diagonals.animate.set_color(BLUE_C))

        self.play(FadeOut(Group(*[m for m in self.mobjects if m != title])))

        # ==========================================
        # PART 3: ARCHITECTURES & FORMULA
        # ==========================================
        step3_title = Text("DeepSet & PointNet Architectures", font_size=28, color=YELLOW).to_edge(UP, buff=1.0)
        eq_text = Text(
            "Basis elements: identity and sum over all elements", font_size=24, color=LIGHT_GREY
        ).next_to(step3_title, DOWN, buff=0.8)

        math_eq = MathTex(
            r"[F(X)]_i = \sigma \Big( ", r"\alpha_1", r" x_i + ", r"\alpha_2", r" \sum_j x_j \Big)",
            font_size=40
        ).next_to(eq_text, DOWN, buff=0.8)

        with self.voiceover(text="This mathematical restriction leads directly to the basis elements of famous architectures like DeepSets and PointNet."):
            self.play(Write(step3_title))
            self.play(Write(eq_text))
            self.play(Write(math_eq))

        citations = Text(
            "DeepSet, PointNet architectures\n(Zaheer et al 2017, Qi et al 2017)",
            font_size=24, color=GREEN_C
        ).next_to(math_eq, DOWN, buff=1.2)

        with self.voiceover(text="The operation elegantly simplifies to a linear combination of the element itself, and the sum over all other elements in the set."):
            self.play(math_eq[1].animate.set_color(RED_C), math_eq[3].animate.set_color(BLUE_C))
            self.play(Write(citations))

        self.play(FadeOut(Group(*self.mobjects)))


## slide 40
What do we get from this?

In [ ]:
%%manim -qh -v WARNING GeneralizationAndLimits40

from manim import *
import networkx as nx
from manim_voiceover import VoiceoverScene
from manim_voiceover.services.gtts import GTTSService

class GeneralizationAndLimits40(VoiceoverScene):
    def construct(self):
        self.camera.background_color = BLACK

        # Khởi tạo dịch vụ tạo giọng đọc tiếng Anh
        self.set_speech_service(EdgeTTSService(voice="en-US-GuyNeural"))

        # ==========================================
        # PART 1: THE NATURE OF "COLORS" (BASIS ELEMENTS)
        # ==========================================
        title = Text("Summary: Consequences and Extensions", font_size=36, color=WHITE, weight=BOLD)

        with self.voiceover(text="Let's discuss the consequences and extensions of parameter sharing."):
            self.play(Write(title))
            self.play(title.animate.to_edge(UP, buff=0.4))

        step1_title = Text("The Nature of 'Colors')", font_size=28, color=YELLOW)
        step1_title.next_to(title, DOWN, buff=0.5)

        with self.voiceover(text="First, let's reflect on the nature of 'colors', also known as basis elements."):
            self.play(Write(step1_title))

        eq_text = Text("Each value in the matrix is the sum of basis parameters:", font_size=24)
        eq_text.next_to(step1_title, DOWN, buff=0.8)

        math_eq = MathTex(
            r"W = \alpha \cdot", r"\begin{bmatrix} 1 & 0 \\ 0 & 1 \end{bmatrix}",
            r"+ \beta \cdot", r"\begin{bmatrix} 0 & 1 \\ 1 & 0 \end{bmatrix}"
        )
        math_eq[0].set_color(RED_C)
        math_eq[2].set_color(BLUE_C)
        math_eq.next_to(eq_text, DOWN, buff=0.8)

        with self.voiceover(text="As we've seen, each value in the weight matrix can be represented as a linear combination of these basis parameters."):
            self.play(FadeIn(eq_text))
            self.play(Write(math_eq))

        desc1 = Text(
            "Number of 'Colors' = Basis Size = Number of learnable parameters.",
            font_size=24, color=GREEN_C
        ).next_to(math_eq, DOWN, buff=1.0)

        with self.voiceover(text="The number of unique colors, or the basis size, dictates the exact number of learnable parameters in the neural network layer."):
            self.play(Write(desc1))

        self.play(FadeOut(Group(*[m for m in self.mobjects if m != title])))

        # ==========================================
        # PART 2: EXTENSION TO GRAPHS (INVARIANT GRAPH NETWORKS)
        # ==========================================
        step2_title = Text("Extension to Graphs (Invariant Graph Networks)", font_size=28, color=YELLOW)
        step2_title.next_to(title, DOWN, buff=0.5).to_edge(LEFT, buff=1.0)

        with self.voiceover(text="Second, we can extend this mathematical framework to Graph Neural Networks, specifically Invariant Graph Networks."):
            self.play(Write(step2_title))

        graph_text = Text("Input: Graph Adjacency Matrix", font_size=24).next_to(step2_title, DOWN, buff=0.5, aligned_edge=LEFT)

        # Graph visualization
        vertices = [1, 2, 3, 4]
        edges = [(1, 2), (2, 3), (3, 4), (4, 1), (1, 3)]
        g = Graph(vertices, edges, layout="circular", labels=True, vertex_config={"color": BLUE})
        g.scale(0.7).move_to(LEFT * 3 + DOWN * 0.2)

        # Adjacency matrix
        adj_matrix = Matrix([
            [0, 1, 1, 1],
            [1, 0, 1, 0],
            [1, 1, 0, 1],
            [1, 0, 1, 0]
        ]).scale(0.6).next_to(g, RIGHT, buff=1.0)
        adj_label = Text("Adjacency Matrix", font_size=20).next_to(adj_matrix, UP)
        adj_group = VGroup(adj_matrix, adj_label)

        with self.voiceover(text="Here, the input is a Graph Adjacency Matrix."):
            self.play(Write(graph_text))
            self.play(Create(g), FadeIn(adj_group))

        sym_text = Text("Symmetry: Permutation of graph nodes", font_size=20, color=LIGHT_GREY).next_to(VGroup(g, adj_group), DOWN, buff=0.4)

        with self.voiceover(text="The symmetry we care about is the permutation of graph nodes. If we swap two nodes, the underlying graph remains structurally identical."):
            self.play(FadeIn(sym_text))
            self.play(
                g.vertices[1].animate.move_to(g.vertices[2].get_center()),
                g.vertices[2].animate.move_to(g.vertices[1].get_center()),
                run_time=1.5
            )
            self.play(
                g.vertices[1].animate.move_to(g.vertices[2].get_center()),
                g.vertices[2].animate.move_to(g.vertices[1].get_center()),
                run_time=1.0
            )

        basis_info = Text("Basis Size = 15", font_size=28, color=RED_C, weight=BOLD).next_to(sym_text, DOWN, buff=0.3)
        basis_desc = Text(
            "Graph's weight matrix requires 15 different colors\n"
            "to represent all symmetric relationships between node pairs!",
            font_size=18, line_spacing=1.2
        ).next_to(basis_info, DOWN, buff=0.2)

        with self.voiceover(text="However, analyzing the weight matrix for graphs reveals that it requires exactly fifteen different basis elements to represent all symmetric relationships between node pairs."):
            self.play(GrowFromCenter(basis_info))
            self.play(Write(basis_desc))

        self.play(FadeOut(Group(*[m for m in self.mobjects if m != title])))

        # ==========================================
        # PART 3: EXTENSIBILITY
        # ==========================================
        step3_title = Text("Extensibility: Tensors & Symmetric Sets", font_size=28, color=YELLOW)
        step3_title.next_to(title, DOWN, buff=0.5).to_edge(LEFT, buff=1.0)
        ext_title = Text("Generalizations:", font_size=24, color=GREEN_C).next_to(step3_title, DOWN, buff=0.5).to_edge(LEFT, buff=1.5)

        with self.voiceover(text="Finally, let's look at extensibility and limitations."):
            self.play(Write(step3_title))
            self.play(Write(ext_title))

        bullet1 = Text("- Higher-order GNNs (Tensors)", font_size=22).next_to(ext_title, DOWN, buff=0.4, aligned_edge=LEFT)

        # Simple Tensor Visual
        tensor_cube = VGroup(*[Square(side_length=0.4, fill_opacity=0.3, color=BLUE) for _ in range(9)]).arrange_in_grid(3, 3, buff=0)
        tensor_cube2 = VGroup(*[Square(side_length=0.4, fill_opacity=0.3, color=GREEN) for _ in range(9)]).arrange_in_grid(3, 3, buff=0).shift(UP*0.2 + RIGHT*0.2)
        tensor_cube3 = VGroup(*[Square(side_length=0.4, fill_opacity=0.3, color=RED) for _ in range(9)]).arrange_in_grid(3, 3, buff=0).shift(UP*0.4 + RIGHT*0.4)
        tensor_group = VGroup(tensor_cube, tensor_cube2, tensor_cube3).scale(0.8).next_to(bullet1, RIGHT, buff=1.0)

        with self.voiceover(text="This mathematical framework can be generalized to higher-order Graph Neural Networks using tensors."):
            self.play(FadeIn(bullet1), FadeIn(tensor_group))

        bullet2 = Text("- Sets of symmetric elements", font_size=22).next_to(bullet1, DOWN, buff=0.8, aligned_edge=LEFT)

        # Sets of elements visual
        set_a = VGroup(*[Circle(radius=0.15, fill_opacity=0.5, color=PURPLE).shift(RIGHT*(i*0.4)) for i in range(3)]).move_to(RIGHT*1)
        set_b = VGroup(*[Circle(radius=0.15, fill_opacity=0.5, color=ORANGE).shift(RIGHT*(i*0.4)) for i in range(3)]).move_to(RIGHT*1 + UP*0.5)
        sets_group = VGroup(set_a, set_b).next_to(bullet2, RIGHT, buff=1.0)

        with self.voiceover(text="It can also be applied to complex sets of symmetric elements."):
            self.play(FadeIn(bullet2), FadeIn(sets_group))

        limit_title = Text("Limitations:", font_size=24, color=RED_C).next_to(bullet2, DOWN, buff=0.8).to_edge(LEFT, buff=1.5)
        limit_desc = Text(
            "- Not always applicable or computationally feasible.\n"
            "- Linear layers alone might lack expressivity for complex symmetries.",
            font_size=20, color=LIGHT_GREY, line_spacing=1.2
        ).next_to(limit_title, DOWN, buff=0.3, aligned_edge=LEFT)

        with self.voiceover(text="But there are limitations. Solving for these basis elements is not always computationally feasible for complex symmetries, and linear layers alone might lack the expressivity needed for advanced geometric tasks."):
            self.play(Write(limit_title))
            self.play(Write(limit_desc))

        # END
        self.play(FadeOut(Group(*self.mobjects)))


## slide 41
Group convolution

In [ ]:
%%manim -qh -v WARNING GroupConvolution41

from manim import *
from manim_voiceover import VoiceoverScene
from manim_voiceover.services.gtts import GTTSService

class GroupConvolution41(VoiceoverScene):
    def construct(self):
        self.camera.background_color = BLACK

        # Khởi tạo dịch vụ tạo giọng đọc tiếng Anh
        self.set_speech_service(EdgeTTSService(voice="en-US-GuyNeural"))

        # Define standard colors
        COLOR_X = RED_C
        COLOR_Y = GREEN_C
        COLOR_VEC = BLUE_C
        COLOR_F = WHITE
        COLOR_PSI = YELLOW

        COLOR_G = RED_C
        COLOR_H = GREEN_C
        COLOR_INV = BLUE_C

        # ==========================================
        # PART 1: REGULAR 2D CONVOLUTION
        # ==========================================
        title = Text("Group Convolution", font_size=36, color=WHITE, weight=BOLD)
        step1_title = Text("Regular 2D Convolution", font_size=28, color=YELLOW)

        with self.voiceover(text="Let's dive into Group Convolution. To understand it, we must first look at the regular 2D convolution we know and love."):
            self.play(Write(title))
            self.play(title.animate.to_edge(UP, buff=0.4))
            step1_title.next_to(title, DOWN, buff=0.2)
            self.play(Write(step1_title))

        eq = MathTex(
            r"f * \psi(", r"x", r") = ", r"\sum", r"_{y}", r" f(", r"y", r") \cdot \psi(", r"y - x", r")",
            font_size=40
        ).next_to(step1_title, DOWN, buff=0.3)

        eq[1].set_color(COLOR_X)
        eq[4].set_color(COLOR_Y)
        eq[6].set_color(COLOR_Y)
        eq[8].set_color(COLOR_VEC)

        with self.voiceover(text="Here is the standard convolution equation, applying a filter psi over an image f."):
            self.play(FadeIn(eq))

        cell_size = 0.8
        grid_f = VGroup(*[Square(side_length=cell_size, stroke_color=DARK_GREY, stroke_width=2) for _ in range(25)])
        grid_f.arrange_in_grid(rows=5, cols=5, buff=0)
        grid_f.move_to(LEFT * 3.5 + DOWN * 0.5)

        title_f = MathTex("f", font_size=32, color=COLOR_F)
        desc_f = Text("Image", font_size=18, color=LIGHT_GREY)
        title_group = VGroup(title_f, desc_f).arrange(RIGHT, buff=0.2).next_to(grid_f, UP, buff=0.2).shift(LEFT * 1)

        with self.voiceover(text="Let's visualize this on a pixel grid."):
            self.play(FadeIn(grid_f), Write(title_f), FadeIn(desc_f))

        right_panel_anchor = RIGHT * 3.0 + UP * 0.5

        exp_group = VGroup(
            Text("x: Filter Center (Current)", font_size=22, color=COLOR_X),
            Text("y: A neighboring pixel of x", font_size=22, color=COLOR_Y),
            Text("y - x: Relative coordinates of y", font_size=22, color=COLOR_VEC)
        ).arrange(DOWN, aligned_edge=LEFT, buff=0.6).move_to(right_panel_anchor)

        filter_box = Square(side_length=cell_size * 3, color=COLOR_PSI, stroke_width=4)
        filter_box.move_to(grid_f[12].get_center())
        filter_lbl = MathTex(r"\psi", font_size=28, color=COLOR_PSI).next_to(filter_box, UP+RIGHT, buff=0.1)

        dot_x = Dot(grid_f[12].get_center(), color=COLOR_X, radius=0.15)
        lbl_x = MathTex("x", font_size=24, color=COLOR_X).next_to(dot_x, DOWN+LEFT, buff=0.1)

        with self.voiceover(text="Let x be the current center position of our filter."):
            self.play(FadeIn(exp_group[0]))
            self.play(Create(filter_box), Write(filter_lbl), FadeIn(dot_x), Write(lbl_x))

        dot_y = Dot(grid_f[8].get_center(), color=COLOR_Y, radius=0.15)
        lbl_y = MathTex("y", font_size=24, color=COLOR_Y).next_to(dot_y, UP+RIGHT, buff=0.1)

        with self.voiceover(text="Let y be a neighboring pixel we are scanning."):
            self.play(FadeIn(exp_group[1]))
            self.play(FadeIn(dot_y), Write(lbl_y))

        vec_yx = Arrow(dot_x.get_center(), dot_y.get_center(), color=COLOR_VEC, buff=0.1, stroke_width=4)
        lbl_vec = MathTex("y - x", font_size=20, color=COLOR_VEC).move_to(vec_yx.get_center() + LEFT*0.4 + UP*0.2)

        with self.voiceover(text="The vector y minus x gives us the exact relative coordinates of y with respect to x."):
            self.play(FadeIn(exp_group[2]))
            self.play(GrowArrow(vec_yx), Write(lbl_vec))

        self.play(FadeOut(exp_group))

        sum_box = SurroundingRectangle(eq[3:5], color=ORANGE, buff=0.1)
        sum_desc = Text(
            "Summation Mechanism (Sigma Scan)\n\n"
            "The Sum sign forces point y to scan through ALL\n"
            "cells within the filter region around x.",
            font_size=22, color=ORANGE, line_spacing=1.3
        ).move_to(right_panel_anchor)

        with self.voiceover(text="The summation sign forces the point y to scan through all surrounding cells within the local filter region."):
            self.play(Create(sum_box))
            self.play(Write(sum_desc))

            neighbors = [6, 7, 11, 13, 16, 17, 18]
            for n_idx in neighbors:
                target_pos = grid_f[n_idx].get_center()
                new_vec = Arrow(dot_x.get_center(), target_pos, color=COLOR_VEC, buff=0.1, stroke_width=4)

                self.play(
                    dot_y.animate.move_to(target_pos),
                    lbl_y.animate.next_to(target_pos, UP+RIGHT, buff=0.1),
                    Transform(vec_yx, new_vec),
                    lbl_vec.animate.move_to((dot_x.get_center() + target_pos)/2 + LEFT*0.3),
                    run_time=0.4
                )

        self.play(FadeOut(sum_desc), FadeOut(sum_box))

        final_conclusion = Text(
            "Conclusion\n(y - x) generates\n"
            "'Translation Equivariance'.\n"
            "Scanning weights only depend\n"
            "on relative vector.",
            font_size=22, color=GREEN_C, line_spacing=1.3
        ).move_to(right_panel_anchor + DOWN * 0.5)

        with self.voiceover(text="The crucial insight here is translation equivariance. The filter weights depend strictly on the relative vector y minus x, not their absolute positions."):
            self.play(Write(final_conclusion))

        self.play(FadeOut(Group(*[m for m in self.mobjects if m != title])))

        # ==========================================
        # PART 2: GENERALIZATION TO GROUP G (VISUALIZED)
        # ==========================================
        step2_title = Text("Generalization to Group G", font_size=28, color=YELLOW)
        step2_title.to_edge(UP, buff=1.0)

        eq_group = MathTex(
            r"f * \psi(", r"g", r") = \sum_{", r"h", r" \in G} f(", r"h", r") \psi(", r"g^{-1} h", r")",
            font_size=36
        ).next_to(step2_title, DOWN, buff=0.5)
        eq_group[1].set_color(COLOR_G)
        eq_group[3].set_color(COLOR_H)
        eq_group[5].set_color(COLOR_H)
        eq_group[7].set_color(COLOR_INV)

        with self.voiceover(text="Now, let's generalize this to an abstract group G. Notice how the subtraction y minus x is replaced by the group operation g inverse h."):
            self.play(Write(step2_title))
            self.play(Write(eq_group))

        # Group space representation
        circle_group = Circle(radius=1.5, color=DARK_GREY, stroke_width=2).move_to(LEFT * 3.5 + DOWN * 0.5)
        ticks = VGroup()
        for angle in [0, PI/2, PI, 3*PI/2]:
            tick = Line(circle_group.get_center(), circle_group.get_center() + RIGHT*1.5, color=DARK_GREY).rotate(angle, about_point=circle_group.get_center())
            ticks.add(tick)

        space_title = Text("Group Space G", font_size=18, color=WHITE).next_to(circle_group, UP, buff=0.4)

        with self.voiceover(text="Imagine our group space represents rotations."):
            self.play(FadeIn(circle_group), FadeIn(ticks), Write(space_title))

        exp_group1 = VGroup(
            Text("g: Current Filter State", font_size=22, color=COLOR_G),
            Text("h: Scanning State", font_size=22, color=COLOR_H)
        ).arrange(DOWN, aligned_edge=LEFT, buff=0.6).move_to(right_panel_anchor)

        vec_g = Arrow(circle_group.get_center(), circle_group.get_center() + UP*1.5, color=COLOR_G, buff=0, stroke_width=6)
        lbl_g = MathTex("g", font_size=28, color=COLOR_G).next_to(vec_g.get_end(), UP+LEFT, buff=0.1)

        with self.voiceover(text="Instead of position x, g represents our current filter's rotational state."):
            self.play(FadeIn(exp_group1[0]))
            self.play(GrowArrow(vec_g), Write(lbl_g))

        vec_h = Arrow(circle_group.get_center(), circle_group.get_center() + RIGHT*1.5, color=COLOR_H, buff=0, stroke_width=6)
        lbl_h = MathTex("h", font_size=28, color=COLOR_H).next_to(vec_h.get_end(), UP+RIGHT, buff=0.1)

        with self.voiceover(text="And h represents the state we are currently scanning."):
            self.play(FadeIn(exp_group1[1]))
            self.play(GrowArrow(vec_h), Write(lbl_h))

        self.play(FadeOut(exp_group1))

        exp_group2 = VGroup(
            Text("Finding relative transform g⁻¹h", font_size=24, color=YELLOW),
            Text("Rotate back by angle g⁻¹", font_size=22, color=COLOR_INV)
        ).arrange(DOWN, aligned_edge=LEFT, buff=0.4).move_to(right_panel_anchor)

        arc_inv = CurvedArrow(vec_g.get_end(), circle_group.get_center() + RIGHT*1.5, angle=-PI/2, color=COLOR_INV)
        lbl_inv_op = MathTex("g^{-1}", font_size=24, color=COLOR_INV).next_to(arc_inv, UP+RIGHT)

        with self.voiceover(text="To find the relative distance, analogous to vector subtraction, we must mathematically 'rewind' the space by applying the inverse of g."):
            self.play(Write(exp_group2))
            self.play(Create(arc_inv), Write(lbl_inv_op))

        target_g = circle_group.get_center() + RIGHT*1.5
        target_h = circle_group.get_center() + DOWN*1.5

        lbl_inv_op.add_updater(lambda m: m.move_to(arc_inv.point_from_proportion(0.5) + normalize(arc_inv.point_from_proportion(0.5) - circle_group.get_center()) * 0.4))

        with self.voiceover(text="Watch as we rotate the entire system back by g inverse."):
            self.play(
                Rotate(vec_g, angle=-PI/2, about_point=circle_group.get_center()),
                Rotate(vec_h, angle=-PI/2, about_point=circle_group.get_center()),
                Rotate(arc_inv, angle=-PI/2, about_point=circle_group.get_center()),
                lbl_g.animate.next_to(target_g, direction=normalize(target_g - circle_group.get_center()), buff=0.2),
                lbl_h.animate.next_to(target_h, direction=normalize(target_h - circle_group.get_center()), buff=0.2),
                run_time=2
            )
        lbl_inv_op.clear_updaters()

        self.play(FadeOut(arc_inv), FadeOut(lbl_inv_op))

        lbl_rel = MathTex("g^{-1}h", font_size=28, color=COLOR_INV).next_to(target_h, direction=normalize(target_h - circle_group.get_center()), buff=0.2)
        desc_rel = Text(
            "This rotation distance is g⁻¹h.\nIt is completely equivalent to (y - x).",
            font_size=20, color=COLOR_INV, line_spacing=1.3
        ).next_to(exp_group2, DOWN, buff=0.5).align_to(exp_group2, LEFT)

        with self.voiceover(text="The new relative position is g inverse h, perfectly mirroring our 2D vector subtraction in the group space."):
            self.play(Transform(lbl_h, lbl_rel), vec_h.animate.set_color(COLOR_INV))
            self.play(Write(desc_rel))

        self.play(FadeOut(Group(*[m for m in self.mobjects if m != title])))

        # ==========================================
        # PART 3: CONCLUSION & REFERENCES
        # ==========================================
        step3_title = Text("G-CNNs (Group Equivariant CNNs)", font_size=28, color=YELLOW)
        step3_title.to_edge(UP, buff=1.0)

        conclusion_text = Text(
            "Group Convolution ensures that:\n"
            "Neural networks are not only translation invariant,\n"
            "but also EQUIVARIANT to other symmetries like Rotation and Reflection.",
            font_size=24, line_spacing=1.5,
            t2c={"EQUIVARIANT": GREEN_C}
        ).next_to(step3_title, DOWN, buff=1.5)

        with self.voiceover(text="This brings us to Group Equivariant CNNs. By using this generalized convolution, neural networks guarantee equivariance to complex symmetries like rotations and reflections."):
            self.play(Write(step3_title))
            self.play(Write(conclusion_text))

        ref_box = SurroundingRectangle(conclusion_text, color=BLUE_E, buff=0.5)
        ref_text = Text(
            "Reference:\n"
            "T. Cohen, M. Welling, 'Group equivariant convolutional networks.' ICML 2016.",
            font_size=18, color=LIGHT_GREY
        ).next_to(ref_box, DOWN, buff=0.5)

        with self.voiceover(text="This beautiful and foundational concept was first introduced by Taco Cohen and Max Welling in their landmark 2016 paper."):
            self.play(Create(ref_box))
            self.play(FadeIn(ref_text))

        self.play(FadeOut(Group(*self.mobjects)))


## slide 43
Invariant polynomials

In [ ]:
%%manim -qh -v WARNING InvariantPolynomials43

class InvariantPolynomials43(VoiceoverScene):
    def construct(self):
        self.camera.background_color = BLACK

        # Set up English voiceover service
        self.set_speech_service(EdgeTTSService(voice="en-US-GuyNeural"))

        # ==========================================
        # PART 1: CONCEPT OF INVARIANT POLYNOMIALS
        # ==========================================
        title = Text("Invariant Polynomials", font_size=36, color=WHITE, weight=BOLD)
        step1_title = Text("The Core Idea", font_size=28, color=YELLOW)

        with self.voiceover(text="Let's explore Invariant Polynomials. Step 1: The core idea."):
            self.play(Write(title))
            self.play(title.animate.to_edge(UP, buff=0.4))
            step1_title.next_to(title, DOWN, buff=0.5)
            self.play(Write(step1_title))

        idea_text = Text(
            "Instead of just using Linear Layers,\n"
            "we can approximate functions using Invariant Polynomials.",
            font_size=24, line_spacing=1.2
        ).next_to(step1_title, DOWN, buff=0.8)

        with self.voiceover(text="Instead of relying solely on linear layers, we can approximate complex functions using invariant polynomials."):
            self.play(FadeIn(idea_text))

        constraint_text = Text("The symmetry constraints are exactly the same:", font_size=24, color=LIGHT_GREY).next_to(idea_text, DOWN, buff=0.8)

        inv_label = Text("Invariant: ", font_size=24, color=GREEN_C)
        inv_math = MathTex(r"f(g \cdot x) = f(x)", font_size=32)
        inv_group = VGroup(inv_label, inv_math).arrange(RIGHT)

        or_text = Text("or", font_size=24, color=WHITE)

        eq_label = Text("Equivariant: ", font_size=24, color=BLUE_C)
        eq_math = MathTex(r"f(g \cdot x) = g \cdot f(x)", font_size=32)
        eq_group = VGroup(eq_label, eq_math).arrange(RIGHT)

        math_eq = VGroup(inv_group, or_text, eq_group).arrange(DOWN, buff=0.3).next_to(constraint_text, DOWN, buff=0.5)

        with self.voiceover(text="The symmetry constraints are exactly the same as before: a polynomial function can be either invariant, or equivariant to a group transformation."):
            self.play(Write(constraint_text))
            self.play(Write(math_eq))

        self.play(FadeOut(Group(*[m for m in self.mobjects if m != title])))

        # ==========================================
        # PART 2: REDUCTION TO SCALARS
        # ==========================================
        step2_title = Text("Using Scalars", font_size=28, color=YELLOW).next_to(title, DOWN, buff=0.5)

        with self.voiceover(text="Using Scalars."):
            self.play(Write(step2_title))

        scalar_desc = Text(
            "Invariant polynomials are expressed as functions of Scalars.",
            font_size=24, line_spacing=1.2
        ).next_to(step2_title, DOWN, buff=0.3)

        with self.voiceover(text="Any invariant polynomial can be expressed cleanly as a function of basic scalar quantities."):
            self.play(Write(scalar_desc))

        # --- LEFT: 2D SPACE ---
        axes = Axes(x_range=[-3, 3, 1], y_range=[-3, 3, 1], x_length=4, y_length=4, axis_config={"color": DARK_GREY})
        axes.move_to(LEFT * 3.5 + DOWN * 1.5)
        self.play(FadeIn(axes))

        theta = ValueTracker(0)
        v1_start = np.array([1.5, 2.0, 0])
        v2_start = np.array([2.5, 0.5, 0])

        vec1 = always_redraw(lambda: Arrow(
            axes.c2p(0,0),
            axes.c2p(*self.rotate_point(v1_start, theta.get_value())),
            color=RED_C, buff=0, stroke_width=4
        ))
        vec2 = always_redraw(lambda: Arrow(
            axes.c2p(0,0),
            axes.c2p(*self.rotate_point(v2_start, theta.get_value())),
            color=BLUE_C, buff=0, stroke_width=4
        ))

        lbl1 = always_redraw(lambda: MathTex("x_1", font_size=24, color=RED_C).next_to(vec1.get_end(), UP+LEFT, buff=0.1))
        lbl2 = always_redraw(lambda: MathTex("x_2", font_size=24, color=BLUE_C).next_to(vec2.get_end(), DOWN+RIGHT, buff=0.1))

        with self.voiceover(text="Let's look at a 2D space with two vectors, x 1 and x 2."):
            self.play(GrowArrow(vec1), GrowArrow(vec2), Write(lbl1), Write(lbl2))

        arc = always_redraw(lambda: Angle(vec2, vec1, radius=0.6, color=YELLOW))
        arc_lbl = always_redraw(lambda: MathTex(r"\theta", font_size=20, color=YELLOW).next_to(arc, UP+RIGHT, buff=0.05))
        dist_line = always_redraw(lambda: DashedLine(vec1.get_end(), vec2.get_end(), color=GREEN_C))

        self.play(Create(arc), Write(arc_lbl), Create(dist_line))

        # --- RIGHT: HUD TABLE ---
        table_center = RIGHT * 3.2 + DOWN * 1.0
        h_y = [1.5, 0.7, -0.1, -0.9, -1.7, -2.5]
        h_lines = VGroup(*[Line(table_center + LEFT*2.8 + UP*y, table_center + RIGHT*2.8 + UP*y, color=DARK_GREY, stroke_width=2) for y in h_y])
        v_x = [-2.8, 0.0, 2.8]
        v_lines = VGroup(*[Line(table_center + RIGHT*x + UP*1.5, table_center + RIGHT*x + DOWN*2.5, color=DARK_GREY, stroke_width=2) for x in v_x])

        table_grid = VGroup(h_lines, v_lines)
        self.play(Create(table_grid), run_time=1.5)

        r_y = [1.1, 0.3, -0.5, -1.3, -2.1]
        header_1 = Text("Quantity", font_size=18, color=WHITE, weight=BOLD).move_to(table_center + UP*r_y[0] + LEFT*1.4)
        header_2 = Text("Value", font_size=18, color=WHITE, weight=BOLD).move_to(table_center + UP*r_y[0] + RIGHT*1.4)
        self.play(Write(header_1), Write(header_2))

        row1_lbl = VGroup(Text("Coords", font_size=16, color=RED_C), MathTex("x_1", font_size=20, color=RED_C)).arrange(RIGHT, buff=0.1).move_to(table_center + UP*r_y[1] + LEFT*1.4)
        row2_lbl = VGroup(Text("Coords", font_size=16, color=BLUE_C), MathTex("x_2", font_size=20, color=BLUE_C)).arrange(RIGHT, buff=0.1).move_to(table_center + UP*r_y[2] + LEFT*1.4)
        row3_lbl = VGroup(Text("Dot Product", font_size=16, color=YELLOW), MathTex(r"\langle x_1, x_2 \rangle", font_size=20, color=YELLOW)).arrange(RIGHT, buff=0.1).move_to(table_center + UP*r_y[3] + LEFT*1.4)
        row4_lbl = VGroup(Text("Distance", font_size=16, color=GREEN_C), MathTex(r"||x_1 - x_2||^2", font_size=20, color=GREEN_C)).arrange(RIGHT, buff=0.1).move_to(table_center + UP*r_y[4] + LEFT*1.4)

        col1_group = VGroup(row1_lbl, row2_lbl, row3_lbl, row4_lbl)
        self.play(FadeIn(col1_group, shift=RIGHT*0.5))

        dot_prod = np.dot(v1_start, v2_start)
        dist_sq = (v1_start[0]-v2_start[0])**2 + (v1_start[1]-v2_start[1])**2

        val1 = always_redraw(lambda: Text(
            f"({self.rotate_point(v1_start, theta.get_value())[0]:.2f}, {self.rotate_point(v1_start, theta.get_value())[1]:.2f})",
            font_size=18, color=RED_C
        ).move_to(table_center + UP*r_y[1] + RIGHT*1.4))

        val2 = always_redraw(lambda: Text(
            f"({self.rotate_point(v2_start, theta.get_value())[0]:.2f}, {self.rotate_point(v2_start, theta.get_value())[1]:.2f})",
            font_size=18, color=BLUE_C
        ).move_to(table_center + UP*r_y[2] + RIGHT*1.4))

        val3 = MathTex(rf"{dot_prod:.2f}", font_size=22, color=YELLOW).move_to(table_center + UP*r_y[3] + RIGHT*1.4)
        val4 = MathTex(rf"{dist_sq:.2f}", font_size=22, color=GREEN_C).move_to(table_center + UP*r_y[4] + RIGHT*1.4)

        col2_group = VGroup(val1, val2, val3, val4)
        self.play(FadeIn(col2_group))

        box_scalar = SurroundingRectangle(VGroup(h_lines[3], h_lines[5]), color=GREEN_C, buff=0.0)

        with self.voiceover(text="Notice the table. As we continuously rotate the system, the individual coordinate values change dramatically. However, the dot product and the squared distance remain completely static. They are invariant scalars!") as tracker:
            self.play(Create(box_scalar))
            self.play(theta.animate.set_value(2*PI), run_time=tracker.duration, rate_func=linear)

        self.play(FadeOut(Group(*[m for m in self.mobjects if m != title])))

        # ==========================================
        # PART 3: UNIVERSALITY
        # ==========================================
        step3_title = Text("Universality", font_size=28, color=YELLOW).next_to(title, DOWN, buff=0.5)

        with self.voiceover(text="Universality."):
            self.play(Write(step3_title))

        quote_text = Text(
            "\"The fact that these quantities are invariant is obvious,\n"
            "but the amazing thing is that this property is Universal!\"",
            font_size=26, color=WHITE, slant=ITALIC, line_spacing=1.5
        ).next_to(step3_title, DOWN, buff=1.0)

        with self.voiceover(text="It is obvious that these scalars are invariant under rotation. But the truly amazing part is their mathematical universality."):
            self.play(FadeIn(quote_text))

        meaning_text = Text(
            "Meaning: Any invariant polynomial can be decomposed\n"
            "as a function of these simple scalar quantities.",
            font_size=22, color=LIGHT_GREY, line_spacing=1.2
        ).next_to(quote_text, DOWN, buff=0.8)

        math_summary = MathTex(r"f(x_1, …, x_n) = P( \langle x_i, x_j \rangle, ||x_i - x_j||^2 )", font_size=36, color=GREEN_C).next_to(meaning_text, DOWN, buff=0.8)

        ref_box = SurroundingRectangle(math_summary, color=BLUE_E, buff=0.3)
        ref_text = Text("(Villar et al. 2021)", font_size=20, color=BLUE_C).next_to(ref_box, DOWN, buff=0.3).align_to(ref_box, RIGHT)

        with self.voiceover(text="This means any generic invariant polynomial can be decomposed and written simply as a function of these basic scalar quantities, like the dot product and distance."):
            self.play(Write(meaning_text))
            self.play(Write(math_summary))
            self.play(Create(ref_box))
            self.play(FadeIn(ref_text))

        self.play(FadeOut(Group(*self.mobjects)))

    def rotate_point(self, point, angle):
        x, y, z = point
        new_x = x * np.cos(angle) - y * np.sin(angle)
        new_y = x * np.sin(angle) + y * np.cos(angle)
        return np.array([new_x, new_y, z])


## slide 44-45

In [ ]:
%%manim -qh -v WARNING ExamplesInvariantPolynomials44

from manim import *
import numpy as np
from manim_voiceover import VoiceoverScene
from manim_voiceover.services.gtts import GTTSService


# ==========================================
# UTILITY FUNCTIONS
# ==========================================
def get_base_cloud_od():
    """Point cloud for Scenes 1 & 2 (O(d))"""
    point_coords = [
        np.array([-1.2, 1.0, 0]),
        np.array([0.6, 1.4, 0]),
        np.array([1.0, -0.6, 0]),
        np.array([-0.8, -1.0, 0])
    ]
    colors = [RED_C, BLUE_C, GREEN_C, ORANGE]
    dots = VGroup(*[Dot(point=coord, color=colors[i], radius=0.1) for i, coord in enumerate(point_coords)])
    labels = VGroup(*[MathTex(f"v_{i+1}", font_size=24).next_to(dots[i], UP+RIGHT, buff=0.1) for i in range(4)])
    return VGroup(dots, labels), dots


def get_base_cloud_trans():
    """Point cloud for Scene 3 (Translation)"""
    point_coords = [
        np.array([-1.5, 0.5, 0]),
        np.array([0.5, 1.2, 0]),
        np.array([0.8, -0.8, 0]),
        np.array([-0.5, -1.0, 0])
    ]
    colors = [YELLOW, BLUE_C, GREEN_C, ORANGE]
    dots = VGroup(*[Dot(point=coord, color=colors[i], radius=0.1) for i, coord in enumerate(point_coords)])
    labels = VGroup(*[MathTex(f"v_{i+1}", font_size=24).next_to(dots[i], UP+RIGHT, buff=0.1) for i in range(4)])
    return VGroup(dots, labels), dots


def rotate_pt(pt, angle):
    return np.array([
        pt[0]*np.cos(angle) - pt[1]*np.sin(angle),
        pt[0]*np.sin(angle) + pt[1]*np.cos(angle),
        0
    ])


class ExamplesInvariantPolynomials44(VoiceoverScene):
    def construct(self):
        self.camera.background_color = BLACK

        # Khởi tạo dịch vụ tạo giọng đọc tiếng Anh
        self.set_speech_service(EdgeTTSService(voice="en-US-GuyNeural"))

        self.scene1_invariant_od()
        self.scene2_equivariant_od()
        self.scene3_translation_invariant()
        self.scene4_final_summary()

    # ==========================================
    # SCENE 1: INVARIANCE WITH O(d)
    # ==========================================
    def scene1_invariant_od(self):
        # --- Titles ---
        title = Text("Examples", font_size=32, color=YELLOW, weight=BOLD).to_edge(UP, buff=0.2)
        subtitle = Text("1. Invariance under Group O(d) (Rotation/Reflection)", font_size=24, color=YELLOW, weight=BOLD).next_to(title, DOWN, buff=0.2)

        with self.voiceover(text="Let's look at some concrete geometric examples. First, invariance under the Orthogonal Group, which includes rotations and reflections."):
            self.play(Write(title), Write(subtitle))

        # --- Geometry on the left ---
        cloud, dots = get_base_cloud_od()
        origin_dot = Dot(ORIGIN, color=WHITE, radius=0.08)
        origin_lbl = MathTex("O", font_size=20).next_to(origin_dot, DOWN+LEFT, buff=0.1)
        geo_group = VGroup(cloud, origin_dot, origin_lbl).move_to(LEFT * 4.0 + UP * 0.5)
        origin_pos = origin_dot.get_center()

        with self.voiceover(text="Consider a point cloud in a 2D space, centered around an origin O."):
            self.play(FadeIn(geo_group))

        v1_line = always_redraw(lambda: Arrow(origin_pos, dots[0].get_center(), buff=0, color=RED_C, stroke_width=4, max_tip_length_to_length_ratio=0.15))
        v2_line = always_redraw(lambda: Arrow(origin_pos, dots[1].get_center(), buff=0, color=BLUE_C, stroke_width=4, max_tip_length_to_length_ratio=0.15))
        angle_arc = always_redraw(lambda: Angle(v1_line, v2_line, radius=0.4, color=YELLOW))
        angle_lbl = always_redraw(lambda: MathTex(r"\theta", font_size=24, color=YELLOW).next_to(angle_arc, UP+RIGHT, buff=0.05))
        self.play(GrowArrow(v1_line), GrowArrow(v2_line))
        self.play(Create(angle_arc), Write(angle_lbl))

        # --- Formula Panel ---
        panel_x = RIGHT * 3.0
        eq_inv = MathTex(r"f(x) = h\Big( \underbrace{v_i^\top v_j}_{\text{Inner Product}} \Big)", font_size=36).move_to(panel_x + UP * 1.5)
        eq_inv[0][8:12].set_color(GREEN_C)

        with self.voiceover(text="An invariant function can be constructed using the inner products of these vectors."):
            self.play(Write(eq_inv))

        eq_breakdown = MathTex(r"v_1^\top v_2 = ", r"||v_1||", r" \cdot ", r"||v_2||", r" \cdot ", r"\cos(\theta)", font_size=32).next_to(eq_inv, DOWN, buff=0.8)
        eq_breakdown[1].set_color(RED_C)
        eq_breakdown[3].set_color(BLUE_C)
        eq_breakdown[5].set_color(YELLOW)

        desc_breakdown = Text(
            "Inner products only depend on:\nVector lengths and the Angle between them.",
            font_size=20, color=LIGHT_GREY, line_spacing=1.3
        ).next_to(eq_breakdown, DOWN, buff=0.5)

        with self.voiceover(text="As we know, the inner product depends strictly on the lengths of the vectors, and the angle between them."):
            self.play(Write(eq_breakdown), FadeIn(desc_breakdown))

        self.play(FadeOut(VGroup(eq_inv, eq_breakdown, desc_breakdown)))

        # --- HUD Table ---
        t_left, t_mid, t_right = 0.5, 3.2, 6.0
        t_top, row_h = 1.5, 0.6
        c1_x = (t_left + t_mid) / 2
        c2_x = (t_mid + t_right) / 2

        h_lines = VGroup(*[Line(np.array([t_left, t_top - i*row_h, 0]), np.array([t_right, t_top - i*row_h, 0]), color=DARK_GREY, stroke_width=2) for i in range(5)])
        v_lines = VGroup(*[Line(np.array([x, t_top, 0]), np.array([x, t_top - 4*row_h, 0]), color=DARK_GREY, stroke_width=2) for x in [t_left, t_mid, t_right]])

        h1 = Text("Quantity", font_size=18, color=WHITE, weight=BOLD).move_to(np.array([c1_x, t_top - 0.5*row_h, 0]))
        h2 = Text("Real-time Value", font_size=18, color=WHITE, weight=BOLD).move_to(np.array([c2_x, t_top - 0.5*row_h, 0]))
        r1_lbl = VGroup(Text("Coordinate", font_size=16, color=RED_C), MathTex("v_1", font_size=28, color=RED_C)).arrange(RIGHT, buff=0.1).move_to(np.array([c1_x, t_top - 1.5*row_h, 0]))
        r2_lbl = VGroup(Text("Length", font_size=16, color=WHITE), MathTex("||v_1||", font_size=28, color=WHITE)).arrange(RIGHT, buff=0.1).move_to(np.array([c1_x, t_top - 2.5*row_h, 0]))
        r3_lbl = VGroup(Text("Angle", font_size=16, color=YELLOW), MathTex(r"\theta", font_size=28, color=YELLOW)).arrange(RIGHT, buff=0.1).move_to(np.array([c1_x, t_top - 3.5*row_h, 0]))

        theta_tracker = ValueTracker(0)
        orig_v1 = np.array([-1.2, 1.0, 0])

        def get_v1_coord_text():
            angle = theta_tracker.get_value()
            new_x = orig_v1[0]*np.cos(angle) - orig_v1[1]*np.sin(angle)
            new_y = orig_v1[0]*np.sin(angle) + orig_v1[1]*np.cos(angle)
            return f"({new_x:.1f}, {new_y:.1f})"

        val1 = always_redraw(lambda: Text(get_v1_coord_text(), font_size=18, color=RED_C).move_to(np.array([c2_x, t_top - 1.5*row_h, 0])))
        val2 = Text("1.56 (Fixed)", font_size=18, color=WHITE).move_to(np.array([c2_x, t_top - 2.5*row_h, 0]))
        val3 = Text("68° (Fixed)", font_size=18, color=YELLOW).move_to(np.array([c2_x, t_top - 3.5*row_h, 0]))

        hud_table = VGroup(VGroup(h_lines, v_lines), h1, h2, r1_lbl, r2_lbl, r3_lbl, val1, val2, val3)

        with self.voiceover(text="Let's observe a real-time dashboard of these values."):
            self.play(FadeIn(hud_table, shift=LEFT*0.5))

        gauge = VGroup(
            Circle(radius=0.4, color=WHITE, stroke_width=3),
            Line(ORIGIN, UP*0.35, color=GREEN_C, stroke_width=4).rotate(-PI/3, about_point=ORIGIN),
            Dot(ORIGIN, color=WHITE, radius=0.05)
        )
        gauge_box = VGroup(gauge, Text("Output f(x)", font_size=20, color=GREEN_C).next_to(gauge, DOWN, buff=0.15))
        gauge_box.move_to(np.array([(t_left + t_right)/2, -2.0, 0]))
        self.play(FadeIn(gauge_box))

        # --- Rotation Animation ---
        action_text = Text("Action: Rotating the point cloud around Origin O", font_size=18, color=WHITE).next_to(hud_table, UP, buff=0.2)

        highlight_box = Rectangle(width=t_right - t_left, height=row_h * 2, color=GREEN_C, stroke_width=4).move_to(np.array([(t_left + t_right)/2, t_top - 3.0*row_h, 0]))

        base_coords = [d.get_center() - origin_pos for d in dots]
        for i in range(4):
            dots[i].add_updater(lambda m, i=i: m.move_to(origin_pos + rotate_pt(base_coords[i], theta_tracker.get_value())))
            cloud[1][i].add_updater(lambda m, i=i: m.next_to(dots[i], UP+RIGHT, buff=0.1))

        with self.voiceover(text="When we rotate the entire point cloud around the origin, notice that while the absolute coordinates change continuously, the length and the angle remain completely fixed.") as tracker:
            self.play(Write(action_text))
            self.play(Create(highlight_box))
            self.play(theta_tracker.animate.set_value(2*PI), run_time=tracker.duration, rate_func=linear)

        for i in range(4):
            dots[i].clear_updaters()
            cloud[1][i].clear_updaters()

        # --- Conclusion ---
        self.play(FadeOut(action_text))
        conclusion = VGroup(
            Text("Result: Coordinates change constantly, but Length and Angle remain static.", font_size=20, color=GREEN_C, weight=BOLD),
            Text("⇒ Output f(x) is INVARIANT under Rotation!", font_size=20, color=GREEN_C, weight=BOLD)
        ).arrange(DOWN, buff=0.15).to_edge(DOWN, buff=0.4)

        with self.voiceover(text="Because the inner products are unchanged, the output of the function f remains perfectly invariant to the rotation!"):
            self.play(Write(conclusion))
            self.play(gauge[1].animate.set_color(WHITE).set_stroke(width=6), run_time=0.5)
            self.play(gauge[1].animate.set_color(GREEN_C).set_stroke(width=4), run_time=0.5)

        self.play(FadeOut(Group(*self.mobjects)))

    # ==========================================
    # SCENE 2: EQUIVARIANCE WITH O(d)
    # ==========================================
    def scene2_equivariant_od(self):
        # --- Titles ---
        title = Text("Examples", font_size=32, color=YELLOW, weight=BOLD).to_edge(UP, buff=0.2)
        subtitle = Text("2. Equivariance under Group O(d) (Rotation/Reflection)", font_size=24, color=YELLOW, weight=BOLD).next_to(title, DOWN, buff=0.2)

        with self.voiceover(text="Example two: Equivariance under the Orthogonal Group."):
            self.play(Write(title), Write(subtitle))

        # --- Geometry on the left ---
        cloud, dots = get_base_cloud_od()
        origin_dot = Dot(ORIGIN, color=WHITE, radius=0.08)
        origin_lbl = MathTex("O", font_size=20).next_to(origin_dot, DOWN+LEFT, buff=0.1)
        geo_group = VGroup(cloud, origin_dot, origin_lbl).move_to(LEFT * 4.0 + UP * 0.5)
        origin_pos = origin_dot.get_center()
        self.play(FadeIn(geo_group))

        # --- Formula Panel ---
        panel_x = RIGHT * 3.0
        eq_equiv = MathTex(r"f(x) = \sum_{t=1}^n \underbrace{h(...)}_{\text{Weight}} \cdot \underbrace{v_t}_{\text{Vector}}", font_size=36).move_to(panel_x + UP * 1.0)
        eq_equiv[0][9:13].set_color(GREEN_C)
        eq_equiv[0][20:22].set_color(RED_C)

        with self.voiceover(text="Here, the output is a linear combination of the original vectors, scaled by invariant weights."):
            self.play(Write(eq_equiv))

        desc_breakdown = Text(
            "Output is a linear combination of the original Vectors.\nSince weights h(...) are INVARIANT (fixed),\nrotating the space forces f(x) to rotate as well!",
            font_size=20, color=LIGHT_GREY, line_spacing=1.3
        ).next_to(eq_equiv, DOWN, buff=0.5)

        with self.voiceover(text="Because the weights are invariant scalars, when the input space rotates, the output vector is algebraically forced to rotate along with it."):
            self.play(FadeIn(desc_breakdown))

        self.play(FadeOut(eq_equiv), FadeOut(desc_breakdown))

        # --- HUD Table ---
        t_left, t_mid, t_right = 0.5, 3.2, 6.0
        t_top, row_h = 1.5, 0.6
        c1_x = (t_left + t_mid) / 2
        c2_x = (t_mid + t_right) / 2

        h_lines = VGroup(*[Line(np.array([t_left, t_top - i*row_h, 0]), np.array([t_right, t_top - i*row_h, 0]), color=DARK_GREY, stroke_width=2) for i in range(4)])
        v_lines = VGroup(*[Line(np.array([x, t_top, 0]), np.array([x, t_top - 3*row_h, 0]), color=DARK_GREY, stroke_width=2) for x in [t_left, t_mid, t_right]])

        h1 = Text("Quantity", font_size=18, color=WHITE, weight=BOLD).move_to(np.array([c1_x, t_top - 0.5*row_h, 0]))
        h2 = Text("Real-time Value", font_size=18, color=WHITE, weight=BOLD).move_to(np.array([c2_x, t_top - 0.5*row_h, 0]))
        r1_lbl = VGroup(Text("Weight", font_size=16, color=GREEN_C), MathTex("h(...)", font_size=28, color=GREEN_C)).arrange(RIGHT, buff=0.1).move_to(np.array([c1_x, t_top - 1.5*row_h, 0]))
        r2_lbl = VGroup(Text("Angle of", font_size=16, color=YELLOW), MathTex("f(x)", font_size=28, color=YELLOW)).arrange(RIGHT, buff=0.1).move_to(np.array([c1_x, t_top - 2.5*row_h, 0]))

        theta_tracker = ValueTracker(0)
        base_output_vec = np.array([1.5, -0.5, 0])

        def get_fx_angle():
            base_angle = np.degrees(np.arctan2(base_output_vec[1], base_output_vec[0]))
            current_angle = (base_angle + np.degrees(theta_tracker.get_value())) % 360
            return f"{current_angle:.1f}°"

        val1 = Text("Fixed", font_size=18, color=WHITE).move_to(np.array([c2_x, t_top - 1.5*row_h, 0]))
        val2 = always_redraw(lambda: Text(get_fx_angle(), font_size=18, color=YELLOW).move_to(np.array([c2_x, t_top - 2.5*row_h, 0])))

        hud_table = VGroup(VGroup(h_lines, v_lines), h1, h2, r1_lbl, r2_lbl, val1, val2)

        with self.voiceover(text="Let's observe this behavior. We have our invariant weights, and the angle of our output vector."):
            self.play(FadeIn(hud_table, shift=LEFT*0.5))

        out_vector = always_redraw(lambda: Arrow(
            origin_pos, origin_pos + rotate_pt(base_output_vec, theta_tracker.get_value()),
            buff=0, color=YELLOW, stroke_width=6, max_tip_length_to_length_ratio=0.15
        ))
        out_lbl = always_redraw(lambda: MathTex("f(x)", font_size=28, color=YELLOW).next_to(out_vector.get_end(), RIGHT, buff=0.1))
        self.play(GrowArrow(out_vector), Write(out_lbl))

        # --- Rotation Animation ---
        action_text = Text("Action: Rotating the point cloud around Origin O", font_size=18, color=WHITE).next_to(hud_table, UP, buff=0.2)
        highlight_box = Rectangle(width=t_right - t_left, height=row_h, color=YELLOW, stroke_width=4).move_to(np.array([(t_left + t_right)/2, t_top - 2.5*row_h, 0]))

        base_coords = [d.get_center() - origin_pos for d in dots]
        for i in range(4):
            dots[i].add_updater(lambda m, i=i: m.move_to(origin_pos + rotate_pt(base_coords[i], theta_tracker.get_value())))
            cloud[1][i].add_updater(lambda m, i=i: m.next_to(dots[i], UP+RIGHT, buff=0.1))

        with self.voiceover(text="As we rotate the point cloud, the angle of the output vector changes by the exact same amount.") as tracker:
            self.play(Write(action_text))
            self.play(Create(highlight_box))
            self.play(theta_tracker.animate.set_value(2*PI), run_time=tracker.duration, rate_func=linear)

        for i in range(4):
            dots[i].clear_updaters()
            cloud[1][i].clear_updaters()

        self.play(FadeOut(action_text))
        conclusion = VGroup(
            Text("Result: As the space rotates, the vector f(x) rotates in sync.", font_size=20, color=GREEN_C, weight=BOLD),
            Text("⇒ This is EQUIVARIANCE under Rotation!", font_size=20, color=GREEN_C, weight=BOLD)
        ).arrange(DOWN, buff=0.15).to_edge(DOWN, buff=0.4)

        with self.voiceover(text="This perfectly demonstrates Equivariance. The output rotates perfectly in sync with the input."):
            self.play(Write(conclusion))
            self.play(out_vector.animate.set_color(WHITE).set_stroke(width=10), run_time=0.5)
            self.play(out_vector.animate.set_color(YELLOW).set_stroke(width=6), run_time=0.5)

        self.play(FadeOut(Group(*self.mobjects)))

    # ==========================================
    # SCENE 3: TRANSLATION INVARIANCE
    # ==========================================
    def scene3_translation_invariant(self):
        # --- Titles ---
        title = Text("Examples", font_size=32, color=YELLOW, weight=BOLD).to_edge(UP, buff=0.2)
        subtitle = Text("3. Invariance under Translation", font_size=24, color=YELLOW, weight=BOLD).next_to(title, DOWN, buff=0.2)

        with self.voiceover(text="Example three: Invariance to Translation."):
            self.play(Write(title), Write(subtitle))

        # --- Geometry on the left ---
        cloud, dots = get_base_cloud_trans()
        base_coords_raw = [d.get_center() for d in dots]
        cloud_group = VGroup(cloud).move_to(LEFT * 4.0 + UP * 0.5)
        offset = cloud_group.get_center() - VGroup(*[Dot(p) for p in base_coords_raw]).get_center()
        base_coords = [p + offset for p in base_coords_raw]
        self.play(FadeIn(cloud_group))

        rel_arrows = VGroup()
        for i in range(1, 4):
            arrow = always_redraw(lambda i=i: Arrow(
                dots[0].get_center(), dots[i].get_center(),
                buff=0.1, color=BLUE_C, stroke_width=3, max_tip_length_to_length_ratio=0.15
            ))
            rel_arrows.add(arrow)

        with self.voiceover(text="To achieve this, we can select one point, say v 1, as the local origin, and represent all other points as relative vectors."):
            self.play(
                dots[0].animate.set_stroke(YELLOW, width=4).scale(1.2),
                LaggedStart(*[GrowArrow(arr) for arr in rel_arrows], lag_ratio=0.2)
            )

        # --- Formula Panel ---
        panel_x = RIGHT * 3.0
        eq_trans = MathTex(r"f(x) = h\Big( ", r"v_2 - v_1", r", \dots, ", r"v_n - v_1", r" \Big)", font_size=36).move_to(panel_x + UP * 1.0)
        eq_trans[1].set_color(BLUE_C)
        eq_trans[3].set_color(BLUE_C)
        desc_trans = Text(
            "Set v_1 as the relative origin.\nConvert all points into relative Vectors.",
            font_size=20, color=LIGHT_GREY, line_spacing=1.3
        ).next_to(eq_trans, DOWN, buff=0.5)
        self.play(Write(eq_trans), FadeIn(desc_trans))
        self.play(FadeOut(eq_trans), FadeOut(desc_trans))

        # --- HUD Table ---
        t_left, t_mid, t_right = 0.5, 3.2, 6.0
        t_top, row_h = 1.5, 0.6
        c1_x = (t_left + t_mid) / 2
        c2_x = (t_mid + t_right) / 2

        h_lines = VGroup(*[Line(np.array([t_left, t_top - i*row_h, 0]), np.array([t_right, t_top - i*row_h, 0]), color=DARK_GREY, stroke_width=2) for i in range(5)])
        v_lines = VGroup(*[Line(np.array([x, t_top, 0]), np.array([x, t_top - 4*row_h, 0]), color=DARK_GREY, stroke_width=2) for x in [t_left, t_mid, t_right]])

        h1 = Text("Quantity", font_size=18, color=WHITE, weight=BOLD).move_to(np.array([c1_x, t_top - 0.5*row_h, 0]))
        h2 = Text("Real-time Value", font_size=18, color=WHITE, weight=BOLD).move_to(np.array([c2_x, t_top - 0.5*row_h, 0]))
        r1_lbl = VGroup(Text("Coordinate", font_size=16, color=YELLOW), MathTex("v_1", font_size=28, color=YELLOW)).arrange(RIGHT, buff=0.1).move_to(np.array([c1_x, t_top - 1.5*row_h, 0]))
        r2_lbl = VGroup(Text("Coordinate", font_size=16, color=WHITE), MathTex("v_2", font_size=28, color=WHITE)).arrange(RIGHT, buff=0.1).move_to(np.array([c1_x, t_top - 2.5*row_h, 0]))
        r3_lbl = VGroup(Text("Vector", font_size=16, color=BLUE_C), MathTex("v_2 - v_1", font_size=28, color=BLUE_C)).arrange(RIGHT, buff=0.1).move_to(np.array([c1_x, t_top - 3.5*row_h, 0]))

        shift_x = ValueTracker(0)
        shift_y = ValueTracker(0)
        rel_v2_v1 = base_coords[1] - base_coords[0]

        val1 = always_redraw(lambda: Text(f"({base_coords[0][0] + shift_x.get_value():.1f}, {base_coords[0][1] + shift_y.get_value():.1f})", font_size=18, color=YELLOW).move_to(np.array([c2_x, t_top - 1.5*row_h, 0])))
        val2 = always_redraw(lambda: Text(f"({base_coords[1][0] + shift_x.get_value():.1f}, {base_coords[1][1] + shift_y.get_value():.1f})", font_size=18, color=WHITE).move_to(np.array([c2_x, t_top - 2.5*row_h, 0])))
        val3 = Text(f"({rel_v2_v1[0]:.1f}, {rel_v2_v1[1]:.1f}) Fixed", font_size=18, color=BLUE_C).move_to(np.array([c2_x, t_top - 3.5*row_h, 0]))

        hud_table = VGroup(VGroup(h_lines, v_lines), h1, h2, r1_lbl, r2_lbl, r3_lbl, val1, val2, val3)

        with self.voiceover(text="Let's track the absolute coordinates of v 1 and v 2, and their relative vector distance."):
            self.play(FadeIn(hud_table, shift=LEFT*0.5))

        gauge = VGroup(
            Circle(radius=0.4, color=WHITE, stroke_width=3),
            Line(ORIGIN, UP*0.35, color=GREEN_C, stroke_width=4).rotate(PI/4, about_point=ORIGIN),
            Dot(ORIGIN, color=WHITE, radius=0.05)
        )
        gauge_box = VGroup(gauge, Text("Output f(x)", font_size=28, color=GREEN_C).next_to(gauge, DOWN, buff=0.15))
        gauge_box.move_to(np.array([(t_left + t_right)/2, -2.0, 0]))
        self.play(FadeIn(gauge_box))

        action_text = Text("Action: Translating (Moving) the point cloud", font_size=18, color=WHITE).next_to(hud_table, UP, buff=0.2)
        highlight_box = Rectangle(width=t_right - t_left, height=row_h, color=BLUE_C, stroke_width=4).move_to(np.array([(t_left + t_right)/2, t_top - 3.5*row_h, 0]))

        for i in range(4):
            dots[i].add_updater(lambda m, i=i: m.move_to(base_coords[i] + np.array([shift_x.get_value(), shift_y.get_value(), 0])))
            cloud[1][i].add_updater(lambda m, i=i: m.next_to(dots[i], UP+RIGHT, buff=0.1))

        with self.voiceover(text="As we translate the entire point cloud across the space, the absolute coordinates constantly change, but the relative distance between them remains completely constant.") as tracker:
            self.play(Write(action_text))
            self.play(Create(highlight_box))
            self.play(
                shift_x.animate.set_value(2.0),
                shift_y.animate.set_value(0.8),
                run_time=tracker.duration,
                rate_func=there_and_back
            )

        for i in range(4):
            dots[i].clear_updaters()
            cloud[1][i].clear_updaters()

        # --- Conclusion ---
        self.play(FadeOut(action_text))
        conclusion = VGroup(
            Text("Result: Absolute coordinates shift, but relative distances remain constant.", font_size=20, color=GREEN_C, weight=BOLD),
            Text("⇒ Output f(x) is completely INVARIANT to Translation!", font_size=20, color=GREEN_C, weight=BOLD)
        ).arrange(DOWN, buff=0.15).to_edge(DOWN, buff=0.4)

        with self.voiceover(text="Because the relative vectors are unchanged, the output function is completely invariant to translation."):
            self.play(Write(conclusion))
            self.play(gauge[1].animate.set_color(WHITE).set_stroke(width=6), run_time=0.5)
            self.play(gauge[1].animate.set_color(GREEN_C).set_stroke(width=4), run_time=0.5)
        self.wait(3)

        self.play(FadeOut(Group(*self.mobjects)))

    # ==========================================
    # SCENE 4: FINAL SUMMARY
    # ==========================================
    def scene4_final_summary(self):
        # --- Titles ---
        title = Text("Examples", font_size=32, color=YELLOW, weight=BOLD).to_edge(UP, buff=0.2)
        subtitle = Text("Summary: Invariance & Equivariance", font_size=24, color=YELLOW, weight=BOLD).next_to(title, DOWN, buff=0.2)

        with self.voiceover(text="To summarize these fundamental principles:"):
            self.play(Write(title), Write(subtitle))

        p1 = Text("• O(d) Invariance: Based on Inner Products", font_size=24).shift(UP * 1)
        p2 = Text("• O(d) Equivariance: Linear combinations of original vectors", font_size=24).next_to(p1, DOWN, buff=0.5)
        p3 = Text("• Translation Invariance: Uses relative coordinates (v_i - v_1)", font_size=24).next_to(p2, DOWN, buff=0.5)

        principles = VGroup(p1, p2, p3).move_to(ORIGIN)

        with self.voiceover(text="Orthogonal Invariance is built using inner products. Orthogonal Equivariance is achieved via linear combinations of the original vectors. And Translation Invariance uses relative coordinates."):
            for p in principles:
                self.play(FadeIn(p, shift=RIGHT * 0.3))

        conclusion = Text(
            "These properties ensure models recognize structures accurately despite transformations.",
            font_size=20, color=BLUE_B, line_spacing=1.5
        ).to_edge(DOWN, buff=1.5)

        with self.voiceover(text="These mathematical constraints guarantee that our geometric models can reliably recognize structures, no matter how the data is transformed in space."):
            self.play(Write(conclusion))
        self.wait(3)

        self.play(FadeOut(Group(*self.mobjects)))


## slide 46
Applications: equivariant GNNs

In [ ]:
%%manim -qh -v WARNING equivariantGNNs46

class equivariantGNNs46(VoiceoverScene):
    def construct(self):
        self.camera.background_color = BLACK

        # Initialize English voiceover service using the custom class
        self.set_speech_service(EdgeTTSService(voice="en-US-GuyNeural"))

        # ==========================================
        # SCENE 1: PROBLEM SETUP - COORDINATES AND FEATURES
        # ==========================================
        title = Text("Applications: Equivariant GNN", font_size=32, color=YELLOW, weight=BOLD)
        title.to_edge(UP, buff=0.4)

        intro_text = Text(
            "In an EGNN, each node has two completely separate data streams:",
            font_size=24, color=LIGHT_GREY
        ).next_to(title, DOWN, buff=0.4)

        with self.voiceover(text="Let's explore Equivariant Graph Neural Networks. In an EGNN, each node processes two completely distinct streams of information."):
            self.play(Write(title))
            self.play(FadeIn(intro_text))

        # Visualizing the Node
        node_center = Dot(point=LEFT * 3.5, radius=0.4, color=BLUE_E)
        node_label = MathTex(r"v_i", font_size=32).move_to(node_center)
        self.play(FadeIn(node_center), Write(node_label))

        # Feature stream h_i
        feat_box = SurroundingRectangle(node_center, color=GREEN_C, buff=0.5)
        feat_text = MathTex(r"h_i", font_size=32, color=GREEN_C).next_to(feat_box, UP, buff=0.2)
        feat_desc = Text(
            "Features (Mass, Charge, etc.)\n→ INVARIANT (Static under rotation)",
            font_size=18, color=GREEN_C, line_spacing=1.3
        ).next_to(feat_box, DOWN, buff=0.3)

        with self.voiceover(text="First, we have node features, denoted as h i, like mass or charge. These are invariant; they don't change when the molecule rotates."):
            self.play(Create(feat_box), Write(feat_text))
            self.play(Write(feat_desc))
        self.wait(0.5)

        # Coordinate stream x_i
        axes = Axes(x_range=[-1, 4, 1], y_range=[-1, 3, 1], x_length=4, y_length=3).next_to(node_center, RIGHT, buff=4)
        origin = axes.c2p(0, 0)
        target = axes.c2p(3, 2)

        node_copy = Dot(point=target, radius=0.2, color=BLUE_E)
        pos_vector = Arrow(start=origin, end=target, color=RED_C, buff=0)
        pos_text = MathTex(r"x_i", font_size=32, color=RED_C).next_to(target, UP)
        pos_desc = Text(
            "3D Spatial Coordinates\n→ EQUIVARIANT (Rotate with frame)",
            font_size=18, color=RED_C, line_spacing=1.3
        ).next_to(axes, DOWN, buff=0.7)

        with self.voiceover(text="Second, we have the spatial coordinates x i. These are equivariant, meaning they rotate in sync with the physical system."):
            self.play(FadeIn(axes), TransformFromCopy(node_center, node_copy))
            self.play(GrowArrow(pos_vector), Write(pos_text))
            self.play(Write(pos_desc))
        self.wait(1.5)

        self.play(FadeOut(Group(*[m for m in self.mobjects if m != title])))

        # ==========================================
        # SCENE 2: DISSECTING THE EGNN FORMULA
        # ==========================================
        eq_title = Text("Dissecting the Coordinate Update Formula", font_size=28, color=YELLOW).next_to(title, DOWN, buff=0.5)

        eq = MathTex(
            r"x_i^{(l+1)} = x_i^{(l)} + C \sum_{j \neq i} ",
            r"\underbrace{(x_i - x_j)}_{\text{Direction}}",
            r" \cdot ",
            r"\underbrace{\phi(h_i, h_j, \|x_i - x_j\|^2)}_{\text{Magnitude (Force)}}",
            font_size=36
        ).next_to(eq_title, DOWN, buff=0.8)

        eq[1].set_color(RED_C)
        eq[3].set_color(GREEN_C)

        with self.voiceover(text="Let's dissect the EGNN coordinate update formula."):
            self.play(Write(eq_title))
            self.play(Write(eq))

        desc_dir = VGroup(
            Text("1. ", font_size=20),
            MathTex(r"(x_i - x_j)", font_size=24),
            Text(": Relative vector. Provides the direction of the interaction.", font_size=20)
        ).arrange(RIGHT, buff=0.1).set_color(RED_C)

        desc_mag = VGroup(
            Text("2. ", font_size=20),
            MathTex(r"\phi(...)", font_size=24),
            Text(": Neural Network scales the force magnitude.", font_size=20)
        ).arrange(RIGHT, buff=0.1).set_color(GREEN_C)

        desc_inv = VGroup(
            Text("   Since ", font_size=20),
            MathTex(r"\phi()"),
            Text(" only takes distances (", font_size=20),
            MathTex(r"d^2"),
            Text(") and features (", font_size=20),
            MathTex(r"h"),
            Text("), the force is INVARIANT.", font_size=20)
        ).arrange(RIGHT, buff=0.1).set_color(LIGHT_GREY)

        desc_group = VGroup(desc_dir, desc_mag, desc_inv).arrange(DOWN, aligned_edge=LEFT, buff=0.3)
        desc_group.next_to(eq, DOWN, buff=1.0)

        with self.voiceover(text="The update has two parts. A relative vector provides direction, and an invariant neural network computes the scalar force magnitude."):
            self.play(Write(desc_dir))
            self.play(Write(desc_mag))
            self.play(FadeIn(desc_inv))
        self.wait(2)

        self.play(FadeOut(Group(*[m for m in self.mobjects if m != title])))

        # ==========================================
        # SCENE 3: VISUALIZING MECHANICS (PUSH/PULL)
        # ==========================================
        mech_title = Text("Mechanics: Sum of Forces (Message Passing)", font_size=28, color=YELLOW).next_to(title, DOWN, buff=0.5)

        node_i = Dot(point=ORIGIN, radius=0.2, color=YELLOW)
        label_i = MathTex(r"v_i", font_size=28).next_to(node_i, DOWN)

        node_j1 = Dot(point=LEFT * 3 + UP * 1.5, radius=0.2, color=BLUE)
        label_j1 = MathTex(r"v_{j_1}", font_size=28).next_to(node_j1, UP)

        node_j2 = Dot(point=RIGHT * 2.5 + UP * 1.2, radius=0.2, color=ORANGE)
        label_j2 = MathTex(r"v_{j_2}", font_size=28).next_to(node_j2, UP)

        with self.voiceover(text="We can visualize this message passing as physical forces acting on node i."):
            self.play(Write(mech_title))
            self.play(FadeIn(node_i, node_j1, node_j2), Write(VGroup(label_i, label_j1, label_j2)))

        vec_j1 = DashedLine(node_j1.get_center(), node_i.get_center(), color=RED_C)
        vec_j2 = DashedLine(node_j2.get_center(), node_i.get_center(), color=RED_C)
        step1_text = Text("Step 1: Define interaction lines (Relative vectors)", font_size=22, color=RED_C).to_edge(DOWN, buff=0.8)

        with self.voiceover(text="First, we define the interaction lines along the relative vectors."):
            self.play(Write(step1_text), Create(vec_j1), Create(vec_j2))

        step2_text = VGroup(
            Text("Step 2: Network ", font_size=22),
            MathTex(r"\phi()"),
            Text(" determines Push (+) or Pull (-)", font_size=22)
        ).arrange(RIGHT, buff=0.1).set_color(GREEN_C).to_edge(DOWN, buff=0.8)

        force_1_vec = (node_i.get_center() - node_j1.get_center()) * 0.5
        force_1 = Arrow(start=node_i.get_center(), end=node_i.get_center() + force_1_vec, color=GREEN_C, buff=0)
        force_1_lbl = Text("j1 Push", font_size=16, color=GREEN_C).next_to(force_1.get_end(), RIGHT)

        force_2_vec = (node_j2.get_center() - node_i.get_center()) * 0.5
        force_2 = Arrow(start=node_i.get_center(), end=node_i.get_center() + force_2_vec, color=GREEN_C, buff=0)
        force_2_lbl = Text("j2 Pull", font_size=16, color=GREEN_C).next_to(force_2.get_end(), UP)

        with self.voiceover(text="The network computes the force magnitude, either pushing or pulling the node."):
            self.play(Transform(step1_text, step2_text))
            self.play(GrowArrow(force_1), Write(force_1_lbl), GrowArrow(force_2), Write(force_2_lbl))

        step3_text = Text("Step 3: Vector Sum determines the New Coordinate", font_size=22, color=YELLOW).to_edge(DOWN, buff=0.8)

        para_line1 = DashedLine(force_1.get_end(), force_1.get_end() + (force_2.get_end() - node_i.get_center()), color=DARK_GREY)
        para_line2 = DashedLine(force_2.get_end(), force_2.get_end() + (force_1.get_center() - node_i.get_center()), color=DARK_GREY)

        net_force_end = force_1.get_end() + (force_2.get_end() - node_i.get_center())
        net_force = Arrow(start=node_i.get_center(), end=net_force_end, color=YELLOW, buff=0, stroke_width=6)

        with self.voiceover(text="Aggregating these forces gives us the final displacement for the coordinate update."):
            self.play(Transform(step1_text, step3_text))
            self.play(Create(para_line1), Create(para_line2), GrowArrow(net_force))
            self.play(
                node_i.animate.move_to(net_force_end).set_color(WHITE),
                label_i.animate.move_to(net_force_end + DOWN*0.3),
                FadeOut(VGroup(force_1, force_2, force_1_lbl, force_2_lbl, para_line1, para_line2, net_force))
            )
        self.wait(1.5)

        self.play(FadeOut(Group(*[m for m in self.mobjects if m != title])))

        # ==========================================
        # SCENE 4: WHY IS IT EQUIVARIANT?
        # ==========================================
        proof_title = Text("Proof of Equivariance", font_size=28, color=YELLOW).next_to(title, DOWN, buff=0.5)

        node_i = Dot(point=UP * 0.2, radius=0.2, color=WHITE)
        node_j = Dot(point=LEFT * 1 + UP * 1, radius=0.2, color=BLUE)
        link = DashedLine(node_j.get_center(), node_i.get_center(), color=LIGHT_GREY)
        force_vec = (node_i.get_center() - node_j.get_center()) * 0.6
        force = Arrow(start=node_i.get_center(), end=node_i.get_center() + force_vec, color=GREEN_C, buff=0)
        system = VGroup(node_i, node_j, link, force)

        with self.voiceover(text="This works because as the graph rotates, relative distances remain static, while the force vectors naturally rotate in sync."):
            self.play(Write(proof_title))
            self.play(FadeIn(system))

            l1 = Text("When the entire graph rotates:", font_size=20, color=LIGHT_GREY)
            l2 = Text("1. Relative distances stay constant → Forces remain the same.", font_size=20, color=LIGHT_GREY)
            l3 = Text("2. Interaction lines rotate along with the graph.", font_size=20, color=LIGHT_GREY)
            logic_text = VGroup(l1, l2, l3).arrange(DOWN, aligned_edge=LEFT, buff=0.1).to_edge(DOWN, buff=0.5)
            self.play(Write(logic_text))

            self.play(Rotate(system, angle=PI/2, about_point=node_i.get_center()), run_time=2)
            self.play(Rotate(system, angle=-PI, about_point=node_i.get_center()), run_time=3)

        self.play(FadeOut(Group(*self.mobjects)))


## slide 47
Application: protein generation models

In [ ]:
%%manim -qh -v WARNING ProteinGeneration47

# Color palette for 8 amino acid beads
AMINO_COLORS = [RED_C, BLUE_C, GREEN_C, ORANGE, PURPLE, PINK, TEAL, GOLD]

# Helper function to get coordinates
def get_polymer_points(state="noisy", radius=1.0, center=ORIGIN):
    np.random.seed(42)
    num_nodes = 8
    points = []
    if state == "noisy":
        points = [center + np.array([np.random.uniform(-radius, radius), np.random.uniform(-radius, radius), 0]) for _ in range(num_nodes)]
    else:
        for i in range(num_nodes):
            angle = i * (2 * np.pi / num_nodes)
            r = radius + 0.25 * np.sin(3 * angle)
            points.append(center + np.array([r * np.cos(angle), r * np.sin(angle), 0]))
    return points

# Helper function to draw edges between dots
def get_edges(dots, is_closed=False):
    edges = VGroup()
    for i in range(len(dots) - 1):
        edges.add(Line(dots[i].get_center(), dots[i+1].get_center(), color=LIGHT_GREY, stroke_width=3, stroke_opacity=0.8))
    if is_closed:
        edges.add(Line(dots[-1].get_center(), dots[0].get_center(), color=LIGHT_GREY, stroke_width=3, stroke_opacity=0.8))
    return edges

class ProteinGeneration47(VoiceoverScene):
    def construct(self):
        self.camera.background_color = BLACK

        # Initialize English voiceover service
        self.set_speech_service(EdgeTTSService(voice="en-US-GuyNeural"))

        # GLOBAL TITLE
        title = Text("Application: Protein Generation Models", font_size=28, color=YELLOW, weight=BOLD).to_edge(UP, buff=0.3)
        self.play(Write(title))

        # ==========================================
        # STAGE 1: MACRO DIFFUSION PROCESS
        # ==========================================
        stg1_title = Text("Macro Generation Process", font_size=20, color=WHITE).next_to(title, DOWN, buff=0.3)

        with self.voiceover(text="Let's apply these geometric concepts to a real-world application: protein generation models. Stage 1 is the Macro Generation process."):
            self.play(FadeIn(stg1_title))

        pos_col = LEFT * 4.5 + DOWN * 0.5
        pos_bb = ORIGIN + DOWN * 0.5
        pos_aa = RIGHT * 4.5 + DOWN * 0.5
        arrow_y_level = pos_col[1]

        noisy_pts = get_polymer_points("noisy", 0.65, pos_col)
        dots1 = VGroup(*[Dot(noisy_pts[i], color=AMINO_COLORS[i], radius=0.15) for i in range(8)])
        edges1 = always_redraw(lambda: get_edges(dots1, is_closed=False))
        lbl_col = Text("Collapsed Polymer", font_size=16).next_to(pos_col, DOWN, buff=1.3)

        with self.voiceover(text="We start with a collapsed, noisy polymer chain representing pure unstructured data."):
            self.play(FadeIn(dots1), FadeIn(edges1), Write(lbl_col))

        arr1 = Arrow(start=[pos_col[0] + 0.8, arrow_y_level, 0], end=[pos_bb[0] - 0.8, arrow_y_level, 0], buff=0.1, color=WHITE)
        txt_arr1 = Text("Reverse Diffusion", font_size=14, color=YELLOW).next_to(arr1, UP, buff=0.15)

        with self.voiceover(text="Through a reverse diffusion process..."):
            self.play(Create(arr1), Write(txt_arr1))

        bb_pts = get_polymer_points("backbone", 0.65, pos_bb)
        dots2 = dots1.copy()
        edges2 = always_redraw(lambda: get_edges(dots2, is_closed=True))
        self.add(edges2, dots2)

        lbl_bb = Text("Backbone", font_size=16).next_to(pos_bb, DOWN, buff=1.3)

        with self.voiceover(text="The model gradually denoises this blob to form the stable 3D backbone of a protein."):
            self.play(
                AnimationGroup(*[dots2[i].animate.move_to(bb_pts[i]) for i in range(8)]),
                FadeIn(lbl_bb), run_time=2.5, rate_func=smooth
            )

        arr2 = Arrow(start=[pos_bb[0] + 0.8, arrow_y_level, 0], end=[pos_aa[0] - 1.0, arrow_y_level, 0], buff=0.1, color=WHITE)
        txt_arr2 = Text("Design Network", font_size=14, color=YELLOW).next_to(arr2, UP, buff=0.15)

        with self.voiceover(text="Finally, a specialized design network..."):
            self.play(Create(arr2), Write(txt_arr2))

        aa_pts = get_polymer_points("backbone", 0.65, pos_aa)
        dots3 = dots2.copy()
        edges3 = always_redraw(lambda: get_edges(dots3, is_closed=True))
        self.add(edges3, dots3)

        lbl_aa = Text("Atomic Completion", font_size=16).next_to(pos_aa, DOWN, buff=1.3)

        with self.voiceover(text="...completes the structure by generating the specific amino acid side chains."):
            self.play(
                AnimationGroup(*[dots3[i].animate.move_to(aa_pts[i]) for i in range(8)]),
                FadeIn(lbl_aa), run_time=1.5
            )

            side_chains = VGroup()
            for i in range(8):
                sc_dir = normalize(aa_pts[i] - pos_aa) * 0.35
                sc_pt = aa_pts[i] + sc_dir
                side_chains.add(Line(aa_pts[i], sc_pt, color=AMINO_COLORS[i], stroke_width=2))
                side_chains.add(Dot(sc_pt, color=AMINO_COLORS[i], radius=0.08))

            self.play(FadeIn(side_chains))

        self.play(FadeOut(Group(*[m for m in self.mobjects if m != title])))

        # ==========================================
        # STAGE 2: MICRO DENOISING PIPELINE
        # ==========================================
        stg2_title = Text("Micro Denoising Step with Equivariant GNN", font_size=20, color=WHITE).next_to(title, DOWN, buff=0.3)

        with self.voiceover(text="Stage 2: Let's zoom into a single micro-step of this denoising process using an Equivariant GNN."):
            self.play(FadeIn(stg2_title))

        g_xt_pts = get_polymer_points("noisy", 0.45, ORIGIN)
        g_xt_dots = VGroup(*[Dot(g_xt_pts[i], color=AMINO_COLORS[i], radius=0.1) for i in range(8)])
        g_xt_edges = get_edges(g_xt_dots)
        step1 = VGroup(g_xt_edges, g_xt_dots, MathTex("x_t", font_size=20).next_to(g_xt_dots, DOWN))

        box_gnn = Rectangle(width=1.2, height=1.0, color=PURPLE, fill_opacity=0.3)
        txt_gnn = Text("Random\nGNN", font_size=14).move_to(box_gnn)
        step2 = VGroup(box_gnn, txt_gnn)

        g_geom_pts = get_polymer_points("noisy", 0.45, ORIGIN)
        g_geom_dots = VGroup(*[Dot(g_geom_pts[i], color=AMINO_COLORS[i], radius=0.1) for i in range(8)])
        g_geom_edges = get_edges(g_geom_dots)
        vecs = VGroup(*[Arrow(d.get_center(), d.get_center() + normalize(np.random.uniform(-1,1,3))*0.35, buff=0, color=PINK, stroke_width=2, max_tip_length_to_length_ratio=0.2) for d in g_geom_dots])
        step3 = VGroup(g_geom_edges, g_geom_dots, vecs, Text("Vector Prediction", font_size=14).next_to(g_geom_dots, DOWN))

        box_solver = Rectangle(width=1.4, height=1.0, color=YELLOW, fill_opacity=0.3)
        txt_solver = Text("Equivariant\nSolver", font_size=14, color=YELLOW).move_to(box_solver)
        step4 = VGroup(box_solver, txt_solver)

        g_x0_pts = get_polymer_points("backbone", 0.45, ORIGIN)
        g_x0_dots = VGroup(*[Dot(g_x0_pts[i], color=AMINO_COLORS[i], radius=0.1) for i in range(8)])
        g_x0_edges = get_edges(g_x0_dots, is_closed=True)
        step5 = VGroup(g_x0_edges, g_x0_dots, MathTex(r"\hat{x}_0(x_t, t)", font_size=20).next_to(g_x0_dots, DOWN))

        micro_pipeline = VGroup(step1, step2, step3, step4, step5).arrange(RIGHT, buff=0.55).move_to(DOWN * 0.5)
        if micro_pipeline.width > config.frame_width - 1:
            micro_pipeline.scale_to_fit_width(config.frame_width - 1)

        arrows_micro = VGroup()
        for i in range(4):
            arrows_micro.add(Arrow(micro_pipeline[i].get_right(), micro_pipeline[i+1].get_left(), buff=0.15, color=WHITE))

        with self.voiceover(text="At step t, the noisy structure x t is passed through a GNN which predicts noise vectors. These are refined by an Equivariant Solver to estimate the original structure."):
            self.play(FadeIn(step1))
            for i in range(4):
                self.play(Create(arrows_micro[i]), FadeIn(micro_pipeline[i+1]))

        hl_box = SurroundingRectangle(VGroup(step2, step3, step4), color=YELLOW, buff=0.15)
        txt_hl = Text("Geometric Deep Learning Core", font_size=16, color=YELLOW).next_to(hl_box, UP)

        with self.voiceover(text="This central core leverages Geometric Deep Learning to ensure the denoising is robust to spatial orientation."):
            self.play(Create(hl_box), Write(txt_hl))

        self.play(FadeOut(Group(*[m for m in self.mobjects if m != title])))

        # ==========================================
        # STAGE 3: CONDITIONING
        # ==========================================
        stg3_title = Text("Mathematical Decomposition of Conditioning", font_size=20, color=WHITE).next_to(title, DOWN, buff=0.3)

        with self.voiceover(text="Stage 3: Understanding how we guide protein generation using specific conditions."):
            self.play(FadeIn(stg3_title))

        eq_bayes = MathTex(
            r"\log p_t(x | y)", r" \approx ", r"\log p_t(x)", r" + ", r"\log p_t(y | x)",
            font_size=36
        ).move_to(UP * 2.0)

        eq_bayes[0].set_color(YELLOW)
        eq_bayes[2].set_color(GREEN_C)
        eq_bayes[4].set_color(PINK)

        with self.voiceover(text="The goal is to generate a protein x that satisfies requirement y. This is a sum of the Prior model and the Conditioner."):
            self.play(Write(eq_bayes))

        desc_target = Text("Target: Generate Protein (x) satisfying (y).", font_size=18, color=YELLOW)
        desc_prior = Text("Prior: Ensures x is a natural, valid protein.", font_size=18, color=GREEN_C)
        desc_cond = Text("Conditioner: Pulls x toward the desired shape y.", font_size=18, color=PINK)

        explanations = VGroup(desc_target, desc_prior, desc_cond).arrange(DOWN, aligned_edge=LEFT, buff=0.5)
        explanations.next_to(eq_bayes, DOWN, buff=0.6).shift(LEFT * 0.5)

        with self.voiceover(text="The Prior handles natural valid protein structures, while the Conditioner ensures it meets our design constraints."):
            self.play(FadeIn(desc_target, shift=DOWN*0.2))
            self.play(Create(SurroundingRectangle(eq_bayes[2], color=GREEN_C)), FadeIn(desc_prior, shift=DOWN*0.2))
            self.play(Create(SurroundingRectangle(eq_bayes[4], color=PINK)), FadeIn(desc_cond, shift=DOWN*0.2))

        cond_box = Rectangle(width=7.5, height=1.6, color=BLUE_C, stroke_width=2).next_to(explanations, DOWN, buff=0.7)
        txt_cond = Text("Possible Guidance Conditions (y):", font_size=16, color=BLUE_C).next_to(cond_box, UP, buff=0.1)

        cond_items = VGroup(
            Text("- Symmetry: Spatial symmetry constraints", font_size=16),
            Text("- Substructure: Fixing part of the original structure", font_size=16),
            Text("- Shape: Requiring the protein to fit a 3D hull", font_size=16)
        ).arrange(DOWN, aligned_edge=LEFT, buff=0.15).move_to(cond_box)

        with self.voiceover(text="Conditions can include symmetry, specific substructures, or global shape requirements."):
            self.play(Create(cond_box), Write(txt_cond))
            self.play(LaggedStart(*[FadeIn(item, shift=RIGHT*0.2) for item in cond_items], lag_ratio=0.3))

        self.wait(2)
        self.play(FadeOut(Group(*self.mobjects)))


## slide 49 Representation theory
Tensor methods I

In [ ]:
%%manim -qh -v WARNING TensorMethodsI49

# ─── MAIN SCENE GRAPH ━━━━━━━━━━━━━━━━━━━━━━━━━━━
class TensorMethodsI49(MovingCameraScene, VoiceoverScene):
    def construct(self):
        self.camera.background_color = BLACK
        self.set_speech_service(EdgeTTSService(voice="en-US-GuyNeural"))

        # Khởi tạo tiêu đề nền tảng
        main_title = Text("Tensor Method I", font_size=36, weight=BOLD).to_edge(UP)
        subtitle = Text("The Power of Standard MLPs", font_size=24, color=YELLOW).next_to(main_title, DOWN, buff=0.2)

        # SỬA LỖI: Ghim chặt tiêu đề vào camera frame để tự động chạy theo hướng di chuyển của camera
        title_group = VGroup(main_title, subtitle)
        self.camera.frame.add(title_group)

        # ==========================================
        # PART 1: INTUITION OF STANDARD MLP POWER
        # ==========================================
        in_layer = VGroup(*[Circle(radius=0.25, color=BLUE_B, fill_opacity=0.2) for _ in range(3)]).arrange(DOWN, buff=0.8).move_to(LEFT * 2.5 + DOWN * 0.5)
        hidden_layer = VGroup(*[Circle(radius=0.25, color=BLUE_C, fill_opacity=0.2) for _ in range(3)]).arrange(DOWN, buff=0.8).move_to(DOWN * 0.5)
        out_layer = VGroup(*[Circle(radius=0.25, color=BLUE_D, fill_opacity=0.2) for _ in range(3)]).arrange(DOWN, buff=0.8).move_to(RIGHT * 2.5 + DOWN * 0.5)
        nn_nodes = VGroup(in_layer, hidden_layer, out_layer)

        free_colors = [RED, ORANGE, YELLOW, GREEN, TEAL, PURPLE, PINK, MAROON_A, BLUE_A]
        edges_l1 = always_redraw(lambda: VGroup(*[
            Line(n1.get_center(), n2.get_center(), stroke_width=2.5, color=free_colors[idx])
            for idx, (n1, n2) in enumerate([(n1, n2) for n1 in in_layer for n2 in hidden_layer])
        ]))
        edges_l2 = always_redraw(lambda: VGroup(*[
            Line(n2.get_center(), n3.get_center(), stroke_width=2.0, color=GRAY_D, stroke_opacity=0.6)
            for n2 in hidden_layer for n3 in out_layer
        ]))

        with self.voiceover(text="Let's begin with our primary object: a standard Multi-Layer Perceptron. Here, every single connection weight is completely independent."):
            self.play(Write(main_title), Write(subtitle))
            self.play(FadeIn(nn_nodes, scale=1.2), Create(edges_l1), Create(edges_l2))

        with self.voiceover(text="With this dense web of unique parameters, signals can flow freely, allowing the model to approximate virtually any complex function."):
            pulses = VGroup(*[Dot(in_layer[i].get_center(), color=YELLOW, radius=0.15) for i in range(3)])
            self.play(FadeIn(pulses))
            self.play(
                pulses[0].animate.move_to(hidden_layer[1].get_center()),
                pulses[1].animate.move_to(hidden_layer[0].get_center()),
                pulses[2].animate.move_to(hidden_layer[2].get_center()),
                run_time=0.6, rate_func=linear
            )
            self.play(
                pulses[0].animate.move_to(out_layer[0].get_center()),
                pulses[1].animate.move_to(out_layer[2].get_center()),
                pulses[2].animate.move_to(out_layer[1].get_center()),
                run_time=0.6, rate_func=linear
            )
            self.play(FadeOut(pulses))

        # ==========================================
        # PART 2: THE TRUE NATURE OF EQUIVARIANCE
        # ==========================================
        self.play(Transform(subtitle, Text("The True Nature of Equivariance", font_size=24, color=YELLOW).next_to(main_title, DOWN, buff=0.2)))

        with self.voiceover(text="But what happens when we demand geometric structure? Let's paint the top neuron red and the bottom neuron blue."):
            self.play(
                in_layer[0].animate.set_color(RED).set_fill(RED, opacity=0.6),
                in_layer[2].animate.set_color(BLUE).set_fill(BLUE, opacity=0.6),
                out_layer[0].animate.set_color(RED).set_fill(RED, opacity=0.6),
                out_layer[2].animate.set_color(BLUE).set_fill(BLUE, opacity=0.6),
            )

        with self.voiceover(text="Equivariance strictly dictates a structural law: if we swap the positions of these input elements..."):
            self.play(Swap(in_layer[0], in_layer[2], path_arc=PI/2), run_time=1.5)

        with self.voiceover(text="The internal transformations must commute the geometry perfectly, forcing the corresponding outputs to swap in the exact same manner."):
            wave = Dot(in_layer[2].get_center(), color=WHITE, radius=0.18)
            self.play(FadeIn(wave))
            self.play(wave.animate.move_to(hidden_layer[1].get_center()), run_time=0.4)
            self.play(wave.animate.move_to(out_layer[0].get_center()), run_time=0.4)
            self.play(FadeOut(wave))
            self.play(Swap(out_layer[0], out_layer[2], path_arc=-PI/2), run_time=1.5)

        # Trả lại góc hoán vị về ban đầu để không làm lệch logic của Part 3
        self.play(Swap(in_layer[0], in_layer[2], path_arc=-PI/2), Swap(out_layer[0], out_layer[2], path_arc=PI/2), run_time=0.5)

        self.play(
            in_layer[0].animate.set_color(BLUE_B).set_fill(opacity=0.2),
            in_layer[2].animate.set_color(BLUE_B).set_fill(opacity=0.2),
            out_layer[0].animate.set_color(BLUE_D).set_fill(opacity=0.2),
            out_layer[2].animate.set_color(BLUE_D).set_fill(opacity=0.2),
        )

        # ==========================================
        # PART 3: THE COST OF SYMMETRY (MATRIX COLLAPSE)
        # ==========================================
        self.play(Transform(subtitle, Text("The Cost of Symmetry (Matrix Collapse)", font_size=24, color=YELLOW).next_to(main_title, DOWN, buff=0.2)))
        self.play(self.camera.frame.animate.move_to(RIGHT * 1.5 + DOWN * 0.5), run_time=1.5)

        mat_free = Matrix([["1.2", "-0.8", "3.4"], ["0.5", "2.1", "-1.1"], ["-2.3", "0.9", "1.7"]]).scale(0.75).move_to(RIGHT * 5.0 + DOWN * 0.5)
        mat_lbl = Text("Weight Matrix Space", font_size=16, color=LIGHT_GRAY).next_to(mat_free, UP, buff=0.3)

        with self.voiceover(text="To maintain this flawless symmetry, we must force the network weights to satisfy the equivariant algebraic constraint."):
            self.play(Write(mat_free), Write(mat_lbl))

        constraint = MathTex(r"\text{Constraint: } R W = W R", font_size=28, color=RED_C).move_to(RIGHT * 5.0 + DOWN * 2.5)
        self.play(FadeIn(constraint, shift=UP*0.2))

        edges_l1.clear_updaters()
        collapsed_edges = VGroup()
        for i, n1 in enumerate(in_layer):
            for j, n2 in enumerate(hidden_layer):
                if i == j: collapsed_edges.add(Line(n1.get_center(), n2.get_center(), stroke_width=4.0, color=GREEN_C))
                else: collapsed_edges.add(Line(n1.get_center(), n2.get_center(), stroke_width=0.5, color=DARK_GREY).set_opacity(0.15))

        mat_collapse = Matrix([[r"\alpha", "0", "0"], ["0", r"\alpha", "0"], ["0", "0", r"\alpha"]]).scale(0.75).move_to(RIGHT * 5.0 + DOWN * 0.5)
        for i in range(3):
            for j in range(3): mat_collapse.get_entries()[i*3 + j].set_color(GREEN_C if i == j else DARK_GREY)

        with self.voiceover(text="By Clebsch-Gordan decomposition, this constraint instantly strips away the matrix's freedom. Look at the network: all cross connections vanish, and the remaining parameters are locked into a single scalar multiplier."):
            lock_txt = Text("EXPRESSIVITY COLLAPSED", font_size=20, color=RED_C, weight=BOLD).move_to(RIGHT * 5.0 + DOWN * 3.2)
            self.play(ReplacementTransform(edges_l1, collapsed_edges), ReplacementTransform(mat_free, mat_collapse), run_time=2.5)
            self.play(Write(lock_txt))
            self.play(Flash(mat_collapse, color=RED, line_length=0.25))

        self.play(FadeOut(mat_collapse), FadeOut(mat_lbl), FadeOut(constraint), FadeOut(lock_txt))

        # ==========================================
        # PART 4: TENSOR PRODUCTS (RESTORING EXPRESSIVITY)
        # ==========================================
        self.play(Transform(subtitle, Text("Tensor Products: Restoring Expressivity", font_size=24, color=YELLOW).next_to(main_title, DOWN, buff=0.2)))
        self.play(self.camera.frame.animate.scale(1.2).move_to(LEFT * 0.5 + DOWN * 0.5), run_time=1.5)

        v_vector = VGroup(*[VGroup(Circle(radius=0.18, color=BLUE_B, fill_opacity=0.6), MathTex(f"x_{i+1}", font_size=14)) for i in range(3)]).arrange(DOWN, buff=0.45).move_to(LEFT * 6.2 + DOWN * 0.5)
        lbl_v_vec = MathTex("X", font_size=24, color=BLUE_B).next_to(v_vector, LEFT, buff=0.2)
        h_vector = VGroup(*[VGroup(Circle(radius=0.18, color=BLUE_B, fill_opacity=0.6), MathTex(f"x_{j+1}", font_size=14)) for j in range(3)]).arrange(RIGHT, buff=0.45).move_to(LEFT * 4.0 + UP * 1.5)
        lbl_h_vec = MathTex("X^\top", font_size=24, color=BLUE_B).next_to(h_vector, UP, buff=0.2)

        with self.voiceover(text="Instead of processing individual elements separately, we duplicate the input vector into a row and a column representation."):
            self.play(TransformFromCopy(in_layer, v_vector), Write(lbl_v_vec))
            self.play(TransformFromCopy(v_vector, h_vector), Write(lbl_h_vec), run_time=1.5)

        tensor_grid = VGroup()
        grid_colors = [BLUE_C, GREEN_C, YELLOW, RED_C, PURPLE, ORANGE, TEAL, PINK, GOLD]
        for r in range(3):
            for c in range(3):
                sq = Square(side_length=0.55, fill_opacity=0.0, stroke_color=GRAY_D, stroke_width=0.5)
                sq.move_to([h_vector[c].get_x(), v_vector[r].get_y(), 0])
                # SỬA LỖI: gán label đúng vào tọa độ tâm của từng ô vuông thay vì để mặc định ở gốc tọa độ
                lbl = MathTex(f"x_{r+1}x_{c+1}", font_size=20, opacity=0).move_to(sq.get_center())
                tensor_grid.add(VGroup(sq, lbl))

        self.play(Create(VGroup(*[cell[0] for cell in tensor_grid])), run_time=1)

        with self.voiceover(text="Watch how the tensor product unfolds. As the row values multiply across the column values, their pairwise interactions populate a new 2D feature matrix."):
            for r in range(3):
                row_glow = SurroundingRectangle(v_vector[r], color=YELLOW, buff=0.04)
                self.play(Create(row_glow), run_time=0.2)
                for c in range(3):
                    col_glow = SurroundingRectangle(h_vector[c], color=YELLOW, buff=0.04)
                    idx = r * 3 + c
                    cell_sq, cell_lbl = tensor_grid[idx]
                    self.play(cell_sq.animate.set_fill(grid_colors[idx], opacity=0.85), cell_lbl.animate.set_opacity(1.0), FadeIn(col_glow), run_time=0.15)
                    self.play(FadeOut(col_glow), run_time=0.05)
                self.play(FadeOut(row_glow), run_time=0.1)

        self.play(
            FadeOut(v_vector), FadeOut(lbl_v_vec), FadeOut(h_vector), FadeOut(lbl_h_vec),
            tensor_grid.animate.scale(0.7).next_to(in_layer, LEFT, buff=0.8),
            run_time=1.5
        )

        restored_edges = VGroup()
        idx = 0
        for sq in tensor_grid:
            for n2 in hidden_layer:
                restored_edges.add(Line(sq[0].get_center(), n2.get_center(), stroke_width=1.5, color=grid_colors[idx % 9]))
                idx += 1

        with self.voiceover(text="From this rich multi-dimensional space, independent parameters explode back into existence. We have restored complete universality while fully honoring our symmetry rules."):
            self.play(FadeOut(collapsed_edges))
            self.play(Create(restored_edges, lag_ratio=0.01), run_time=2.5)
            insight_txt = Text("Universality Restored!", font_size=20, color=GREEN_C).to_edge(DOWN, buff=0.4)
            self.play(Write(insight_txt))
            self.wait(1.5)

        self.play(FadeOut(Group(*self.mobjects)), self.camera.frame.animate.scale(1/1.2).move_to(ORIGIN))


## slide 50
Tensor methods II

In [ ]:
%%manim -qh -v WARNING TensorMethodsII50

class TensorMethodsII50(VoiceoverScene):
    def construct(self):
        self.camera.background_color = BLACK

        # Khởi tạo dịch vụ tạo giọng đọc tiếng Anh
        self.set_speech_service(EdgeTTSService(voice="en-US-GuyNeural"))

        # ===================================================================
        # SCENE 1: THE CONTEXT & TENSOR POWER
        # ===================================================================
        main_title1 = Text("Tensor Method II", font_size=36, weight=BOLD).to_edge(UP)
        title1 = Text("Tensor Methods in Geometric Deep Learning", font_size=28, color=YELLOW).next_to(main_title1, DOWN)

        with self.voiceover(text="Next to Tensor Methods Part Two. Let's briefly recap how tensors expand our feature space."):
            self.play(Write(main_title1), Write(title1))

        # Khởi tạo Vector 1D chuyên nghiệp (Node Column)
        vec_1d_nodes = VGroup(*[Circle(radius=0.15, fill_color=BLUE_C, fill_opacity=0.8, stroke_color=WHITE, stroke_width=1) for _ in range(4)]).arrange(DOWN, buff=0.3)
        vec_1d_lines = VGroup(*[Line(vec_1d_nodes[i].get_center(), vec_1d_nodes[i+1].get_center(), stroke_width=2, color=GRAY) for i in range(3)])
        vector_x = VGroup(vec_1d_lines, vec_1d_nodes).move_to(ORIGIN)

        label_x = MathTex(r"x \in \mathbb{R}^d").next_to(vector_x, UP, buff=0.5)

        with self.voiceover(text="We start with a standard 1D feature vector x."):
            self.play(FadeIn(vector_x, shift=UP*0.5), FadeIn(label_x))
            self.wait(0.5)

        # Lưới ma trận tính năng 2D (2D Neural Mesh)
        grid_2d_nodes = VGroup(*[Circle(radius=0.1, fill_color=TEAL, fill_opacity=0.8, stroke_color=WHITE, stroke_width=0.5) for _ in range(16)]).arrange_in_grid(rows=4, cols=4, buff=0.4)
        grid_2d_lines = VGroup()
        for r in range(4):
            for c in range(4):
                if c < 3: grid_2d_lines.add(Line(grid_2d_nodes[r*4+c].get_center(), grid_2d_nodes[r*4+c+1].get_center(), color=TEAL_E, stroke_width=1.5))
                if r < 3: grid_2d_lines.add(Line(grid_2d_nodes[r*4+c].get_center(), grid_2d_nodes[(r+1)*4+c].get_center(), color=TEAL_E, stroke_width=1.5))
        grid_2d = VGroup(grid_2d_lines, grid_2d_nodes).move_to(ORIGIN)

        label_x2 = MathTex(r"x^{\otimes 2}").move_to(label_x.get_center())

        with self.voiceover(text="Taking the tensor product squares the space into a 2D matrix, capturing pairwise interactions."):
            self.play(ReplacementTransform(vector_x, grid_2d), Transform(label_x, label_x2), run_time=1.5)
            self.wait(0.5)

        # Tinh thể Lưới Tensor 3D (3D Tensor Lattice)
        tensor_cube = VGroup()
        for depth in range(3):
            layer_nodes = VGroup(*[Circle(radius=0.06, fill_color=BLUE_D, fill_opacity=0.9, stroke_width=0) for _ in range(16)]).arrange_in_grid(rows=4, cols=4, buff=0.3)
            layer_lines = VGroup()
            for r in range(4):
                for c in range(4):
                    if c < 3: layer_lines.add(Line(layer_nodes[r*4+c].get_center(), layer_nodes[r*4+c+1].get_center(), color=BLUE_E, stroke_width=1))
                    if r < 3: layer_lines.add(Line(layer_nodes[r*4+c].get_center(), layer_nodes[(r+1)*4+c].get_center(), color=BLUE_E, stroke_width=1))
            layer = VGroup(layer_lines, layer_nodes).shift(UR * depth * 0.25)
            tensor_cube.add(layer)

        # Đảo ngược thứ tự để layer đằng trước không bị che
        tensor_cube = VGroup(*reversed(tensor_cube.submobjects)).move_to(ORIGIN)
        label_xk = MathTex(r"x^{\otimes k} \in (\mathbb{R}^d)^k").move_to(label_x.get_center())

        with self.voiceover(text="Repeating this process K times creates a massive, high-dimensional tensor cube."):
            self.play(ReplacementTransform(grid_2d, tensor_cube), Transform(label_x, label_xk), run_time=2)
            self.wait(1)

        # Chuyển cảnh
        self.play(*[FadeOut(m) for m in self.mobjects])
        self.wait(0.5)

        # ===================================================================
        # SCENE 2: EQUIVARIANCE & UNIVERSALITY (MATH VISUALIZATION)
        # ===================================================================
        main_title2 = Text("Tensor Method II", font_size=36, weight=BOLD).to_edge(UP)
        title2 = Text("Equivariance & Universality", font_size=28, color=YELLOW).next_to(main_title2, DOWN)
        self.add(main_title2, title2)

        input_label = Text("Input Tensor Space", font_size=20).shift(LEFT * 4.5 + UP * 2)

        # 1. TENSOR INPUT LÀ MỘT MA TRẬN ĐIỂM (POINT CLOUD MATRIX)
        tensor_in = VGroup(*[
            Dot(radius=0.08, color=interpolate_color(BLUE_C, TEAL_C, np.random.rand()))
            for _ in range(36)
        ]).arrange_in_grid(rows=6, cols=6, buff=0.2).move_to(LEFT * 4.5)

        # 2. MLP KHÔNG CÒN LÀ HỘP, MÀ LÀ MỘT MẠNG NƠ-RON THỰC SỰ
        mlp_label = Text("Equivariant MLP", font_size=20, color=GREEN_A).shift(UP * 2)
        mlp_layers = [4, 5, 4]
        mlp_neurons = VGroup()
        mlp_edges = VGroup()

        for i, num in enumerate(mlp_layers):
            layer = VGroup(*[Circle(radius=0.1, fill_color=GREEN_E, fill_opacity=0.8, stroke_color=GREEN_C, stroke_width=1) for _ in range(num)])
            layer.arrange(DOWN, buff=0.3).move_to(LEFT * 0.8 + RIGHT * 1.5 * i)
            mlp_neurons.add(layer)

        for i in range(len(mlp_layers) - 1):
            for n1 in mlp_neurons[i]:
                for n2 in mlp_neurons[i+1]:
                    mlp_edges.add(Line(n1.get_center(), n2.get_center(), stroke_width=1, color=DARK_GREY))

        mlp_group = VGroup(mlp_edges, mlp_neurons).move_to(ORIGIN)

        output_label = Text("Output Target Function", font_size=20).shift(RIGHT * 4.5 + UP * 2)
        polar_grid = PolarPlane(radius_max=1.5, azimuth_step=6, color=GRAY_E).move_to(RIGHT * 4.5)

        output_shape = ParametricFunction(
            lambda t: np.array([1.2 * np.cos(3*t) * np.cos(t), 1.2 * np.cos(3*t) * np.sin(t), 0]),
            t_range=[0, PI], color=PURPLE_B, stroke_width=4
        ).move_to(RIGHT * 4.5)

        with self.voiceover("When we feed these high-dimensional tensors into an Equivariant Multi-Layer Perceptron..."):
            self.play(FadeIn(tensor_in), Write(input_label))
            self.play(Write(mlp_label), Create(mlp_edges, lag_ratio=0.1), FadeIn(mlp_neurons, lag_ratio=0.1), run_time=1.5)
            self.play(FadeIn(polar_grid), Create(output_shape), Write(output_label))

        sync_text = Text("Equivariance: Geometry is strictly preserved", font_size=22, color=ORANGE).to_edge(DOWN, buff=1.0)

        with self.voiceover("...the network preserves symmetry perfectly. Notice how rotating the input point cloud dictates a perfectly synchronized rotation in the output function."):
            self.play(Write(sync_text))

            # Sóng kích hoạt truyền qua MLP để mô phỏng "đang xử lý"
            self.play(
                Rotate(tensor_in, angle=PI/2, run_time=2, rate_func=smooth),
                LaggedStart(*[e.animate.set_color(ORANGE).set_stroke(width=2) for e in mlp_edges], lag_ratio=0.01, run_time=1),
                Rotate(output_shape, angle=PI/2, run_time=2, rate_func=smooth)
            )
            # Khôi phục màu MLP
            self.play(mlp_edges.animate.set_color(DARK_GREY).set_stroke(width=1), run_time=0.5)
            self.play(FadeOut(sync_text))

        uni_text = Text("Universality: Linear combinations of basis functions", font_size=22, color=YELLOW).to_edge(DOWN, buff=1.0)

        # 3. CHUYỂN TENSOR THÀNH CÁC CỤM ĐIỂM (BASIS CLOUDS) ĐỂ CHUẨN BỊ XẤP XỈ
        basis_clouds = VGroup()
        for _ in range(8):
            cloud = VGroup(*[Dot(radius=0.06, color=TEAL_C) for _ in range(4)]).arrange_in_grid(rows=2, buff=0.1)
            cloud.move_to(tensor_in.get_center() + np.array([np.random.uniform(-1.0, 1.0), np.random.uniform(-1.0, 1.0), 0]))
            basis_clouds.add(cloud)

        with self.voiceover("Furthermore, this method guarantees Universality. We can decompose the input space into fundamental, orthogonal basis features."):
            self.play(Write(uni_text))
            self.play(ReplacementTransform(tensor_in, basis_clouds), run_time=1.5)

        target_shapes = [
            ParametricFunction(lambda t: np.array([1.2 * (1 - np.sin(t)) * np.cos(t), 1.2 * (1 - np.sin(t)) * np.sin(t), 0]) + UP*0.5, t_range=[0, TAU], color=RED_C, stroke_width=4).move_to(RIGHT * 4.5),
            ParametricFunction(lambda t: np.array([1.2 * np.sin(4*t), 1.2 * np.sin(3*t), 0]), t_range=[0, TAU], color=GOLD, stroke_width=4).move_to(RIGHT * 4.5)
        ]

        with self.voiceover("By feeding enough of these tensor components into the network, it can mathematically approximate literally ANY continuous equivariant function."):
            for shape in target_shapes:
                # Mô phỏng các cụm hạt "bị hút" vào mạng nơ-ron
                self.play(
                    basis_clouds.animate.scale(0.2).move_to(mlp_neurons[0].get_center()).set_opacity(0),
                    run_time=0.8, rate_func=rush_into
                )

                # MLP nhấp nháy bùng nổ năng lượng
                self.play(
                    *[n.animate.set_fill(YELLOW, opacity=1).set_stroke(YELLOW_A) for layer in mlp_neurons for n in layer],
                    mlp_edges.animate.set_color(YELLOW).set_stroke(width=2),
                    run_time=0.3
                )

                # Xuất ra hàm mới
                self.play(
                    ReplacementTransform(output_shape, shape),
                    *[n.animate.set_fill(GREEN_E, opacity=0.8).set_stroke(GREEN_C) for layer in mlp_neurons for n in layer],
                    mlp_edges.animate.set_color(DARK_GREY).set_stroke(width=1),
                    run_time=1.2
                )
                output_shape = shape

                # Tái tạo lại Basis Clouds cho chu kỳ sau
                basis_clouds.set_opacity(1).scale(5)
                for cloud in basis_clouds:
                    cloud.move_to(LEFT * 4.5 + np.array([np.random.uniform(-1.0, 1.0), np.random.uniform(-1.0, 1.0), 0]))
                self.wait(0.5)

        final_text = Text("With infinite tensor order (k → ∞), the model capacity spans the entire function space.", font_size=20, color=WHITE).to_edge(DOWN, buff=1.0)

        # 4. ĐÁM MÂY ĐIỂM DÀY ĐẶC (DENSE CLOUD) TƯỢNG TRƯNG CHO DUNG LƯỢNG VÔ HẠN
        dense_cloud = VGroup(*[
            Dot(radius=0.03, color=BLUE_B, fill_opacity=0.6).move_to(LEFT * 4.5 + np.array([np.random.uniform(-1.5, 1.5), np.random.uniform(-1.5, 1.5), 0]))
            for _ in range(250)
        ])

        with self.voiceover("Theoretically, as the tensor order K approaches infinity, the parameter space becomes dense enough to model anything."):
            self.play(FadeOut(basis_clouds), FadeOut(uni_text), Write(final_text))
            self.play(FadeIn(dense_cloud, lag_ratio=0.01), run_time=2.5)

        # Chuyển cảnh sang Scene 3
        self.play(*[FadeOut(m) for m in self.mobjects])
        self.wait(0.5)

        # ===================================================================
        # SCENE 3: THE CHALLENGE OF TENSOR SPACE (EXPONENTIAL EXPLOSION)
        # ===================================================================
        main_title3 = Text("Tensor Method II", font_size=36, weight=BOLD).to_edge(UP)
        title3 = Text("The Challenge of Tensor Space", font_size=28, color=YELLOW).next_to(main_title3, DOWN)
        self.add(main_title3, title3)

        dim_text = MathTex(r"\text{Dimension of } (\mathbb{R}^d)^k \text{ scales as } d^k", font_size=32).next_to(title3, DOWN)

        with self.voiceover(text="But there's a catch. The dimension of this tensor space scales exponentially, as d to the power of k."):
            self.play(Write(dim_text))

        grid_k1 = VGroup(*[Square(side_length=0.4, fill_opacity=0.6, color=BLUE) for _ in range(3)]).arrange(RIGHT, buff=0.1)
        label_k1 = MathTex(r"k=1 \Rightarrow d^1 = 3 \text{ (Lightweight)}", font_size=28).next_to(grid_k1, DOWN)
        group_k1 = VGroup(grid_k1, label_k1).move_to(LEFT * 4 + UP * 1)

        with self.voiceover(text="For k equals one, it's lightweight."):
            self.play(FadeIn(grid_k1, lag_ratio=0.1), Write(label_k1))

        grid_k2 = VGroup(*[Square(side_length=0.2, fill_opacity=0.6, color=TEAL) for _ in range(9)]).arrange_in_grid(rows=3, buff=0.05)
        label_k2 = MathTex(r"k=2 \Rightarrow d^2 = 9", font_size=28).next_to(grid_k2, DOWN)
        group_k2 = VGroup(grid_k2, label_k2).move_to(ORIGIN + UP * 1)

        with self.voiceover(text="For k equals two, the dimension squares."):
            self.play(TransformFromCopy(grid_k1, grid_k2), Write(label_k2))

        grid_k3 = VGroup(*[Square(side_length=0.08, fill_opacity=0.8, color=ORANGE) for _ in range(81)]).arrange_in_grid(rows=9, buff=0.02)
        label_k3 = MathTex(r"k=4 \Rightarrow d^4 = 81", font_size=28).next_to(grid_k3, DOWN)
        group_k3 = VGroup(grid_k3, label_k3).move_to(RIGHT * 4 + UP * 1)

        with self.voiceover(text="At k equals four, it explodes."):
            self.play(TransformFromCopy(grid_k2, grid_k3), Write(label_k3))

        self.play(FadeOut(group_k1), FadeOut(group_k2), FadeOut(label_k3), FadeOut(dim_text))

        massive_grid = VGroup(*[
            Square(side_length=0.15, fill_opacity=np.random.uniform(0.3, 1), color=RED, stroke_width=0.5)
            for _ in range(400)
        ]).arrange_in_grid(rows=20, buff=0.02).move_to(DOWN * 0.5)

        warning_text = Text("Large k: Massive & redundant space!", font_size=32, color=RED).next_to(massive_grid, UP, buff=0.5)

        with self.voiceover(text="A large K creates a massive, computational nightmare filled with redundant symmetries.") as tracker:
            self.play(ReplacementTransform(grid_k3, massive_grid), run_time=1.5)
            self.play(FadeIn(warning_text))

            for _ in range(4):
                self.play(massive_grid.animate.shift(RIGHT * 0.1).set_color(RED_A), run_time=0.1)
                self.play(massive_grid.animate.shift(LEFT * 0.2).set_color(RED_E), run_time=0.1)
                self.play(massive_grid.animate.shift(RIGHT * 0.1), run_time=0.1)

        solution_text = Text("Solution: Decompose into independent blocks", font_size=32, color=GREEN).next_to(massive_grid, UP, buff=0.5)

        with self.voiceover(text="The mathematical solution is to decompose this giant matrix into irreducible, independent blocks using Clebsch-Gordan decomposition."):
            self.play(ReplacementTransform(warning_text, solution_text))
            self.play(massive_grid.animate.set_opacity(0.1).set_color(GRAY), run_time=1.5)

            block_1 = Rectangle(width=1.5, height=1.5, color=GREEN, fill_opacity=0.8).move_to(massive_grid.get_corner(UL) + DOWN*0.75 + RIGHT*0.75)
            block_2 = Rectangle(width=1, height=1, color=BLUE, fill_opacity=0.8).next_to(block_1, DR, buff=0)
            block_3 = Rectangle(width=0.6, height=0.6, color=YELLOW, fill_opacity=0.8).next_to(block_2, DR, buff=0)
            blocks = VGroup(block_1, block_2, block_3)

            self.play(FadeIn(blocks, shift=UP*0.5, lag_ratio=0.3), run_time=2)

        math_formula = MathTex(
            r"(\mathbb{R}^d)^k", r"\cong", r"\mathbf{V_1}", r"\oplus", r"\mathbf{V_2}", r"\oplus", r"\mathbf{V_3}", r"\oplus \dots"
        ).scale(1.2).next_to(massive_grid, DOWN, buff=0.5)

        math_formula[0].set_color(GRAY)
        math_formula[2].set_color(GREEN)
        math_formula[4].set_color(BLUE)
        math_formula[6].set_color(YELLOW)

        with self.voiceover(text="We project the tensor space into a direct sum of these smaller, orthogonal representations."):
            self.play(Write(math_formula))
            self.wait(1)

        clean_text = Text("Data is now compact and manageable!", font_size=28, color=WHITE).next_to(massive_grid, UP, buff=0.5)
        aligned_blocks = VGroup(
            Rectangle(width=1.5, height=0.5, color=GREEN, fill_opacity=0.8),
            Rectangle(width=1, height=0.5, color=BLUE, fill_opacity=0.8),
            Rectangle(width=0.6, height=0.5, color=YELLOW, fill_opacity=0.8)
        ).arrange(RIGHT, buff=0.2).move_to(ORIGIN)

        with self.voiceover(text="Now, the data is incredibly compact, redundancy-free, and computationally manageable."):
            self.play(FadeOut(massive_grid), ReplacementTransform(solution_text, clean_text))
            self.play(Transform(blocks, aligned_blocks), FadeOut(math_formula))
            self.wait(1)

        # Chuyển cảnh
        self.play(*[FadeOut(m) for m in self.mobjects])
        self.wait(0.5)

        # ===================================================================
        # SCENE 4: CONTINUOUS VS FINITE GROUPS
        # ===================================================================
        main_title4 = Text("Tensor Method II", font_size=36, weight=BOLD).to_edge(UP)
        title4 = Text("Continuous vs Finite Groups", font_size=28, color=YELLOW).next_to(main_title4, DOWN, buff=0.2)

        with self.voiceover(text="Finally, let's address a foundational dichotomy in geometric deep learning: Continuous versus Finite Groups."):
            self.play(Write(main_title4), Write(title4))

        # --- BÊN TRÁI: CONTINUOUS GROUPS SO(d) ---
        # Đã sửa lỗi tọa độ UP * 3 -> UP * 1.2 để vừa vặn trong khung hình
        lbl_cont = MathTex(r"\text{Continuous } SO(d)", font_size=28, color=TEAL).move_to(LEFT * 3.5 + UP * 1.2)
        desc_cont = Text("Smooth Spherical Harmonics", font_size=18, color=TEAL_A).next_to(lbl_cont, DOWN, buff=0.2)

        # Mô phỏng khối cầu 3D mượt mà (Pseudo-3D Sphere)
        sphere_bg = Circle(radius=1.2, color=TEAL_E, fill_opacity=0.2)
        orbit1 = Ellipse(width=2.4, height=0.6, color=TEAL_A, stroke_width=2)
        orbit2 = Ellipse(width=0.6, height=2.4, color=TEAL_C, stroke_width=2)
        orbit3 = Ellipse(width=2.0, height=2.0, color=TEAL_B, stroke_width=1).rotate(PI/4)

        sphere_group = VGroup(sphere_bg, orbit1, orbit2, orbit3).move_to(LEFT * 3.5 + DOWN * 1.0)

        with self.voiceover(text="For continuous symmetry groups, like 3D rotations, the transformations are buttery smooth."):
            self.play(Write(lbl_cont), FadeIn(desc_cont))
            self.play(FadeIn(sphere_bg), Create(orbit1), Create(orbit2), Create(orbit3))

            # Xoay liên tục, mượt mà (Linear/Smooth rotation)
            self.play(
                Rotate(orbit1, angle=PI, axis=RIGHT),
                Rotate(orbit2, angle=PI, axis=UP),
                run_time=2.5, rate_func=linear
            )

        # Mô phỏng Sóng điều hòa cầu (Basis Functions) bứt ra từ khối cầu
        harmonics = VGroup(*[
            ParametricFunction(lambda t, f=f: np.array([t, 0.3 * np.sin(f * t), 0]), t_range=[-1.2, 1.2], color=TEAL_B, stroke_width=3)
            for f in [3, 6, 9]
        ]).arrange(DOWN, buff=0.3).move_to(LEFT * 3.5 + DOWN * 1.0)

        with self.voiceover(text="We can elegantly decompose this continuous space using well-understood analytical functions, known as spherical harmonics."):
            self.play(
                ReplacementTransform(sphere_group, harmonics),
                run_time=1.5
            )
            # Sóng điều hòa dao động nhẹ
            self.play(LaggedStart(*[h.animate.stretch(1.2, dim=1) for h in harmonics], lag_ratio=0.2), run_time=1, rate_func=there_and_back)

        # --- BÊN PHẢI: FINITE GROUPS G ---
        # Căn chỉnh đối xứng với bên trái
        lbl_fin = MathTex(r"\text{Arbitrary Finite } G", font_size=28, color=ORANGE).move_to(RIGHT * 3.5 + UP * 1.2)
        desc_fin = Text("Discrete Block Matrices", font_size=18, color=ORANGE).next_to(lbl_fin, DOWN, buff=0.2)

        # Đa giác mô phỏng nhóm rời rạc
        polygon = RegularPolygon(n=6, radius=1.2, color=ORANGE, fill_opacity=0.2, stroke_width=4).move_to(RIGHT * 3.5 + DOWN * 1.0)

        with self.voiceover(text="In stark contrast, arbitrary finite groups operate through rigid, discrete jumps. There is no smooth transition, only sudden permutations."):
            self.play(Write(lbl_fin), FadeIn(desc_fin))
            self.play(DrawBorderThenFill(polygon))

            # Hiệu ứng quay giật cục (Snapping)
            for _ in range(3):
                self.play(Rotate(polygon, angle=PI/3, about_point=polygon.get_center()), run_time=0.2, rate_func=rush_into)
                self.wait(0.3)

        # Trực quan hóa Ma trận rời rạc "bắn" ra từ đa giác
        discrete_matrices = VGroup()
        for _ in range(3):
            mat = VGroup(*[Square(side_length=0.2, fill_color=ORANGE, fill_opacity=0.8, stroke_width=1, stroke_color=WHITE) for _ in range(4)]).arrange_in_grid(rows=2, buff=0.05)
            discrete_matrices.add(mat)
        discrete_matrices.arrange(RIGHT, buff=0.4).move_to(RIGHT * 3.5 + DOWN * 1.0)

        with self.voiceover(text="Because of this jumpy nature, creating a generalized mathematical decomposition for finite groups requires entirely different, often combinatorial algebraic strategies."):
            # Thu nhỏ đa giác và biến thành các khối ma trận rời rạc
            self.play(
                polygon.animate.scale(0.5).set_opacity(0),
                FadeIn(discrete_matrices, shift=UP*0.5, lag_ratio=0.2),
                run_time=1.5
            )
            self.play(Flash(discrete_matrices, color=ORANGE, line_length=0.4))
            self.wait(1.5)

        # Kết thúc phân cảnh gọn gàng
        self.play(FadeOut(Group(*self.mobjects)))


## slide 51
Atomistic systems and geometric ML I

In [ ]:
%%manim -qh -v WARNING AtomisticGeoMLI51

class AtomisticGeoMLI51(VoiceoverScene):
    def construct(self):
        self.camera.background_color = BLACK

        # Khởi tạo dịch vụ tạo giọng đọc tiếng Anh
        self.set_speech_service(EdgeTTSService(voice="en-US-GuyNeural"))

        # ===================================================================
        # SCENE 0: MỞ ĐẦU
        # ===================================================================
        title = Text("Atomistic Systems and Geometric ML I", font_size=36, color=WHITE, weight=BOLD).to_edge(UP)

        with self.voiceover(text="We further explore how Geometric Machine Learning is revolutionizing our understanding of atomistic systems."):
            self.play(Write(title), run_time=1.5)

        chem_structure = RegularPolygon(n=6, radius=1.5, color=WHITE, stroke_width=2)
        vertices = chem_structure.get_vertices()
        inner_scale = 0.75
        chem_double_lines = VGroup(
            Line(vertices[0] * inner_scale, vertices[1] * inner_scale, color=WHITE, stroke_width=2),
            Line(vertices[2] * inner_scale, vertices[3] * inner_scale, color=WHITE, stroke_width=2),
            Line(vertices[4] * inner_scale, vertices[5] * inner_scale, color=WHITE, stroke_width=2)
        )
        traditional_chem = VGroup(chem_structure, chem_double_lines).shift(LEFT * 3)
        chem_text = Text("Chemical Structure", font_size=24, color=GRAY).next_to(traditional_chem, DOWN, buff=0.5)

        with self.voiceover(text="Traditionally, we represent molecules using 2D chemical structures."):
            self.play(FadeIn(traditional_chem), Write(chem_text), run_time=1.5)

        nodes = VGroup(*[
            Dot(point=v, radius=0.15, color=BLUE_D if i % 2 == 0 else TEAL_D)
            for i, v in enumerate(chem_structure.get_vertices())
        ])
        edges = VGroup(*[
            Line(nodes[i].get_center(), nodes[(i + 1) % 6].get_center(), stroke_width=5, color=GRAY_C)
            for i in range(6)
        ])
        graph = VGroup(edges, nodes).shift(LEFT * 1)
        graph_text = Text("Graph of Atoms", font_size=24, color=BLUE).next_to(graph, DOWN, buff=0.5)

        with self.voiceover(text="However, for machine learning, it is much more powerful to view them as 3D Graphs of Atoms, where nodes are atoms and edges represent their interactions."):
            self.play(
                ReplacementTransform(traditional_chem, graph),
                ReplacementTransform(chem_text, graph_text),
                run_time=2.0
            )
            self.play(graph.animate.shift(RIGHT * 7), graph_text.animate.shift(RIGHT * 7), run_time=1.5)

        # ===================================================================
        # SCENE 1: VẤN ĐỀ CỦA AI TRUYỀN THỐNG (MLP Fails)
        # ===================================================================
        self.play(*[FadeOut(m) for m in self.mobjects if m != title], run_time=0.5)

        # SETUP RIGHT: Thực tại vật lý
        target_benzene = RegularPolygon(n=6, radius=1.5, color=GREEN_C, stroke_width=4)
        vertices_t = target_benzene.get_vertices()
        target_lines = VGroup(
            Line(vertices_t[0] * inner_scale, vertices_t[1] * inner_scale, color=GREEN_C, stroke_width=3),
            Line(vertices_t[2] * inner_scale, vertices_t[3] * inner_scale, color=GREEN_C, stroke_width=3),
            Line(vertices_t[4] * inner_scale, vertices_t[5] * inner_scale, color=GREEN_C, stroke_width=3)
        )
        benzene_right = VGroup(target_benzene, target_lines).shift(RIGHT * 4 + DOWN * 0.5)
        right_label = Text("Physical Reality", font_size=20, color=GREEN_C).next_to(benzene_right, UP, buff=0.5)

        # SETUP MIDDLE: MLP
        mlp_layers = VGroup()
        for num_nodes in [3, 4, 3]:
            layer = VGroup(*[Dot(radius=0.15, color=GRAY) for _ in range(num_nodes)]).arrange(DOWN, buff=0.5)
            mlp_layers.add(layer)
        mlp_layers.arrange(RIGHT, buff=1.2).shift(DOWN * 0.5)

        mlp_edges = VGroup()
        for i in range(2):
            for n1 in mlp_layers[i]:
                for n2 in mlp_layers[i + 1]:
                    mlp_edges.add(Line(n1.get_center(), n2.get_center(), stroke_width=1.5, stroke_opacity=0.3, color=WHITE))

        mlp = VGroup(mlp_edges, mlp_layers)
        mlp_text = Text("MLP", font_size=20, color=WHITE).next_to(mlp, UP, buff=0.5)

        # SETUP LEFT: Input Data
        axes = Axes(
            x_range=[-3, 3, 1], y_range=[-3, 3, 1], x_length=4, y_length=4,
            axis_config={"color": GRAY_D, "stroke_width": 2}
        ).shift(LEFT * 4 + DOWN * 0.5)

        input_benzene = RegularPolygon(n=6, radius=1.2, color=BLUE_C, stroke_width=3).move_to(axes.c2p(0, 0))
        input_nodes = VGroup(*[Dot(point=v, radius=0.1, color=BLUE) for v in input_benzene.get_vertices()])
        node_labels = VGroup(*[
            Text(str(i), font_size=16, color=WHITE).move_to(
                input_nodes[i].get_center() + normalize(input_nodes[i].get_center() - axes.c2p(0, 0)) * 0.3
            )
            for i in range(6)
        ])
        input_group = VGroup(axes, input_benzene, input_nodes, node_labels)
        left_label = Text("Input Data", font_size=20, color=BLUE_C).next_to(axes, UP, buff=0.2)

        with self.voiceover(text="Why can't we just use a standard neural network, like an MLP, to process this data?"):
            self.play(
                FadeIn(benzene_right), Write(right_label),
                FadeIn(mlp), Write(mlp_text),
                FadeIn(input_group), Write(left_label),
                run_time=2.0
            )

        # ---- Hàm mô phỏng dòng chảy MLP ----
        def simulate_flow(success=True, message=""):
            status_text = Text(message, font_size=20, color=GREEN if success else RED).to_edge(DOWN, buff=1)
            self.play(Write(status_text), run_time=0.5)

            signals_in = VGroup(*[Dot(radius=0.08, color=YELLOW).move_to(input_benzene.get_center()) for _ in range(5)])
            self.play(*[sig.animate.move_to(mlp_layers[0][np.random.randint(0, 3)].get_center()) for sig in signals_in], run_time=0.8)
            self.remove(*signals_in)

            if success:
                self.play(mlp_edges.animate.set_color(GREEN).set_opacity(0.8), run_time=0.5)
                self.play(mlp_edges.animate.set_color(WHITE).set_opacity(0.3), run_time=0.5)
            else:
                self.play(mlp_edges.animate.set_color(RED).set_opacity(1), mlp_layers.animate.set_color(RED), run_time=0.3)
                self.play(mlp_edges.animate.set_color(WHITE).set_opacity(0.3), mlp_layers.animate.set_color(GRAY), run_time=0.3)

            signals_out = VGroup(*[
                Dot(radius=0.08, color=GREEN if success else RED).move_to(mlp_layers[-1][np.random.randint(0, 3)].get_center())
                for _ in range(3)
            ])
            self.play(*[sig.animate.move_to(benzene_right.get_center()) for sig in signals_out], run_time=0.8)
            self.remove(*signals_out)

            if success:
                check_mark = Text("✔ Match", font_size=16, color=GREEN).next_to(benzene_right, DOWN, buff=0.5)
                self.play(Write(check_mark), benzene_right.animate.set_color(GREEN_A), run_time=0.5)
                self.play(FadeOut(check_mark), benzene_right.animate.set_color(GREEN_C), FadeOut(status_text), run_time=0.5)
            else:
                cross_mark = Text("✘ Mismatch", font_size=16, color=RED).next_to(benzene_right, DOWN, buff=0.5)
                self.play(Write(cross_mark), benzene_right.animate.set_color(RED), run_time=0.5)
                self.play(FadeOut(cross_mark), benzene_right.animate.set_color(GREEN_C), FadeOut(status_text), run_time=0.5)

        # Baseline
        with self.voiceover(text="At first, the MLP simply memorizes the specific spatial layout of the training data."):
            simulate_flow(success=True, message="Baseline: MLP memorizes this specific layout.")

        # Translation Test
        with self.voiceover(text="But what happens if we shift the molecule?"):
            self.play(
                input_benzene.animate.shift(UP * 1.5 + RIGHT * 0.5),
                input_nodes.animate.shift(UP * 1.5 + RIGHT * 0.5),
                node_labels.animate.shift(UP * 1.5 + RIGHT * 0.5),
                run_time=1.0
            )

        with self.voiceover(text="The coordinates have changed completely. The MLP fails to recognize it as the same molecule."):
            simulate_flow(success=False, message="1. Translation: Coordinates changed. MLP fails!")

        self.play(
            input_benzene.animate.shift(DOWN * 1.5 + LEFT * 0.5),
            input_nodes.animate.shift(DOWN * 1.5 + LEFT * 0.5),
            node_labels.animate.shift(DOWN * 1.5 + LEFT * 0.5),
            run_time=0.5
        )

        # Rotation Test
        with self.voiceover(text="What if we rotate it?"):
            self.play(
                Rotate(input_benzene, angle=PI / 3, about_point=axes.c2p(0, 0)),
                Rotate(input_nodes, angle=PI / 3, about_point=axes.c2p(0, 0)),
                Rotate(node_labels, angle=PI / 3, about_point=axes.c2p(0, 0)),
                run_time=1.5
            )

        with self.voiceover(text="Physically it's identical, but mathematically the input matrix is rotated. The MLP fails again."):
            simulate_flow(success=False, message="2. Rotation: Identical molecule, but rotated matrix. MLP fails!")

        self.play(
            Rotate(input_benzene, angle=-PI / 3, about_point=axes.c2p(0, 0)),
            Rotate(input_nodes, angle=-PI / 3, about_point=axes.c2p(0, 0)),
            Rotate(node_labels, angle=-PI / 3, about_point=axes.c2p(0, 0)),
            run_time=0.5
        )

        # Permutation Test
        label_0, label_1 = node_labels[0], node_labels[1]

        with self.voiceover(text="And what if we simply swap the order we feed the atoms into the network?"):
            self.play(
                Swap(label_0, label_1, path_arc=PI / 2),
                input_nodes[0].animate.set_color(YELLOW),
                input_nodes[1].animate.set_color(YELLOW),
                run_time=1.0
            )

        with self.voiceover(text="The MLP treats permutation as completely different data."):
            simulate_flow(success=False, message="3. Permutation: Atom order swapped. MLP sees different data!")

        self.play(
            Swap(label_0, label_1, path_arc=-PI / 2),
            input_nodes[0].animate.set_color(BLUE),
            input_nodes[1].animate.set_color(BLUE),
            run_time=0.5
        )

        # ===================================================================
        # SCENE 2: HỌC MÁY HÌNH HỌC (Geometric ML) - NÂNG CẤP CHUYÊN NGHIỆP
        # ===================================================================
        # 1. TẠO LÕI GNN DẠNG ĐỒ THỊ TRUYỀN THÔNG ĐIỆP (MESSAGE PASSING CORE)
        geoml_box = RoundedRectangle(width=3.5, height=2.8, color=TEAL_E, fill_color=BLACK, fill_opacity=0.6, stroke_width=3)
        geoml_box.move_to(mlp.get_center())

        # Tạo mạng nơ-ron đồ thị thu nhỏ bên trong Box
        gnn_nodes = VGroup(*[Dot(radius=0.08, color=TEAL_C) for _ in range(6)])
        # Sắp xếp các nơ-ron thành một đồ thị mini
        gnn_positions = [UP*0.6, UR*0.4, DR*0.5, DOWN*0.6, DL*0.5, UL*0.4]
        for node, pos in zip(gnn_nodes, gnn_positions):
            node.move_to(geoml_box.get_center() + pos)

        gnn_edges = VGroup()
        for i in range(len(gnn_nodes)):
            for j in range(i + 1, len(gnn_nodes)):
                if np.random.rand() > 0.4: # Tạo kết nối ngẫu nhiên nhưng đẹp mắt
                    gnn_edges.add(Line(gnn_nodes[i].get_center(), gnn_nodes[j].get_center(), stroke_width=1.5, color=TEAL_E))

        gnn_core = VGroup(gnn_edges, gnn_nodes)
        geoml_group = VGroup(geoml_box, gnn_core)

        geoml_label = Text("Geometric ML (Message Passing)", font_size=20, color=TEAL_A).next_to(geoml_box, UP, buff=0.2)

        with self.voiceover(text="This is why we need Geometric Machine Learning. By replacing the dense MLP with an Equivariant Graph Neural Network, we bake physical symmetries directly into the internal architecture."):
            self.play(
                ReplacementTransform(mlp, geoml_box),
                FadeIn(gnn_core, lag_ratio=0.1),
                ReplacementTransform(mlp_text, geoml_label),
                run_time=2
            )

        # ---- HÀM MÔ PHỎNG DÒNG CHẢY TOÁN HỌC (MESSAGE PASSING FLOW) ----
        def message_passing_flow(message="", success=True):
            status_text = Text(message, font_size=18, color=YELLOW).to_edge(DOWN, buff=1)
            self.play(Write(status_text), run_time=0.5)

            # Bước 1: Trích xuất đặc trưng cục bộ (Ripples từ Input Nodes)
            ripples = VGroup(*[Circle(radius=0.1, color=TEAL_C, stroke_width=3).move_to(n.get_center()) for n in input_nodes])
            self.play(ripples.animate.scale(4).set_opacity(0), run_time=1)

            # Bước 2: Dòng dữ liệu hội tụ về GNN
            data_stream = VGroup(*[Line(n.get_center(), geoml_box.get_center(), color=TEAL_B, stroke_width=2) for n in input_nodes])
            self.play(ShowPassingFlash(data_stream, time_width=0.5), run_time=0.8)

            # Bước 3: GNN tính toán (Cập nhật đồ thị)
            self.play(
                gnn_edges.animate.set_color(YELLOW).set_stroke(width=3),
                gnn_nodes.animate.set_color(WHITE).scale(1.5),
                run_time=0.4
            )
            self.play(
                gnn_edges.animate.set_color(TEAL_E).set_stroke(width=1.5),
                gnn_nodes.animate.set_color(TEAL_C).scale(1/1.5),
                run_time=0.4
            )

            # Bước 4: Đồng bộ đầu ra
            out_stream = VGroup(*[Line(geoml_box.get_center(), benzene_right.get_center(), color=GREEN_C, stroke_width=3)])
            self.play(ShowPassingFlash(out_stream, time_width=0.5), run_time=0.6)

            if success:
                check = MathTex(r"\checkmark", color=GREEN).scale(1.5).next_to(benzene_right, UP, buff=0.3)
                self.play(Write(check), benzene_right.animate.set_color(GREEN_B), run_time=0.5)
                self.play(FadeOut(check), benzene_right.animate.set_color(GREEN_E), FadeOut(status_text), run_time=0.5)

        # Baseline Test
        with self.voiceover(text="A Graph Neural Network operates by passing messages between connected atoms, inherently understanding the underlying structure."):
            message_passing_flow("Baseline: Message passing reads the graph structure.")

        # --- BÀI TEST 1: TRANSLATION (MINH HỌA BẰNG VECTOR TƯƠNG ĐỐI) ---
        with self.voiceover(text="When the molecule is translated through space..."):
            # Vẽ vector khoảng cách tương đối r_ij để giải thích toán học
            vec_rij = Arrow(input_nodes[0].get_center(), input_nodes[1].get_center(), buff=0, color=YELLOW, stroke_width=3, max_tip_length_to_length_ratio=0.15)
            math_rij = MathTex(r"\vec{r}_{ij} = x_i - x_j", font_size=24, color=YELLOW).next_to(vec_rij, LEFT)

            self.play(Create(vec_rij), Write(math_rij))

            # Tịnh tiến toàn bộ phân tử kèm theo Vector (Vector không đổi hướng/độ dài)
            self.play(
                VGroup(input_benzene, input_nodes, node_labels, vec_rij, math_rij).animate.shift(UP * 1.5 + RIGHT * 0.5),
                run_time=1.5, rate_func=smooth
            )

        with self.voiceover(text="...the relative distance vectors between atoms remain completely unchanged. The GNN naturally ignores absolute coordinates. Success!"):
            self.play(Indicate(vec_rij, color=RED))
            message_passing_flow("Translation: Relative vectors r_ij remain invariant.")

        # Dọn dẹp vector và đưa về vị trí cũ
        self.play(
            FadeOut(vec_rij), FadeOut(math_rij),
            VGroup(input_benzene, input_nodes, node_labels).animate.shift(DOWN * 1.5 + LEFT * 0.5),
            run_time=0.8
        )

        # --- BÀI TEST 2: ROTATION (ĐỒNG BIẾN - EQUIVARIANCE) ---
        with self.voiceover(text="Crucially, when the input molecule is rotated..."):
            # ĐỒNG BỘ: Input xoay -> Output xoay theo ngay lập tức (Minh họa chữ Equivariant)
            self.play(
                Rotate(input_benzene, angle=PI / 3, about_point=axes.c2p(0, 0)),
                Rotate(input_nodes, angle=PI / 3, about_point=axes.c2p(0, 0)),
                Rotate(node_labels, angle=PI / 3, about_point=axes.c2p(0, 0)),
                # Xoay output đồng bộ
                Rotate(benzene_right, angle=PI / 3, about_point=benzene_right.get_center()),
                run_time=2, rate_func=smooth
            )

        with self.voiceover(text="...the model's geometric features adapt automatically, rotating the output target in perfect synchronization. Equivariance in action!"):
            message_passing_flow("Rotation: Output synchronously rotates (Equivariance).")

        self.play(
            Rotate(input_benzene, angle=-PI / 3, about_point=axes.c2p(0, 0)),
            Rotate(input_nodes, angle=-PI / 3, about_point=axes.c2p(0, 0)),
            Rotate(node_labels, angle=-PI / 3, about_point=axes.c2p(0, 0)),
            Rotate(benzene_right, angle=-PI / 3, about_point=benzene_right.get_center()),
            run_time=0.8
        )

        # --- BÀI TEST 3: PERMUTATION (BẤT BIẾN - INVARIANCE) ---
        label_0, label_1 = node_labels[0], node_labels[1]
        with self.voiceover(text="Finally, if we arbitrarily swap the index labels of the atoms in our data array..."):
            self.play(
                Swap(label_0, label_1, path_arc=PI / 2),
                input_nodes[0].animate.set_color(RED),
                input_nodes[1].animate.set_color(ORANGE),
                run_time=1.2
            )
            # Làm mờ tất cả các con số (label) để cho thấy GNN chỉ quan tâm đến hình học
            self.play(node_labels.animate.set_opacity(0.1), run_time=0.5)

        with self.voiceover(text="...the message passing routes are mathematically identical. The network is fundamentally order-independent."):
            message_passing_flow("Permutation: Topology rules. Order is ignored (Invariance).")

        # Trả lại trạng thái
        self.play(
            node_labels.animate.set_opacity(1),
            Swap(label_0, label_1, path_arc=-PI / 2),
            input_nodes[0].animate.set_color(BLUE),
            input_nodes[1].animate.set_color(BLUE),
            run_time=0.8
        )

        # Chuyển cảnh
        self.play(
            FadeOut(input_group), FadeOut(left_label), FadeOut(right_label),
            FadeOut(geoml_group), FadeOut(geoml_label), FadeOut(benzene_right),
            run_time=1.5
        )

        # ===================================================================
        # SCENE 3: VÒNG LẶP KHOA HỌC & CỖ MÁY GIA TỐC
        # ===================================================================
        goal_text = Text("Goal: Advancing atomistic science", font_size=32, color=WHITE).next_to(title, DOWN, buff=0.5)

        with self.voiceover(text="Ultimately, our goal is to advance atomistic science."):
            self.play(Write(goal_text), run_time=1.5)

        box_system = Rectangle(width=3, height=1.5, color=TEAL, fill_opacity=0.3).shift(LEFT * 4 + DOWN * 1)
        text_system = Text("Atomistic\nSystem", font_size=24).move_to(box_system.get_center())
        sys_group = VGroup(box_system, text_system)

        box_props = Rectangle(width=3.5, height=1.5, color=GREEN_E, fill_opacity=0.3).shift(RIGHT * 4 + DOWN * 1)
        text_props = Text("Properties\n(energy, forces)", font_size=24).move_to(box_props.get_center())
        prop_group = VGroup(box_props, text_props)

        with self.voiceover(text="We often need to map an atomistic system to its physical properties, like energy and forces."):
            self.play(FadeIn(sys_group), FadeIn(prop_group), run_time=1.5)

        arc_top = ArcBetweenPoints(box_system.get_top() + RIGHT * 0.5, box_props.get_top() + LEFT * 0.5, angle=-PI / 4)
        path_slow = DashedVMobject(arc_top, num_dashes=20, color=GRAY)
        arrow_tip_slow = RegularPolygon(n=3, fill_opacity=1, color=GRAY).scale(0.15).rotate(-PI / 6).move_to(arc_top.get_end())
        text_slow = Text("Scientific computing", font_size=20, color=GRAY).next_to(arc_top, UP, buff=0.1)

        with self.voiceover(text="Traditional scientific computing methods do this, but they are notoriously slow and expensive."):
            self.play(Create(path_slow), FadeIn(arrow_tip_slow), Write(text_slow), run_time=2.5)

        arc_bottom = CurvedArrow(
            box_props.get_bottom() + LEFT * 0.5, box_system.get_bottom() + RIGHT * 0.5,
            angle=-PI / 4, color=WHITE
        )
        text_bottom = Text("Design (medicine)", font_size=20, color=WHITE).next_to(arc_bottom, DOWN, buff=0.2)

        with self.voiceover(text="This loop is critical for fields like medicine design."):
            self.play(Create(arc_bottom), Write(text_bottom), run_time=1.5)

        fast_arrow = CurvedArrow(
            box_system.get_right(), box_props.get_left(),
            angle=-PI / 6, color=RED, stroke_width=6
        )
        accel_text = Text("GeoML can accelerate this!", font_size=20, color=RED).next_to(fast_arrow, UP, buff=0.2)

        with self.voiceover(text="Geometric Machine Learning acts as a powerful accelerator, drastically speeding up these computations while preserving strict physical laws."):
            self.play(Write(accel_text), run_time=1.0)
            self.play(
                FadeOut(path_slow), FadeOut(arrow_tip_slow), FadeOut(text_slow),
                Create(fast_arrow, rate_func=rush_into),
                run_time=0.6
            )
            self.play(box_props.animate.set_color(RED).set_fill(opacity=0.6), run_time=0.3)
            self.play(box_props.animate.set_color(GREEN_E).set_fill(opacity=0.3), run_time=0.3)

        self.play(*[FadeOut(m) for m in self.mobjects], run_time=1.5)


In [ ]:
# %%manim -qh -v WARNING AtomisticGeoMLI51

# from manim import *
# import numpy as np
# from manim_voiceover import VoiceoverScene
# from manim_voiceover.services.gtts import GTTSService
# import os
# import shutil
# from pathlib import Path

# # ===================================================================
# # ─── HIGH-QUALITY 3D ATOMISTIC GEOMETRIC ML PRODUCTION
# # ===================================================================
# class AtomisticGeoMLI51(ThreeDScene, VoiceoverScene):
#     def construct(self):
#         self.camera.background_color = BLACK

#         # Tự động dọn dẹp các tàn dư bộ nhớ đệm âm thanh cũ nếu có
#         if os.path.exists("voiceover_cache"):
#             shutil.rmtree("voiceover_cache")

#         # Thiết lập dịch vụ giọng đọc tiếng Anh chuẩn hệ thống
#         self.set_speech_service(GTTSService(lang="en"))

#         # Khởi tạo góc nhìn camera chuyên nghiệp (Góc phối cảnh có chiều sâu)
#         self.set_camera_orientation(phi=55 * DEGREES, theta=-50 * DEGREES)

#         # ===================================================================
#         # SCENE 0: MỞ ĐẦU (INTRODUCTION)
#         # ===================================================================
#         title = Text("Atomistic Systems and Geometric ML I", font_size=36, color=WHITE, weight=BOLD).to_edge(UP)
#         self.add_fixed_in_frame_mobjects(title)

#         with self.voiceover(text="We further explore how Geometric Machine Learning is revolutionizing our understanding of atomistic systems."):
#             self.play(Write(title), run_time=1.5)

#         # Định nghĩa lõi cấu trúc phân tử không gian 3D thực sự
#         base_hexagon = RegularPolygon(n=6, radius=1.3)
#         vertices_2d = base_hexagon.get_vertices()

#         # Thêm độ lệch trục Z cho từng nguyên tử để tạo cấu trúc tinh thể lập thể
#         z_offsets = [0.35, -0.25, 0.4, -0.3, 0.25, -0.15]
#         atoms_pos = [np.array([v[0], v[1], z]) for v, z in zip(vertices_2d, z_offsets)]

#         # Tạo nhóm các nút nguyên tử 3D
#         nodes = VGroup(*[
#             Dot3D(point=pos, radius=0.14, color=BLUE_D if i % 2 == 0 else TEAL_D)
#             for i, pos in enumerate(atoms_pos)
#         ]).shift(LEFT * 1.5 + DOWN * 0.5)

#         # Dựng các cạnh liên kết dạng hình trụ 3D (Sửa tham số width thành thickness)
#         edges = VGroup(*[
#             Line3D(start=nodes[i].get_center(), end=nodes[(i + 1) % 6].get_center(), thickness=0.035, color=GRAY_C)
#             for i in range(6)
#         ])

#         graph_3d = VGroup(edges, nodes)
#         graph_text = Text("3D Graph of Atoms", font_size=24, color=BLUE).move_to([2.5, -2, 0])
#         self.add_fixed_in_frame_mobjects(graph_text)

#         with self.voiceover(text="Traditionally represented as flat drawings, machine learning views molecules as rich 3D graphs of atoms, where nodes represent atoms and edges capture their spatial interactions."):
#             self.play(FadeIn(graph_3d, lag_ratio=0.1))
#             self.play(Write(graph_text))
#             # Hiệu ứng xoay khối không gian bộc lộ góc nhìn 3D đỉnh cao
#             self.play(Rotate(graph_3d, angle=PI/2, axis=UP), run_time=2)
#             self.play(graph_3d.animate.move_to(LEFT * 4 + DOWN * 0.5), run_time=1)

#         # ===================================================================
#         # SCENE 1: THỬ NGHIỆM VÀ SỰ THẤT BẠI CỦA MLP TRUYỀN THỐNG
#         # ===================================================================
#         self.play(FadeOut(graph_text))

#         # Mặt phẳng đích bên phải (Thực tại vật lý)
#         target_benzene = RegularPolygon(n=6, radius=1.0, color=GREEN_C).move_to([4.5, -0.5, 0])
#         right_label = Text("Physical Reality", font_size=20, color=GREEN_C).next_to(target_benzene, UP, buff=0.4)
#         self.add_fixed_in_frame_mobjects(right_label)

#         # Mạng nơ-ron MLP cổ điển ở trung tâm màn hình
#         mlp_layers = VGroup()
#         for num_nodes in [3, 4, 3]:
#             layer = VGroup(*[Dot(radius=0.12, color=GRAY) for _ in range(num_nodes)]).arrange(DOWN, buff=0.4)
#             mlp_layers.add(layer)
#         mlp_layers.arrange(RIGHT, buff=1.0).move_to([0, -0.5, 0])

#         mlp_edges = VGroup()
#         for i in range(2):
#             for n1 in mlp_layers[i]:
#                 for n2 in mlp_layers[i + 1]:
#                     mlp_edges.add(Line(n1.get_center(), n2.get_center(), stroke_width=1.2, stroke_opacity=0.3, color=WHITE))

#         mlp_text = Text("Standard MLP", font_size=20, color=WHITE).next_to(mlp_layers, UP, buff=0.4)
#         self.add_fixed_in_frame_mobjects(mlp_text, mlp_layers, mlp_edges)

#         left_label = Text("Input Molecule", font_size=20, color=BLUE_C).move_to([-4.5, 1.8, 0])
#         self.add_fixed_in_frame_mobjects(left_label)

#         # Thiết lập nhãn số nguyên tử dùng kỹ thuật khóa hướng nhìn 3D phẳng (Fixed Orientation)
#         node_labels = VGroup()
#         for i in range(6):
#             lbl = Text(str(i), font_size=16, color=WHITE)
#             lbl.move_to(graph_3d[1][i].get_center() + RIGHT * 0.25 + UP * 0.25)
#             node_labels.add(lbl)
#             self.add_fixed_orientation_mobjects(lbl)

#         # Gom toàn bộ phân tử và nhãn thành một khối đồng bộ hình học
#         input_group = VGroup(graph_3d, node_labels)

#         with self.voiceover(text="Why can't we just use a standard multilayer perceptron to process this geometric data?"):
#             self.play(
#                 FadeIn(target_benzene), Write(right_label),
#                 FadeIn(mlp_layers), Create(mlp_edges), Write(mlp_text),
#                 Write(left_label)
#             )

#         def simulate_flow(success=True, message=""):
#             status_text = Text(message, font_size=18, color=GREEN if success else RED).to_edge(DOWN, buff=0.8)
#             self.add_fixed_in_frame_mobjects(status_text)
#             self.play(Write(status_text), run_time=0.5)

#             signals_in = VGroup(*[Dot(radius=0.08, color=YELLOW).move_to(input_group.get_center()) for _ in range(4)])
#             self.play(*[sig.animate.move_to(mlp_layers[0][np.random.randint(0, 3)].get_center()) for sig in signals_in], run_time=0.6)
#             self.play(FadeOut(signals_in, run_time=0.2))

#             if not success:
#                 self.play(mlp_edges.animate.set_color(RED).set_opacity(0.8), run_time=0.2)
#                 self.play(mlp_edges.animate.set_color(WHITE).set_opacity(0.3), run_time=0.2)

#             signals_out = VGroup(*[Dot(radius=0.08, color=GREEN if success else RED).move_to(mlp_layers[-1][np.random.randint(0, 3)].get_center()) for _ in range(3)])
#             self.play(*[sig.animate.move_to(target_benzene.get_center()) for sig in signals_out], run_time=0.6)
#             self.play(FadeOut(signals_out, run_time=0.2))

#             self.play(target_benzene.animate.set_color(GREEN if success else RED), run_time=0.3)
#             self.play(target_benzene.animate.set_color(GREEN_C), FadeOut(status_text), run_time=0.3)

#         with self.voiceover(text="At first, the MLP simply memorizes the specific coordinate layout of the training dataset."):
#             simulate_flow(success=True, message="Baseline: MLP memorizes this layout.")

#         # Bài Test 1: Tịnh tiến (Translation)
#         with self.voiceover(text="But what happens if we shift the molecule in space?"):
#             self.play(input_group.animate.shift(UP * 0.8 + RIGHT * 0.3), run_time=1.0)

#         with self.voiceover(text="The absolute coordinates have completely changed. The standard MLP fails to recognize it."):
#             simulate_flow(success=False, message="1. Translation: Absolute positions changed. MLP fails!")

#         self.play(input_group.animate.shift(DOWN * 0.8 + LEFT * 0.3), run_time=0.5)

#         # Bài Test 2: Xoay khối tự do (Rotation)
#         with self.voiceover(text="What if the molecule is rotated?"):
#             # Xoay phân tử quanh một trục 3D bất kỳ để kiểm tra năng lực của MLP
#             self.play(Rotate(input_group, angle=PI / 3, axis=OUT, about_point=input_group.get_center()), run_time=1.2)

#         with self.voiceover(text="Physically it is identical, but the input values are altered. The network fails again."):
#             simulate_flow(success=False, message="2. Rotation: Matrix rotated. MLP fails!")

#         self.play(Rotate(input_group, angle=-PI / 3, axis=OUT, about_point=input_group.get_center()), run_time=0.5)

#         # Bài Test 3: Hoán vị thứ tự mảng (Permutation)
#         with self.voiceover(text="And what if we simply swap the order we feed the atoms into the system?"):
#             # KHẮC PHỤC LỖI: Hoán đổi vị trí thủ công mượt mà thay thế hoàn toàn cho hàm Swap lỗi
#             pos_0 = node_labels[0].get_center().copy()
#             pos_1 = node_labels[1].get_center().copy()
#             self.play(
#                 node_labels[0].animate.move_to(pos_1),
#                 node_labels[1].animate.move_to(pos_0),
#                 input_group[0][1][0].animate.set_color(YELLOW),
#                 input_group[0][1][1].animate.set_color(YELLOW),
#                 run_time=1.0
#             )

#         with self.voiceover(text="The structural topology is the same, but the MLP treats this permutation as completely new data."):
#             simulate_flow(success=False, message="3. Permutation: Atom indexing changed. MLP fails!")

#         # Khôi phục vị trí nhãn nguyên bản
#         self.play(
#             node_labels[0].animate.move_to(pos_0),
#             node_labels[1].animate.move_to(pos_1),
#             input_group[0][1][0].animate.set_color(BLUE_D),
#             input_group[0][1][1].animate.set_color(TEAL_D),
#             run_time=0.5
#         )

#         # ===================================================================
#         # SCENE 2: GIẢI PHÁP HỌC MÁY HÌNH HỌC (GEOMETRIC ML)
#         # ===================================================================
#         geoml_box = RoundedRectangle(width=2.5, height=3.0, color=TEAL_E, fill_color=BLACK, fill_opacity=0.8, stroke_width=3)
#         geoml_box.move_to(mlp_layers.get_center())

#         geoml_label = Text("Geometric GNN", font_size=20, color=TEAL_A).next_to(geoml_box, UP, buff=0.4)
#         self.add_fixed_in_frame_mobjects(geoml_box, geoml_label)

#         with self.voiceover(text="This is why we need Geometric Machine Learning. By replacing dense layers with an Equivariant Graph Neural Network, we bake symmetries directly into the architecture."):
#             self.play(
#                 ReplacementTransform(mlp_layers, geoml_box),
#                 FadeOut(mlp_edges),
#                 ReplacementTransform(mlp_text, geoml_label),
#                 run_time=1.5
#             )

#         def geometric_flow(message=""):
#             status_text = Text(message, font_size=18, color=YELLOW).to_edge(DOWN, buff=0.8)
#             self.add_fixed_in_frame_mobjects(status_text)
#             self.play(Write(status_text), run_time=0.4)

#             # Tạo làn sóng tin nhắn lan tỏa mượt mà không vệt mờ
#             ripples = VGroup(*[
#                 Circle(radius=0.05, color=TEAL_C, stroke_width=2).move_to(atom.get_center())
#                 for atom in input_group[0][1]
#             ])
#             self.play(FadeOut(ripples, scale=4.0), run_time=0.8)

#             stream = Line(input_group.get_center(), geoml_box.get_center(), color=TEAL_B, stroke_width=3)
#             self.play(ShowPassingFlash(stream, time_width=0.4), run_time=0.6)

#             out_stream = Line(geoml_box.get_center(), target_benzene.get_center(), color=GREEN, stroke_width=3)
#             self.play(ShowPassingFlash(out_stream, time_width=0.4), run_time=0.6)

#             self.play(target_benzene.animate.set_color(GREEN), run_time=0.2)
#             self.play(target_benzene.animate.set_color(GREEN_C), FadeOut(status_text), run_time=0.3)

#         with self.voiceover(text="The GNN processes data by passing messages along graph edges, inherently respecting molecular structure."):
#             geometric_flow("GNN Baseline: Processing underlying graph topology.")

#         # GNN Bài Test 1: Tịnh tiến thành công (Bất biến khoảng cách tương đối)
#         with self.voiceover(text="Now, when the molecule translates..."):
#             # Tạo vector 3D khoảng cách tương đối thực sự bằng Arrow3D
#             vec_rij = Arrow3D(start=input_group[0][1][0].get_center(), end=input_group[0][1][1].get_center(), color=YELLOW, thickness=0.04)
#             math_rij = MathTex(r"\vec{r}_{ij} = x_i - x_j", font_size=20, color=YELLOW).next_to(input_group.get_center(), UP, buff=0.8)
#             self.add_fixed_in_frame_mobjects(math_rij)

#             self.play(Create(vec_rij), Write(math_rij))
#             moving_elements = VGroup(input_group, vec_rij)
#             self.play(moving_elements.animate.shift(UP * 0.8 + RIGHT * 0.4), run_time=1.2)

#         with self.voiceover(text="The relative vectors between atoms remain perfectly invariant. The network recognizes it instantly. Success!"):
#             geometric_flow("Translation: Invariant relative vectors. Success!")

#         self.play(
#             FadeOut(vec_rij), FadeOut(math_rij),
#             input_group.animate.shift(DOWN * 0.8 + LEFT * 0.4),
#             run_time=0.5
#         )

#         # GNN Bài Test 2: Xoay thành công (Đồng biến đồng bộ hoàn hảo)
#         with self.voiceover(text="When the input molecule is rotated, the internal features rotate synchronously."):
#             self.play(
#                 Rotate(input_group, angle=PI / 2, axis=OUT, about_point=input_group.get_center()),
#                 Rotate(target_benzene, angle=PI / 2, axis=OUT, about_point=target_benzene.get_center()),
#                 run_time=1.8
#             )

#         with self.voiceover(text="The target adjusts in perfect symmetry. This is equivariance in action!"):
#             geometric_flow("Rotation: Robust equivariant symmetry. Success!")

#         self.play(*[FadeOut(m) for m in self.mobjects], run_time=1.0)

#         # ===================================================================
#         # SCENE 3: SƠ ĐỒ QUY TRÌNH KHOA HỌC (SCIENTIFIC COMPUTING LOOP)
#         # ===================================================================
#         # BÍ QUYẾT: Hạ tiêu cự camera về dạng 2D trực diện để vẽ sơ đồ phẳng sắc nét nhất
#         self.move_camera(phi=0, theta=-90 * DEGREES, run_time=1.2)

#         goal_text = Text("Goal: Advancing atomistic science", font_size=32, color=WHITE).to_edge(UP, buff=0.5)
#         self.add_fixed_in_frame_mobjects(goal_text)

#         with self.voiceover(text="Ultimately, our goal is to advance atomistic science."):
#             self.play(Write(goal_text), run_time=1.5)

#         box_system = Rectangle(width=3, height=1.5, color=TEAL, fill_opacity=0.3).shift(LEFT * 4 + DOWN * 0.5)
#         text_system = Text("Atomistic\nSystem", font_size=24).move_to(box_system.get_center())
#         sys_group = VGroup(box_system, text_system)

#         box_props = Rectangle(width=3.5, height=1.5, color=GREEN_E, fill_opacity=0.3).shift(RIGHT * 4 + DOWN * 0.5)
#         text_props = Text("Properties\n(energy, forces)", font_size=24).move_to(box_props.get_center())
#         prop_group = VGroup(box_props, text_props)

#         self.add_fixed_in_frame_mobjects(sys_group, prop_group)

#         with self.voiceover(text="We often need to map an atomistic system to its physical properties, like energy and forces."):
#             self.play(FadeIn(sys_group), FadeIn(prop_group), run_time=1.5)

#         arc_top = ArcBetweenPoints(box_system.get_top() + RIGHT * 0.5, box_props.get_top() + LEFT * 0.5, angle=-PI / 4)
#         path_slow = DashedVMobject(arc_top, num_dashes=20, color=GRAY)
#         arrow_tip_slow = RegularPolygon(n=3, fill_opacity=1, color=GRAY).scale(0.15).rotate(-PI / 6).move_to(arc_top.get_end())
#         text_slow = Text("Scientific computing", font_size=20, color=GRAY).next_to(arc_top, UP, buff=0.1)

#         self.add_fixed_in_frame_mobjects(path_slow, arrow_tip_slow, text_slow)

#         with self.voiceover(text="Traditional scientific computing methods do this, but they are notoriously slow and expensive."):
#             self.play(Create(path_slow), FadeIn(arrow_tip_slow), Write(text_slow), run_time=2.5)

#         arc_bottom = CurvedArrow(box_props.get_bottom() + LEFT * 0.5, box_system.get_bottom() + RIGHT * 0.5, angle=-PI / 4, color=WHITE)
#         text_bottom = Text("Design (medicine)", font_size=20, color=WHITE).next_to(arc_bottom, DOWN, buff=0.2)

#         self.add_fixed_in_frame_mobjects(arc_bottom, text_bottom)

#         with self.voiceover(text="This loop is critical for fields like medicine design."):
#             self.play(Create(arc_bottom), Write(text_bottom), run_time=1.5)

#         fast_arrow = CurvedArrow(box_system.get_right(), box_props.get_left(), angle=-PI / 6, color=RED, stroke_width=6)
#         accel_text = Text("GeoML can accelerate this!", font_size=20, color=RED).next_to(fast_arrow, UP, buff=0.2)

#         self.add_fixed_in_frame_mobjects(fast_arrow, accel_text)

#         with self.voiceover(text="Geometric Machine Learning acts as a powerful accelerator, drastically speeding up these computations while preserving strict physical laws."):
#             self.play(Write(accel_text), run_time=1.0)
#             self.play(
#                 FadeOut(path_slow), FadeOut(arrow_tip_slow), FadeOut(text_slow),
#                 Create(fast_arrow, rate_func=rush_into),
#                 run_time=0.6
#             )
#             self.play(box_props.animate.set_color(RED).set_fill(opacity=0.6), run_time=0.3)
#             self.play(box_props.animate.set_color(GREEN_E).set_fill(opacity=0.3), run_time=0.3)

#         self.play(*[FadeOut(m) for m in self.mobjects], run_time=1.5)
#         self.wait(0.5)


In [ ]:
# %%manim -qh -v WARNING AtomisticGeoMLI51

# from manim import *
# import numpy as np
# from manim_voiceover import VoiceoverScene
# from manim_voiceover.services.gtts import GTTSService
# import os
# import shutil
# from pathlib import Path

# # ===================================================================
# # ─── HIGH-QUALITY 3D ATOMISTIC GEOMETRIC ML PRODUCTION
# # ===================================================================
# class AtomisticGeoMLI51(ThreeDScene, VoiceoverScene):
#     def construct(self):
#         self.camera.background_color = BLACK

#         # Tự động dọn dẹp các tàn dư bộ nhớ đệm âm thanh cũ nếu có
#         if os.path.exists("voiceover_cache"):
#             shutil.rmtree("voiceover_cache")

#         # Thiết lập dịch vụ giọng đọc tiếng Anh chuẩn hệ thống
#         self.set_speech_service(GTTSService(lang="en"))

#         # Khởi tạo góc nhìn camera chuyên nghiệp (Góc phối cảnh có chiều sâu)
#         self.set_camera_orientation(phi=55 * DEGREES, theta=-50 * DEGREES)

#         # ===================================================================
#         # SCENE 0: MỞ ĐẦU (INTRODUCTION)
#         # ===================================================================
#         title = Text("Atomistic Systems and Geometric ML I", font_size=36, color=WHITE, weight=BOLD).to_edge(UP)
#         self.add_fixed_in_frame_mobjects(title)

#         with self.voiceover(text="We further explore how Geometric Machine Learning is revolutionizing our understanding of atomistic systems."):
#             self.play(Write(title), run_time=1.5)

#         # Định nghĩa lõi cấu trúc phân tử không gian 3D thực sự
#         base_hexagon = RegularPolygon(n=6, radius=1.3)
#         vertices_2d = base_hexagon.get_vertices()

#         # Thêm độ lệch trục Z cho từng nguyên tử để tạo cấu trúc tinh thể lập thể
#         z_offsets = [0.35, -0.25, 0.4, -0.3, 0.25, -0.15]
#         atoms_pos = [np.array([v[0], v[1], z]) for v, z in zip(vertices_2d, z_offsets)]

#         # Tạo nhóm các nút nguyên tử 3D
#         nodes = VGroup(*[
#             Dot3D(point=pos, radius=0.14, color=BLUE_D if i % 2 == 0 else TEAL_D)
#             for i, pos in enumerate(atoms_pos)
#         ]).shift(LEFT * 1.5 + DOWN * 0.5)

#         # Dựng các cạnh liên kết dạng hình trụ 3D (Vá lỗi width thành thickness)
#         edges = VGroup(*[
#             Line3D(start=nodes[i].get_center(), end=nodes[(i + 1) % 6].get_center(), thickness=0.035, color=GRAY_C)
#             for i in range(6)
#         ])

#         graph_3d = VGroup(edges, nodes)
#         graph_text = Text("3D Graph of Atoms", font_size=24, color=BLUE).move_to([2.5, -2, 0])
#         self.add_fixed_in_frame_mobjects(graph_text)

#         with self.voiceover(text="Traditionally represented as flat drawings, machine learning views molecules as rich 3D graphs of atoms, where nodes represent atoms and edges capture their spatial interactions."):
#             self.play(FadeIn(graph_3d, lag_ratio=0.1))
#             self.play(Write(graph_text))
#             # Hiệu ứng xoay khối không gian bộc lộ góc nhìn 3D đỉnh cao
#             self.play(Rotate(graph_3d, angle=PI/2, axis=UP), run_time=2)
#             self.play(graph_3d.animate.move_to(LEFT * 4 + DOWN * 0.5), run_time=1)

#         # ===================================================================
#         # SCENE 1: THỬ NGHIỆM VÀ SỰ THẤT BẠI CỦA MLP TRUYỀN THỐNG
#         # ===================================================================
#         self.play(FadeOut(graph_text))

#         # -----------------------------------------------------------------
#         # CẢI THIỆN: NỔI "PHYSICAL REALITY" LÊN TRÊN ĐỂ CHỐNG CHỒNG LẤN & KHÔNG "DẸP"
#         # -----------------------------------------------------------------
#         target_benzene_face = RegularPolygon(n=6, radius=1.0, color=GREEN_C, fill_opacity=0.3)
#         target_benzene_edges = VGroup(*[
#             Line3D(
#                 start=target_benzene_face.get_vertices()[i],
#                 end=target_benzene_face.get_vertices()[(i + 1) % 6],
#                 thickness=0.03, color=GREEN_C, shade_in_3d=True
#             )
#             for i in range(6)
#         ])

#         # Gom lại và shift OUT * 2.0 để nổi hẳn lên trên bề mặt Z=0 chống chồng lấn
#         benzene_right = VGroup(target_benzene_face, target_benzene_edges)
#         benzene_right.move_to([4.5, -0.5, 0]).shift(OUT * 2.0)

#         target_benzene = benzene_right[0] # Alias cho animation đổi màu

#         right_label = Text("Physical Reality", font_size=20, color=GREEN_C).next_to(benzene_right, UP, buff=0.4)
#         self.add_fixed_in_frame_mobjects(right_label)

#         # Mạng nơ-ron MLP cổ điển ở trung tâm màn hình, cũng shift nổi lên Z=2
#         mlp_layers = VGroup()
#         for num_nodes in [3, 4, 3]:
#             layer = VGroup(*[Dot(radius=0.12, color=GRAY, shade_in_3d=True) for _ in range(num_nodes)]).arrange(DOWN, buff=0.4)
#             mlp_layers.add(layer)
#         mlp_layers.arrange(RIGHT, buff=1.0).move_to([0, -0.5, 0]).shift(OUT * 2.0)

#         mlp_edges = VGroup()
#         for i in range(2):
#             for n1 in mlp_layers[i]:
#                 for n2 in mlp_layers[i + 1]:
#                     mlp_edges.add(Line(n1.get_center(), n2.get_center(), stroke_width=1.2, stroke_opacity=0.3, color=WHITE, shade_in_3d=True))

#         mlp_text = Text("Standard MLP", font_size=20, color=WHITE).next_to(mlp_layers, UP, buff=0.4)
#         self.add_fixed_in_frame_mobjects(mlp_text)

#         # Cụm dữ liệu Input: graph_3d nằm gần Z=0, node_labels được khóa hướng nhìn
#         left_label = Text("Input Molecule", font_size=20, color=BLUE_C).move_to([-4.5, 1.8, 0])
#         self.add_fixed_in_frame_mobjects(left_label)

#         # Vá lỗi Permutation Swap: Thiết lập nhãn với add_fixed_orientation_mobjects
#         node_labels = VGroup()
#         for i in range(6):
#             lbl = Text(str(i), font_size=16, color=WHITE)
#             lbl.move_to(graph_3d[1][i].get_center() + RIGHT * 0.25 + UP * 0.25)
#             node_labels.add(lbl)
#             self.add_fixed_orientation_mobjects(lbl)

#         input_group = VGroup(graph_3d, node_labels)

#         with self.voiceover(text="Why can't we just use a standard multilayer perceptron to process this geometric data?"):
#             self.play(
#                 FadeIn(benzene_right), Write(right_label),
#                 FadeIn(mlp_layers), Create(mlp_edges), Write(mlp_text),
#                 Write(left_label)
#             )

#         def simulate_flow(success=True, message=""):
#             status_text = Text(message, font_size=18, color=GREEN if success else RED).to_edge(DOWN, buff=0.8)
#             self.add_fixed_in_frame_mobjects(status_text)
#             self.play(Write(status_text), run_time=0.5)

#             # Đẩy tín hiệu và pass-flashes nổi lên Z=2 để khớp luồng chảy
#             signals_in = VGroup(*[Dot(radius=0.08, color=YELLOW, shade_in_3d=True).move_to(input_group.get_center()) for _ in range(4)])
#             self.play(*[sig.animate.move_to(mlp_layers[0][np.random.randint(0, 3)].get_center()) for sig in signals_in], run_time=0.6)
#             self.play(FadeOut(signals_in, run_time=0.2))

#             if not success:
#                 self.play(mlp_edges.animate.set_color(RED).set_opacity(0.8), run_time=0.2)
#                 self.play(mlp_edges.animate.set_color(WHITE).set_opacity(0.3), run_time=0.2)

#             signals_out = VGroup(*[Dot(radius=0.08, color=GREEN if success else RED, shade_in_3d=True).move_to(mlp_layers[-1][np.random.randint(0, 3)].get_center()) for _ in range(3)])
#             self.play(*[sig.animate.move_to(benzene_right.get_center()) for sig in signals_out], run_time=0.6)
#             self.play(FadeOut(signals_out, run_time=0.2))

#             self.play(target_benzene.animate.set_color(GREEN if success else RED), run_time=0.3)
#             self.play(target_benzene.animate.set_color(GREEN_C), FadeOut(status_text), run_time=0.3)

#         with self.voiceover(text="At first, the MLP simply memorizes the specific coordinate layout of the training dataset."):
#             simulate_flow(success=True, message="Baseline: MLP memorizes this layout.")

#         # Bài Test 1: Tịnh tiến (Translation)
#         with self.voiceover(text="But what happens if we shift the molecule in space?"):
#             self.play(input_group.animate.shift(UP * 0.8 + RIGHT * 0.3), run_time=1.0)

#         with self.voiceover(text="The absolute coordinates have completely changed. The standard MLP fails to recognize it."):
#             simulate_flow(success=False, message="1. Translation: Absolute positions changed. MLP fails!")

#         self.play(input_group.animate.shift(DOWN * 0.8 + LEFT * 0.3), run_time=0.5)

#         # Bài Test 2: Xoay khối tự do (Rotation)
#         with self.voiceover(text="What if the molecule is rotated?"):
#             # Xoay khối phân tử 3D thực sự quanh trục thẳng đứng để bộc lộ chiều sâu
#             self.play(Rotate(input_group, angle=PI / 3, axis=OUT, about_point=input_group.get_center()), run_time=1.2)

#         with self.voiceover(text="Physically it is identical, but the input values are altered. The network fails again."):
#             simulate_flow(success=False, message="2. Rotation: Matrix rotated. MLP fails!")

#         self.play(Rotate(input_group, angle=-PI / 3, axis=OUT, about_point=input_group.get_center()), run_time=0.5)

#         # Bài Test 3: Hoán vị thứ tự mảng (Permutation)
#         with self.voiceover(text="And what if we simply swap the order we feed the atoms into the system?"):
#             # VÁ LỖI Swap không tồn tại bằng move_to song song mượt mà
#             pos_0 = node_labels[0].get_center().copy()
#             pos_1 = node_labels[1].get_center().copy()
#             self.play(
#                 node_labels[0].animate.move_to(pos_1),
#                 node_labels[1].animate.move_to(pos_0),
#                 input_group[0][1][0].animate.set_color(YELLOW),
#                 input_group[0][1][1].animate.set_color(YELLOW),
#                 run_time=1.0
#             )

#         with self.voiceover(text="The structural topology is the same, but the MLP treats this permutation as completely new data."):
#             simulate_flow(success=False, message="3. Permutation: Atom indexing changed. MLP fails!")

#         # Khôi phục vị trí nhãn nguyên bản
#         self.play(
#             node_labels[0].animate.move_to(pos_0),
#             node_labels[1].animate.move_to(pos_1),
#             input_group[0][1][0].animate.set_color(BLUE_D),
#             input_group[0][1][1].animate.set_color(TEAL_D),
#             run_time=0.5
#         )

#         # ===================================================================
#         # SCENE 2: GIẢI PHÁP HỌC MÁY HÌNH HỌC (GEOMETRIC ML SAVES THE DAY)
#         # ===================================================================
#         # -----------------------------------------------------------------
#         # CẢI THIỆN 1: ĐỊNH NGHĨA LÕI ĐỒ THỊ GNN ĐỂ HIỂN THỊ TRONG BOX
#         # -----------------------------------------------------------------
#         geoml_box = RoundedRectangle(width=2.5, height=3.0, color=TEAL_E, fill_color=BLACK, fill_opacity=0.8, stroke_width=3)
#         geoml_box.move_to(mlp_layers.get_center())

#         # Mạng nơ-ron đồ thị thu nhỏ bên trong Box đại diện cho Message Passing
#         gnn_core_nodes = VGroup(*[Dot(radius=0.06, color=TEAL_C) for _ in range(6)])
#         gnn_positions = [UP*0.5, UR*0.35, DR*0.35, DOWN*0.5, DL*0.35, UL*0.35]
#         for node, pos in zip(gnn_core_nodes, gnn_positions):
#             node.move_to(geoml_box.get_center() + pos)
#         gnn_core_edges = VGroup(*[
#             Line(gnn_core_nodes[i].get_center(), gnn_core_nodes[j].get_center(), stroke_width=1.0, color=TEAL_E)
#             for i in range(6) for j in range(i+1, 6) if np.random.rand() > 0.5 # Kết nối ngẫu nhiên đẹp mắt
#         ])
#         gnn_core_group = VGroup(gnn_core_edges, gnn_core_nodes)

#         geoml_label = Text("Geometric GNN", font_size=20, color=TEAL_A).next_to(geoml_box, UP, buff=0.4)
#         self.add_fixed_in_frame_mobjects(geoml_box, geoml_label)

#         with self.voiceover(text="This is why we need Geometric Machine Learning. By replacing dense layers with an Equivariant Graph Neural Network, we bake symmetries directly into the architecture."):
#             self.play(
#                 ReplacementTransform(mlp_layers, geoml_box),
#                 FadeOut(mlp_edges),
#                 # FadeIn cấu trúc lõi GNN vào bên trong hộp đồng thời
#                 FadeIn(gnn_core_group, lag_ratio=0.1),
#                 ReplacementTransform(mlp_text, geoml_label),
#                 run_time=1.5
#             )

#         def geometric_flow(message=""):
#             status_text = Text(message, font_size=18, color=YELLOW).to_edge(DOWN, buff=0.8)
#             self.add_fixed_in_frame_mobjects(status_text)
#             self.play(Write(status_text), run_time=0.4)

#             # Hiệu ứng Ripples Message Passing lan tỏa mượt mà tan vào nền đen
#             ripples = VGroup(*[
#                 Circle(radius=0.05, color=TEAL_C, stroke_width=2).move_to(atom.get_center())
#                 for atom in input_group[0][1]
#             ])
#             self.play(FadeOut(ripples, scale=4.0), run_time=0.8)

#             stream = Line(input_group.get_center(), geoml_box.get_center(), color=TEAL_B, stroke_width=3)
#             self.play(ShowPassingFlash(stream, time_width=0.4), run_time=0.6)

#             out_stream = Line(geoml_box.get_center(), target_benzene.get_center(), color=GREEN, stroke_width=3)
#             self.play(ShowPassingFlash(out_stream, time_width=0.4), run_time=0.6)

#             self.play(target_benzene.animate.set_color(GREEN), run_time=0.2)
#             self.play(target_benzene.animate.set_color(GREEN_C), FadeOut(status_text), run_time=0.3)

#         with self.voiceover(text="The GNN processes data by passing messages along graph edges, inherently respecting molecular structure."):
#             geometric_flow("GNN Baseline: Processing underlying graph topology.")

#         # GNN Bài Test 1: Tịnh tiến thành công
#         with self.voiceover(text="Now, when the molecule translates..."):
#             # -----------------------------------------------------------------
#             # CẢI THIỆN 3: DÙNG ARROW3D & SHIFT Z CAO HẲN ĐỂ CHỐNG CHỒNG LẤN
#             # -----------------------------------------------------------------
#             # shift Z lên OUT * 4.0 để nổi hẳn lên trên tất cả các hộp Z=2, chống z-fighting răng cưa
#             vec_rij = Arrow3D(
#                 start=input_group[0][1][0].get_center(),
#                 end=input_group[0][1][1].get_center(),
#                 color=YELLOW, thickness=0.04
#             ).shift(OUT * 2.0)

#             math_rij = MathTex(r"\vec{r}_{ij} = x_i - x_j", font_size=20, color=YELLOW).next_to(input_group.get_center(), UP, buff=0.8).shift(OUT * 4.0)
#             self.add_fixed_in_frame_mobjects(math_rij)

#             self.play(Create(vec_rij), Write(math_rij))

#             # Gom lại để dịch chuyển đồng bộ
#             moving_elements = VGroup(input_group, vec_rij)
#             self.play(moving_elements.animate.shift(UP * 0.8 + RIGHT * 0.4), run_time=1.2)

#         with self.voiceover(text="The relative vectors between atoms remain perfectly invariant. The network recognizes it instantly. Success!"):
#             geometric_flow("Translation: Invariant relative vectors. Success!")

#         self.play(
#             FadeOut(vec_rij), FadeOut(math_rij),
#             input_group.animate.shift(DOWN * 0.8 + LEFT * 0.4),
#             run_time=0.5
#         )

#         # GNN Bài Test 2: Xoay thành công (Đồng biến đồng bộ hoàn hảo)
#         with self.voiceover(text="When the input molecule is rotated, the internal features rotate synchronously."):
#             self.play(
#                 Rotate(input_group, angle=PI / 2, axis=OUT, about_point=input_group.get_center()),
#                 # Kịch bản đồng bộ: Đầu vào xoay thì hệ thống đầu ra cũng xoay nhịp nhàng theo
#                 Rotate(benzene_right, angle=PI / 2, axis=OUT, about_point=benzene_right.get_center()),
#                 run_time=1.8
#             )

#         with self.voiceover(text="The target adjusts in perfect symmetry. This is equivariance in action!"):
#             geometric_flow("Rotation: Robust equivariant symmetry. Success!")

#         # Dọn dẹp nhóm kết thúc phần 2
#         self.play(*[FadeOut(m) for m in self.mobjects], run_time=1.0)

#         # ===================================================================
#         # SCENE 3: SƠ ĐỒ QUY TRÌNH KHOA HỌC (SCIENTIFIC COMPUTING LOOP)
#         # ===================================================================
#         # -----------------------------------------------------------------
#         # CẢI THIỆN: CHUYỂN CAMERA TOP-DOWN phẳng lặng để sơ đồ sắc nét
#         # -----------------------------------------------------------------
#         self.move_camera(phi=0, theta=-90 * DEGREES, run_time=1.2)

#         goal_text = Text("Goal: Advancing atomistic science", font_size=32, color=WHITE).to_edge(UP, buff=0.5)
#         self.add_fixed_in_frame_mobjects(goal_text)

#         with self.voiceover(text="Ultimately, our goal is to advance atomistic science."):
#             self.play(Write(goal_text), run_time=1.5)

#         box_system = Rectangle(width=3, height=1.5, color=TEAL, fill_opacity=0.3).shift(LEFT * 4 + DOWN * 0.5)
#         text_system = Text("Atomistic\nSystem", font_size=24).move_to(box_system.get_center())
#         sys_group = VGroup(box_system, text_system)

#         box_props = Rectangle(width=3.5, height=1.5, color=GREEN_E, fill_opacity=0.3).shift(RIGHT * 4 + DOWN * 0.5)
#         text_props = Text("Properties\n(energy, forces)", font_size=24).move_to(box_props.get_center())
#         prop_group = VGroup(box_props, text_props)

#         self.add_fixed_in_frame_mobjects(sys_group, prop_group)

#         with self.voiceover(text="We often need to map an atomistic system to its physical properties, like energy and forces."):
#             self.play(FadeIn(sys_group), FadeIn(prop_group), run_time=1.5)

#         arc_top = ArcBetweenPoints(box_system.get_top() + RIGHT * 0.5, box_props.get_top() + LEFT * 0.5, angle=-PI / 4)
#         path_slow = DashedVMobject(arc_top, num_dashes=20, color=GRAY)
#         arrow_tip_slow = RegularPolygon(n=3, fill_opacity=1, color=GRAY).scale(0.15).rotate(-PI / 6).move_to(arc_top.get_end())
#         text_slow = Text("Scientific computing", font_size=20, color=GRAY).next_to(arc_top, UP, buff=0.1)

#         self.add_fixed_in_frame_mobjects(path_slow, arrow_tip_slow, text_slow)

#         with self.voiceover(text="Traditional scientific computing methods do this, but they are notoriously slow and expensive."):
#             self.play(Create(path_slow), FadeIn(arrow_tip_slow), Write(text_slow), run_time=2.5)

#         arc_bottom = CurvedArrow(box_props.get_bottom() + LEFT * 0.5, box_system.get_bottom() + RIGHT * 0.5, angle=-PI / 4, color=WHITE)
#         text_bottom = Text("Design (medicine)", font_size=20, color=WHITE).next_to(arc_bottom, DOWN, buff=0.2)

#         self.add_fixed_in_frame_mobjects(arc_bottom, text_bottom)

#         with self.voiceover(text="This loop is critical for fields like medicine design."):
#             self.play(Create(arc_bottom), Write(text_bottom), run_time=1.5)

#         fast_arrow = CurvedArrow(box_system.get_right(), box_props.get_left(), angle=-PI / 6, color=RED, stroke_width=6)
#         accel_text = Text("GeoML can accelerate this!", font_size=20, color=RED).next_to(fast_arrow, UP, buff=0.2)

#         self.add_fixed_in_frame_mobjects(fast_arrow, accel_text)

#         with self.voiceover(text="Geometric Machine Learning acts as a powerful accelerator, drastically speeding up these computations while preserving strict physical laws."):
#             self.play(Write(accel_text), run_time=1.0)
#             self.play(
#                 FadeOut(path_slow), FadeOut(arrow_tip_slow), FadeOut(text_slow),
#                 Create(fast_arrow, rate_func=rush_into),
#                 run_time=0.6
#             )
#             self.play(box_props.animate.set_color(RED).set_fill(opacity=0.6), run_time=0.3)
#             self.play(box_props.animate.set_color(GREEN_E).set_fill(opacity=0.3), run_time=0.3)

#         self.play(*[FadeOut(m) for m in self.mobjects], run_time=1.5)


In [ ]:
# %%manim -qh -v WARNING AtomisticGeoMLI

# from manim import *
# import numpy as np
# from manim_voiceover import VoiceoverScene
# from manim_voiceover.services.gtts import GTTSService
# import os
# import shutil
# from pathlib import Path

# # ===================================================================
# # ─── HIGH-QUALITY 3D ATOMISTIC GEOMETRIC ML PRODUCTION
# # ===================================================================
# class AtomisticGeoMLI(ThreeDScene, VoiceoverScene):
#     def construct(self):
#         self.camera.background_color = BLACK

#         # Tự động dọn dẹp các tàn dư bộ nhớ đệm âm thanh cũ nếu có
#         if os.path.exists("voiceover_cache"):
#             shutil.rmtree("voiceover_cache")

#         # Thiết lập dịch vụ giọng đọc tiếng Anh chuẩn hệ thống
#         self.set_speech_service(GTTSService(lang="en"))

#         # Khởi tạo góc nhìn camera chuyên nghiệp (Góc phối cảnh có chiều sâu)
#         self.set_camera_orientation(phi=55 * DEGREES, theta=-50 * DEGREES)

#         # ===================================================================
#         # SCENE 0: MỞ ĐẦU (INTRODUCTION)
#         # ===================================================================
#         title = Text("Atomistic Systems and Geometric ML I", font_size=36, color=WHITE, weight=BOLD).to_edge(UP)
#         self.add_fixed_in_frame_mobjects(title)

#         with self.voiceover(text="We further explore how Geometric Machine Learning is revolutionizing our understanding of atomistic systems."):
#             self.play(Write(title), run_time=1.5)

#         # Định nghĩa lõi cấu trúc phân tử không gian 3D thực sự
#         base_hexagon = RegularPolygon(n=6, radius=1.3)
#         vertices_2d = base_hexagon.get_vertices()

#         # Thêm độ lệch trục Z cho từng nguyên tử để tạo cấu trúc tinh thể lập thể
#         z_offsets = [0.35, -0.25, 0.4, -0.3, 0.25, -0.15]
#         atoms_pos = [np.array([v[0], v[1], z]) for v, z in zip(vertices_2d, z_offsets)]

#         # Tạo nhóm các nút nguyên tử 3D
#         nodes = VGroup(*[
#             Dot3D(point=pos, radius=0.14, color=BLUE_D if i % 2 == 0 else TEAL_D)
#             for i, pos in enumerate(atoms_pos)
#         ]).shift(LEFT * 1.5 + DOWN * 0.5)

#         # Dựng các cạnh liên kết dạng hình trụ 3D (Vá lỗi width thành thickness)
#         edges = VGroup(*[
#             Line3D(start=nodes[i].get_center(), end=nodes[(i + 1) % 6].get_center(), thickness=0.035, color=GRAY_C)
#             for i in range(6)
#         ])

#         graph_3d = VGroup(edges, nodes)
#         graph_text = Text("3D Graph of Atoms", font_size=24, color=BLUE).move_to([2.5, -2, 0])
#         self.add_fixed_in_frame_mobjects(graph_text)

#         with self.voiceover(text="Traditionally represented as flat drawings, machine learning views molecules as rich 3D graphs of atoms, where nodes represent atoms and edges capture their spatial interactions."):
#             self.play(FadeIn(graph_3d, lag_ratio=0.1))
#             self.play(Write(graph_text))
#             # Hiệu ứng xoay khối không gian bộc lộ góc nhìn 3D đỉnh cao
#             self.play(Rotate(graph_3d, angle=PI/2, axis=UP), run_time=2)
#             self.play(graph_3d.animate.move_to(LEFT * 4 + DOWN * 0.5), run_time=1)

#         # ===================================================================
#         # SCENE 1: THỬ NGHIỆM VÀ SỰ THẤT BẠI CỦA MLP TRUYỀN THỐNG
#         # ===================================================================
#         self.play(FadeOut(graph_text))

#         # -----------------------------------------------------------------
#         # CẢI THIỆN: NỔI "PHYSICAL REALITY" LÊN TRÊN ĐỂ CHỐNG CHỒNG LẤN & KHÔNG "DẸP"
#         # -----------------------------------------------------------------
#         target_benzene_face = RegularPolygon(n=6, radius=1.0, color=GREEN_C, fill_opacity=0.3)
#         target_benzene_edges = VGroup(*[
#             Line3D(
#                 start=target_benzene_face.get_vertices()[i],
#                 end=target_benzene_face.get_vertices()[(i + 1) % 6],
#                 thickness=0.03, color=GREEN_C, shade_in_3d=True
#             )
#             for i in range(6)
#         ])

#         # Gom lại và shift OUT * 2.0 để nổi hẳn lên trên bề mặt Z=0 chống chồng lấn
#         benzene_right = VGroup(target_benzene_face, target_benzene_edges)
#         benzene_right.move_to([4.5, -0.5, 0]).shift(OUT * 2.0)

#         target_benzene = benzene_right[0] # Alias cho animation đổi màu

#         right_label = Text("Physical Reality", font_size=20, color=GREEN_C).next_to(benzene_right, UP, buff=0.4)
#         self.add_fixed_in_frame_mobjects(right_label)

#         # Mạng nơ-ron MLP cổ điển ở trung tâm màn hình, cũng shift nổi lên Z=2
#         mlp_layers = VGroup()
#         for num_nodes in [3, 4, 3]:
#             layer = VGroup(*[Dot(radius=0.12, color=GRAY, shade_in_3d=True) for _ in range(num_nodes)]).arrange(DOWN, buff=0.4)
#             mlp_layers.add(layer)
#         mlp_layers.arrange(RIGHT, buff=1.0).move_to([0, -0.5, 0]).shift(OUT * 2.0)

#         mlp_edges = VGroup()
#         for i in range(2):
#             for n1 in mlp_layers[i]:
#                 for n2 in mlp_layers[i + 1]:
#                     mlp_edges.add(Line(n1.get_center(), n2.get_center(), stroke_width=1.2, stroke_opacity=0.3, color=WHITE, shade_in_3d=True))

#         mlp_text = Text("Standard MLP", font_size=20, color=WHITE).next_to(mlp_layers, UP, buff=0.4)
#         self.add_fixed_in_frame_mobjects(mlp_text)

#         # Cụm dữ liệu Input: graph_3d nằm gần Z=0, node_labels được khóa hướng nhìn
#         left_label = Text("Input Molecule", font_size=20, color=BLUE_C).move_to([-4.5, 1.8, 0])
#         self.add_fixed_in_frame_mobjects(left_label)

#         # Vá lỗi Permutation Swap: Thiết lập nhãn với add_fixed_orientation_mobjects
#         node_labels = VGroup()
#         for i in range(6):
#             lbl = Text(str(i), font_size=16, color=WHITE)
#             lbl.move_to(graph_3d[1][i].get_center() + RIGHT * 0.25 + UP * 0.25)
#             node_labels.add(lbl)
#             self.add_fixed_orientation_mobjects(lbl)

#         input_group = VGroup(graph_3d, node_labels)

#         with self.voiceover(text="Why can't we just use a standard multilayer perceptron to process this geometric data?"):
#             self.play(
#                 FadeIn(benzene_right), Write(right_label),
#                 FadeIn(mlp_layers), Create(mlp_edges), Write(mlp_text),
#                 Write(left_label)
#             )

#         def simulate_flow(success=True, message=""):
#             status_text = Text(message, font_size=18, color=GREEN if success else RED).to_edge(DOWN, buff=0.8)
#             self.add_fixed_in_frame_mobjects(status_text)
#             self.play(Write(status_text), run_time=0.5)

#             # Đẩy tín hiệu và pass-flashes nổi lên Z=2 để khớp luồng chảy
#             signals_in = VGroup(*[Dot(radius=0.08, color=YELLOW, shade_in_3d=True).move_to(input_group.get_center()) for _ in range(4)])
#             self.play(*[sig.animate.move_to(mlp_layers[0][np.random.randint(0, 3)].get_center()) for sig in signals_in], run_time=0.6)
#             self.play(FadeOut(signals_in, run_time=0.2))

#             if not success:
#                 self.play(mlp_edges.animate.set_color(RED).set_opacity(0.8), run_time=0.2)
#                 self.play(mlp_edges.animate.set_color(WHITE).set_opacity(0.3), run_time=0.2)

#             signals_out = VGroup(*[Dot(radius=0.08, color=GREEN if success else RED, shade_in_3d=True).move_to(mlp_layers[-1][np.random.randint(0, 3)].get_center()) for _ in range(3)])
#             self.play(*[sig.animate.move_to(benzene_right.get_center()) for sig in signals_out], run_time=0.6)
#             self.play(FadeOut(signals_out, run_time=0.2))

#             self.play(target_benzene.animate.set_color(GREEN if success else RED), run_time=0.3)
#             self.play(target_benzene.animate.set_color(GREEN_C), FadeOut(status_text), run_time=0.3)

#         with self.voiceover(text="At first, the MLP simply memorizes the specific coordinate layout of the training dataset."):
#             simulate_flow(success=True, message="Baseline: MLP memorizes this layout.")

#         # Bài Test 1: Tịnh tiến (Translation)
#         with self.voiceover(text="But what happens if we shift the molecule in space?"):
#             self.play(input_group.animate.shift(UP * 0.8 + RIGHT * 0.3), run_time=1.0)

#         with self.voiceover(text="The absolute coordinates have completely changed. The standard MLP fails to recognize it."):
#             simulate_flow(success=False, message="1. Translation: Absolute positions changed. MLP fails!")

#         self.play(input_group.animate.shift(DOWN * 0.8 + LEFT * 0.3), run_time=0.5)

#         # Bài Test 2: Xoay khối tự do (Rotation)
#         with self.voiceover(text="What if the molecule is rotated?"):
#             # Xoay khối phân tử 3D thực sự quanh trục thẳng đứng để bộc lộ chiều sâu
#             self.play(Rotate(input_group, angle=PI / 3, axis=OUT, about_point=input_group.get_center()), run_time=1.2)

#         with self.voiceover(text="Physically it is identical, but the input values are altered. The network fails again."):
#             simulate_flow(success=False, message="2. Rotation: Matrix rotated. MLP fails!")

#         self.play(Rotate(input_group, angle=-PI / 3, axis=OUT, about_point=input_group.get_center()), run_time=0.5)

#         # Bài Test 3: Hoán vị thứ tự mảng (Permutation)
#         with self.voiceover(text="And what if we simply swap the order we feed the atoms into the system?"):
#             # VÁ LỖI Swap không tồn tại bằng move_to song song mượt mà
#             pos_0 = node_labels[0].get_center().copy()
#             pos_1 = node_labels[1].get_center().copy()
#             self.play(
#                 node_labels[0].animate.move_to(pos_1),
#                 node_labels[1].animate.move_to(pos_0),
#                 input_group[0][1][0].animate.set_color(YELLOW),
#                 input_group[0][1][1].animate.set_color(YELLOW),
#                 run_time=1.0
#             )

#         with self.voiceover(text="The structural topology is the same, but the MLP treats this permutation as completely new data."):
#             simulate_flow(success=False, message="3. Permutation: Atom indexing changed. MLP fails!")

#         # Khôi phục vị trí nhãn nguyên bản
#         self.play(
#             node_labels[0].animate.move_to(pos_0),
#             node_labels[1].animate.move_to(pos_1),
#             input_group[0][1][0].animate.set_color(BLUE_D),
#             input_group[0][1][1].animate.set_color(TEAL_D),
#             run_time=0.5
#         )

#         # ===================================================================
#         # SCENE 2: GIẢI PHÁP HỌC MÁY HÌNH HỌC (GEOMETRIC ML SAVES THE DAY)
#         # ===================================================================
#         # -----------------------------------------------------------------
#         # CẢI THIỆN 1: ĐỊNH NGHĨA LÕI ĐỒ THỊ GNN ĐỂ HIỂN THỊ TRONG BOX
#         # -----------------------------------------------------------------
#         geoml_box = RoundedRectangle(width=2.5, height=3.0, color=TEAL_E, fill_color=BLACK, fill_opacity=0.8, stroke_width=3)
#         geoml_box.move_to(mlp_layers.get_center())

#         # Mạng nơ-ron đồ thị thu nhỏ bên trong Box đại diện cho Message Passing
#         gnn_core_nodes = VGroup(*[Dot(radius=0.06, color=TEAL_C) for _ in range(6)])
#         gnn_positions = [UP*0.5, UR*0.35, DR*0.35, DOWN*0.5, DL*0.35, UL*0.35]
#         for node, pos in zip(gnn_core_nodes, gnn_positions):
#             node.move_to(geoml_box.get_center() + pos)
#         gnn_core_edges = VGroup(*[
#             Line(gnn_core_nodes[i].get_center(), gnn_core_nodes[j].get_center(), stroke_width=1.0, color=TEAL_E)
#             for i in range(6) for j in range(i+1, 6) if np.random.rand() > 0.5 # Kết nối ngẫu nhiên đẹp mắt
#         ])
#         gnn_core_group = VGroup(gnn_core_edges, gnn_core_nodes)

#         geoml_label = Text("Geometric GNN", font_size=20, color=TEAL_A).next_to(geoml_box, UP, buff=0.4)
#         self.add_fixed_in_frame_mobjects(geoml_box, geoml_label)

#         with self.voiceover(text="This is why we need Geometric Machine Learning. By replacing dense layers with an Equivariant Graph Neural Network, we bake symmetries directly into the architecture."):
#             self.play(
#                 ReplacementTransform(mlp_layers, geoml_box),
#                 FadeOut(mlp_edges),
#                 # FadeIn cấu trúc lõi GNN vào bên trong hộp đồng thời
#                 FadeIn(gnn_core_group, lag_ratio=0.1),
#                 ReplacementTransform(mlp_text, geoml_label),
#                 run_time=1.5
#             )

#         def geometric_flow(message=""):
#             status_text = Text(message, font_size=18, color=YELLOW).to_edge(DOWN, buff=0.8)
#             self.add_fixed_in_frame_mobjects(status_text)
#             self.play(Write(status_text), run_time=0.4)

#             # Hiệu ứng Ripples Message Passing lan tỏa mượt mà tan vào nền đen
#             ripples = VGroup(*[
#                 Circle(radius=0.05, color=TEAL_C, stroke_width=2).move_to(atom.get_center())
#                 for atom in input_group[0][1]
#             ])
#             self.play(FadeOut(ripples, scale=4.0), run_time=0.8)

#             stream = Line(input_group.get_center(), geoml_box.get_center(), color=TEAL_B, stroke_width=3)
#             self.play(ShowPassingFlash(stream, time_width=0.4), run_time=0.6)

#             out_stream = Line(geoml_box.get_center(), target_benzene.get_center(), color=GREEN, stroke_width=3)
#             self.play(ShowPassingFlash(out_stream, time_width=0.4), run_time=0.6)

#             self.play(target_benzene.animate.set_color(GREEN), run_time=0.2)
#             self.play(target_benzene.animate.set_color(GREEN_C), FadeOut(status_text), run_time=0.3)

#         with self.voiceover(text="The GNN processes data by passing messages along graph edges, inherently respecting molecular structure."):
#             geometric_flow("GNN Baseline: Processing underlying graph topology.")

#         # GNN Bài Test 1: Tịnh tiến thành công
#         with self.voiceover(text="Now, when the molecule translates..."):
#             # -----------------------------------------------------------------
#             # CẢI THIỆN 3: DÙNG ARROW3D & SHIFT Z CAO HẲN ĐỂ CHỐNG CHỒNG LẤN
#             # -----------------------------------------------------------------
#             # shift Z lên OUT * 4.0 để nổi hẳn lên trên tất cả các hộp Z=2, chống z-fighting răng cưa
#             vec_rij = Arrow3D(
#                 start=input_group[0][1][0].get_center(),
#                 end=input_group[0][1][1].get_center(),
#                 color=YELLOW, thickness=0.04
#             ).shift(OUT * 2.0)

#             math_rij = MathTex(r"\vec{r}_{ij} = x_i - x_j", font_size=20, color=YELLOW).next_to(input_group.get_center(), UP, buff=0.8).shift(OUT * 4.0)
#             self.add_fixed_in_frame_mobjects(math_rij)

#             self.play(Create(vec_rij), Write(math_rij))

#             # Gom lại để dịch chuyển đồng bộ
#             moving_elements = VGroup(input_group, vec_rij)
#             self.play(moving_elements.animate.shift(UP * 0.8 + RIGHT * 0.4), run_time=1.2)

#         with self.voiceover(text="The relative vectors between atoms remain perfectly invariant. The network recognizes it instantly. Success!"):
#             geometric_flow("Translation: Invariant relative vectors. Success!")

#         self.play(
#             FadeOut(vec_rij), FadeOut(math_rij),
#             input_group.animate.shift(DOWN * 0.8 + LEFT * 0.4),
#             run_time=0.5
#         )

#         # GNN Bài Test 2: Xoay thành công (Đồng biến đồng bộ hoàn hảo)
#         with self.voiceover(text="When the input molecule is rotated, the internal features rotate synchronously."):
#             self.play(
#                 Rotate(input_group, angle=PI / 2, axis=OUT, about_point=input_group.get_center()),
#                 # Kịch bản đồng bộ: Đầu vào xoay thì hệ thống đầu ra cũng xoay nhịp nhàng theo
#                 Rotate(benzene_right, angle=PI / 2, axis=OUT, about_point=benzene_right.get_center()),
#                 run_time=1.8
#             )

#         with self.voiceover(text="The target adjusts in perfect symmetry. This is equivariance in action!"):
#             geometric_flow("Rotation: Robust equivariant symmetry. Success!")

#         # Dọn dẹp nhóm kết thúc phần 2
#         self.play(*[FadeOut(m) for m in self.mobjects], run_time=1.0)

#         # ===================================================================
#         # SCENE 3: SƠ ĐỒ QUY TRÌNH KHOA HỌC (SCIENTIFIC COMPUTING LOOP)
#         # ===================================================================
#         # -----------------------------------------------------------------
#         # CẢI THIỆN: CHUYỂN CAMERA TOP-DOWN phẳng lặng để sơ đồ sắc nét
#         # -----------------------------------------------------------------
#         self.move_camera(phi=0, theta=-90 * DEGREES, run_time=1.2)

#         goal_text = Text("Goal: Advancing atomistic science", font_size=32, color=WHITE).to_edge(UP, buff=0.5)
#         self.add_fixed_in_frame_mobjects(goal_text)

#         with self.voiceover(text="Ultimately, our goal is to advance atomistic science."):
#             self.play(Write(goal_text), run_time=1.5)

#         box_system = Rectangle(width=3, height=1.5, color=TEAL, fill_opacity=0.3).shift(LEFT * 4 + DOWN * 0.5)
#         text_system = Text("Atomistic\nSystem", font_size=24).move_to(box_system.get_center())
#         sys_group = VGroup(box_system, text_system)

#         box_props = Rectangle(width=3.5, height=1.5, color=GREEN_E, fill_opacity=0.3).shift(RIGHT * 4 + DOWN * 0.5)
#         text_props = Text("Properties\n(energy, forces)", font_size=24).move_to(box_props.get_center())
#         prop_group = VGroup(box_props, text_props)

#         self.add_fixed_in_frame_mobjects(sys_group, prop_group)

#         with self.voiceover(text="We often need to map an atomistic system to its physical properties, like energy and forces."):
#             self.play(FadeIn(sys_group), FadeIn(prop_group), run_time=1.5)

#         arc_top = ArcBetweenPoints(box_system.get_top() + RIGHT * 0.5, box_props.get_top() + LEFT * 0.5, angle=-PI / 4)
#         path_slow = DashedVMobject(arc_top, num_dashes=20, color=GRAY)
#         arrow_tip_slow = RegularPolygon(n=3, fill_opacity=1, color=GRAY).scale(0.15).rotate(-PI / 6).move_to(arc_top.get_end())
#         text_slow = Text("Scientific computing", font_size=20, color=GRAY).next_to(arc_top, UP, buff=0.1)

#         self.add_fixed_in_frame_mobjects(path_slow, arrow_tip_slow, text_slow)

#         with self.voiceover(text="Traditional scientific computing methods do this, but they are notoriously slow and expensive."):
#             self.play(Create(path_slow), FadeIn(arrow_tip_slow), Write(text_slow), run_time=2.5)

#         arc_bottom = CurvedArrow(box_props.get_bottom() + LEFT * 0.5, box_system.get_bottom() + RIGHT * 0.5, angle=-PI / 4, color=WHITE)
#         text_bottom = Text("Design (medicine)", font_size=20, color=WHITE).next_to(arc_bottom, DOWN, buff=0.2)

#         self.add_fixed_in_frame_mobjects(arc_bottom, text_bottom)

#         with self.voiceover(text="This loop is critical for fields like medicine design."):
#             self.play(Create(arc_bottom), Write(text_bottom), run_time=1.5)

#         fast_arrow = CurvedArrow(box_system.get_right(), box_props.get_left(), angle=-PI / 6, color=RED, stroke_width=6)
#         accel_text = Text("GeoML can accelerate this!", font_size=20, color=RED).next_to(fast_arrow, UP, buff=0.2)

#         self.add_fixed_in_frame_mobjects(fast_arrow, accel_text)

#         with self.voiceover(text="Geometric Machine Learning acts as a powerful accelerator, drastically speeding up these computations while preserving strict physical laws."):
#             self.play(Write(accel_text), run_time=1.0)
#             self.play(
#                 FadeOut(path_slow), FadeOut(arrow_tip_slow), FadeOut(text_slow),
#                 Create(fast_arrow, rate_func=rush_into),
#                 run_time=0.6
#             )
#             self.play(box_props.animate.set_color(RED).set_fill(opacity=0.6), run_time=0.3)
#             self.play(box_props.animate.set_color(GREEN_E).set_fill(opacity=0.3), run_time=0.3)

#         self.play(*[FadeOut(m) for m in self.mobjects], run_time=1.5)


## slide 52

In [ ]:
#@title ref
# %%manim -qh -v WARNING AtomisticGeoMLIII

# class AtomisticGeoMLIII(VoiceoverScene):
#     def construct(self):
#         # Khởi tạo giọng đọc tiếng Anh chuyên nghiệp (en-US-AndrewNeural)
#         self.set_speech_service(EdgeTTSService(voice="en-US-AndrewNeural"))

#         # --- 1. TIÊU ĐỀ CHÍNH ---
#         title = Text("Equivariant Graph Transformers in Atomistic ML", font_size=32, weight=BOLD).to_edge(UP)
#         self.play(FadeIn(title, shift=DOWN * 0.2))

#         # Khởi tạo cấu trúc phân tử H2O thực tế (Oxygen ở giữa, 2 Hydrogen 2 bên)
#         pos_O = np.array([0, 0.5, 0])
#         pos_H1 = np.array([-1.2, -0.6, 0])
#         pos_H2 = np.array([1.2, -0.6, 0])

#         atom_O = Dot(pos_O, radius=0.28, color=RED_D)
#         lbl_O = MathTex(r"\text{O}", font_size=24, color=WHITE).move_to(pos_O)
#         node_O = VGroup(atom_O, lbl_O)

#         atom_H1 = Dot(pos_H1, radius=0.20, color=WHITE)
#         lbl_H1 = MathTex(r"\text{H}_1", font_size=20, color=BLACK).move_to(pos_H1)
#         node_H1 = VGroup(atom_H1, lbl_H1)

#         atom_H2 = Dot(pos_H2, radius=0.20, color=WHITE)
#         lbl_H2 = MathTex(r"\text{H}_2", font_size=20, color=BLACK).move_to(pos_H2)
#         node_H2 = VGroup(atom_H2, lbl_H2)

#         bonds = VGroup(
#             Line(pos_O, pos_H1, stroke_width=4, color=GRAY_C),
#             Line(pos_O, pos_H2, stroke_width=4, color=GRAY_C)
#         )

#         # Thước đo đơn vị thực tế (Angstrom Scale Bar)
#         scale_line = Line(LEFT*0.5, RIGHT*0.5, stroke_width=2, color=GRAY_A).to_corner(DL).shift(UP*0.5 + RIGHT*0.5)
#         scale_lbl = MathTex(r"1.0 \text{ \AA}", font_size=18).next_to(scale_line, UP, buff=0.1)
#         scale_bar = VGroup(scale_line, scale_lbl)

#         molecule = VGroup(bonds, node_O, node_H1, node_H2)

#         with self.voiceover("In atomistic systems, molecules are modeled as geometric graphs where node positions are measured precisely in Angstroms."):
#             self.play(Create(bonds))
#             self.play(FadeIn(node_O), FadeIn(node_H1), FadeIn(node_H2))
#             self.play(FadeIn(scale_bar))
#             self.wait(0.5)

#         # --- 2. DỊCH CHUYỂN VÀ GIẢI THÍCH PHƯƠNG TRÌNH NĂNG LƯỢNG LỰC ---
#         with self.voiceover("Our core goal is to predict the total potential energy E of the system, represented as a scalar invariant value."):
#             # Dịch chuyển phân tử và thước đo sang cánh trái màn hình
#             self.play(
#                 molecule.animate.shift(LEFT * 3.5),
#                 scale_bar.animate.shift(LEFT * 1.5),
#                 run_time=1.2
#             )

#             # Khởi tạo phương trình toán học chi tiết kèm đơn vị vật lý (eV, eV/A)
#             energy_eq = MathTex(r"E_{\text{predicted}} \in \mathbb{R} \quad [\text{eV}]", font_size=32, color=YELLOW).move_to(RIGHT * 3 + UP * 1.0)
#             force_eq = MathTex(r"F_i = -\nabla_{r_i} E \quad \left[\frac{\text{eV}}{\text{\AA}}\right]", font_size=32, color=ORANGE).next_to(energy_eq, DOWN, buff=0.6)
#             formulas = VGroup(energy_eq, force_eq)

#             self.play(Write(energy_eq))

#         with self.voiceover("By leveraging conservative physics and automatic differentiation, the atomic forces F-sub-i are derived directly as the negative gradient of energy."):
#             self.play(Write(force_eq))
#             self.wait(0.5)

#         # --- 3. VECTOR LỰC ĐẦU RA (FORCE VECTORS) ---
#         # Thiết lập các vector lực dựa theo vị trí mới sau khi đã tịnh tiến phân tử sang trái
#         f_vec_O = Arrow(node_O.get_center(), node_O.get_center() + UP*1.2 + RIGHT*0.4, color=ORANGE, stroke_width=4, buff=0)
#         f_vec_H1 = Arrow(node_H1.get_center(), node_H1.get_center() + DOWN*0.8 + LEFT*0.6, color=ORANGE, stroke_width=4, buff=0)
#         f_vec_H2 = Arrow(node_H2.get_center(), node_H2.get_center() + DOWN*0.6 + RIGHT*0.5, color=ORANGE, stroke_width=4, buff=0)

#         lbl_f_O = MathTex(r"F_{\text{O}}", font_size=20, color=ORANGE).next_to(f_vec_O.get_end(), UR, buff=0.05)
#         lbl_f_H1 = MathTex(r"F_{\text{H1}}", font_size=20, color=ORANGE).next_to(f_vec_H1.get_end(), DL, buff=0.05)
#         lbl_f_H2 = MathTex(r"F_{\text{H2}}", font_size=20, color=ORANGE).next_to(f_vec_H2.get_end(), DR, buff=0.05)

#         forces = VGroup(f_vec_O, f_vec_H1, f_vec_H2, lbl_f_O, lbl_f_H1, lbl_f_H2)

#         with self.voiceover("This elegant formulation guarantees that the predicted force vectors align perfectly with the true quantum mechanical landscape."):
#             self.play(Create(VGroup(f_vec_O, f_vec_H1, f_vec_H2)))
#             self.play(Write(VGroup(lbl_f_O, lbl_f_H1, lbl_f_H2)))
#             self.wait(0.5)

#         # --- 4. CHỨNG MINH TÍNH ĐỒNG BIẾN (EQUIVARIANCE PROOF ANIMATION) ---
#         # Gộp toàn bộ hệ thống phân tử và vector lực thành một khối liên kết hình học thống nhất
#         equivariant_system = VGroup(molecule, forces)

#         equiv_definition = MathTex(r"f(R \cdot r) = R \cdot f(r)", font_size=36, color=YELLOW).to_edge(DOWN, buff=0.6)

#         with self.voiceover("A fundamental constraint here is equivariance. If the molecule rotates in space, the predicted force vectors must rotate in exact lockstep."):
#             self.play(Write(equiv_definition))
#             # 3b1b Style: Xoay toàn bộ hệ thống 90 độ để chứng minh cấu trúc không đổi
#             self.play(Rotate(equivariant_system, angle=PI/2, about_point=molecule.get_center()), run_time=2.0, rate_func=smooth)
#             self.play(Indicate(equiv_definition, color=YELLOW))
#             self.wait(1)

#         # Dọn dẹp không gian để chuyển sang phân cảnh kiến trúc Equiformer
#         self.play(FadeOut(equivariant_system), FadeOut(formulas), FadeOut(equiv_definition), FadeOut(scale_bar))

#         # --- 5. KIẾN TRÚC EQUIFORMER VÀ CƠ CHẾ CHÚ Ý ĐỒ THỊ (EQUIVARIANT GRAPH ATTENTION) ---
#         equi_text = Text("Equiformer: Equivariant Graph Transformers", color=GREEN, font_size=26).next_to(title, DOWN, buff=0.3)
#         self.play(Write(equi_text))

#         # Đưa phân tử quay trở lại tâm màn hình để phân tích cơ chế Attention nội bộ
#         molecule.move_to(DOWN * 0.5)
#         self.play(FadeIn(molecule))

#         # Tạo mạng lưới liên kết Attention giữa tất cả các cặp nguyên tử (Fully-connected Graph)
#         attn_line_1 = DashedLine(node_O.get_center(), node_H1.get_center(), color=GREEN_A, dash_length=0.08)
#         attn_line_2 = DashedLine(node_O.get_center(), node_H2.get_center(), color=GREEN_A, dash_length=0.08)
#         attn_line_3 = DashedLine(node_H1.get_center(), node_H2.get_center(), color=GREEN_A, dash_length=0.08)

#         lbl_alpha_1 = MathTex(r"\alpha_{O, H1}", font_size=20, color=GREEN_A).move_to(attn_line_1.get_center() + LEFT*0.3 + UP*0.2)
#         lbl_alpha_2 = MathTex(r"\alpha_{O, H2}", font_size=20, color=GREEN_A).move_to(attn_line_2.get_center() + RIGHT*0.3 + UP*0.2)

#         attention_machinery = VGroup(attn_line_1, attn_line_2, attn_line_3, lbl_alpha_1, lbl_alpha_2)

#         with self.voiceover("Equiformer achieves this by leveraging Equivariant Graph Attention. Instead of scalar weights, it processes directional tensor messages."):
#             self.play(Create(VGroup(attn_line_1, attn_line_2, attn_line_3)))
#             self.play(Write(lbl_alpha_1), Write(lbl_alpha_2))

#         with self.voiceover("By passing irreducible representations across these attention boundaries, the model captures non-local quantum interactions cleanly, guaranteeing geometric rigor."):
#             # Tạo hiệu ứng sóng năng lượng chạy dọc theo các liên kết Attention
#             self.play(
#                 ShowPassingFlash(attn_line_1.copy().set_color(YELLOW).set_stroke(width=4), time_width=0.4),
#                 ShowPassingFlash(attn_line_2.copy().set_color(YELLOW).set_stroke(width=4), time_width=0.4),
#                 ShowPassingFlash(attn_line_3.copy().set_color(YELLOW).set_stroke(width=4), time_width=0.4),
#                 run_time=1.5
#             )
#             self.play(Indicate(molecule, color=GREEN_A))
#             self.wait(2)


In [ ]:
%%manim -qh -v WARNING AtomisticGeoMLII52

from manim import *
import numpy as np
import asyncio
import threading
import hashlib
import os
from pathlib import Path
from manim_voiceover import VoiceoverScene
from manim_voiceover.services.base import SpeechService

# ─── CUSTOM EDGETTS SERVICE (ĐẢM BẢO CHẠY MƯỢT MÀ TRÊN COLAB) ───
class EdgeTTSService(SpeechService):
    def __init__(self, voice="en-US-GuyNeural", **kwargs):
        self.voice = voice
        super().__init__(**kwargs)

    def generate_from_text(self, text: str, cache_dir: str = None, path: str = None) -> dict:
        import edge_tts
        if cache_dir is None: cache_dir = "media/voiceovers"
        cache_dir_path = Path(cache_dir)
        os.makedirs(cache_dir_path, exist_ok=True)
        if path is None:
            hash_name = hashlib.md5(text.encode()).hexdigest()
            path = f"{hash_name}.mp3"
        final_path_str = str(cache_dir_path / path)
        async def _download():
            communicate = edge_tts.Communicate(text, self.voice)
            await communicate.save(final_path_str)
        def _worker():
            loop = asyncio.new_event_loop()
            asyncio.set_event_loop(loop)
            try: loop.run_until_complete(_download())
            finally: loop.close()
        thread = threading.Thread(target=_worker)
        thread.start()
        thread.join()
        return {"original_audio": path}

# ─── MAIN SCENE GRAPH ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
class AtomisticGeoMLII52(VoiceoverScene):
    def construct(self):
        self.camera.background_color = BLACK
        self.set_speech_service(EdgeTTSService(voice="en-US-GuyNeural"))

        # --- 1. TITLE ---
        title = Text("Atomistic Systems: Energy & Equivariant Forces", font_size=36, weight=BOLD).to_edge(UP)
        self.play(FadeIn(title))

        # Khởi tạo đồ thị nguyên tử (Molecule) chuyên nghiệp hơn
        node_coords = [np.array([-1.5, -1, 0]), np.array([1, 1.5, 0]), np.array([1.5, -1, 0]), np.array([0, 0, 0])]
        nodes = VGroup(*[Circle(radius=0.25, fill_color=BLUE_D, fill_opacity=0.8, stroke_color=BLUE_B, stroke_width=2).move_to(pos) for pos in node_coords])
        edges = VGroup(*[Line(nodes[i].get_center(), nodes[j].get_center(), color=GRAY_B, stroke_width=2)
                         for i in range(4) for j in range(i+1, 4)
                         if np.linalg.norm(nodes[i].get_center() - nodes[j].get_center()) < 2.5])

        molecule = VGroup(edges, nodes)

        with self.voiceover(text="In atomistic machine learning, our primary goal is to predict the potential energy of a molecule, and the forces acting on each atom."):
            self.play(Create(edges), FadeIn(nodes, lag_ratio=0.1), run_time=1.5)
            self.play(molecule.animate.shift(LEFT * 3.5))

        # --- 2. THE MATHEMATICAL PROOF (CHAIN RULE) ---
        proof_title = Text("Mathematical Guarantee via Auto-Grad", font_size=20, color=YELLOW).move_to(RIGHT * 3.0 + UP * 2.0)
        self.play(Write(proof_title))

        # Bước 1: Năng lượng là đại lượng vô hướng bất biến (Invariant Scalar)
        eq_energy = MathTex(r"1.\, \text{Energy is Invariant: }", r"E(", r"R", r"X", r")", r"=", r"E(", r"X", r")", font_size=32).next_to(proof_title, DOWN, buff=0.5, aligned_edge=LEFT)
        eq_energy.set_color_by_tex("R", ORANGE)

        # Bước 2: Lực là đạo hàm của năng lượng
        eq_force = MathTex(r"2.\, \text{Force is the Gradient: }", r"F(", r"X", r")", r"=", r"-\nabla_{", r"X", r"} E(", r"X", r")", font_size=32).next_to(eq_energy, DOWN, buff=0.4, aligned_edge=LEFT)

        with self.voiceover(text="Physics tells us that Energy E is a scalar value that is strictly invariant to 3D rotations."):
            self.play(Write(eq_energy))

        with self.voiceover(text="And the Force F is a vector field, defined exactly as the negative gradient of the Energy with respect to atomic coordinates X."):
            self.play(Write(eq_force))

        # Bước 3: Chứng minh (Chain Rule)
        proof_step1 = MathTex(r"\Rightarrow F(", r"R", r"X", r") = -\nabla_{", r"R", r"X", r"} E(", r"R", r"X", r")", font_size=32).next_to(eq_force, DOWN, buff=0.6, aligned_edge=LEFT).shift(RIGHT*0.5)
        proof_step1.set_color_by_tex("R", ORANGE)

        proof_step2 = MathTex(r"= -", r"R", r" \cdot \nabla_{", r"X", r"} E(", r"X", r") \quad \text{(Chain Rule)}", font_size=32).next_to(proof_step1, DOWN, buff=0.3, aligned_edge=LEFT).align_to(proof_step1[3], LEFT)
        proof_step2.set_color_by_tex("R", ORANGE)

        proof_step3 = MathTex(r"= ", r"R", r" \cdot F(", r"X", r") \quad \textbf{(Equivariant!)}", font_size=32, color=GREEN_C).next_to(proof_step2, DOWN, buff=0.3, aligned_edge=LEFT).align_to(proof_step1[3], LEFT)
        proof_step3.set_color_by_tex("R", ORANGE)

        with self.voiceover(text="Let's see what happens if we rotate the molecule by a matrix R. By substituting R X into the gradient..."):
            self.play(Write(proof_step1))

        with self.voiceover(text="...the multi-variable chain rule naturally pulls the rotation matrix R outside of the gradient operator."):
            self.play(Write(proof_step2))
            self.play(Indicate(proof_step2[1], color=ORANGE, scale_factor=1.5)) # Nhấn mạnh R văng ra ngoài

        with self.voiceover(text="This elegantly proves that modeling an invariant energy automatically guarantees a perfectly equivariant force field via backpropagation."):
            self.play(Write(proof_step3))
            self.play(Flash(proof_step3, color=GREEN_C, line_length=0.3))
            self.wait(0.5)

        # --- 3. TRỰC QUAN HÓA LỰC (FORCE VECTORS & ROTATION) ---
        np.random.seed(42)
        force_vectors = VGroup()
        for node in nodes:
            direction = np.random.uniform(-1, 1, 3)
            direction[2] = 0
            direction = direction / np.linalg.norm(direction) * 1.2
            # Sử dụng vector update để nó dính chặt vào node khi xoay
            arrow = Arrow(start=ORIGIN, end=direction, color=RED_C, buff=0, stroke_width=4).shift(node.get_center())
            force_vectors.add(arrow)

        # Gắn mác F cho các vector
        force_labels = VGroup(*[MathTex(r"\vec{F}", font_size=20, color=RED_A).next_to(arrow.get_end(), normalize(arrow.get_end() - arrow.get_start()), buff=0.1) for arrow in force_vectors])

        # Nhóm toàn bộ phân tử và vector lực lại để xoay cùng nhau
        physics_system = VGroup(molecule, force_vectors, force_labels)

        with self.voiceover(text="Visually, these auto-grad force vectors are attached to each atom."):
            self.play(Create(force_vectors), FadeIn(force_labels))

        with self.voiceover(text="If we rotate the input geometry, the force vectors gracefully rotate in absolute synchrony. That is the essence of Equivariance."):
            # Sửa lỗi: Đổi in_and_out thành there_and_back
            self.play(
                Rotate(physics_system, angle=PI*2/3, about_point=LEFT * 3.5),
                run_time=2.5, rate_func=there_and_back
            )
            self.wait(1)

        # --- 4. EQUIFORMER VÀ CƠ CHẾ GLOBAL ATTENTION ---
        # Dọn dẹp bảng toán học bên phải
        self.play(FadeOut(VGroup(proof_title, eq_energy, eq_force, proof_step1, proof_step2, proof_step3)))

        equi_text = Text("Equivariant Transformers (Equiformer)", color=TEAL_C, font_size=28).next_to(title, DOWN, buff=0.4)
        attn_formula = MathTex(r"m_{ij} = \text{Attn}(h_i, h_j) \otimes Y_l^m(\hat{r}_{ij})", font_size=36, color=YELLOW).move_to(RIGHT * 3.5 + UP * 0.5)
        desc_attn = Text("Spherical Harmonics ensure geometric routing.", font_size=18, color=LIGHT_GREY).next_to(attn_formula, DOWN, buff=0.3)

        with self.voiceover(text="To maximize this capability, modern architectures like the Equiformer replace local message passing with a Global Attention mechanism."):
            self.play(Write(equi_text))
            self.play(physics_system.animate.move_to(LEFT * 3.5 + DOWN * 0.5)) # Dịch nhẹ để cân bằng

            # Xóa các vector lực để tập trung vào Attention
            self.play(FadeOut(force_vectors), FadeOut(force_labels))

        with self.voiceover(text="Every atom is directly connected to every other atom. By modulating these connections with Spherical Harmonics, the model captures complex 3D structures flawlessly."):
            self.play(Write(attn_formula), FadeIn(desc_attn))

            # Tạo các đường nét đứt (Global Attention) kết nối mọi node (Đồ thị đầy đủ - Complete Graph)
            attn_lines = VGroup()
            for i in range(len(nodes)):
                for j in range(len(nodes)):
                    if i != j: # Tất cả các cặp
                        line = DashedLine(nodes[i].get_center(), nodes[j].get_center(), color=YELLOW_E, dash_length=0.08, stroke_width=1.5)
                        attn_lines.add(line)

            # Vẽ các đường Attention mượt mà và nhấp nháy năng lượng
            self.play(Create(attn_lines, lag_ratio=0.05), run_time=2)
            self.play(
                attn_lines.animate.set_color(YELLOW_A).set_stroke(width=3),
                nodes.animate.set_fill(YELLOW_D).set_stroke(YELLOW_A),
                run_time=0.5
            )
            self.play(
                attn_lines.animate.set_color(YELLOW_E).set_stroke(width=1.5),
                nodes.animate.set_fill(BLUE_D).set_stroke(BLUE_B),
                run_time=1.0
            )

        # Chuyển cảnh kết thúc
        self.play(FadeOut(Group(*self.mobjects)))


## slide 53-54
Symmetries in robotics I + II

In [ ]:
%%manim -qh -v WARNING symmetriesInRobotics53

class symmetriesInRobotics53(ThreeDScene, VoiceoverScene):
    def construct(self):
        self.set_speech_service(EdgeTTSService(voice="en-US-GuyNeural"))

        # --- 1. CAMERA & FORMULA SETUP ---
        self.set_camera_orientation(phi=60 * DEGREES, theta=-45 * DEGREES)

        title = Text("Symmetries in robotics", font_size=36, weight=BOLD).to_edge(UP)

        eq = MathTex(
            r"\pi^*(", r"g \cdot ", r"\text{state}", r")", r"=",
            r"g \cdot ", r"\pi^*(", r"\text{state}", r")",
            font_size=42, color=WHITE
        ).to_edge(DOWN, buff=0.8)

        eq[1].set_color(GREEN)
        eq[5].set_color(GREEN)
        eq[2].set_color(YELLOW)
        eq[7].set_color(YELLOW)

        self.add_fixed_in_frame_mobjects(title, eq)
        self.remove(title, eq)

        # --- 2. ENVIRONMENT & ROBOT ---
        floor = VGroup(*[
            Square(side_length=1, fill_color=GRAY_A if (i + j) % 2 == 0 else GRAY_C, fill_opacity=0.8, stroke_width=0.5).move_to([i, j, 0])
            for i in range(-2, 3) for j in range(-2, 3)
        ])

        base = Cylinder(radius=0.4, height=0.5, color=DARK_GREY).move_to([0, 0, 0.25])

        t1 = ValueTracker(-PI/4)
        t2 = ValueTracker(PI/4)
        g_z = ValueTracker(0.5)
        g_open = ValueTracker(0.25) # Bộ điều khiển khoảng cách mở của 2 cần gắp

        # Khởi tạo cấu trúc cố định (8 thành phần mesh đặc)
        arm1 = Cylinder(radius=0.14, height=2, color=BLUE_D)
        joint1 = Sphere(radius=0.18, color=ORANGE)
        arm2 = Cylinder(radius=0.10, height=1.5, color=BLUE_B)
        joint2 = Sphere(radius=0.14, color=ORANGE)
        gripper_stem = Cylinder(radius=0.06, height=0.5, color=LIGHT_GREY)
        gripper_palm = Cylinder(radius=0.04, height=0.6, color=GRAY_A)
        finger1 = Cylinder(radius=0.03, height=0.25, color=LIGHT_GREY)
        finger2 = Cylinder(radius=0.03, height=0.25, color=LIGHT_GREY)

        robot_parts = VGroup(arm1, joint1, arm2, joint2, gripper_stem, gripper_palm, finger1, finger2)

        def update_robot(mob):
            th1 = t1.get_value()
            th2 = t2.get_value()
            z = g_z.get_value()
            d_open = g_open.get_value()

            pt0 = np.array([0, 0, 0.5])
            pt1 = np.array([2 * np.cos(th1), 2 * np.sin(th1), 0.5])
            pt2 = pt1 + np.array([1.5 * np.cos(th1 + th2), 1.5 * np.sin(th1 + th2), 0])

            # 1. Cập nhật Arm 1
            new_arm1 = Cylinder(radius=0.14, height=2, color=BLUE_D)
            new_arm1.rotate(PI/2, axis=UP).rotate(th1, axis=OUT)
            new_arm1.move_to((pt0 + pt1) / 2)
            mob[0].match_points(new_arm1)

            # 2. Cập nhật Joint 1
            mob[1].move_to(pt1)

            # 3. Cập nhật Arm 2
            new_arm2 = Cylinder(radius=0.10, height=1.5, color=BLUE_B)
            new_arm2.rotate(PI/2, axis=UP).rotate(th1 + th2, axis=OUT)
            new_arm2.move_to((pt1 + pt2) / 2)
            mob[2].match_points(new_arm2)

            # 4. Cập nhật Joint 2
            mob[3].move_to(pt2)

            # 5. Cập nhật Trục tịnh tiến Gripper Stem
            g_height = max(0.5 - z, 0.01)
            new_gripper_stem = Cylinder(radius=0.06, height=g_height, color=LIGHT_GREY)
            new_gripper_stem.move_to([pt2[0], pt2[1], (0.5 + z) / 2])
            mob[4].match_points(new_gripper_stem)

            # Điểm trung tâm của bàn tay kẹp
            pt_wrist = np.array([pt2[0], pt2[1], z])
            phi_perp = (th1 + th2) + PI/2
            vec_perp = np.array([np.cos(phi_perp), np.sin(phi_perp), 0])

            # 6. Cập nhật Thanh ngang khớp gắp (Gripper Palm)
            new_palm = Cylinder(radius=0.04, height=0.6, color=GRAY_A)
            new_palm.rotate(PI/2, axis=UP).rotate(phi_perp, axis=OUT)
            new_palm.move_to(pt_wrist)
            mob[5].match_points(new_palm)

            # 7. Cập nhật Cần gắp 1 (Finger 1)
            pt_f1 = pt_wrist + vec_perp * d_open - np.array([0, 0, 0.125])
            new_finger1 = Cylinder(radius=0.03, height=0.25, color=LIGHT_GREY)
            new_finger1.move_to(pt_f1)
            mob[6].match_points(new_finger1)

            # 8. Cập nhật Cần gắp 2 (Finger 2)
            pt_f2 = pt_wrist - vec_perp * d_open - np.array([0, 0, 0.125])
            new_finger2 = Cylinder(radius=0.03, height=0.25, color=LIGHT_GREY)
            new_finger2.move_to(pt_f2)
            mob[7].match_points(new_finger2)

        robot_parts.add_updater(update_robot)

        # Đồng bộ hóa tọa độ hình học khối mục tiêu chuẩn xác 100% với tầm gắp
        target_block = Cube(side_length=0.3, fill_color=YELLOW, fill_opacity=1, stroke_width=1).move_to([2, 1.5, 0.15])

        with self.voiceover(text="Let's build an articulated robot arm to see exactly how equivariance ties the math to physical motion."):
            self.play(FadeIn(floor), FadeIn(base))
            self.play(FadeIn(robot_parts), FadeIn(target_block))
            self.play(Write(title), Write(eq))
            self.wait(0.5)

        # --- 3. ĐỒNG BỘ HÀNH ĐỘNG GẮP THỰC TẾ ---
        with self.voiceover(text="First, the optimal policy maps the current state to a specific sequence of joint angles."):
            self.play(Indicate(eq[6:9], scale_factor=1.2, color=BLUE))
            # Di chuyển hệ thống cánh tay đến vị trí phía trên vật thể
            self.play(t1.animate.set_value(0), t2.animate.set_value(PI/2), run_time=1.5)
            # Hạ trục kẹp gắp xuống vị trí mục tiêu (Cần gắp vẫn đang mở)
            self.play(g_z.animate.set_value(0.25), run_time=0.5)
            # Thực hiện hành động gắp: Khép chặt 2 cần kẹp vào khối Cube
            self.play(g_open.animate.set_value(0.14), run_time=0.3)
            # Nhấc bổng kẹp gắp lên cao
            self.play(g_z.animate.set_value(0.5), run_time=0.5)

        with self.voiceover(text="Now, let's return the arm to its resting position."):
            # Mở cần kẹp để giải phóng không gian vật thể trước khi thu cánh tay về
            self.play(g_open.animate.set_value(0.25), run_time=0.3)
            self.play(t1.animate.set_value(-PI/4), t2.animate.set_value(PI/4), run_time=1)

        # --- 4. BIẾN ĐỔI XOAY ĐỒNG BỘ 90 ĐỘ ---
        with self.voiceover(text="What happens if the target state undergoes a ninety-degree rotational transformation?"):
            self.play(Indicate(eq[1:3], scale_factor=1.2, color=YELLOW))
            self.play(Rotate(target_block, angle=PI/2, about_point=ORIGIN), run_time=1.5)

        # --- 5. ĐỒNG BIẾN THỰC THI (EQUIVARIANT GRABBING) ---
        with self.voiceover(text="Thanks to the equivariant architecture, the policy doesn't need to learn a new path from scratch."):
            self.play(Indicate(eq[5:9], scale_factor=1.2, color=BLUE))

        with self.voiceover(text="It perfectly commutes the geometry. The base joint simply absorbs the ninety-degree shift, while the relative elbow angle remains completely unchanged."):
            # Di chuyển xoay góc nền hấp thụ hoàn toàn biến đổi 90 độ
            self.play(t1.animate.set_value(PI/2), t2.animate.set_value(PI/2), run_time=2)
            # Hạ kẹp gắp đặc xuống mục tiêu mới sau dịch chuyển
            self.play(g_z.animate.set_value(0.25), run_time=0.5)
            # Khép chặt 2 cần kẹp gắp vật thể lần hai tại góc xoay mới
            self.play(g_open.animate.set_value(0.14), run_time=0.3)
            # Hoàn thành chu kỳ gắp đồng biến
            self.play(g_z.animate.set_value(0.5), run_time=0.5)

        self.wait(1)


## slide 55

In [ ]:
%%manim -qh -v WARNING SymmetryBreaking55

# ─── MAIN SCENE GRAPH ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
class SymmetryBreaking55(VoiceoverScene):
    def construct(self):
        self.camera.background_color = BLACK
        self.set_speech_service(EdgeTTSService(voice="en-US-GuyNeural"))

        # --- 1. TIÊU ĐỀ & KHẲNG ĐỊNH ---
        title = Text("The Symmetry Breaking", font_size=40, weight=BOLD).to_edge(UP, buff=0.2)
        claim = Text("Equivariant architectures cannot break symmetry!", color=RED_C, font_size=24)
        claim.next_to(title, DOWN, buff=0.2)

        self.play(FadeIn(title), Write(claim))

        # --- 2. THÍ NGHIỆM VẬT LÝ: CÂY BÚT CHÌ THĂNG BẰNG TỐI ĐA ---
        # Tạo mặt bàn 3D giả lập (Ellipse)
        table = Ellipse(width=5.0, height=1.5, color=DARK_GREY, fill_opacity=0.6, stroke_width=2).move_to(LEFT * 3.5 + DOWN * 1.5)

        # Tạo cây bút chì đứng thẳng hoàn hảo tại tâm
        pencil_body = Rectangle(width=0.15, height=2.0, fill_color=YELLOW_D, fill_opacity=1, stroke_width=1).move_to(table.get_center() + UP * 1.0)
        pencil_tip = Polygon(
            pencil_body.get_corner(DL),
            pencil_body.get_corner(DR),
            pencil_body.get_center() + DOWN * 1.3,
            fill_color=DARK_BROWN, fill_opacity=1, stroke_width=0
        )
        pencil_lead = Polygon(
            pencil_tip.get_vertices()[0]*0.2 + pencil_tip.get_vertices()[2]*0.8,
            pencil_tip.get_vertices()[1]*0.2 + pencil_tip.get_vertices()[2]*0.8,
            pencil_tip.get_vertices()[2],
            fill_color=BLACK, fill_opacity=1, stroke_width=0
        )
        pencil = VGroup(pencil_body, pencil_tip, pencil_lead)

        label_x = MathTex(r"\text{Input State } x", font_size=24, color=YELLOW).next_to(pencil, UP)

        with self.voiceover(text="Imagine a pencil balancing perfectly upright on a table. This system has perfect rotational symmetry. You can look at it from any angle, and the state x remains identical."):
            self.play(FadeIn(table), FadeIn(pencil), Write(label_x))

            # Mô phỏng góc nhìn camera quay quanh bút chì (Bằng cách vẽ quỹ đạo quay Ellipse mượt)
            orbit = Ellipse(width=3.5, height=1.0, color=YELLOW_A, stroke_width=2, stroke_opacity=0.5).move_to(table.get_center())
            camera_eye = Dot(color=YELLOW).move_to(orbit.point_from_proportion(0))
            self.play(Create(orbit), FadeIn(camera_eye))
            self.play(MoveAlongPath(camera_eye, orbit), run_time=2.5, rate_func=linear)
            self.play(FadeOut(orbit), FadeOut(camera_eye))

        # --- 3. CHỨNG MINH TOÁN HỌC TẠI SAO BỊ KẸT ---
        proof_bg = RoundedRectangle(width=6.0, height=3.5, corner_radius=0.2, fill_color=GRAY_E, fill_opacity=0.4, stroke_color=GRAY).move_to(RIGHT * 3.5 + DOWN * 0.5)

        eq1 = MathTex(r"\text{Output } y = f(x)", font_size=28).move_to(proof_bg.get_top() + DOWN * 0.5)
        eq2 = MathTex(r"\text{Isotropy Group: } G_x = \{g : g \cdot x = x\}", font_size=24, color=YELLOW).next_to(eq1, DOWN, buff=0.3)

        with self.voiceover(text="If we ask an Equivariant Neural Network to predict how this pencil will fall, the math traps it completely."):
            self.play(FadeIn(proof_bg), Write(eq1))
            self.play(Write(eq2))

        eq3_1 = MathTex(r"g \in G_x \implies", font_size=28)
        eq3_2 = MathTex(r"g \cdot y", font_size=28)
        eq3_3 = MathTex(r"= g \cdot f(x)", font_size=28)
        eq3_4 = MathTex(r"= f(g \cdot x)", font_size=28)
        eq3_5 = MathTex(r"= f(x)", font_size=28)
        eq3_6 = MathTex(r"= y", font_size=28)

        # Trình bày toán học theo luồng chạy logic (3B1B Style)
        flow_1 = VGroup(eq3_1, eq3_2, eq3_3).arrange(RIGHT, buff=0.15).next_to(eq2, DOWN, buff=0.4)
        flow_2 = VGroup(eq3_4, eq3_5, eq3_6).arrange(RIGHT, buff=0.15).next_to(flow_1, DOWN, buff=0.2)

        with self.voiceover(text="Take any transformation g that keeps the pencil upright. Applying g to the predicted output y gives g times f of x."):
            self.play(Write(flow_1))

        with self.voiceover(text="Because the network is strictly equivariant, this is equal to f of g times x. And since g leaves x unchanged, the output collapses back to just f of x, which is exactly y."):
            self.play(Write(eq3_4))
            self.play(Indicate(eq2, color=YELLOW), Write(eq3_5))
            self.play(Write(eq3_6))

        eq4 = MathTex(r"g \cdot y = y \implies G_x \subseteq G_y", color=RED_C, font_size=32).next_to(flow_2, DOWN, buff=0.4)

        with self.voiceover(text="This proves that any symmetry belonging to the input must be violently inherited by the output. The network cannot break symmetry."):
            self.play(Write(eq4))
            self.play(Flash(eq4, color=RED, line_length=0.2))

        # --- 4. HẬU QUẢ VẬT LÝ VÀ PHÁ VỠ ĐỐI XỨNG ---
        with self.voiceover(text="In physical reality, the pencil falls in one random direction. But the AI is forbidden to pick a side. Thus, it predicts the pencil falling in all directions simultaneously, forming a blurred circle, or just standing still forever."):
            # Tạo hiệu ứng đa thực thể ảo ảnh đổ rạp xuống xung quanh
            ghost_pencils = VGroup()
            for angle in np.linspace(0, TAU, 12, endpoint=False):
                ghost = pencil.copy().set_opacity(0.15).set_color(GRAY)
                # Dùng thuộc tính xoay gốc mỏ neo tại tâm bàn
                ghost.rotate(angle=angle, axis=OUT, about_point=table.get_center())
                ghost.rotate(angle=PI/2 - 0.2, axis=RIGHT, about_point=table.get_center())
                ghost_pencils.add(ghost)

            self.play(FadeIn(ghost_pencils), run_time=1.5)
            self.wait(1)

        break_title = Text("How to Break It?", font_size=24, color=GREEN_C).next_to(proof_bg, DOWN, buff=0.6)
        break_desc = Text("Input Noise Injection", font_size=20, color=WHITE).next_to(break_title, DOWN, buff=0.2)

        with self.voiceover(text="To escape this trap, researchers use Weight Perturbation, injecting a tiny drop of random noise to shatter the perfect balance, allowing the physics to finally collapse into reality."):
            self.play(Write(break_title), FadeIn(break_desc))

            # Gửi một viên đạn nhỏ (Nhiễu ngẫu nhiên) bắn vào bút chì
            noise_bullet = Dot(color=GREEN_C, radius=0.1).move_to(LEFT * 6 + UP * 1)
            self.play(FadeIn(noise_bullet))
            self.play(noise_bullet.animate.move_to(pencil.get_center()), run_time=0.5)

            # Bút chì vật lý đổ xuống một hướng cố định, các ảo ảnh biến mất
            self.play(FadeOut(ghost_pencils), FadeOut(noise_bullet))
            self.play(
                Rotate(pencil, angle=-PI/2.5, about_point=table.get_center()),
                run_time=0.8, rate_func=rush_into
            )
            self.wait(2)

        # Dọn dẹp khung hình
        self.play(FadeOut(Group(*self.mobjects)))


## slide 57-59
Flexible symmetries: adaptivity + symmetry discovery

In [ ]:
%%manim -qh -v WARNING FlexibleSymmetries57

from manim import *
import numpy as np
from manim_voiceover import VoiceoverScene
from manim_voiceover.services.gtts import GTTSService

class FlexibleSymmetries57(ThreeDScene, VoiceoverScene):
    def construct(self):
        self.set_speech_service(EdgeTTSService(voice="en-US-GuyNeural"))

        # Set 3D camera orientation
        self.set_camera_orientation(phi=65 * DEGREES, theta=-45 * DEGREES)

        # --- PART 1: CONTEXTUAL ADAPTIVITY ---
        title1 = Text("Flexible Symmetries: Adaptivity", font_size=36, weight=BOLD).to_edge(UP)
        self.add_fixed_in_frame_mobjects(title1)

        with self.voiceover("Hard-coding symmetries beforehand can limit the power of foundation models. Instead, models should flexibly encode them and adapt based on the data context."):
            self.play(FadeIn(title1))

        # Visualize Contextual World Models using abstract geometric shapes
        context_text = Text("Contextual Selection", font_size=24, color=BLUE).to_corner(UL).shift(DOWN * 1.5)
        self.add_fixed_in_frame_mobjects(context_text)

        # Example 1: Horizontal flip & Color change
        sq1 = Square(fill_color=ORANGE, fill_opacity=0.8).scale(0.5).move_to([-3, 1, 0])
        sq2 = Square(fill_color=GREEN, fill_opacity=0.8).scale(0.5).move_to([-1.5, 1, 0])
        arrow1 = Arrow(sq1.get_right(), sq2.get_left(), buff=0.1, color=WHITE)

        # Example 2: 3D Rotation
        cube1 = Cube(fill_color=BLUE_D, fill_opacity=0.8).scale(0.5).move_to([1.5, 1, 0.5])

        with self.voiceover("For example, when encountering data type A, the model automatically selects horizontal flip and color change symmetries. But when encountering data type B, it activates 3D rotational symmetry."):
            self.play(FadeIn(context_text))

            # Scene 1
            self.play(FadeIn(sq1))
            self.play(GrowArrow(arrow1))
            self.play(TransformFromCopy(sq1, sq2))
            self.play(sq2.animate.flip(RIGHT))

            # Scene 2
            self.play(FadeIn(cube1))
            self.play(Rotate(cube1, angle=PI, axis=UP + RIGHT), run_time=2)
            self.wait(1)

        # Clean up for transition
        self.play(FadeOut(sq1), FadeOut(sq2), FadeOut(arrow1), FadeOut(cube1), FadeOut(context_text))

        # --- PART 2: SYMMETRY BREAKING WITH VECTOR V (Any-subgroup) ---
        eq_bg = BackgroundRectangle(title1, color=BLACK, fill_opacity=0) # Dummy object for alignment
        equation = MathTex(
            r"f(x) = h(x, ", r"\mathbf{v}", r")",
            font_size=42
        ).to_edge(DOWN, buff=1)
        equation[1].set_color(RED)

        condition = MathTex(
            r"\text{if } g \cdot \mathbf{v} = \mathbf{v}, \text{ then } f(g \cdot x) = g \cdot f(x)",
            font_size=32, color=YELLOW
        ).next_to(equation, UP, buff=0.5)

        self.add_fixed_in_frame_mobjects(equation, condition)

        # Draw 3D Axes and Sphere
        axes = ThreeDAxes(x_range=[-2, 2], y_range=[-2, 2], z_range=[-2, 2], x_length=4, y_length=4, z_length=4)
        sphere = Surface(
            lambda u, v: np.array([
                1.5 * np.cos(u) * np.cos(v),
                1.5 * np.cos(u) * np.sin(v),
                1.5 * np.sin(u)
            ]), v_range=[0, TAU], u_range=[-PI / 2, PI / 2],
            checkerboard_colors=[BLUE_D, BLUE_E], resolution=(15, 32), fill_opacity=0.6
        )

        # Vector v pointing along Z axis
        vector_v = Arrow3D(start=np.array([0, 0, 0]), end=np.array([0, 0, 2]), color=RED, thickness=0.05)
        v_label = MathTex(r"\mathbf{v}", color=RED, font_size=36).move_to(np.array([0, 0, 2.3]))
        # Ensure v label always faces the camera
        self.add_fixed_orientation_mobjects(v_label)

        with self.voiceover("Another elegant mathematical approach is to provide the model with a special input, called vector v, to break the general symmetry."):
            self.play(Write(equation))
            self.play(Create(axes), FadeIn(sphere))
            self.play(Create(vector_v), Write(v_label))

        with self.voiceover("Look at this condition: The transformation g only retains its equivariance if it preserves vector v. This means..."):
            self.play(Write(condition))

        with self.voiceover("If we rotate the entire system around the axis of v, vector v remains unchanged. This subgroup symmetry is accepted."):
            # Rotate around Z axis (same direction as v) -> Valid
            self.play(Indicate(condition[0][2:7], color=GREEN)) # Highlight g.v = v
            self.play(Rotate(sphere, angle=PI, axis=OUT, about_point=ORIGIN), run_time=2)

        with self.voiceover("However, if we rotate along a different axis that tilts vector v, the equivariant symmetry is immediately broken for that rotation direction."):
            # Rotate around X axis tilting v -> Invalid
            group_to_rotate = VGroup(sphere, vector_v, v_label)
            self.play(Rotate(group_to_rotate, angle=PI/3, axis=RIGHT, about_point=ORIGIN), run_time=1.5)
            self.play(Rotate(group_to_rotate, angle=-PI/3, axis=RIGHT, about_point=ORIGIN), run_time=1.5)

        # Clean up for part 3
        self.play(
            FadeOut(axes), FadeOut(sphere), FadeOut(vector_v), FadeOut(v_label),
            FadeOut(equation), FadeOut(condition), FadeOut(title1)
        )

        # --- PART 3: SYMMETRY DISCOVERY ---
        # Return to 2D view (Camera straight down)
        self.move_camera(phi=0 * DEGREES, theta=-90 * DEGREES, run_time=1)

        title2 = Text("Flexible Symmetries: Discovery", font_size=36, weight=BOLD).to_edge(UP)
        self.add_fixed_in_frame_mobjects(title2)

        question = Text("What if I don't know the symmetries a priori?", font_size=28)
        answer = Text("Learn them!", font_size=36, color=YELLOW, weight=BOLD).next_to(question, DOWN, buff=0.5)

        with self.voiceover("Finally, a major question arises: What if we have absolutely no prior knowledge of the symmetries in the physical system or dataset?"):
            self.play(FadeIn(title2), Write(question))

        with self.voiceover("The answer is simple: Let the neural network discover and learn them on its own!"):
            self.play(TransformFromCopy(question, answer))
            self.play(Flash(answer, color=YELLOW, line_length=0.4))
            self.wait(1)

        methods = VGroup(
            Text("1. Learn input transformations (Data Augmentation)", font_size=24),
            Text("2. Learn architectural constraints (Weight Sharing)", font_size=24),
            Text("3. Probe trained networks (Discover from patterns)", font_size=24)
        ).arrange(DOWN, aligned_edge=LEFT, buff=0.4).next_to(answer, DOWN, buff=0.8)

        with self.voiceover("We can achieve this by forcing the model to learn data augmentations, automatically share weights, or by probing a trained neural network."):
            for item in methods:
                self.play(FadeIn(item, shift=RIGHT))
            self.wait(2)


In [ ]:
%%manim -qh -v WARNING FlexibleSymmetries57

from manim import *
import numpy as np
from manim_voiceover import VoiceoverScene

class FlexibleSymmetries57(ThreeDScene, VoiceoverScene):
    def construct(self):
        self.camera.background_color = BLACK
        self.set_speech_service(EdgeTTSService(voice="en-US-GuyNeural"))
        # ==========================================================
        # PART 1: CONTEXTUAL ADAPTIVITY (Mạng tự học đối xứng)
        # ==========================================================
        # Khóa Camera 2D ở phần này
        self.set_camera_orientation(phi=0 * DEGREES, theta=-90 * DEGREES)

        title1 = Text("Flexible Symmetries: Contextual Adaptivity", font_size=36, weight=BOLD).to_edge(UP)
        self.play(FadeIn(title1, shift=DOWN*0.2))

        with self.voiceover(text="Hard-coding symmetries beforehand can limit the power of foundation models. Instead, models should flexibly encode them and adapt based on the data context."):
            self.wait(0.5)

        # Xây dựng Sơ đồ Điều hướng theo Context (Contextual Router)
        data_in = Dot(radius=0.2, color=WHITE).move_to(LEFT * 5)
        lbl_in = Text("Input Data", font_size=16).next_to(data_in, UP)

        router = Polygon(
            np.array([0, 0.8, 0]), np.array([1, 0, 0]), np.array([0, -0.8, 0]), np.array([-1, 0, 0]),
            color=BLUE_E, fill_opacity=0.6, stroke_width=2
        ).move_to(LEFT * 2)
        lbl_router = Text("Context\nRouter", font_size=16, color=BLUE_A).move_to(router.get_center())

        # Các nhánh Symmetries (Flip, Color, Rotation)
        branch_paths = VGroup(
            Line(router.get_right(), RIGHT * 1.5 + UP * 1.5, color=GREY),
            Line(router.get_right(), RIGHT * 1.5, color=GREY),
            Line(router.get_right(), RIGHT * 1.5 + DOWN * 1.5, color=GREY)
        )

        # Các phép biến đổi hiển thị tại các nút cuối
        sym_nodes = VGroup(
            VGroup(Square(side_length=0.4, color=ORANGE, fill_opacity=0.8), MathTex(r"\leftrightarrow", font_size=24), Square(side_length=0.4, color=GREEN, fill_opacity=0.8)).arrange(RIGHT, buff=0.2).move_to(RIGHT * 3.5 + UP * 1.5),
            VGroup(Triangle(color=YELLOW, fill_opacity=0.8).scale(0.3), Arrow(LEFT*0.2, RIGHT*0.2), Triangle(color=YELLOW, fill_opacity=0.8).scale(0.3).rotate(PI)).arrange(RIGHT, buff=0.2).move_to(RIGHT * 3.5),
            VGroup(Circle(radius=0.25, color=RED, fill_opacity=0.8), MathTex(r"\circlearrowleft", font_size=24)).arrange(RIGHT, buff=0.2).move_to(RIGHT * 3.5 + DOWN * 1.5)
        )

        sym_labels = VGroup(
            Text("Color & Flip Sym", font_size=16, color=GREEN_B).next_to(sym_nodes[0], RIGHT),
            Text("Inversion Sym", font_size=16, color=YELLOW_B).next_to(sym_nodes[1], RIGHT),
            Text("Rotational Sym", font_size=16, color=RED_B).next_to(sym_nodes[2], RIGHT)
        )

        with self.voiceover(text="For example, a neural router analyzes the input data context..."):
            self.play(FadeIn(data_in), FadeIn(lbl_in))
            self.play(data_in.animate.move_to(router.get_left()), run_time=1)
            self.play(FadeIn(router), Write(lbl_router), FadeIn(branch_paths))

        with self.voiceover(text="...when encountering data type A, it automatically selects color-flip symmetries. But for data type B, it activates 3D rotational symmetry."):
            # Mô phỏng dòng dữ liệu chạy theo nhánh 1
            flow_1 = Line(router.get_right(), RIGHT * 1.5 + UP * 1.5, color=GREEN_C, stroke_width=4)
            self.play(ShowPassingFlash(flow_1, time_width=0.5), run_time=0.8)
            self.play(FadeIn(sym_nodes[0]), Write(sym_labels[0]))

            # Mô phỏng dòng dữ liệu chạy theo nhánh 3
            flow_2 = Line(router.get_right(), RIGHT * 1.5 + DOWN * 1.5, color=RED_C, stroke_width=4)
            self.play(ShowPassingFlash(flow_2, time_width=0.5), run_time=0.8)
            self.play(FadeIn(sym_nodes[2]), Write(sym_labels[2]))
            self.wait(1)

        # Dọn dẹp chuyển sang 3D
        self.play(FadeOut(VGroup(data_in, lbl_in, router, lbl_router, branch_paths, sym_nodes, sym_labels)))


        # ==========================================================
        # PART 2: SYMMETRY BREAKING WITH VECTOR V (Any-subgroup)
        # ==========================================================
        self.play(Transform(title1, Text("Flexible Symmetries: Any-subgroup via Vector v", font_size=36, weight=BOLD).to_edge(UP)))

        # Sắp xếp công thức gọn gàng ở nửa trên
        equation = MathTex(
            r"f(x) = h(x, ", r"\mathbf{v}", r")",
            font_size=42
        ).next_to(title1, DOWN, buff=0.5)
        equation[1].set_color(RED)

        condition = MathTex(
            r"\text{Equivariance valid if: } g \cdot \mathbf{v} = \mathbf{v}",
            font_size=32, color=YELLOW
        ).next_to(equation, DOWN, buff=0.3)

        self.add_fixed_in_frame_mobjects(equation, condition)

        # Chuẩn bị không gian 3D mượt mà
        self.move_camera(phi=65 * DEGREES, theta=-45 * DEGREES, run_time=1.5)

        axes = ThreeDAxes(x_range=[-2, 2], y_range=[-2, 2], z_range=[-2, 2], x_length=4, y_length=4, z_length=4).shift(DOWN * 0.5)

        # Bề mặt cầu (Sphere) mượt mà hơn (Opaque Surface thay vì lưới rác)
        sphere = Surface(
            lambda u, v: axes.c2p(1.5 * np.cos(u) * np.cos(v), 1.5 * np.cos(u) * np.sin(v), 1.5 * np.sin(u)),
            u_range=[-PI/2, PI/2], v_range=[0, TAU],
            resolution=(40, 40), fill_color=BLUE_E, fill_opacity=0.7, stroke_color=BLUE_E, stroke_width=0.5
        )

        # Thêm các vĩ tuyến để người xem dễ thấy quả cầu đang xoay
        meridians = VGroup(*[
            ParametricFunction(lambda t, v=v_val: axes.c2p(1.5 * np.cos(t) * np.cos(v), 1.5 * np.cos(t) * np.sin(v), 1.5 * np.sin(t)), t_range=[-PI/2, PI/2], color=TEAL_C, stroke_width=1.5)
            for v_val in np.linspace(0, TAU, 8, endpoint=False)
        ])

        globe = VGroup(sphere, meridians)

        # Vector v phá vỡ đối xứng (Hướng lên trục Z)
        vector_v = Arrow3D(start=axes.c2p(0,0,0), end=axes.c2p(0,0,2.2), color=RED, thickness=0.04)
        v_label = MathTex(r"\mathbf{v}", color=RED, font_size=36).move_to(axes.c2p(0.2, 0, 2.5))
        self.add_fixed_orientation_mobjects(v_label)

        with self.voiceover(text="Another elegant mathematical approach is to provide the model with a special directional input, called vector v, acting like a magnetic field to break general symmetry."):
            self.play(FadeIn(equation), FadeIn(condition))
            self.play(Create(axes), FadeIn(globe))
            self.play(Create(vector_v), Write(v_label))

        with self.voiceover(text="Look at this condition: The neural network is only equivariant to transformations that preserve vector v."):
            self.play(Indicate(condition, color=YELLOW, scale_factor=1.2))

        with self.voiceover(text="If we rotate the entire system around the axis of v, the vector remains unchanged. This subgroup symmetry is accepted."):
            # Nhấn mạnh vector v đứng yên
            self.play(vector_v.animate.set_color(GREEN_A))
            self.play(Rotate(globe, angle=PI, axis=OUT, about_point=axes.c2p(0,0,0)), run_time=2.5, rate_func=smooth)
            self.play(vector_v.animate.set_color(RED))

        with self.voiceover(text="However, if we rotate along a different axis that tilts vector v, the symmetry is immediately broken, and the network behaves differently."):
            # Xoay quanh trục X làm vector V bị nghiêng
            group_to_rotate = VGroup(globe, vector_v, v_label)
            self.play(Rotate(group_to_rotate, angle=PI/3, axis=RIGHT, about_point=axes.c2p(0,0,0)), run_time=1.5, rate_func=smooth)
            # Khôi phục trạng thái
            self.play(Rotate(group_to_rotate, angle=-PI/3, axis=RIGHT, about_point=axes.c2p(0,0,0)), run_time=1.5, rate_func=smooth)

        # Dọn dẹp chuyển sang Phần 3
        self.play(
            FadeOut(axes), FadeOut(globe), FadeOut(vector_v), FadeOut(v_label),
            FadeOut(equation), FadeOut(condition), FadeOut(title1)
        )


        # ==========================================================
        # PART 3: SYMMETRY DISCOVERY (Tự động khám phá)
        # ==========================================================
        self.move_camera(phi=0 * DEGREES, theta=-90 * DEGREES, run_time=1.5)

        title2 = Text("Flexible Symmetries: Discovery", font_size=36, weight=BOLD).to_edge(UP)
        self.add_fixed_in_frame_mobjects(title2)

        question = Text("What if we don't know the symmetries a priori?", font_size=28).move_to(UP * 1.5)
        answer = Text("Learn them directly from data!", font_size=36, color=YELLOW, weight=BOLD).next_to(question, DOWN, buff=0.5)

        with self.voiceover(text="Finally, a major question arises: What if we have absolutely no prior knowledge of the symmetries in the physical system or dataset?"):
            self.play(FadeIn(title2), Write(question))

        with self.voiceover(text="The answer is simple: Let the neural network discover and learn them on its own!"):
            self.play(FadeIn(answer, shift=UP*0.2))
            self.play(Flash(answer, color=YELLOW, line_length=0.4))

        # Hiển thị các phương pháp dạng danh sách học thuật
        methods = VGroup(
            Text("1. Learn input transformations (e.g., Data Augmentation Grids)", font_size=22),
            Text("2. Learn architectural constraints (e.g., Soft Weight Sharing)", font_size=22),
            Text("3. Probe trained networks (e.g., Hessian Spectrum Analysis)", font_size=22)
        ).arrange(DOWN, aligned_edge=LEFT, buff=0.4).next_to(answer, DOWN, buff=1.0)

        with self.voiceover(text="We can achieve this by forcing the model to learn augmentations, automatically sharing weights, or by probing the hidden geometry of a trained neural network."):
            for item in methods:
                self.play(FadeIn(item, shift=RIGHT*0.5), run_time=0.8)
            self.wait(2)

        self.play(FadeOut(Group(*self.mobjects)))


## slide 60-63
Neural Parameter Symmetries

In [ ]:
%%manim -qh -v WARNING NeuralParameterSymmetries60

from manim import *
import numpy as np
from manim_voiceover import VoiceoverScene

class NeuralParameterSymmetries60(VoiceoverScene):
    def construct(self):
        self.camera.background_color = BLACK

        # --- 1. TITLE & CORE PARADOX ---
        self.set_speech_service(EdgeTTSService(voice="en-US-GuyNeural"))
        title = Text("Neural Parameter Symmetries", font_size=36, weight=BOLD).to_edge(UP)

        eq_core = MathTex(
            r"f_{ ", r"\theta", r" } = f_{ ", r"g \cdot \theta", r" }",
            font_size=46
        ).move_to(UP * 1.5)
        eq_core[1].set_color(BLUE)
        eq_core[3].set_color(ORANGE)

        with self.voiceover(text="When we train a neural network, we usually think of a unique set of parameters leading to a unique function. But in reality, neural networks possess massive parameter symmetries."):
            self.play(FadeIn(title))
            self.play(Write(eq_core))
            self.wait(1)

        with self.voiceover(text="This means two entirely different arrangements of weights, theta and g times theta, can express the exact same mathematical function."):
            self.play(Indicate(eq_core[1], color=BLUE), Indicate(eq_core[3], color=ORANGE))
            self.wait(0.5)

        self.play(eq_core.animate.scale(0.8).to_corner(UR))

        # --- 2A. PERMUTATION INVARIANCE IN MLPs ---
        mlp_title = Text("1. Permutation Invariance", color=BLUE_B, font_size=28).to_edge(LEFT).shift(UP * 2)
        eq_perm = MathTex(
            r"W_2 \sigma(W_1 x) = W_2 ", r"P^\top", r" \sigma(", r"P", r" W_1 x)",
            font_size=32
        ).next_to(mlp_title, DOWN, aligned_edge=LEFT, buff=0.3)
        eq_perm[1].set_color(RED)
        eq_perm[3].set_color(RED)

        self.play(Write(mlp_title), Write(eq_perm))

        input_layer = VGroup(*[Dot(radius=0.15, color=GRAY_B) for _ in range(3)]).arrange(DOWN, buff=0.4).move_to(LEFT * 3 + DOWN * 1)
        hidden_layer = VGroup(*[Dot(radius=0.15, color=TEAL) for _ in range(3)]).arrange(DOWN, buff=0.4).move_to(LEFT * 1 + DOWN * 1)
        output_layer = VGroup(*[Dot(radius=0.15, color=GRAY_B) for _ in range(2)]).arrange(DOWN, buff=0.4).move_to(RIGHT * 1 + DOWN * 1)

        hidden_layer[0].set_color(ORANGE)
        hidden_layer[1].set_color(GREEN)

        in_links = VGroup(*[always_redraw(lambda i=i, h=h: Line(i.get_center(), h.get_center(), stroke_width=1.5, color=GRAY_C)) for i in input_layer for h in hidden_layer])
        out_links = VGroup(*[always_redraw(lambda h=h, o=o: Line(h.get_center(), o.get_center(), stroke_width=1.5, color=GRAY_C)) for h in hidden_layer for o in output_layer])

        nn_graph = VGroup(in_links, out_links, input_layer, hidden_layer, output_layer)

        w1_matrix = Matrix([["w_{11}", "w_{12}", "w_{13}"], ["w_{21}", "w_{22}", "w_{23}"], ["w_{31}", "w_{32}", "w_{33}"]], element_to_mobject_config={"font_size": 20}).scale(0.8).move_to(RIGHT * 3.5 + UP * 0.2)
        w2_matrix = Matrix([["v_{11}", "v_{12}", "v_{13}"], ["v_{21}", "v_{22}", "v_{23}"]], element_to_mobject_config={"font_size": 20}).scale(0.8).move_to(RIGHT * 3.5 + DOWN * 1.8)

        # Tạo viền màu cho dễ phân biệt Hàng/Cột sắp bị đổi
        w1_matrix.get_rows()[0].set_color(ORANGE)
        w1_matrix.get_rows()[1].set_color(GREEN)

        w2_matrix.get_columns()[0].set_color(ORANGE)
        w2_matrix.get_columns()[1].set_color(GREEN)

        w1_label = MathTex("W_1", font_size=24).next_to(w1_matrix, LEFT, buff=0.1)
        w2_label = MathTex("W_2", font_size=24).next_to(w2_matrix, LEFT, buff=0.1)
        matrices_group = VGroup(w1_matrix, w2_matrix, w1_label, w2_label)

        with self.voiceover(text="Consider a standard multi layer perceptron. If we swap the positions of these two internal hidden neurons..."):
            self.play(FadeIn(nn_graph), FadeIn(matrices_group))
            self.play(Indicate(hidden_layer[0], color=ORANGE), Indicate(hidden_layer[1], color=GREEN))

        with self.voiceover(text="We must simultaneously permute the rows of the incoming weight matrix double-u one, and the columns of the outgoing matrix double-u two."):
            self.play(
                # Đổi chỗ Nơ-ron đồ thị
                hidden_layer[0].animate.move_to(hidden_layer[1].get_center()),
                hidden_layer[1].animate.move_to(hidden_layer[0].get_center()),

                # FIX: HOÁN VỊ HÀNG CỦA MA TRẬN W1
                Swap(w1_matrix.get_rows()[0], w1_matrix.get_rows()[1]),

                # FIX: HOÁN VỊ CỘT CỦA MA TRẬN W2
                Swap(w2_matrix.get_columns()[0], w2_matrix.get_columns()[1]),

                run_time=1.5
            )
            self.play(Flash(eq_perm[1], color=RED), Flash(eq_perm[3], color=RED))

        # Khôi phục trạng thái
        self.play(
            hidden_layer[0].animate.move_to(hidden_layer[1].get_center()),
            hidden_layer[1].animate.move_to(hidden_layer[0].get_center()),
            Swap(w1_matrix.get_rows()[0], w1_matrix.get_rows()[1]),
            Swap(w2_matrix.get_columns()[0], w2_matrix.get_columns()[1]),
            run_time=0.5
        )
        self.play(FadeOut(mlp_title), FadeOut(eq_perm))

        # --- 2B. SCALE INVARIANCE IN MLPs ---
        scale_title = Text("2. Scale Invariance", color=TEAL_B, font_size=28).to_edge(LEFT).shift(UP * 2)
        eq_scale = MathTex(
            r"W_2 \sigma(W_1 x) = ", r"\alpha", r" W_2 \sigma(", r"\frac{1}{\alpha}", r" W_1 x)",
            font_size=32
        ).next_to(scale_title, DOWN, aligned_edge=LEFT, buff=0.3)
        eq_scale[1].set_color(RED_C)
        eq_scale[3].set_color(GREEN_C)

        with self.voiceover(text="Moreover, if the network uses homogeneous activation functions, we discover Scale Invariance."):
            self.play(Write(scale_title), Write(eq_scale))

        # Ngắt luôn updater của các đường line để chúng có thể nhận animation thay đổi độ dày/màu sắc
        for line in in_links: line.clear_updaters()
        for line in out_links: line.clear_updaters()

        # Tạo Label alpha
        alpha_w2 = MathTex(r"\alpha", font_size=24, color=RED_C)
        inv_alpha_w1 = MathTex(r"\frac{1}{\alpha}", font_size=24, color=GREEN_C)

        # FIX: Gắn Updater cho W1_Label và W2_Label để tự động né dấu ngoặc [] khi ma trận phình to
        w1_label.add_updater(lambda m: m.next_to(w1_matrix, LEFT, buff=0.15))
        w2_label.add_updater(lambda m: m.next_to(w2_matrix, LEFT, buff=0.15))

        # Gắn Updater cho alpha để luôn bám theo nhãn W
        alpha_w2.add_updater(lambda m: m.next_to(w2_label, LEFT, buff=0.1))
        inv_alpha_w1.add_updater(lambda m: m.next_to(w1_label, LEFT, buff=0.1))

        with self.voiceover(text="By scaling up the output weights by a factor of alpha, and simultaneously scaling down the input weights by the exact inverse of alpha..."):
            self.play(Indicate(eq_scale[1], color=RED_C), Indicate(eq_scale[3], color=GREEN_C))

            self.play(
                # Đồ thị thay đổi
                in_links.animate.set_stroke(width=0.5, color=GREEN_C, opacity=0.3),
                out_links.animate.set_stroke(width=4.0, color=RED_C, opacity=0.9),
                *[h.animate.scale(1.5).set_color(RED_C) for h in hidden_layer],

                # CHỈ CẦN ANIMATE MA TRẬN: Hệ thống updaters sẽ tự lo việc xê dịch nhãn để tránh chồng lấn
                w1_matrix.animate.scale(0.8).set_color(GREEN_C).shift(UP * 0.5),
                w2_matrix.animate.scale(1.2).set_color(RED_C).shift(DOWN * 0.5),

                FadeIn(inv_alpha_w1, shift=RIGHT*0.2),
                FadeIn(alpha_w2, shift=RIGHT*0.2),

                run_time=2
            )

        with self.voiceover(text="The mathematical effect perfectly cancels out. The internal geometry shifts entirely, yet the final network output is absolutely invariant."):
            self.play(Indicate(output_layer, color=YELLOW, scale_factor=1.2))
            self.wait(1)

        # Xóa updaters an toàn trước khi FadeOut
        w1_label.clear_updaters()
        w2_label.clear_updaters()
        alpha_w2.clear_updaters()
        inv_alpha_w1.clear_updaters()

        self.play(
            FadeOut(nn_graph), FadeOut(matrices_group),
            FadeOut(inv_alpha_w1), FadeOut(alpha_w2),
            FadeOut(scale_title), FadeOut(eq_scale)
        )

        # --- 3. GL(r)-INVARIANCE (LoRA & ATTENTION) ---
        lora_title = Text("3. Matrix Products & GL(r)-Invariance", color=GREEN_B, font_size=28).to_edge(LEFT).shift(UP * 2)
        eq_lora = MathTex(
            r"U", r"V^\top", r"=", r"U", r"R", r"R^{-1}", r"V^\top",
            font_size=38
        ).next_to(lora_title, DOWN, aligned_edge=LEFT, buff=0.4)
        eq_lora[4].set_color(YELLOW)
        eq_lora[5].set_color(ORANGE)

        self.play(Write(lora_title), Write(eq_lora[:3]))

        mat_U = Rectangle(width=0.6, height=1.8, fill_color=BLUE, fill_opacity=0.6, stroke_width=1).move_to(LEFT * 2 + DOWN * 1)
        mat_V = Rectangle(width=1.8, height=0.6, fill_color=PURPLE, fill_opacity=0.6, stroke_width=1).next_to(mat_U, RIGHT, buff=0.2)

        lbl_U = MathTex("U", font_size=28).move_to(mat_U.get_center())
        lbl_V = MathTex("V^\top", font_size=28).move_to(mat_V.get_center())

        with self.voiceover(text="Another fascinating symmetry occurs inside matrix products, widely seen in low-rank adaptations, or LoRAs, and Attention blocks."):
            self.play(Create(mat_U), Create(mat_V), Write(lbl_U), Write(lbl_V))

        mat_R = Rectangle(width=0.6, height=0.6, fill_color=YELLOW, fill_opacity=0.8, stroke_width=1).next_to(mat_U, RIGHT, buff=0.5)
        mat_R_inv = Rectangle(width=0.6, height=0.6, fill_color=ORANGE, fill_opacity=0.8, stroke_width=1).next_to(mat_R, RIGHT, buff=0.1)
        lbl_R = MathTex("R", font_size=24).move_to(mat_R.get_center())
        lbl_R_inv = MathTex("R^{-1}", font_size=24).move_to(mat_R_inv.get_center())

        with self.voiceover(text="We can inject any arbitrary invertible matrix R and its inverse right into the bottleneck layer."):
            self.play(Write(eq_lora[3:]))
            self.play(mat_V.animate.shift(RIGHT * 2.5), lbl_V.animate.shift(RIGHT * 2.5))
            self.play(FadeIn(mat_R), FadeIn(mat_R_inv), Write(lbl_R), Write(lbl_R_inv))

        with self.voiceover(text="If we group R with U, and R-inverse with V-transpose, the absolute weight parameter values shift drastically, yet their collective dot product remains entirely unchanged."):
            box_left = SurroundingRectangle(VGroup(mat_U, mat_R), color=YELLOW, buff=0.05)
            box_right = SurroundingRectangle(VGroup(mat_R_inv, mat_V), color=ORANGE, buff=0.05)
            self.play(Create(box_left), Create(box_right))
            self.wait(1)

        # --- 4. SUMMARY & CONCLUSION ---
        with self.voiceover(text="Whether it is scaling factors, head permutations in Transformers, or orthogonal groups in RMS Norm, parameter symmetries define the complex geometry of our loss landscapes."):
            self.play(FadeOut(box_left), FadeOut(box_right), FadeOut(mat_R), FadeOut(mat_R_inv), FadeOut(lbl_R), FadeOut(lbl_R_inv))
            self.play(FadeOut(mat_U), FadeOut(mat_V), FadeOut(lbl_U), FadeOut(lbl_V), FadeOut(eq_lora), FadeOut(lora_title))

            summary = VGroup(
                Text("- Permutations within MLPs", font_size=24),
                Text("- Scaling Invariance with Homogeneous Activations", font_size=24),
                Text("- Permutation of Attention Heads", font_size=24),
                Text("- LoRA General Linear GL(r) Invariance", font_size=24)
            ).arrange(DOWN, aligned_edge=LEFT, buff=0.3).move_to(DOWN * 0.5)

            for item in summary:
                self.play(FadeIn(item, shift=UP * 0.2), run_time=0.6)

            self.wait(2)


## slide 64

In [ ]:
%%manim -qh -v WARNING TypesOfSymmetries64

class TypesOfSymmetries64(ThreeDScene, VoiceoverScene):
    def construct(self):
        self.camera.background_color = BLACK
        self.set_speech_service(EdgeTTSService(voice="en-US-GuyNeural"))
        # --- 1. SET UP TITLE AND FORMULAS (2D FLAT OVERLAY) ---
        title = Text("Types of Symmetries", font_size=36, weight=BOLD).to_edge(UP)

        # Functional Symmetry (Bên trái)
        func_sym = MathTex(r"f_\theta(x) = f_{g \cdot \theta}(x)", font_size=36)
        func_label = Text("Functional Symmetry", font_size=24, color=BLUE_C).next_to(func_sym, UP, buff=0.2)
        func_group = VGroup(func_label, func_sym).move_to(UP * 1.8 + LEFT * 3.5)

        # Loss Symmetry (Bên phải)
        loss_sym = MathTex(r"\mathcal{L}(\theta) = \mathcal{L}(g \cdot \theta)", font_size=36)
        loss_label = Text("Loss Symmetry", font_size=24, color=ORANGE).next_to(loss_sym, UP, buff=0.2)
        loss_group = VGroup(loss_label, loss_sym).move_to(UP * 1.8 + RIGHT * 3.5)

        # Ghim cố định UI vào màn hình phẳng
        self.add_fixed_in_frame_mobjects(title, func_group, loss_group)

        with self.voiceover(text="Symmetries in neural networks can be categorized into two main types. The first is functional symmetry, where transforming the parameters results in the exact same mathematical output for any input."):
            self.play(FadeIn(title))
            self.play(Write(func_group))
            self.wait(1)

        with self.voiceover(text="However, a much broader category is loss symmetry. Here, the internal functions might behave differently, but they ultimately yield the exact same objective loss value."):
            self.play(Write(loss_group))
            self.wait(1)

        # --- 2. 3D LOSS LANDSCAPE (BÊN PHẢI MÀN HÌNH) ---
        self.move_camera(phi=65 * DEGREES, theta=-60 * DEGREES, run_time=1.5)

        # Đẩy công thức thu gọn lên góc trái để nhường chỗ cho 3D
        self.play(
            func_group.animate.scale(0.7).to_corner(UL).shift(DOWN*0.5 + RIGHT*0.5),
            loss_group.animate.scale(0.7).next_to(func_group, DOWN, buff=0.5, aligned_edge=LEFT),
            run_time=1.5
        )

        axes3d = ThreeDAxes(
            x_range=[-3, 3], y_range=[-3, 3], z_range=[0, 3],
            x_length=6, y_length=6, z_length=2.5
        ).shift(DOWN * 1.5 + RIGHT * 1.5)

        # Hàm đồi núi với đỉnh đối xứng
        def loss_landscape(u, v):
            z = 0.5 * (np.cos(u*1.5) + np.sin(v*1.5)) + 0.1 * (u**2 + v**2) + 0.5
            return axes3d.c2p(u, v, z)

        surface = Surface(
            loss_landscape,
            u_range=[-2.5, 2.5], v_range=[-2.5, 2.5],
            resolution=(40, 40),
            fill_color=BLUE_E, fill_opacity=0.8,
            stroke_color=BLUE_E, stroke_width=0.5
        )

        with self.voiceover(text="In the vast optimization landscape, this means there are multiple distinct parameter configurations that sit at the exact same altitude."):
            self.play(Create(axes3d), Create(surface), run_time=2)

        # Hai điểm đối xứng trên sườn đồi
        theta_A_coord = [-1.5, 1.0]
        theta_B_coord = [1.0, -1.5]

        dot_A = Sphere(radius=0.1, color=ORANGE).move_to(axes3d.c2p(*theta_A_coord, loss_landscape(*theta_A_coord)[2] + 0.1))
        lbl_A = MathTex(r"\theta", font_size=32, color=ORANGE).next_to(dot_A, UP)

        dot_B = Sphere(radius=0.1, color=ORANGE).move_to(axes3d.c2p(*theta_B_coord, loss_landscape(*theta_B_coord)[2] + 0.1))
        lbl_B = MathTex(r"g \cdot \theta", font_size=32, color=ORANGE).next_to(dot_B, UP)

        self.add_fixed_orientation_mobjects(lbl_A, lbl_B)

        with self.voiceover(text="Point theta and its symmetric counterpart g-theta are physically distant, yet perfectly equivalent in terms of performance."):
            self.play(FadeIn(dot_A), Write(lbl_A))
            self.play(FadeIn(dot_B), Write(lbl_B))

            # Quỹ đạo vòng cung chứng minh cùng cao độ Loss
            arc_path = ParametricFunction(
                lambda t: axes3d.c2p(-1.5*np.cos(t) - 1.0*np.sin(t), 1.0*np.cos(t) - 1.5*np.sin(t), loss_landscape(-1.5, 1.0)[2] + 0.1),
                t_range=[0, PI/2], color=YELLOW, stroke_width=3
            )
            self.play(Create(arc_path), run_time=1.5)
            self.wait(1)

        # Dọn dẹp đồ họa 3D để tập trung vào Ví dụ
        self.play(FadeOut(VGroup(axes3d, surface, dot_A, dot_B, lbl_A, lbl_B, arc_path)))


        # --- 3. CONTRASTIVE LOSS EXAMPLE (TOÁN HỌC TRỰC QUAN BÊN PHẢI) ---
        # Đưa màn hình về 2D phẳng
        self.move_camera(phi=0, theta=-90 * DEGREES, run_time=1.5)

        ex_title = Text("Example: Contrastive Loss", font_size=28, color=YELLOW).move_to(UP * 1.5 + RIGHT * 2.5)
        self.add_fixed_in_frame_mobjects(ex_title)

        ax2d = Axes(x_range=[-2, 2], y_range=[-2, 2], x_length=4, y_length=4).move_to(RIGHT * 2.5 + DOWN * 1.0)
        self.add_fixed_in_frame_mobjects(ax2d)

        # Vector output ban đầu
        vec1 = Arrow(ax2d.c2p(0, 0), ax2d.c2p(1.5, 0.5), color=TEAL_C, buff=0)
        vec2 = Arrow(ax2d.c2p(0, 0), ax2d.c2p(0.8, 1.6), color=BLUE_C, buff=0)

        lbl_v1 = MathTex(r"f_\theta(x)", font_size=24, color=TEAL_C).next_to(vec1.get_end(), RIGHT, buff=0.1)
        lbl_v2 = MathTex(r"f_\theta(x')", font_size=24, color=BLUE_C).next_to(vec2.get_end(), UP, buff=0.1)

        # Góc giữa 2 vector (Similarity)
        angle_arc = Angle(vec1, vec2, radius=0.6, color=YELLOW)
        similarity_text = MathTex(r"\text{Similarity} \propto f_\theta(x)^\top f_\theta(x')", font_size=24, color=YELLOW).next_to(ax2d, DOWN, buff=0.3)

        self.add_fixed_in_frame_mobjects(vec1, vec2, lbl_v1, lbl_v2, angle_arc, similarity_text)

        with self.voiceover(text="A classic example of loss symmetry is contrastive learning. The network outputs embedding vectors for different inputs."):
            self.play(FadeIn(ex_title), Create(ax2d))
            self.play(GrowArrow(vec1), Write(lbl_v1))
            self.play(GrowArrow(vec2), Write(lbl_v2))
            self.play(Create(angle_arc), Write(similarity_text))

        with self.voiceover(text="If a symmetry operation, like a rotation, alters both output vectors simultaneously..."):
            # Thực hiện phép xoay đồng bộ (Mô phỏng phép biến đổi g)
            rotation_angle = PI / 3
            self.play(
                Rotate(vec1, angle=rotation_angle, about_point=ax2d.c2p(0, 0)),
                Rotate(vec2, angle=rotation_angle, about_point=ax2d.c2p(0, 0)),
                Rotate(lbl_v1, angle=rotation_angle, about_point=ax2d.c2p(0, 0)),
                Rotate(lbl_v2, angle=rotation_angle, about_point=ax2d.c2p(0, 0)),
                Rotate(angle_arc, angle=rotation_angle, about_point=ax2d.c2p(0, 0)),
                run_time=2, rate_func=smooth
            )

            # Cập nhật Label thành f_{g * theta}
            lbl_v1_new = MathTex(r"f_{g \cdot \theta}(x)", font_size=24, color=TEAL_A).move_to(lbl_v1.get_center())
            lbl_v2_new = MathTex(r"f_{g \cdot \theta}(x')", font_size=24, color=BLUE_A).move_to(lbl_v2.get_center())
            self.add_fixed_in_frame_mobjects(lbl_v1_new, lbl_v2_new)
            self.play(Transform(lbl_v1, lbl_v1_new), Transform(lbl_v2, lbl_v2_new))

        with self.voiceover(text="...the individual outputs change, breaking functional symmetry. But the angle and inner product between them remain exactly the same. Thus, the loss is perfectly symmetric."):
            self.play(Indicate(angle_arc, color=RED, scale_factor=1.5))
            self.play(Indicate(similarity_text, color=YELLOW, scale_factor=1.2))
            self.wait(1.5)

        # Lời kết Distribution
        dist_note = Text("*Note: Invariance can also hold merely in expectation over the data distribution.", font_size=18, slant=ITALIC, color=GRAY_B).to_edge(DOWN, buff=0.3)
        self.add_fixed_in_frame_mobjects(dist_note)

        with self.voiceover(text="So, keep in mind that in complex real-world models, these symmetries might only hold over the entire data distribution, rather than for every single point."):
            self.play(FadeIn(dist_note, shift=UP*0.2))
            self.wait(2)

        self.play(FadeOut(Group(*self.mobjects)))


## slide 65-69
Loss landscapes and parameter symmetries

In [ ]:
%%manim -qh -v WARNING LossLandscapes65

class LossLandscapes65(ThreeDScene, VoiceoverScene):
    def construct(self):
        # Clean up any potential cache issues
        if os.path.exists("voiceover_cache"):
            shutil.rmtree("voiceover_cache")

        # Set up English voiceover service
        self.set_speech_service(EdgeTTSService(voice="en-US-GuyNeural"))

        # --- PART 1: CORE FORMULA ---
        title = Text("Loss Landscapes & Parameter Symmetries", font_size=36, weight=BOLD).to_edge(UP)
        self.play(FadeIn(title))

        main_eq = MathTex(
            r"L(", r"f_{ ", r"\theta", r" }", r") = L(", r"f_{ ", r"g \cdot \theta", r" }", r")",
            font_size=48
        ).move_to(UP * 0.5)

        main_eq[2].set_color(BLUE)
        main_eq[6].set_color(ORANGE)

        with self.voiceover(text="The core equation of parameter symmetry states that: If theta is a local minimum providing an optimal loss..."):
            self.play(Write(main_eq))
            self.wait(0.5)

        with self.voiceover(text="...then applying a symmetry transformation g results in a new state that preserves the exact same loss value."):
            self.play(Indicate(main_eq[2], color=BLUE), Indicate(main_eq[6], color=ORANGE))
            self.wait(1)

        self.play(main_eq.animate.scale(0.7).to_corner(UL).shift(DOWN * 0.8))

        # --- PART 2: 3D VISUALIZATION - CONTINUOUS SYMMETRY ---
        subtitle_cont = Text(" Continuous Symmetries - Mode Connectivity", font_size=24, color=BLUE_B).next_to(title, DOWN, buff=0.2)
        self.add_fixed_in_frame_mobjects(title, main_eq, subtitle_cont)
        self.play(Write(subtitle_cont))

        self.set_camera_orientation(phi=65 * DEGREES, theta=-60 * DEGREES)

        axes3d = ThreeDAxes(
            x_range=[-2, 2], y_range=[-1, 3], z_range=[0, 4],
            x_length=4.5, y_length=4.5, z_length=2.5
        ).shift(DOWN * 1.2)

        def valley_math(u, v):
            z = min(0.6 * (v - u**2)**2, 3.8)
            return axes3d.c2p(u, v, z)

        valley_surface = Surface(
            valley_math,
            u_range=[-1.8, 1.8],
            v_range=[-0.5, 2.5],
            resolution=(64, 64),
            fill_color=BLUE_E,
            fill_opacity=0.7,
            stroke_width=0,
            shade_in_3d=True
        )

        contours = VGroup()
        for z_val in [0.2, 0.8, 1.8, 3.0]:
            offset = np.sqrt(z_val / 0.6)
            limit_sq1 = 2.5 - offset
            if limit_sq1 >= 0:
                t_max1 = min(1.8, np.sqrt(limit_sq1))
                contours.add(ParametricFunction(lambda t, z=z_val, o=offset: axes3d.c2p(t, t**2 + o, z + 0.02), t_range=[-t_max1, t_max1], color=TEAL_A, stroke_width=2))
            max_limit_sq2 = 2.5 + offset
            min_limit_sq2 = offset - 0.5
            t_max2 = min(1.8, np.sqrt(max_limit_sq2))
            t_min2 = np.sqrt(min_limit_sq2) if min_limit_sq2 > 0 else 0
            if t_min2 < t_max2:
                if t_min2 > 0:
                    contours.add(ParametricFunction(lambda t, z=z_val, o=offset: axes3d.c2p(t, t**2 - o, z + 0.02), t_range=[t_min2, t_max2], color=TEAL_A, stroke_width=2))
                    contours.add(ParametricFunction(lambda t, z=z_val, o=offset: axes3d.c2p(t, t**2 - o, z + 0.02), t_range=[-t_max2, -t_min2], color=TEAL_A, stroke_width=2))
                else:
                    contours.add(ParametricFunction(lambda t, z=z_val, o=offset: axes3d.c2p(t, t**2 - o, z + 0.02), t_range=[-t_max2, t_max2], color=TEAL_A, stroke_width=2))

        dot_a_3d = Dot(axes3d.c2p(-1, 1, 0.05), color=RED, radius=0.1)
        dot_b_3d = Dot(axes3d.c2p(1, 1, 0.05), color=RED, radius=0.1)

        path_curve = ParametricFunction(
            lambda t: axes3d.c2p(t, t**2, 0.05),
            t_range=[-1.0, 1.0], color=YELLOW, stroke_width=5
        )

        with self.voiceover(text="Observe this smooth three-dimensional loss landscape. At the base of this quiet valley, we find two local minima, theta A and theta B, highlighted here."):
            self.play(Create(axes3d), Create(valley_surface))
            self.play(Create(contours), FadeIn(dot_a_3d), FadeIn(dot_b_3d), run_time=1.5)

        with self.voiceover(text="Thanks to continuous symmetry, these points are actually connected by a path of invariant low loss. This is the phenomenon of Mode Connectivity."):
            self.play(Create(path_curve), run_time=2)
            self.wait(1)

        self.play(FadeOut(subtitle_cont), FadeOut(axes3d), FadeOut(path_curve))

        # --- PART 3: DISCRETE SYMMETRIES & 2D PROJECTION ---
        subtitle_disc = Text(" Discrete Symmetries - Linear Mode Connectivity", font_size=24, color=GREEN_B).next_to(title, DOWN, buff=0.2)
        self.add_fixed_in_frame_mobjects(subtitle_disc)
        self.play(Write(subtitle_disc))

        axes2d = Axes(
            x_range=[0, 1, 0.2], y_range=[0, 0.4, 0.1],
            x_length=6.5, y_length=3.5,
        ).shift(DOWN * 0.8).shift(OUT * 4.0)

        three_d_group = VGroup(valley_surface, contours, dot_a_3d, dot_b_3d)
        scale_factor = 6.5 / 2.25
        target_y = axes2d.c2p(0, 0.08)[1]
        tem_y = dot_a_3d.get_center()[1] * scale_factor
        shift_y = target_y - tem_y

        with self.voiceover(text="Looking from above, we scale the background valley map so that the distance between the two solutions perfectly matches our new two-dimensional axes."):
            self.move_camera(
                phi=0,
                theta=-90 * DEGREES,
                run_time=2,
                added_anims=[
                    three_d_group.animate.scale(scale_factor, about_point=ORIGIN).shift(UP * shift_y)
                ]
            )

        x_label = axes2d.get_x_axis_label(Text("Interpolation", font_size=18), edge=DOWN, direction=DOWN, buff=0.3)
        y_label = axes2d.get_y_axis_label(Text("Test Loss", font_size=18), edge=LEFT, direction=LEFT, buff=0.3)

        dot_a_2d = Dot(axes2d.c2p(0, 0.08), color=BLUE, radius=0.08)
        dot_b_2d = Dot(axes2d.c2p(1, 0.08), color=BLUE, radius=0.08)

        lbl_a_2d = MathTex(r"\theta_A", font_size=28).next_to(dot_a_2d, UP, buff=0.15)
        lbl_b_2d = MathTex(r"\theta_B", font_size=28).next_to(dot_b_2d, UP, buff=0.15)

        self.play(Create(axes2d), Write(x_label), Write(y_label))

        self.play(
            Transform(dot_a_3d, dot_a_2d),
            Transform(dot_b_3d, dot_b_2d),
            FadeIn(lbl_a_2d), FadeIn(lbl_b_2d),
            run_time=1
        )

        not_aligned_curve = axes2d.plot(lambda x: 0.08 + 0.28 * np.exp(-((x - 0.5) / 0.08)**2), color=RED, stroke_width=4)
        aligned_curve = axes2d.plot(lambda x: 0.08 + 0.01 * np.sin(x * PI), color=GREEN, stroke_width=4)

        lbl_barrier = Text("Severe Loss Barrier (Not Aligned)", color=RED, font_size=18).next_to(not_aligned_curve.get_top(), UP, buff=0.2)
        lbl_flat = Text("Flattened by Permutation (Aligned)", color=GREEN, font_size=18).next_to(aligned_curve, UP, buff=0.25)

        with self.voiceover(text="If we take a straight line path between the two models without alignment, the interpolation hits a massive loss barrier due to structural misalignment."):
            self.play(Create(not_aligned_curve), run_time=2)
            self.play(Write(lbl_barrier))

        dot_b_aligned = Dot(axes2d.c2p(1, 0.08), color=GREEN, radius=0.08)
        lbl_b_aligned = MathTex(r"\pi(\theta_B)", font_size=28, color=GREEN).next_to(dot_b_aligned, DOWN, buff=0.2)

        with self.voiceover(text="However, by using a permutation matrix pi to align the neuron ordering, the barrier vanishes, revealing a flat and safe path through parameter space."):
            self.play(TransformFromCopy(dot_b_3d, dot_b_aligned), Write(lbl_b_aligned))
            self.play(Create(aligned_curve), run_time=2)
            self.play(Write(lbl_flat))
            self.wait(2)


## slide 68-69
Implications for optimization + Other

In [ ]:
%%manim -qh -v WARNING AdvancedOptimizationDynamics68

class AdvancedOptimizationDynamics68(ThreeDScene, VoiceoverScene):
    def construct(self):
        self.set_speech_service(EdgeTTSService(voice="en-US-GuyNeural"))

        # --- PHẦN 1: QUỸ ĐẠO GRADIENT DESCENT VÀ ĐỘ CONG (3D) ---
        self.set_camera_orientation(phi=70 * DEGREES, theta=-55 * DEGREES)

        header1 = VGroup(
            Text("Implications for Optimization", font_size=36, weight=BOLD),
            Text("Learning Dynamics: Curvature Distortions", font_size=24, color=BLUE_B)
        ).arrange(DOWN, buff=0.2).to_edge(UP)
        self.add_fixed_in_frame_mobjects(header1)

        axes3d = ThreeDAxes(
            x_range=[-3, 3], y_range=[-2, 2], z_range=[0, 4],
            x_length=6, y_length=5, z_length=3
        ).shift(DOWN * 4)

        # ---------------------------------------------------------
        # FIX: TRỰC QUAN HÓA BỀ MẶT TRƠN NHẴN (SMOOTH OPAQUE SURFACE)
        # ---------------------------------------------------------
        # ---------------------------------------------------------
        # THE 3B1B SMOOTH SURFACE TRICK
        # ---------------------------------------------------------
        surface = Surface(
            lambda u, v: axes3d.c2p(u, v, 0.15 * u**2 + 0.8 * v**2),
            u_range=[-2.8, 2.8], v_range=[-1.8, 1.8],
            resolution=(128, 128),                  # Lưới chia càng nhỏ, bề mặt càng mượt (smooth)
            checkerboard_colors=[BLUE_E, BLUE_E], # Ép dùng 1 màu duy nhất để triệt tiêu hoàn toàn caro
            fill_opacity=1.0,                     # BẮT BUỘC: Đặc 100% để tránh lỗi cộng dồn Alpha tạo viền đen
            stroke_width=0                        # Xóa nét viền
        )

        # FIX: Nâng Z lên +0.02 để đường contour không bị chìm vào bề mặt đặc
        level_set = ParametricFunction(
            lambda t: axes3d.c2p(np.sqrt(10) * np.cos(t), np.sqrt(1.875) * np.sin(t), 1.5 + 0.02),
            t_range=[0, 2*PI], color=YELLOW, stroke_width=4
        )

        with self.voiceover("Let's visualize why learning dynamics differ so drastically. Here is a loss landscape where the curvature is extremely steep in one direction and very flat in the other."):
            self.play(FadeIn(header1))
            self.play(Create(axes3d), Create(surface))
            self.play(Create(level_set))

        # Khởi tạo mô phỏng thuật toán Gradient Descent
        lr = 1.15
        steps = 15

        # Quỹ đạo A (Độ cong lớn → Zig-zag)
        x_a, y_a = 0.2, 1.36
        # FIX: Nâng Z lên +0.02
        pts_a = [axes3d.c2p(x_a, y_a, 0.15 * x_a**2 + 0.8 * y_a**2 + 0.02)]
        for _ in range(steps):
            x_a -= lr * (0.3 * x_a)
            y_a -= lr * (1.6 * y_a)
            pts_a.append(axes3d.c2p(x_a, y_a, 0.15 * x_a**2 + 0.8 * y_a**2 + 0.02))

        path_a = VMobject(color=RED, stroke_width=4).set_points_as_corners(pts_a)
        dot_a = Sphere(radius=0.08, color=RED).move_to(pts_a[0])
        lbl_a = MathTex(r"\theta", font_size=36, color=RED).next_to(dot_a, UP, buff=0.1)
        self.add_fixed_orientation_mobjects(lbl_a)
        lbl_a.add_updater(lambda m: m.next_to(dot_a, UP, buff=0.1))

        # Quỹ đạo B (Độ cong thấp → Mượt mà thẳng đích)
        x_b, y_b = 3.1, 0.1
        # FIX: Nâng Z lên +0.02
        pts_b = [axes3d.c2p(x_b, y_b, 0.15 * x_b**2 + 0.8 * y_b**2 + 0.02)]
        for _ in range(steps):
            x_b -= lr * (0.3 * x_b)
            y_b -= lr * (1.6 * y_b)
            pts_b.append(axes3d.c2p(x_b, y_b, 0.15 * x_b**2 + 0.8 * y_b**2 + 0.02))

        path_b = VMobject(color=PURPLE, stroke_width=4).set_points_as_corners(pts_b)
        dot_b = Sphere(radius=0.08, color=PURPLE).move_to(pts_b[0])
        lbl_b = MathTex(r"g \cdot \theta", font_size=36, color=PURPLE).next_to(dot_b, RIGHT, buff=0.1)
        self.add_fixed_orientation_mobjects(lbl_b)
        lbl_b.add_updater(lambda m: m.next_to(dot_b, RIGHT, buff=0.1))

        with self.voiceover("Take point theta. It sits on a steep slope. If we apply standard gradient descent, the high curvature causes severe oscillations, slowing down convergence."):
            self.play(FadeIn(dot_a), Write(lbl_a))
            self.play(Create(path_a), MoveAlongPath(dot_a, path_a), run_time=2.5)

        with self.voiceover("But its symmetric counterpart, g-theta, lies on the exact same contour line in a flatter region. From here, the optimization trajectory is perfectly smooth and direct."):
            self.play(FadeIn(dot_b), Write(lbl_b))
            self.play(Create(path_b), MoveAlongPath(dot_b, path_b), run_time=2.5)
            self.wait(1)

        lbl_a.clear_updaters()
        lbl_b.clear_updaters()

        solution_text = Text("Solution: Search for better g, or use Path-SGD", font_size=24, color=YELLOW).to_edge(DOWN, buff=0.5)
        self.add_fixed_in_frame_mobjects(solution_text)

        with self.voiceover("This visually proves why actively searching for symmetric transformations, or using invariant algorithms like Path-SGD, can dramatically accelerate deep learning."):
            self.play(Write(solution_text))
            self.wait(2)

        self.play(
            FadeOut(axes3d), FadeOut(surface), FadeOut(level_set),
            FadeOut(path_a), FadeOut(dot_a), FadeOut(lbl_a),
            FadeOut(path_b), FadeOut(dot_b), FadeOut(lbl_b),
            FadeOut(header1), FadeOut(solution_text)
        )

        self.clear()

        # --- PHẦN 2: ĐẠI LƯỢNG BẢO TOÀN (2D SIMULATION) ---
        self.move_camera(phi=0, theta=-90 * DEGREES, run_time=1)

        header2 = VGroup(
            Text("Other Implications", font_size=36, weight=BOLD),
            Text("Symmetries Lead to Conserved Quantities", font_size=24, color=ORANGE)
        ).arrange(DOWN, buff=0.2).to_edge(UP)

        self.play(FadeIn(header2))

        conserved_eq = MathTex(
            r"W_\ell W_\ell^\top", r"-", r"W_{\ell+1}^\top W_{\ell+1}", r"=", r"\text{Constant}"
        ).scale(1.1).next_to(header2, DOWN, buff=0.5)

        conserved_eq[0].set_color(BLUE)
        conserved_eq[2].set_color(RED)
        conserved_eq[4].set_color(YELLOW)
        self.play(Write(conserved_eq))

        growth_factor = ValueTracker(0)

        bar1 = always_redraw(lambda: Rectangle(
            width=1.5, height=1.5 + growth_factor.get_value(),
            fill_color=BLUE, fill_opacity=0.8, stroke_width=0
        ).move_to(DOWN * 2.0 + LEFT * 2, DOWN))

        bar2 = always_redraw(lambda: Rectangle(
            width=1.5, height=0.5 + growth_factor.get_value(),
            fill_color=RED, fill_opacity=0.8, stroke_width=0
        ).move_to(DOWN * 2.0 + RIGHT * 2, DOWN))

        lbl_bar1 = always_redraw(lambda: MathTex(r"W_\ell W_\ell^\top").next_to(bar1, DOWN))
        lbl_bar2 = always_redraw(lambda: MathTex(r"W_{\ell+1}^\top W_{\ell+1}").next_to(bar2, DOWN))

        diff_line = always_redraw(lambda: DashedLine(
            start=bar1.get_top(),
            end=bar2.get_top() + UP * 1,
            color=YELLOW, stroke_width=4
        ))

        diff_brace = always_redraw(lambda: BraceBetweenPoints(
            bar2.get_top(), bar2.get_top() + UP * 1, direction=RIGHT, color=YELLOW
        ))
        const_label = always_redraw(lambda: MathTex(r"\text{Constant}", color=YELLOW, font_size=28).next_to(diff_brace, RIGHT))

        with self.voiceover("Another profound implication is the emergence of conserved quantities. Much like energy in physics, certain properties of the network remain constant during optimization."):
            self.wait(0.5)

        with self.voiceover("Consider the weight matrices of two consecutive layers. As the network trains, their individual magnitudes might grow substantially."):
            self.play(FadeIn(bar1), FadeIn(bar2), Write(lbl_bar1), Write(lbl_bar2))
            self.play(Create(diff_line), Create(diff_brace), Write(const_label))

        with self.voiceover("However, because of the underlying parameter symmetries, the difference between them is locked. This invariant controls the optimization trajectory, preventing the layers from becoming severely imbalanced."):
            self.play(growth_factor.animate.set_value(1.5), run_time=5, rate_func=linear)
            self.play(Indicate(conserved_eq, color=YELLOW))
            self.wait(2)


## slide 71-72

In [ ]:
%%manim -qh -v WARNING NeuralNetworksAsDataMP71

class NeuralNetworksAsDataMP71(ThreeDScene, VoiceoverScene):
    def construct(self):
        self.set_speech_service(EdgeTTSService(voice="en-US-GuyNeural"))


        # --- CHƯƠNG 1: METANETWORKS & THU GỌN KHÔNG GIAN TRỌNG SỐ ---
        self.set_camera_orientation(phi=0 * DEGREES, theta=-90 * DEGREES)

        header1 = VGroup(
            Text("Neural Networks as Data", font_size=36, weight=BOLD),
            Text("Weight Space Learning & Metanetworks", font_size=24, color=BLUE_B)
        ).arrange(DOWN, buff=0.2).to_edge(UP)
        self.add_fixed_in_frame_mobjects(header1)
        self.play(FadeIn(header1, shift=DOWN * 0.2))

        # Vẽ một mạng nơ-ron với độ chi tiết cao hơn
        layers = [3, 5, 5, 2]
        neurons = VGroup()
        edges = VGroup()

        for i, num_neurons in enumerate(layers):
            layer = VGroup(*[Dot(radius=0.12, color=GRAY_C) for _ in range(num_neurons)])
            layer.arrange(DOWN, buff=0.35).move_to(LEFT * 4.5 + RIGHT * 1.5 * i)
            neurons.add(layer)

        for i in range(len(layers) - 1):
            for n1 in neurons[i]:
                for n2 in neurons[i+1]:
                    edges.add(Line(n1.get_center(), n2.get_center(), stroke_width=0.8, color=GRAY_E))

        standard_nn = VGroup(edges, neurons).move_to(LEFT * 3.5)

        with self.voiceover(text="We are now entering a fascinating paradigm shift in machine learning: treating neural networks themselves as data."):
            self.play(Create(neurons), run_time=1)
            self.play(Create(edges, lag_ratio=0.05), run_time=1.5)

        with self.voiceover(text="A traditional network processes data layer by layer..."):
            self.play(LaggedStart(*[layer.animate.set_color(YELLOW) for layer in neurons], lag_ratio=0.3), run_time=1.2)
            self.play(LaggedStart(*[layer.animate.set_color(GRAY_C) for layer in neurons], lag_ratio=0.3), run_time=1)

        data_point = Dot(radius=0.25, color=BLUE_D).move_to(LEFT * 3.5)
        weight_vector_label = MathTex(r"\Theta \in \mathbb{R}^D", font_size=28, color=BLUE_B).next_to(data_point, DOWN)
        glowing_aura = Dot(radius=0.6, color=BLUE_B, fill_opacity=0.3).move_to(data_point)

        with self.voiceover(text="But imagine taking the entire weight space of this trained network, and collapsing it into a single, high-dimensional point."):
            self.play(
                standard_nn.animate.scale(0.05).move_to(LEFT * 3.5).set_color(BLUE),
                run_time=2, rate_func=rush_into
            )
            self.play(
                ReplacementTransform(standard_nn, data_point),
                FadeIn(glowing_aura),
                Write(weight_vector_label)
            )
            self.play(glowing_aura.animate.scale(1.5).set_opacity(0), run_time=1.5)

        # ---------------------------------------------------------
        # NÂNG CẤP: XÂY DỰNG KIẾN TRÚC METANETWORK CHUYÊN NGHIỆP
        # ---------------------------------------------------------
        meta_layers = [1, 3, 3, 1]
        meta_nodes = VGroup()

        # Hàm tạo các nút Metanetwork dưới dạng Lưới Ma trận (Weight Grids)
        def create_meta_node(color=PURPLE_E, opacity=0.4):
            return VGroup(*[Square(side_length=0.12, fill_color=color, fill_opacity=opacity, stroke_width=0.5, stroke_color=PURPLE_A) for _ in range(4)]).arrange_in_grid(rows=2, buff=0.02)

        for i, num in enumerate(meta_layers):
            layer = VGroup(*[create_meta_node() for _ in range(num)]).arrange(DOWN, buff=0.6)
            layer.move_to(RIGHT * (0.5 + i * 1.5))
            meta_nodes.add(layer)

        # Căn giữa toàn bộ Metanetwork
        meta_nodes.move_to(RIGHT * 2.5)

        meta_edges = VGroup()
        for i in range(len(meta_layers) - 1):
            for n1 in meta_nodes[i]:
                for n2 in meta_nodes[i+1]:
                    edge = Line(n1.get_right(), n2.get_left(), stroke_width=1.2, color=PURPLE_E, stroke_opacity=0.5)
                    meta_edges.add(edge)

        meta_network_group = VGroup(meta_edges, meta_nodes)

        # Hộp viền mờ ảo bọc ngoài tạo cảm giác một Black-box kiến trúc cấp cao
        meta_bounding_box = RoundedRectangle(width=5.5, height=4, corner_radius=0.4, fill_color=BLACK, fill_opacity=0.6, stroke_color=PURPLE_C, stroke_width=1.5).move_to(meta_network_group.get_center())
        meta_label = Text("Meta-Architecture (Neural Functional)", font_size=20, color=PURPLE_A).next_to(meta_bounding_box, DOWN, buff=0.2)

        with self.voiceover(text="We then feed this massive weight vector into a higher-level architecture called a Metanetwork. Notice how its internal states process entire grids of parameters."):
            self.play(FadeIn(meta_bounding_box), Write(meta_label))
            self.play(FadeIn(meta_network_group, lag_ratio=0.1), run_time=2)

            self.play(
                data_point.animate.move_to(meta_nodes[0][0].get_center()).scale(0.5),
                FadeOut(weight_vector_label),
                run_time=1.5, rate_func=smooth
            )

            # Điểm dữ liệu được hấp thụ vào nút đầu tiên của Metanetwork
            self.play(
                FadeOut(data_point),
                *[sq.animate.set_fill(YELLOW, opacity=0.8).set_stroke(YELLOW_A) for sq in meta_nodes[0][0]],
                run_time=0.8
            )

        goals_text = Text("Goals: Analyze, Understand, Edit Models", font_size=20, color=YELLOW).next_to(meta_bounding_box, DOWN * 2, buff=0.3)

        with self.voiceover(text="Its goal is to deeply analyze and edit the fundamental symmetries of these models. As the functional processes the model..."):
            self.play(Write(goals_text))

            # Làn sóng phân tích (Analytical Flow) chạy qua Metanetwork
            step1_edges = VGroup(*[e for e in meta_edges if e.get_start()[0] < meta_nodes[1][0].get_x()])
            self.play(LaggedStart(*[e.animate.set_color(YELLOW).set_stroke(width=3, opacity=1) for e in step1_edges], lag_ratio=0.1), run_time=1)
            self.play(
                *[sq.animate.set_fill(YELLOW, opacity=0.8) for node in meta_nodes[1] for sq in node],
                *[e.animate.set_color(PURPLE_E).set_stroke(width=1.2, opacity=0.5) for e in step1_edges],
                run_time=0.8
            )

            step2_edges = VGroup(*[e for e in meta_edges if meta_nodes[1][0].get_x() < e.get_start()[0] < meta_nodes[2][0].get_x()])
            self.play(LaggedStart(*[e.animate.set_color(TEAL).set_stroke(width=3, opacity=1) for e in step2_edges], lag_ratio=0.1), run_time=1)
            self.play(
                *[sq.animate.set_fill(TEAL, opacity=0.8) for node in meta_nodes[2] for sq in node],
                *[e.animate.set_color(PURPLE_E).set_stroke(width=1.2, opacity=0.5) for e in step2_edges],
                run_time=0.8
            )

        with self.voiceover(text="...it eventually outputs a highly optimized, structurally edited version of the original network."):
            step3_edges = VGroup(*[e for e in meta_edges if e.get_start()[0] > meta_nodes[1][0].get_x()])
            self.play(LaggedStart(*[e.animate.set_color(GREEN).set_stroke(width=3, opacity=1) for e in step3_edges], lag_ratio=0.1), run_time=1)

            # Nút Output cuối cùng biến thành một mạng đã được Edit (Màu xanh lục rực rỡ)
            self.play(
                *[sq.animate.set_fill(GREEN, opacity=1).set_stroke(GREEN_A, width=1) for node in meta_nodes[-1] for sq in node],
                *[e.animate.set_color(PURPLE_E).set_stroke(width=1.2, opacity=0.5) for e in step3_edges],
                run_time=0.8
            )

            # Chớp sáng xác nhận chỉnh sửa thành công
            self.play(Flash(meta_nodes[-1][0], color=GREEN_A, line_length=0.4))
            self.wait(1.5)

        # Dọn dẹp chuyển cảnh
        self.play(FadeOut(Group(*self.mobjects)))

        # --- CHƯƠNG 2: IMPLICIT NEURAL REPRESENTATIONS (3D ĐIỆN Ảnh) ---
        header2 = VGroup(
            Text("Neural Networks as Data", font_size=36, weight=BOLD),
            Text("Model Populations: Implicit Neural Representations", font_size=24, color=TEAL)
        ).arrange(DOWN, buff=0.2).to_edge(UP)
        self.add_fixed_in_frame_mobjects(header2)
        self.play(FadeIn(header2))

        self.move_camera(phi=65 * DEGREES, theta=-30 * DEGREES, run_time=1.5)

        # Bắt đầu xoay camera điện ảnh liên tục
        self.begin_ambient_camera_rotation(rate=0.08)

        axes3d = ThreeDAxes(x_range=[-2, 2], y_range=[-2, 2], z_range=[-2, 2], x_length=4, y_length=4, z_length=4)
        target_object = Surface(
            lambda u, v: axes3d.c2p(
                1.2 * np.cos(u) * np.cos(v),
                1.2 * np.cos(u) * np.sin(v),
                1.2 * np.sin(u) + 0.3 * np.cos(3*v)
            ),
            u_range=[-PI/2, PI/2], v_range=[0, TAU],
            resolution=(25, 25), fill_color=YELLOW_D, fill_opacity=0.7, stroke_width=0.3, stroke_color=YELLOW_B
        )

        camera_plane = Rectangle(width=2, height=2, color=GRAY_C, fill_opacity=0.2).move_to(axes3d.c2p(-3, -3, 1))
        self.add_fixed_orientation_mobjects(camera_plane)

        with self.voiceover("A perfect example is Implicit Neural Representations, like NeRFs. Instead of storing 3D scenes as pixels or voxels..."):
            self.play(Create(axes3d), Create(target_object))
            self.play(FadeIn(camera_plane, scale=0.5))

        # Hiệu ứng tia sáng bắn ra
        ray_line = DashedLine(start=axes3d.c2p(-3, -3, 1), end=axes3d.c2p(0.5, 0.5, 0), color=RED, dash_length=0.1)
        photon = Sphere(radius=0.05, color=WHITE).move_to(ray_line.get_start())
        intersection_dot = Sphere(radius=0.1, color=RED).move_to(axes3d.c2p(0.5, 0.5, 0))

        with self.voiceover("We train a tiny, unique neural network to map spatial coordinates and viewing rays directly to color and density."):
            self.play(Create(ray_line))
            self.play(photon.animate.move_to(ray_line.get_end()), run_time=0.8, rate_func=linear)

            # Vụ nổ khi chạm bề mặt
            flash = Sphere(radius=0.02, color=YELLOW).move_to(intersection_dot)
            self.play(FadeOut(photon), FadeIn(intersection_dot), flash.animate.scale(10).set_opacity(0), run_time=0.5)

        # Công thức
        nerf_eq = MathTex(r"(x, y, z, \theta, \phi) \rightarrow", r"F_\Theta", r"\rightarrow (\text{RGB}, \sigma)", font_size=32)
        nerf_eq[1].set_color(TEAL)
        nerf_box = SurroundingRectangle(nerf_eq, color=TEAL, buff=0.2)
        nerf_group = VGroup(nerf_box, nerf_eq).to_corner(DR).shift(UP * 0.5 + LEFT * 0.5)
        self.add_fixed_in_frame_mobjects(nerf_group)

        self.play(Write(nerf_group))

        with self.voiceover("Multiply this by thousands of 3D objects, and your dataset is no longer just images. It is a dataset composed entirely of neural networks."):
            # Nhảy ra hàng loạt mô hình (Populations)
            self.play(
                target_object.animate.scale(0.4).shift(UP * 1.5 + LEFT * 1.5),
                FadeOut(ray_line), FadeOut(camera_plane), FadeOut(intersection_dot),
                run_time=1.5
            )
            clones = VGroup(*[target_object.copy().shift(RIGHT * np.random.uniform(-1, 3) + DOWN * np.random.uniform(0, 3)).scale(np.random.uniform(0.6, 1.2)) for _ in range(6)])
            self.play(LaggedStart(*[FadeIn(clone, scale=0.2) for clone in clones], lag_ratio=0.15), run_time=2)
            self.wait(1)

        # Dừng camera ambient trước khi dọn dẹp để tránh lỗi
        self.stop_ambient_camera_rotation()
        self.play(
            FadeOut(axes3d), FadeOut(target_object), FadeOut(clones),
            FadeOut(nerf_group), FadeOut(header2)
        )

        # --- CHƯƠNG 3: VŨ TRỤ HUGGING FACE (GALAXY EFFECT) ---
        self.move_camera(phi=0, theta=-90 * DEGREES, run_time=1)

        header3 = VGroup(
            Text("Neural Networks as Data", font_size=36, weight=BOLD),
            Text("The Wild: Hugging Face Model Populations", font_size=24, color=ORANGE)
        ).arrange(DOWN, buff=0.2).to_edge(UP)
        self.add_fixed_in_frame_mobjects(header3)
        self.play(FadeIn(header3))

        hf_text = Text("Models: 2,243,592+", font_size=28, color=WHITE).to_corner(UL).shift(DOWN * 1.0)
        self.add_fixed_in_frame_mobjects(hf_text)
        self.play(Write(hf_text))

        np.random.seed(42)

        def create_cluster(center, num_nodes, color, spread=1.2):
            nodes = VGroup()
            for _ in range(num_nodes):
                pos = center + np.array([np.random.normal(0, spread), np.random.normal(0, spread), 0])
                radius = np.random.uniform(0.04, 0.2)
                nodes.add(Dot(pos, radius=radius, color=color, fill_opacity=0.9))
            return nodes

        # Phân bố các cụm theo thiên hà (hạ thấp để không đè title)
        vision_cluster = create_cluster(LEFT * 3.5 + DOWN * 1.5, 80, TEAL_C, spread=1.0)
        nlp_cluster = create_cluster(RIGHT * 3.5 + DOWN * 1.5, 120, PURPLE_B, spread=1.2)
        audio_cluster = create_cluster(DOWN * 0.2, 60, GREEN_C, spread=0.8)

        all_nodes = VGroup(vision_cluster, nlp_cluster, audio_cluster)

        # Kết nối nội bộ
        edges = VGroup()
        for cluster in [vision_cluster, nlp_cluster, audio_cluster]:
            for i in range(len(cluster)):
                if np.random.random() > 0.9:
                    target = cluster[np.random.randint(0, len(cluster))]
                    edges.add(Line(cluster[i].get_center(), target.get_center(), stroke_width=0.3, color=WHITE, stroke_opacity=0.15))

        # Kết nối chéo (Vision-Language, Audio-Language)
        cross_edges = VGroup()
        for _ in range(25):
            n1 = vision_cluster[np.random.randint(0, len(vision_cluster))]
            n2 = nlp_cluster[np.random.randint(0, len(nlp_cluster))]
            cross_edges.add(Line(n1.get_center(), n2.get_center(), stroke_width=0.8, color=ORANGE, stroke_opacity=0.6))

        for _ in range(15):
            n1 = audio_cluster[np.random.randint(0, len(audio_cluster))]
            n2 = nlp_cluster[np.random.randint(0, len(nlp_cluster))]
            cross_edges.add(Line(n1.get_center(), n2.get_center(), stroke_width=0.8, color=YELLOW, stroke_opacity=0.6))

        with self.voiceover("And this concept extends far beyond 3D scenes. Today, platforms like Hugging Face host over two million open-source models."):
            # Sinh ra hệ sinh thái như các vì sao (LaggedStart với GrowFromCenter)
            self.play(
                LaggedStart(*[GrowFromCenter(node) for node in all_nodes.submobjects], lag_ratio=0.005),
                run_time=3.5
            )
            self.play(FadeIn(edges, run_time=1))

        lbl_vision = Text("Computer Vision", font_size=20, weight=BOLD).next_to(vision_cluster, DOWN)
        lbl_nlp = Text("Large Language Models", font_size=20, weight=BOLD).next_to(nlp_cluster, DOWN)
        lbl_audio = Text("Audio & Speech", font_size=20, weight=BOLD).next_to(audio_cluster, UP)

        self.add_fixed_in_frame_mobjects(lbl_vision, lbl_nlp, lbl_audio)

        with self.voiceover("From computer vision architectures to massive language models, they form an evolving, deeply interconnected population of weight spaces."):
            self.play(FadeIn(lbl_vision), FadeIn(lbl_nlp), FadeIn(lbl_audio))

        with self.voiceover("To build algorithms that can automatically search, merge, or generate new models within this vast universe, we must first decipher the mathematical symmetries hidden inside them."):
            # Bắn tia sáng dọc theo các liên kết chéo
            self.play(Create(cross_edges, lag_ratio=0.1), run_time=2.5)
            self.play(cross_edges.animate.set_stroke(width=2).set_color(RED), run_time=1)
            self.wait(2)


## slide 73-75

In [ ]:
%%manim -qh -v WARNING NeuralNetworksAsDataWhy73

# ─── MAIN SCENE GRAPH ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
class NeuralNetworksAsDataWhy73(VoiceoverScene):
    def construct(self):
        self.camera.background_color = BLACK
        self.set_speech_service(EdgeTTSService(voice="en-US-GuyNeural"))

        # --- TIÊU ĐỀ ---
        header = VGroup(
            Text("Neural Networks as Data", font_size=36, weight=BOLD),
            Text("Why? The Power of Metanetworks", font_size=24, color=BLUE_B)
        ).arrange(DOWN, buff=0.2).to_edge(UP)
        self.play(FadeIn(header, shift=DOWN * 0.2))

        # Hàm tiện ích vẽ Mạng Nơ-ron chuẩn
        def create_nn(layers_sizes, color_theme=BLUE):
            neurons = VGroup()
            edges = VGroup()
            for i, num in enumerate(layers_sizes):
                layer = VGroup(*[Circle(radius=0.1, fill_color=color_theme, fill_opacity=0.6, stroke_width=1) for _ in range(num)])
                layer.arrange(DOWN, buff=0.2).move_to(RIGHT * 1.2 * i)
                neurons.add(layer)
            for i in range(len(layers_sizes) - 1):
                for n1 in neurons[i]:
                    for n2 in neurons[i+1]:
                        edges.add(Line(n1.get_center(), n2.get_center(), stroke_width=1.0, color=GRAY_E))
            nn_group = VGroup(edges, neurons).move_to(ORIGIN)
            return nn_group, neurons, edges

        # Hàm tiện ích vẽ Ma trận Trọng số (Weight Matrix)
        def create_matrix_grid(rows, cols, color_theme=BLUE_C):
            return VGroup(*[
                Square(side_length=0.12, fill_color=color_theme, fill_opacity=0.7, stroke_color=WHITE, stroke_width=0.5)
                for _ in range(rows * cols)
            ]).arrange_in_grid(rows=rows, cols=cols, buff=0.03)

        # --- XÂY DỰNG KIẾN TRÚC METANETWORK TRUNG TÂM (THU GỌN VÀ NÂNG CAO) ---
        meta_math = MathTex(r"\mathcal{F}_\Phi(", r"\Theta", r")", font_size=36).move_to(UP * 1.0)
        meta_math[0].set_color(PURPLE_B)
        meta_math[2].set_color(PURPLE_B)

        meta_core = VGroup()
        for c in [PURPLE_D, PURPLE_C, PURPLE_A]:
            grid = create_matrix_grid(3, 3, color_theme=c)
            meta_core.add(grid)
        meta_core.arrange(RIGHT, buff=0.3).next_to(meta_math, DOWN, buff=0.2)

        meta_edges = VGroup(*[Line(meta_core[i].get_right(), meta_core[i+1].get_left(), color=PURPLE_B, stroke_width=2) for i in range(2)])
        meta_group = VGroup(meta_math, meta_core, meta_edges).move_to(ORIGIN)

        meta_box = RoundedRectangle(width=meta_group.width + 0.6, height=meta_group.height + 0.6, corner_radius=0.2, color=PURPLE_E, fill_opacity=0.1, stroke_width=2).move_to(meta_group)
        meta_lbl = Text("Metanetwork Architecture", font_size=16, color=PURPLE_A).next_to(meta_box, UP, buff=0.1)

        # Đặt Metanetwork cố định ở khu vực trên-phải để lấy toàn bộ không gian dưới làm sân diễn
        full_meta = VGroup(meta_box, meta_group, meta_lbl).scale(0.85).move_to(RIGHT * 3.0 + UP * 0.5)

        with self.voiceover(text="Why should we treat neural networks as data? What is the practical use of a metanetwork? Let's explore three revolutionary applications."):
            self.play(DrawBorderThenFill(meta_box), Write(meta_lbl))
            self.play(Write(meta_math), FadeIn(meta_core, lag_ratio=0.2), Create(meta_edges))
            self.wait(0.5)

        # =====================================================================
        # ỨNG DỤNG 1: DỰ ĐOÁN NHANH (PERFORMANCE PREDICTION)
        # =====================================================================
        # Định vị ở khu vực dưới-trái
        nn_1, nodes_1, edges_1 = create_nn([3, 4, 3], color_theme=BLUE_D)
        nn_1.scale(0.7).move_to(LEFT * 4.0 + DOWN * 1.5)
        lbl_nn1 = Text("Standard Model", font_size=14, color=BLUE_B).next_to(nn_1, DOWN, buff=0.15)

        theta_matrix = create_matrix_grid(4, 4, BLUE_C).move_to(LEFT * 1.5 + DOWN * 1.5)
        lbl_theta = MathTex(r"\Theta", font_size=24, color=BLUE_B).next_to(theta_matrix, DOWN, buff=0.15)

        with self.voiceover(text="First, predicting model performance. Normally, evaluating a model requires running millions of test images through its layers."):
            self.play(FadeIn(nn_1), Write(lbl_nn1))
            scan_line = Line(nn_1.get_top() + UP*0.1, nn_1.get_bottom() + DOWN*0.1, color=YELLOW).move_to(nn_1.get_left())
            self.play(Create(scan_line))
            self.play(scan_line.animate.move_to(nn_1.get_right()), run_time=1.2)
            self.play(FadeOut(scan_line))
            self.play(TransformFromCopy(nn_1, theta_matrix), Write(lbl_theta))

        # Định vị kết quả ở khu vực dưới-phải
        accuracy_box = RoundedRectangle(width=2.5, height=1.0, fill_color=GREEN_E, fill_opacity=0.4, stroke_color=GREEN_B).move_to(RIGHT * 3.0 + DOWN * 2.0)
        accuracy_text = Text("Accuracy: 99.1%", font_size=16, color=GREEN).move_to(accuracy_box.get_center() + UP*0.1)
        speed_text = Text("≥ 50,000x Faster", font_size=16, color=YELLOW, weight=BOLD).next_to(accuracy_text, DOWN, buff=0.1)

        with self.voiceover(text="Instead, a metanetwork can simply ingest the model's weight matrix, analyze its structural geometry, and instantly predict its accuracy."):
            self.play(
                theta_matrix.animate.scale(0.4).move_to(meta_core[0].get_center()).set_opacity(0),
                FadeOut(lbl_theta),
                run_time=1
            )
            # Sóng phân tích
            self.play(meta_core[0].animate.set_color(YELLOW), meta_edges[0].animate.set_color(YELLOW), run_time=0.25)
            self.play(meta_core[1].animate.set_color(YELLOW), meta_edges[1].animate.set_color(YELLOW), run_time=0.25)
            self.play(meta_core[2].animate.set_color(GREEN), run_time=0.25)

            # Xuất kết quả
            self.play(DrawBorderThenFill(accuracy_box), Write(accuracy_text))

        with self.voiceover(text="This bypasses the data pipeline entirely, making evaluation up to fifty thousand times faster."):
            self.play(FadeIn(speed_text, shift=UP*0.1))
            self.play(Flash(speed_text, color=YELLOW, line_length=0.2))
            self.wait(1)

        # DỌN DẸP KHUNG HÌNH (WIPE CLEAN) TRƯỚC KHI SANG ỨNG DỤNG 2
        self.play(
            meta_core[0].animate.set_color(PURPLE_D), meta_edges[0].animate.set_color(PURPLE_B),
            meta_core[1].animate.set_color(PURPLE_C), meta_edges[1].animate.set_color(PURPLE_B),
            meta_core[2].animate.set_color(PURPLE_A),
            FadeOut(nn_1), FadeOut(lbl_nn1), FadeOut(accuracy_box), FadeOut(accuracy_text), FadeOut(speed_text)
        )

        # =====================================================================
        # ỨNG DỤNG 2: CHỈNH SỬA MẠNG (NETWORK EDITING / PRUNING)
        # =====================================================================
        with self.voiceover(text="Second, we can directly edit networks."):
            theta_matrix.scale(2.5).move_to(LEFT * 4.0 + DOWN * 1.5).set_opacity(1).set_color(BLUE_C)
            self.play(FadeIn(theta_matrix), FadeIn(lbl_theta.next_to(theta_matrix, DOWN, buff=0.15)))

        pruned_theta = create_matrix_grid(4, 4, BLUE_C).move_to(LEFT * 4.0 + DOWN * 1.5)
        for idx in [2, 5, 8, 11, 14]:
            pruned_theta[idx].set_fill(BLACK, opacity=0.8).set_stroke(GRAY_E)

        lbl_pruned_theta = MathTex(r"\Theta_{pruned}", font_size=24, color=TEAL_C).next_to(pruned_theta, DOWN, buff=0.15)

        with self.voiceover(text="A metanetwork can ingest a dense, heavy model, intelligently manipulate its internal weight space, and output a structurally pruned weight matrix."):
            self.play(
                theta_matrix.animate.scale(0.4).move_to(meta_core[0].get_center()).set_opacity(0),
                FadeOut(lbl_theta),
                run_time=1
            )
            # Sóng phân tích Pruning
            self.play(meta_core[0].animate.set_color(TEAL), meta_edges[0].animate.set_color(TEAL), run_time=0.25)
            self.play(meta_core[1].animate.set_color(TEAL), meta_edges[1].animate.set_color(TEAL), run_time=0.25)
            self.play(meta_core[2].animate.set_color(TEAL), run_time=0.25)

            pruned_theta.scale(0.4).move_to(meta_core[2].get_center()).set_opacity(0)
            self.play(
                pruned_theta.animate.scale(2.5).move_to(RIGHT * 0.5 + DOWN * 1.5).set_opacity(1),
                FadeIn(lbl_pruned_theta.move_to(RIGHT * 0.5 + DOWN * 2.3)),
                run_time=1.5
            )

        nn_2, nodes_2, edges_2 = create_nn([3, 4, 3], color_theme=TEAL_D)
        nn_2.scale(0.7).move_to(RIGHT * 4.5 + DOWN * 1.5)
        lbl_nn2 = Text("Pruned Model", font_size=14, color=TEAL_B).next_to(nn_2, DOWN, buff=0.15)

        for idx in [1, 4, 7, 10, 15, 18]:
            edges_2[idx].set_stroke(width=0.2, opacity=0.2)

        with self.voiceover(text="This instantly generates a brand new, highly compressed, and efficient model without any retraining. Perfect for on-device deployment."):
            self.play(ReplacementTransform(pruned_theta, nn_2), ReplacementTransform(lbl_pruned_theta, lbl_nn2))
            self.play(Flash(nn_2, color=TEAL, line_length=0.3))
            self.wait(1)

        # DỌN DẸP KHUNG HÌNH (WIPE CLEAN) TRƯỚC KHI SANG ỨNG DỤNG 3
        self.play(
            meta_core[0].animate.set_color(PURPLE_D), meta_edges[0].animate.set_color(PURPLE_B),
            meta_core[1].animate.set_color(PURPLE_C), meta_edges[1].animate.set_color(PURPLE_B),
            meta_core[2].animate.set_color(PURPLE_A),
            FadeOut(nn_2), FadeOut(lbl_nn2)
        )

        # =====================================================================
        # ỨNG DỤNG 3: ĐIỀU TRA MÔ HÌNH (MODEL FORENSICS)
        # =====================================================================
        wild_theta = create_matrix_grid(4, 4, RED_D).move_to(LEFT * 4.0 + DOWN * 1.5)
        for sq in wild_theta:
            sq.set_fill(interpolate_color(RED_D, ORANGE, np.random.rand()), opacity=0.9)
        lbl_wild = Text("Unknown 'Wild' Model", font_size=14, color=RED_A).next_to(wild_theta, DOWN, buff=0.15)

        with self.voiceover(text="Finally, as the open-source model population explodes, we desperately need model forensics."):
            self.play(FadeIn(wild_theta), Write(lbl_wild))

        fingerprint = VGroup(*[Rectangle(width=0.06, height=0.7, fill_color=ORANGE, fill_opacity=np.random.uniform(0.3, 1), stroke_width=0) for _ in range(20)]).arrange(RIGHT, buff=0.05)
        fingerprint.move_to(RIGHT * 3.0 + DOWN * 1.2)

        checklist = VGroup(
            Text("✘ Copyrighted Data Detected", font_size=16, color=RED_B),
            Text("✔ Base Architecture: LLaMA", font_size=16, color=GREEN_B),
            Text("! High Bias Estimation", font_size=16, color=YELLOW_B)
        ).arrange(DOWN, aligned_edge=LEFT, buff=0.2).next_to(fingerprint, DOWN, buff=0.3)

        with self.voiceover(text="By ingesting an unknown model from the internet, the metanetwork analyzes the structural fingerprint of its weights."):
            self.play(
                wild_theta.animate.scale(0.4).move_to(meta_core[0].get_center()).set_opacity(0),
                FadeOut(lbl_wild),
                run_time=1
            )
            # Sóng Forensic Analysis
            self.play(meta_core[0].animate.set_color(RED_E), meta_edges[0].animate.set_color(RED_A), run_time=0.25)
            self.play(meta_core[1].animate.set_color(ORANGE), meta_edges[1].animate.set_color(ORANGE), run_time=0.25)
            self.play(meta_core[2].animate.set_color(YELLOW), run_time=0.25)

            self.play(FadeIn(fingerprint, lag_ratio=0.1), run_time=1)

        with self.voiceover(text="It can decode this fingerprint to answer critical questions: Was it trained on copyrighted data? Is it safe? Does it carry hidden biases?"):
            scan_laser = Line(fingerprint.get_top() + UP*0.1, fingerprint.get_bottom() + DOWN*0.1, color=WHITE).move_to(fingerprint.get_left())
            self.play(Create(scan_laser))
            self.play(scan_laser.animate.move_to(fingerprint.get_right()), run_time=1.2)
            self.play(FadeOut(scan_laser))

            self.play(LaggedStart(*[FadeIn(item, shift=LEFT * 0.2) for item in checklist], lag_ratio=0.4), run_time=1.5)
            self.wait(1)

        conclusion = Text("Turning the wild model jungle into a searchable atlas.", font_size=24, color=WHITE).to_edge(DOWN, buff=0.5)

        with self.voiceover(text="Ultimately, weight space learning and metanetworks turn the chaotic, wild jungle of millions of models into a structured, searchable, and editable atlas."):
            self.play(Write(conclusion))
            self.play(
                full_meta.animate.set_color(TEAL),
                checklist.animate.set_opacity(0.3),
                fingerprint.animate.set_opacity(0.3),
                run_time=2
            )
            self.wait(2)

        self.play(FadeOut(Group(*self.mobjects)))


## slide 76-77
Weight space learning: example techniques

In [ ]:
%%manim -qh -v WARNING WeightSpaceTechniques76

class WeightSpaceTechniques76(VoiceoverScene, ThreeDScene):
    def construct(self):
        self.set_speech_service(EdgeTTSService(voice="en-US-GuyNeural"))

        # --- TIÊU ĐỀ CHÍNH ---
        header = VGroup(
            Text("Weight Space Learning: Advanced Techniques", font_size=36, weight=BOLD),
            Text("Deep Weight Spaces & Functional Architectures", font_size=24, color=BLUE_B)
        ).arrange(DOWN, buff=0.2).to_edge(UP)
        self.play(FadeIn(header, shift=DOWN * 0.2))

        # --- HỒI 1: LINEAR EQUIVARIANT LAYERS (DWSN) ---
        subtitle1 = Text("1. Permutation Equivariant Layers (DWSN)", font_size=24, color=YELLOW).next_to(header, DOWN, buff=0.4).to_edge(LEFT, buff=1)
        self.play(Write(subtitle1))

        neurons_l1 = VGroup(*[Dot(radius=0.15, color=GRAY) for _ in range(3)]).arrange(DOWN, buff=0.5).move_to(LEFT * 4 + DOWN * 0.5)
        neurons_l2 = VGroup(*[Dot(radius=0.15, color=GRAY) for _ in range(3)]).arrange(DOWN, buff=0.5).move_to(LEFT * 2 + DOWN * 0.5)

        edges = VGroup()
        for n1 in neurons_l1:
            for n2 in neurons_l2:
                edges.add(Line(n1.get_center(), n2.get_center(), stroke_width=1.5, color=GRAY_D))
        nn_group = VGroup(edges, neurons_l1, neurons_l2)

        matrix_cells = VGroup()
        for r in range(3):
            for c in range(3):
                cell = Square(side_length=0.6, stroke_width=1, stroke_color=WHITE)
                cell.set_fill(TEAL_D if r == c else TEAL_A, opacity=0.8)
                cell.move_to(RIGHT * (c*0.6) + DOWN * (r*0.6))
                matrix_cells.add(cell)
        matrix_cells.center().move_to(RIGHT * 3 + DOWN * 0.5)
        matrix_lbl = Text("Weight Matrix Blocks", font_size=20).next_to(matrix_cells, UP)

        with self.voiceover("According to Navon et al, deep weight space networks structure parameters into distinct blocks to preserve permutation equivariance."):
            self.play(Create(nn_group), FadeIn(matrix_cells), Write(matrix_lbl))

        with self.voiceover("When neurons in the hidden layers are permuted, the equivariance constraint forces the corresponding rows and columns in the weight block matrices to swap simultaneously."):
            self.play(neurons_l2[0].animate.set_color(RED), neurons_l2[1].animate.set_color(GREEN))
            self.play(
                neurons_l2[0].animate.move_to(neurons_l2[1].get_center()),
                neurons_l2[1].animate.move_to(neurons_l2[0].get_center()),
                run_time=1.2
            )
            row_0 = VGroup(matrix_cells[0], matrix_cells[1], matrix_cells[2])
            row_1 = VGroup(matrix_cells[3], matrix_cells[4], matrix_cells[5])
            self.play(row_0.animate.shift(DOWN * 0.6), row_1.animate.shift(UP * 0.6), run_time=1.2)
            self.wait(0.5)

        self.play(FadeOut(subtitle1), FadeOut(nn_group), FadeOut(matrix_cells), FadeOut(matrix_lbl))

        # --- HỒI 2: NEURAL FUNCTIONAL TRANSFORMERS (NFT) ---
        subtitle2 = Text("2. Neural Functional Transformers", font_size=24, color=PURPLE_B).next_to(header, DOWN, buff=0.4).to_edge(LEFT, buff=1)
        self.play(Write(subtitle2))

        # Hàm tạo lưới ma trận ô vuông màu sắc chuẩn phong cách 3b1b
        def create_weight_grid(color, size=0.3):
            grid = VGroup()
            for r in range(3):
                for c in range(3):
                    cell = Square(side_length=size, stroke_width=0.5, stroke_color=WHITE)
                    cell.set_fill(color, opacity=0.7)
                    cell.move_to(RIGHT * size * c + DOWN * size * r)
                    grid.add(cell)
            grid.center() # Đồng băng tâm hình học
            return grid

        # Khởi tạo ma trận trọng số đầu vào dưới dạng các Token hình học
        w_grid_1 = create_weight_grid(BLUE_D).move_to(LEFT * 4.5 + UP * 1.0)
        w_grid_2 = create_weight_grid(TEAL_D).move_to(LEFT * 4.5 + DOWN * 1.0)

        lbl_w1 = MathTex(r"W^{(\ell-1)}", font_size=24).next_to(w_grid_1, LEFT, buff=0.2)
        lbl_w2 = MathTex(r"W^{(\ell)}", font_size=24).next_to(w_grid_2, LEFT, buff=0.2)
        token_group = VGroup(w_grid_1, w_grid_2, lbl_w1, lbl_w2)

        with self.voiceover("To scale up, Neural Functional Transformers represent entire weight matrices as structured tokens, treating layers as sequence elements."):
            self.play(LaggedStart(*[FadeIn(item, shift=RIGHT * 0.3) for item in token_group], lag_ratio=0.15))
            self.wait(0.5)

        # 3b1b Style: Nhân bản và biến hình ma trận thành không gian Q, K, V
        q_matrix = create_weight_grid(YELLOW_D, size=0.25)
        k_matrix = create_weight_grid(GREEN_D, size=0.25)
        v_matrix = create_weight_grid(ORANGE, size=0.25)

        # Sắp xếp bố cục hình học gọn gàng theo chiều dọc để tránh chồng lấn
        q_matrix.move_to(RIGHT * 1.0 + UP * 1.5)
        k_matrix.move_to(RIGHT * 1.0 + ORIGIN)
        v_matrix.move_to(RIGHT * 1.0 + DOWN * 1.5)

        lbl_q = MathTex("Q", color=YELLOW, font_size=24).next_to(q_matrix, UP, buff=0.1)
        lbl_k = MathTex(r"K^\top", color=GREEN, font_size=24).next_to(k_matrix, UP, buff=0.1)
        lbl_v = MathTex("V", color=ORANGE, font_size=24).next_to(v_matrix, UP, buff=0.1)

        with self.voiceover("Each token undergoes an equivariant projection, spinning out distinct Query, Key, and Value weight-space representations."):
            self.play(
                TransformFromCopy(w_grid_2, q_matrix),
                TransformFromCopy(w_grid_1, k_matrix),
                TransformFromCopy(w_grid_2, v_matrix),
                FadeIn(lbl_q), FadeIn(lbl_k), FadeIn(lbl_v),
                run_time=2
            )
            self.wait(0.5)

        # Khởi tạo ma trận kết quả Attention trống tại giao điểm của Q và K^T
        attn_grid = VGroup()
        for r in range(3):
            for c in range(3):
                cell = Square(side_length=0.5, stroke_width=0.8, stroke_color=GRAY_A)
                cell.set_fill(PURPLE_E, opacity=0.1)
                cell.move_to(RIGHT * 0.5 * c + DOWN * 0.5 * r)
                attn_grid.add(cell)
        attn_grid.center()
        attn_grid.move_to(RIGHT * 4.0 + DOWN * 0.5)

        attn_title = MathTex(r"\text{Softmax}\left(\frac{QK^\top}{\sqrt{d}}\right)", font_size=24).next_to(attn_grid, UP, buff=0.4)

        with self.voiceover("Now observe the matrix multiplication. The rows of Query interact directly with the columns of Key transpose."):
            self.play(Create(attn_grid), Write(attn_title))

        # NÂNG CẤP ĐẠO DIỄN: Quét toàn bộ dòng song song tính toán tương tác (Parallel Sweep Computing)
        with self.voiceover("As they scan across the layers, their inner products intersect to map out the raw attention coefficients cell by cell."):
            for i in range(3):
                row_highlight = SurroundingRectangle(VGroup(*[q_matrix[i*3 + j] for j in range(3)]), color=YELLOW, buff=0.02)
                col_highlights = VGroup(*[SurroundingRectangle(VGroup(*[k_matrix[j*3 + c] for j in range(3)]), color=GREEN, buff=0.02) for c in range(3)])

                self.play(Create(row_highlight), Create(col_highlights), run_time=0.5)

                target_cells = VGroup(*[attn_grid[i*3 + c] for c in range(3)])
                self.play(
                    target_cells.animate.set_fill(PURPLE_A, opacity=0.6),
                    *[Flash(cell, color=PURPLE_A, line_length=0.15) for cell in target_cells],
                    run_time=0.5
                )
                self.play(FadeOut(row_highlight), FadeOut(col_highlights), run_time=0.2)

        # HIỆU ỨNG SÓNG SOFTMAX NÂNG CAO
        with self.voiceover("Finally, a row-wise Softmax normalization passes through, smoothly scaling the weights into a fluid global activation map."):
            softmax_wave = Line(
                attn_grid.get_corner(UL) + LEFT * 0.1,
                attn_grid.get_corner(DL) + LEFT * 0.1,
                color=YELLOW, stroke_width=4
            ).set_opacity(0.8)

            opacities = [0.9, 0.2, 0.4, 0.3, 0.8, 0.1, 0.2, 0.5, 0.9]
            softmax_animations = [attn_grid[idx].animate.set_fill(GOLD, opacity=opacities[idx]) for idx in range(9)]

            self.play(
                softmax_wave.animate.move_to(attn_grid.get_right() + RIGHT * 0.1),
                *softmax_animations,
                run_time=2,
                rate_func=smooth
            )
            self.play(FadeOut(softmax_wave))
            self.wait(1)

        # Dọn dẹp hồi 2 sạch sẽ không để lại rác bộ nhớ lớp hiển thị
        self.play(
            FadeOut(subtitle2), FadeOut(token_group), FadeOut(q_matrix), FadeOut(k_matrix),
            FadeOut(v_matrix), FadeOut(lbl_q), FadeOut(lbl_k), FadeOut(lbl_v), FadeOut(attn_grid), FadeOut(attn_title)
        )


        # --- HỒI 3: GRAPH METANETWORKS FOR DIVERSE ARCHITECTURES (Lim et al., ICLR 2024) ---
        subtitle3 = Text("3. Graph Metanetworks", font_size=24, color=GREEN_B).next_to(header, DOWN, buff=0.4).to_edge(LEFT, buff=1)
        self.play(Write(subtitle3))

        # --- PHÂN CẢNH 3A: XUẤT HIỆN TUẦN TỰ VÀ ĐỐI LẬP KIẾN TRÚC ---
        # 1. Khởi tạo mạng A (Tuần tự phẳng)
        arch_a_title = Text("Architecture A: Standard MLP", font_size=16, color=BLUE_A).move_to(LEFT * 4.2 + UP * 1.5)
        in_a = VGroup(*[Dot(radius=0.12, color=BLUE_A) for _ in range(2)]).arrange(DOWN, buff=0.6).move_to(LEFT * 5.2 + DOWN * 0.5)
        hid_a = VGroup(*[Dot(radius=0.12, color=BLUE_B) for _ in range(3)]).arrange(DOWN, buff=0.4).move_to(LEFT * 4.0 + DOWN * 0.5)
        out_a = VGroup(*[Dot(radius=0.12, color=BLUE_C) for _ in range(1)]).arrange(DOWN, buff=0.6).move_to(LEFT * 2.8 + DOWN * 0.5)

        edges_a = VGroup()
        for i in in_a:
            for h in hid_a:
                edges_a.add(Line(i.get_center(), h.get_center(), stroke_width=1.2, color=GRAY_C))
        for h in hid_a:
            for o in out_a:
                edges_a.add(Line(h.get_center(), o.get_center(), stroke_width=1.2, color=GRAY_C))
        network_a = VGroup(arch_a_title, edges_a, in_a, hid_a, out_a)

        with self.voiceover("But what if we aren't just dealing with fixed MLPs? Consider a standard, structurally rigid model like Architecture A."):
            self.play(Write(arch_a_title))
            self.play(Create(VGroup(in_a, hid_a, out_a)), Create(edges_a, lag_ratio=0.05), run_time=1.5)

        # 2. Khởi tạo mạng B (Dị thể, bất đối xứng)
        arch_b_title = Text("Architecture B: Irregular (With Skip Connection)", font_size=16, color=ORANGE).move_to(RIGHT * 3.5 + UP * 1.5)
        in_b = VGroup(*[Dot(radius=0.12, color=ORANGE) for _ in range(2)]).arrange(DOWN, buff=0.8).move_to(RIGHT * 1.5 + DOWN * 0.5)
        hid_b1 = VGroup(*[Dot(radius=0.12, color=YELLOW_D) for _ in range(2)]).arrange(DOWN, buff=0.5).move_to(RIGHT * 2.8 + DOWN * 0.5)
        hid_b2 = VGroup(*[Dot(radius=0.12, color=YELLOW_A) for _ in range(4)]).arrange(DOWN, buff=0.3).move_to(RIGHT * 4.1 + DOWN * 0.5)
        out_b = VGroup(*[Dot(radius=0.12, color=RED_D) for _ in range(1)]).arrange(DOWN, buff=0.8).move_to(RIGHT * 5.4 + DOWN * 0.5)

        edges_b = VGroup()
        for i in in_b:
            for h in hid_b1:
                edges_b.add(Line(i.get_center(), h.get_center(), stroke_width=1.2, color=GRAY_C))
        for h in hid_b1:
            for h2 in hid_b2:
                edges_b.add(Line(h.get_center(), h2.get_center(), stroke_width=1.2, color=GRAY_C))
        for h2 in hid_b2:
            for o in out_b:
                edges_b.add(Line(h2.get_center(), o.get_center(), stroke_width=1.2, color=GRAY_C))

        skip_line = ArcBetweenPoints(in_b[0].get_center(), out_b[0].get_center(), radius=-3.5, color=RED_A, stroke_width=2.5)
        edges_b.add(skip_line)
        network_b = VGroup(arch_b_title, edges_b, in_b, hid_b1, hid_b2, out_b)

        with self.voiceover("Now contrast it with Architecture B, featuring unpredictable layer widths and non sequential skip connections."):
            self.play(Write(arch_b_title))
            self.play(Create(VGroup(in_b, hid_b1, hid_b2, out_b)), Create(edges_b, lag_ratio=0.05), run_time=1.8)
            # Làm rực sáng đường kết nối tắt để nhấn mạnh điểm dị thể
            self.play(Indicate(skip_line, color=RED, scale_factor=1.2))

        # 3. Chỉ ra lỗi của phương pháp cũ
        with self.voiceover("Traditional weight space networks completely fail here, because they expect fixed matrix shapes and cannot handle irregular topologies."):
            # Tạo hiệu ứng cảnh báo đỏ rung lắc trên phần cấu trúc dị thể của mạng B
            failure_box = SurroundingRectangle(VGroup(hid_b2, skip_line), color=RED, buff=0.15)
            self.play(Create(failure_box))
            self.play(VGroup(hid_b2, skip_line, failure_box).animate.shift(UP*0.1), rate_func=there_and_back, run_time=0.2)
            self.play(VGroup(hid_b2, skip_line, failure_box).animate.shift(DOWN*0.1), rate_func=there_and_back, run_time=0.2)
            self.play(FadeOut(failure_box))

        # --- PHÂN CẢNH 3B: BIẾN HÌNH TOPO THÀNH ĐỒ THỊ THAM SỐ TOÀN CỤC ---
        # Tạo Đồ thị Đa cấu trúc hợp nhất ở giữa màn hình
        graph_centers = [
            UP*1.2+LEFT*1.5, UP*0.8+RIGHT*0.2, DOWN*0.4+LEFT*2.2,
            DOWN*1.2+LEFT*0.3, DOWN*0.8+RIGHT*1.8, UP*0.2+RIGHT*2.4, DOWN*0.2+LEFT*0.4
        ]
        gmn_nodes = VGroup(*[Dot(pos, radius=0.16, color=GREEN_C) for pos in graph_centers])

        gmn_edges = VGroup()
        connections = [(0,1), (0,2), (1,3), (2,3), (3,4), (1,5), (2,6), (6,4), (0,5)]
        for u, v in connections:
            gmn_edges.add(Line(gmn_nodes[u].get_center(), gmn_nodes[v].get_center(), stroke_width=2.5, color=GREEN_E))

        gmn_graph = VGroup(gmn_edges, gmn_nodes).move_to(DOWN * 0.5)
        gmn_math_label = MathTex(r"G = (\mathcal{V}_{\text{neurons}}, \mathcal{E}_{\text{weights}})", font_size=28, color=GREEN_A).next_to(gmn_graph, UP, buff=0.4)

        with self.voiceover("This is where Graph Metanetworks introduce a revolutionary shift. Instead of forcing weights into rigid tensor blocks, every model is unified into a single graph representation."):
            self.play(
                FadeOut(arch_a_title), FadeOut(arch_b_title),
                ReplacementTransform(network_a[1:], gmn_graph),
                ReplacementTransform(network_b[1:], gmn_graph),
                run_time=2.2
            )
            self.play(Write(gmn_math_label))

        # --- PHÂN CẢNH 3C: GIẢI THÍCH CHI TIẾT ĐỈNH VÀ CẠNH ---
        with self.voiceover("In this unified parameter graph, the original neurons themselves act as the vertices."):
            # Phóng to một đỉnh bất kỳ và gắn nhãn chú thích rõ ràng
            target_node = gmn_nodes[1]
            node_anote = Text("Vertex = Neuron", font_size=16, color=YELLOW).next_to(target_node, UP, buff=0.2)
            self.play(target_node.animate.scale(2.0).set_color(YELLOW), Write(node_anote))
            self.wait(0.5)
            self.play(target_node.animate.scale(0.5).set_color(GREEN_C), FadeOut(node_anote))

        with self.voiceover("And the weight values reside directly as attributes on the connecting edges."):
            # Phóng to một cạnh bất kỳ và gắn nhãn chú thích rõ ràng
            target_edge = gmn_edges[0]
            edge_anote = Text("Edge Attribute = Weight Value", font_size=16, color=ORANGE).next_to(target_edge, UR, buff=0.1)
            self.play(target_edge.animate.set_stroke(width=6, color=ORANGE), Write(edge_anote))
            self.wait(0.5)
            self.play(target_edge.animate.set_stroke(width=2.5, color=GREEN_E), FadeOut(edge_anote))

        # --- PHÂN CẢNH 3D: LAN TRUYỀN THÔNG ĐIỆP ĐỒ THỊ ĐA LUỒNG ---
        gmn_operator = MathTex(r"f_{\text{GMN}}(G) \rightarrow \text{Predicted Property}", font_size=32, color=YELLOW).to_edge(DOWN, buff=0.6)

        with self.voiceover("Because Graph Neural Networks are inherently agnostic to size and layer boundaries, a single meta model can now compute predictions across any arbitrary topology."):
            self.play(Write(gmn_operator))

        with self.voiceover("Watch how messages flow. Node features are aggregated along the boundaries of the weights, creating structural awareness across the entire network architecture."):
            pulse_loops = 2
            for loop in range(pulse_loops):
                wave_animations = []
                for edge in gmn_edges:
                    pulse = ShowPassingFlash(
                        edge.copy().set_color(YELLOW if loop == 0 else ORANGE).set_stroke(width=4.5),
                        time_width=0.5
                    )
                    wave_animations.append(pulse)

                self.play(*wave_animations, run_time=1.2, rate_func=linear)
                self.play(
                    gmn_nodes.animate.set_color(YELLOW if loop == 0 else ORANGE),
                    run_time=0.3
                )

            self.play(Flash(gmn_nodes, color=ORANGE, line_length=0.25, flash_radius=0.3), run_time=1.0)
            self.wait(1)

        # --- KẾT THÚC TOÀN BỘ BÀI GIẢNG ---
        with self.voiceover("By learning directly on this parameter graph, Graph Metanetworks successfully unlock deep structural understanding across completely diverse model universes."):
            self.play(
                FadeOut(subtitle3), FadeOut(gmn_graph),
                FadeOut(gmn_math_label), FadeOut(gmn_operator),
                # header.animate.scale(1.2).move_to(ORIGIN),
                run_time=1
            )
            self.wait(1)


## slide 79-80
Further questions & comments

In [ ]:
%%manim -qh -v WARNING QuestionNComment79

from manim import *
import numpy as np
import asyncio
import threading
import hashlib
import os
from pathlib import Path
from manim_voiceover import VoiceoverScene
from manim_voiceover.services.base import SpeechService

# ─── CUSTOM EDGETTS SERVICE (ĐẢM BẢO CHẠY MƯỢT MÀ TRÊN COLAB) ───
class EdgeTTSService(SpeechService):
    def __init__(self, voice="en-US-GuyNeural", **kwargs):
        self.voice = voice
        super().__init__(**kwargs)

    def generate_from_text(self, text: str, cache_dir: str = None, path: str = None) -> dict:
        import edge_tts
        if cache_dir is None: cache_dir = "media/voiceovers"
        cache_dir_path = Path(cache_dir)
        os.makedirs(cache_dir_path, exist_ok=True)
        if path is None:
            hash_name = hashlib.md5(text.encode()).hexdigest()
            path = f"{hash_name}.mp3"
        final_path_str = str(cache_dir_path / path)
        async def _download():
            communicate = edge_tts.Communicate(text, self.voice)
            await communicate.save(final_path_str)
        def _worker():
            loop = asyncio.new_event_loop()
            asyncio.set_event_loop(loop)
            try: loop.run_until_complete(_download())
            finally: loop.close()
        thread = threading.Thread(target=_worker)
        thread.start()
        thread.join()
        return {"original_audio": path}

# ─── MAIN SCENE GRAPH ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
class QuestionNComment79(VoiceoverScene):
    def construct(self):
        self.camera.background_color = BLACK
        self.set_speech_service(EdgeTTSService(voice="en-US-GuyNeural"))

        # --- TIÊU ĐỀ ---
        header = VGroup(
            Text("Further Questions & Comments", font_size=36, weight=BOLD),
            Text("Empirical Proof & Future Horizons", font_size=24, color=BLUE_B)
        ).arrange(DOWN, buff=0.2).to_edge(UP)
        self.play(FadeIn(header, shift=DOWN * 0.2))

        # ===================================================================
        # PHẦN 1: MINH HỌA THỰC NGHIỆM (THE LORA BENCHMARK)
        # ===================================================================
        subtitle1 = Text("Does Equivariance Help?", font_size=22, color=YELLOW).next_to(header, DOWN, buff=0.4)
        self.play(Write(subtitle1))

        # Thiết kế 2 hộp Dashboard chuyên nghiệp
        box_width = 5.5
        box_height = 4.2
        naive_box = RoundedRectangle(width=box_width, height=box_height, corner_radius=0.3, fill_color=GRAY_E, fill_opacity=0.3, stroke_color=BLUE_D).move_to(LEFT * 3.2 + DOWN * 0.8)
        invariant_box = RoundedRectangle(width=box_width, height=box_height, corner_radius=0.3, fill_color=GRAY_E, fill_opacity=0.3, stroke_color=GREEN_D).move_to(RIGHT * 3.2 + DOWN * 0.8)

        # Labels cho hộp Naive
        naive_lbl = Text("Naive Models (Symmetry-Agnostic)", font_size=18, color=BLUE_B).move_to(naive_box.get_top() + DOWN * 0.4)
        naive_model_name = MathTex(r"\text{Transformer}([U, V])", font_size=22, color=GRAY_A).next_to(naive_lbl, DOWN, buff=0.2)

        # Labels cho hộp Invariant
        invariant_lbl = Text("Efficient Invariant (GL-net)", font_size=18, color=GREEN_B).move_to(invariant_box.get_top() + DOWN * 0.4)
        invariant_model_name = MathTex(r"\text{GL-net } (\sigma(UV^\top))", font_size=22, color=GRAY_A).next_to(invariant_lbl, DOWN, buff=0.2)

        dashboard = VGroup(naive_box, naive_lbl, naive_model_name, invariant_box, invariant_lbl, invariant_model_name)

        with self.voiceover(text="To conclude, we must ask the critical question: Does all this mathematical equivariance actually help in practice?"):
            self.play(Create(naive_box), Create(invariant_box))
            self.play(Write(naive_lbl), Write(invariant_lbl))
            self.play(Write(naive_model_name), Write(invariant_model_name))

        # -------------------------------------------------------------
        # FIX: SỬ DỤNG BARCHART CHUẨN ĐỂ VẼ BIỂU ĐỒ SO SÁNH R^2
        # -------------------------------------------------------------
        # Biểu đồ Naive R^2 = 0.63
        chart_naive = BarChart(
            values=[0.63],
            bar_names=["R² Score"],
            y_range=[0, 1, 0.5],
            y_length=2.0, x_length=1.5,
            bar_colors=[BLUE_E]
        ).next_to(naive_model_name, DOWN, buff=0.4)
        lbl_r2_naive = MathTex(r"0.630", font_size=24).next_to(chart_naive.bars[0], UP, buff=0.1)

        # Biểu đồ Invariant R^2 = 0.998
        chart_inv = BarChart(
            values=[0.998],
            bar_names=["R² Score"],
            y_range=[0, 1, 0.5],
            y_length=2.0, x_length=1.5,
            bar_colors=[GREEN_E]
        ).next_to(invariant_model_name, DOWN, buff=0.4)
        lbl_r2_inv = MathTex(r"0.998", font_size=26, color=GREEN_A).next_to(chart_inv.bars[0], UP, buff=0.1)

        with self.voiceover(text="Recent benchmarks on LoRA metanetworks reveal a stark contrast. Naive architectures that ignore parameter symmetries suffer from poor generalization accuracy."):
            self.play(Create(chart_naive), Write(lbl_r2_naive))
            self.play(Indicate(chart_naive.bars[0], color=RED))

        with self.voiceover(text="Worse, when scaling up to massive models like LLaMA, naive configurations completely run out of memory, collapsing entirely."):
            oom_text = Text("OOM CRASH", font_size=24, color=RED, weight=BOLD).move_to(chart_naive.get_center())
            self.play(
                FadeOut(chart_naive), FadeOut(lbl_r2_naive),
                FadeIn(oom_text, shift=DOWN*0.2)
            )
            self.play(Flash(oom_text, color=RED, line_length=0.3))

        with self.voiceover(text="In contrast, symmetry-aware methods like GL-net maintain absolute stability, achieving a near perfect R-squared score of point nine nine eight while remaining highly efficient."):
            self.play(Create(chart_inv), Write(lbl_r2_inv))
            self.play(Indicate(chart_inv.bars[0], scale_factor=1.1, color=GREEN_A))
            self.wait(1.5)

        # Xóa phân cảnh thực nghiệm gọn gàng
        self.play(
            FadeOut(dashboard), FadeOut(oom_text),
            FadeOut(chart_inv), FadeOut(lbl_r2_inv), FadeOut(subtitle1)
        )

        # ===================================================================
        # PHẦN 2: TẦM NHÌN TƯƠNG LAI (TREE LAYOUT CHUẨN)
        # ===================================================================
        subtitle2 = Text("Open Horizons & Future Pathways", font_size=24, color=ORANGE).next_to(header, DOWN, buff=0.4)
        self.play(Write(subtitle2))

        # Layout dạng cây (Tree): Nút trung tâm đặt bên trái
        center_node = Dot(LEFT * 4.5 + DOWN * 0.5, radius=0.15, color=ORANGE)
        center_lbl = Text("Future\nResearch", font_size=18, weight=BOLD, color=YELLOW).next_to(center_node, UP, buff=0.2)
        self.play(Create(center_node), Write(center_lbl))

        # Các nhánh tỏa sang bên phải xếp dọc không bao giờ bị đè
        y_positions = [1.2, 0.1, -1.0, -2.1]
        node_colors = [BLUE_B, PURPLE_B, TEAL_B, GREEN_B]

        branch_nodes = VGroup(*[Dot(LEFT * 1.5 + UP * y, radius=0.1, color=c) for y, c in zip(y_positions, node_colors)])

        texts = [
            "1. Scalability limits in massive weight spaces",
            "2. When to choose invariance over equivariance",
            "3. Designing universal invariant descriptors",
            "4. Exploiting the unified model atlas ecosystem"
        ]
        branch_texts = VGroup(*[Text(t, font_size=20).next_to(n, RIGHT, buff=0.3) for t, n in zip(texts, branch_nodes)])
        horizons = VGroup(*[VGroup(n, t) for n, t in zip(branch_nodes, branch_texts)])

        # Đường dẫn từ trung tâm (Left) tỏa ra các nhánh (Right)
        connecting_lines = VGroup(*[
            Line(center_node.get_right(), n.get_left(), stroke_width=1.5, color=GRAY_C, stroke_opacity=0.4)
            for n in branch_nodes
        ])

        with self.voiceover(text="Yet, as we look to the horizon, several profound questions remain open for the scientific community."):
            self.play(Create(connecting_lines, lag_ratio=0.2), run_time=1.5)

        with self.voiceover(text="First, how do we scale these metanetworks to process foundational models containing hundreds of billions of parameters?"):
            self.play(FadeIn(horizons[0], shift=RIGHT*0.2))
            self.play(Indicate(branch_nodes[0], color=BLUE_B))

        with self.voiceover(text="Second, how do we formalize strict mathematical rules to determine exactly when to enforce permutation invariance versus full layers equivariance?"):
            self.play(FadeIn(horizons[1], shift=RIGHT*0.2))
            self.play(Indicate(branch_nodes[1], color=PURPLE_B))

        with self.voiceover(text="And finally, how can we leverage these geometric properties to seamlessly navigate, parse, and exploit the massive, interconnected model atlas of the future?"):
            self.play(FadeIn(horizons[2], shift=RIGHT*0.2))
            self.play(FadeIn(horizons[3], shift=RIGHT*0.2))
            self.play(Indicate(branch_nodes[3], color=GREEN_B))
            self.wait(1)

        # Hồi kết lắng đọng trường không gian
        with self.voiceover(text="Unlocking these hidden parameter symmetries will not only protect model security, but fundamentally reshape how humanity understands deep neural functions."):
            self.play(
                FadeOut(horizons), FadeOut(connecting_lines), FadeOut(center_node), FadeOut(center_lbl), FadeOut(subtitle2),
                header.animate.scale(1.2).move_to(ORIGIN),
                run_time=2.5
            )
            self.wait(2)


In [ ]:
!ls /content/drive/MyDrive/lab01


In [ ]:
import os
import re
import subprocess
from pathlib import Path

# Đường dẫn thư mục chứa các file bài giảng gốc của bạn
source_directory = Path("/content/drive/MyDrive/lab01")

mp4_files = [
    "EquivariantVsInvariant29.mp4",
    "LinearEquivariantTheory30.mp4",
    "Shiftin1D35.mp4",
    "AllpermutationsOfAset37.mp4",
    "AdvancedOptimizationDynamics68.mp4",
    "AtomisticGeoMLI51.mp4",
    "AtomisticGeoMLII52.mp4",
    "equivariantGNNs46.mp4",
    "ExamplesInvariantPolynomials44.mp4",
    "FlexibleSymmetries57.mp4",
    "GeneralizationAndLimits40.mp4",
    "GroupConvolution41.mp4",
    "InvariantPolynomials43.mp4",
    "LossLandscapes65.mp4",
    "NeuralNetworksAsDataMP71.mp4",
    "NeuralNetworksAsDataWhy73.mp4",
    "NeuralParameterSymmetries60.mp4",
    "ProteinGeneration47.mp4",
    "QuestionNComment79.mp4",
    "symmetriesInRobotics53.mp4",
    "SymmetryBreaking55.mp4",
    "TensorMethodsI49.mp4",
    "TensorMethodsII50.mp4",
    "TypesOfSymmetries64.mp4",
    "WeightSpaceTechniques76.mp4"
]

# Hàm tách số ở cuối tên file để sắp xếp chính xác
def extract_number(filename):
    match = re.search(r"(\d+)\.mp4$", filename)
    return int(match.group(1)) if match else 0

# Sắp xếp danh sách theo thứ tự tăng dần của số phía sau
sorted_mp4_files = sorted(mp4_files, key=extract_number)

# Tạo đường dẫn thư mục đích lưu kết quả
output_directory = source_directory / "result"
os.makedirs(output_directory, exist_ok=True)
final_output_path = output_directory / "result.mp4"

# Tạo file danh sách tạm thời cấu hình đường dẫn TUYỆT ĐỐI đầy đủ cho ffmpeg
text_list_path = "ffmpeg_concat_list.txt"
with open(text_list_path, "w", encoding="utf-8") as text_file:
    for file_name in sorted_mp4_files:
        # SỬA ĐỔI: Sử dụng đường dẫn tuyệt đối đầy đủ dẫn thẳng vào Google Drive
        full_file_path = source_directory / file_name
        text_file.write(f"file '{str(full_file_path)}'\n")

print("--- THỨ TỰ TIẾN HÀNH GHÉP FILE ---")
for index, name in enumerate(sorted_mp4_files, 1):
    print(f"{index}. {name}")
print("──────────────────────────────────")

# Gọi lệnh ffmpeg ghép nối trực tiếp luồng stream không cần re-encode (Cực kỳ nhanh)
ffmpeg_command = [
    "ffmpeg", "-y", "-f", "concat", "-safe", "0",
    "-i", text_list_path, "-c", "copy", str(final_output_path)
]

try:
    print("\nĐang tiến hành gộp dữ liệu...")
    result = subprocess.run(ffmpeg_command, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    print(f"\nThành công! Video hoàn chỉnh của chuỗi bài giảng đã được lưu tại đường dẫn:\n{final_output_path}")
except subprocess.CalledProcessError as error:
    print(f"\nĐã xảy ra lỗi trong quá trình xử lý ffmpeg: {error.stderr.decode('utf-8')}")
finally:
    # Dọn dẹp file cấu hình tạm thời sau khi xử lý xong
    if os.path.exists(text_list_path):
        os.remove(text_list_path)


In [ ]:
!ls /content/drive/MyDrive/lab01/result/


## Ghi file vào Google Drive

In [ ]:
import os

# Chuyển đường dẫn làm việc
os.chdir('/content/drive/MyDrive/lab01/result/')

# Tạo file danh sách các video cần nối
with open("file_list.txt", "w") as f:
    f.write("file 'Final_MQ.mov'\n")
    f.write("file 'result.mp4'\n")
    f.write("file 'Lab2.mp4'\n")

# Chạy lệnh ffmpeg để nối file mà không re-encode
!ffmpeg -f concat -safe 0 -i file_list.txt -c copy hpq.mp4

# Xóa file danh sách tạm
os.remove("file_list.txt")


---
